# Notebook 03 · Build matched inputs and verify evaluator feasibility

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import platform
import sys


NOTEBOOK_NAME = "03_pair_and_evaluator_feasibility"

PREFERRED_STAGE_02_DATASET_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/"
    "prose-operation-stage-02-slices-fresh"
)

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

STAGE_03_OUTPUT_ROOT = Path(
    "/kaggle/working/stage_03_fresh_dataset"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def locate_stage_02_dataset_root() -> Path:
    preferred_manifest = (
        PREFERRED_STAGE_02_DATASET_ROOT
        / "stage_02_manifest.json"
    )

    if preferred_manifest.is_file():
        return PREFERRED_STAGE_02_DATASET_ROOT

    input_root = Path("/kaggle/input")

    if not input_root.is_dir():
        raise RuntimeError(
            "Kaggle input directory does not exist: "
            f"{input_root}"
        )

    matches = sorted(
        input_root.rglob("stage_02_manifest.json"),
        key=lambda path: str(path),
    )

    if not matches:
        raise RuntimeError(
            "Could not locate stage_02_manifest.json. "
            "Attach the prose-operation-stage-02-slices-fresh "
            "dataset to this notebook."
        )

    if len(matches) > 1:
        raise RuntimeError(
            "Multiple Stage 02 manifests were found:\n"
            + "\n".join(str(path) for path in matches)
        )

    return matches[0].parent


def locate_stage_02_project_root(
    dataset_root: Path,
) -> Path:
    candidates = [
        (
            dataset_root
            / "stage_02_operation_slicing_v1"
            / "prose_operation_experiment"
        ),
        dataset_root / "prose_operation_experiment",
    ]

    for candidate in candidates:
        if (
            candidate
            / "registries"
            / "02_manifest.json"
        ).is_file():
            return candidate

    matches = sorted(
        dataset_root.rglob(
            "prose_operation_experiment/"
            "registries/02_manifest.json"
        ),
        key=lambda path: str(path),
    )

    if not matches:
        raise RuntimeError(
            "Could not locate the expanded Stage 02 "
            "project directory."
        )

    if len(matches) > 1:
        raise RuntimeError(
            "Multiple expanded Stage 02 project "
            "directories were found:\n"
            + "\n".join(str(path) for path in matches)
        )

    return matches[0].parents[1]


STAGE_02_DATASET_ROOT = (
    locate_stage_02_dataset_root()
)

STAGE_02_PROJECT_ROOT = (
    locate_stage_02_project_root(
        STAGE_02_DATASET_ROOT
    )
)

STAGE_02_EXTERNAL_MANIFEST_PATH = (
    STAGE_02_DATASET_ROOT
    / "stage_02_manifest.json"
)

STAGE_02_INTERNAL_MANIFEST_PATH = (
    STAGE_02_PROJECT_ROOT
    / "registries"
    / "02_manifest.json"
)

STAGE_02_ELIGIBLE_OPERATIONS_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data"
    / "inventory"
    / "final_eligible_operations.jsonl"
)

STAGE_02_OPERATIONS_AUDIT_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data"
    / "audits"
    / "operations.csv"
)

STAGE_02_EXCLUSION_LEDGER_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data"
    / "inventory"
    / "operation_exclusions.jsonl"
)


external_manifest = read_json(
    STAGE_02_EXTERNAL_MANIFEST_PATH
)

internal_manifest = read_json(
    STAGE_02_INTERNAL_MANIFEST_PATH
)

observed_internal_manifest_sha256 = (
    sha256_file(
        STAGE_02_INTERNAL_MANIFEST_PATH
    )
)

expected_internal_manifest_sha256 = (
    external_manifest
    .get("internal_manifest", {})
    .get("sha256")
)

external_counts = external_manifest.get(
    "final_counts",
    {},
)

expected_eligible_operation_count = int(
    external_counts.get(
        "eligible_operation_count",
        -1,
    )
)

internal_eligible_operation_count = int(
    internal_manifest.get(
        "eligible_operation_count",
        -1,
    )
)


problems = []

if external_manifest.get("stage") != "02":
    problems.append(
        "External manifest stage is not 02."
    )

if external_manifest.get("stage_status") != "PASS":
    problems.append(
        "Stage 02 external status is not PASS."
    )

if not external_manifest.get(
    "may_begin_notebook_03",
    False,
):
    problems.append(
        "Stage 02 does not authorize Notebook 03."
    )

if internal_manifest.get("stage") != "02":
    problems.append(
        "Internal manifest stage is not 02."
    )

if internal_manifest.get("stage_status") != "PASS":
    problems.append(
        "Stage 02 internal status is not PASS."
    )

if (
    observed_internal_manifest_sha256
    != expected_internal_manifest_sha256
):
    problems.append(
        "Stage 02 internal manifest SHA-256 "
        "does not match the external manifest."
    )

if expected_eligible_operation_count <= 0:
    problems.append("Stage 02 must declare a positive candidate count.")

if (
    internal_eligible_operation_count
    != expected_eligible_operation_count
):
    problems.append(
        "External and internal eligible-operation "
        "counts do not match."
    )

required_inputs = [
    STAGE_02_ELIGIBLE_OPERATIONS_PATH,
    STAGE_02_OPERATIONS_AUDIT_PATH,
    STAGE_02_EXCLUSION_LEDGER_PATH,
]

for required_path in required_inputs:
    if not required_path.is_file():
        problems.append(
            f"Required Stage 02 input is missing: "
            f"{required_path}"
        )
    elif required_path.stat().st_size == 0:
        problems.append(
            f"Required Stage 02 input is empty: "
            f"{required_path}"
        )


cell_1_report = {
    "notebook": NOTEBOOK_NAME,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "kaggle_kernel_run_type": os.environ.get(
        "KAGGLE_KERNEL_RUN_TYPE",
        "UNKNOWN",
    ),
    "accelerator_required": False,
    "stage_02_dataset_root": str(
        STAGE_02_DATASET_ROOT
    ),
    "stage_02_project_root": str(
        STAGE_02_PROJECT_ROOT
    ),
    "external_manifest": str(
        STAGE_02_EXTERNAL_MANIFEST_PATH
    ),
    "external_status": external_manifest.get(
        "stage_status"
    ),
    "internal_manifest": str(
        STAGE_02_INTERNAL_MANIFEST_PATH
    ),
    "internal_status": internal_manifest.get(
        "stage_status"
    ),
    "internal_manifest_sha256": (
        observed_internal_manifest_sha256
    ),
    "eligible_operation_count": (
        expected_eligible_operation_count
    ),
    "eligible_specification_count": (
        external_counts.get(
            "eligible_specification_count"
        )
    ),
    "eligible_native_openapi_count": (
        external_counts.get(
            "eligible_native_openapi_count"
        )
    ),
    "eligible_converted_swagger_count": (
        external_counts.get(
            "eligible_converted_swagger_count"
        )
    ),
    "stage_02_valid_slice_count": (
        external_counts.get(
            "valid_slice_count"
        )
    ),
    "stage_02_excluded_operation_count": (
        external_counts.get(
            "excluded_operation_count"
        )
    ),
    "strata_thresholds_selected": (
        internal_manifest.get(
            "strata_thresholds_selected"
        )
    ),
    "experimental_generation_authorized": (
        external_manifest.get(
            "may_begin_experimental_generation"
        )
    ),
    "eligible_operations_registry": str(
        STAGE_02_ELIGIBLE_OPERATIONS_PATH
    ),
    "project_root": str(PROJECT_ROOT),
    "stage_03_output_root": str(
        STAGE_03_OUTPUT_ROOT
    ),
    "problems": problems,
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

print(
    json.dumps(
        cell_1_report,
        indent=2,
        ensure_ascii=False,
    )
)

if problems:
    raise RuntimeError(
        "Notebook 03 input qualification failed:\n"
        + "\n".join(problems)
    )

print(
    "\nNOTEBOOK 03 — CELL 1: PASS"
)
if internal_manifest.get("design_id") != "openapi-fresh-pilot-v1":
    raise RuntimeError("Attach the fresh Stage 02 release.")

import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)

verify_artifact_provenance(internal_manifest)


{
  "notebook": "03_pair_and_evaluator_feasibility",
  "python_version": "3.13.18",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "kaggle_kernel_run_type": "Interactive",
  "accelerator_required": false,
  "stage_02_dataset_root": "/kaggle/input/datasets/anonuser01/stage-02-provenance",
  "stage_02_project_root": "/kaggle/input/datasets/anonuser01/stage-02-provenance/stage_02_provenance_v1/prose_operation_experiment",
  "external_manifest": "/kaggle/input/datasets/anonuser01/stage-02-provenance/stage_02_manifest.json",
  "external_status": "PASS",
  "internal_manifest": "/kaggle/input/datasets/anonuser01/stage-02-provenance/stage_02_provenance_v1/prose_operation_experiment/registries/02_manifest.json",
  "internal_status": "PASS",
  "internal_manifest_sha256": "538df541290ea421a1dab61cdd3a78a072d63d4ab109f2d902b805d7e728b5fd",
  "eligible_operation_count": 52922,
  "eligible_specification_count": 1925,
  "eligible_native_openapi_count": 50405,
  "eligible_converted_swagger_co

In [2]:
# Notebook 03 — Cell 2
# Self-contained Stage 02 artifact and expanded-payload verification.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import gzip
import hashlib
import json
import os
import time


CELL_2_STARTED = time.perf_counter()

EXPECTED_ARTIFACT_COUNT = read_json(STAGE_02_INTERNAL_MANIFEST_PATH)["artifact_count"]
EXPECTED_NONCHUNK_COUNT = EXPECTED_ARTIFACT_COUNT - 235
EXPECTED_PROMPT_CHUNK_COUNT = 235
EXPECTED_MEASUREMENT_COUNT = int(external_counts["operation_count"])
EXPECTED_PAYLOAD_COUNT = int(external_counts["valid_slice_count"])
EXPECTED_INVALID_OPERATION_ID = (
    "op_e0fd428bf92d567920c7bca4"
)
EXPECTED_INTERNAL_MANIFEST_SHA256 = read_json(STAGE_02_EXTERNAL_MANIFEST_PATH)["internal_manifest"]["sha256"]

REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/upstream_stage_02_verification.json"
)
CHUNK_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "stage_02_expanded_prompt_chunk_"
    "verification_corrected.jsonl"
)
PROMPT_INDEX_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data/audits/rendered_prompt_payload_index.jsonl"
)
PROMPT_MEASUREMENTS_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data/audits/rendered_prompt_measurements.jsonl"
)


def canonical_json_bytes_cell2(value) -> bytes:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def sha256_value_cell2(value) -> str:
    return hashlib.sha256(
        canonical_json_bytes_cell2(value)
    ).hexdigest()


def sha256_file_cell2(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(chunk_size):
            digest.update(block)

    return digest.hexdigest()


def sha256_text_cell2(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def read_json_cell2(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def read_jsonl_cell2(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}."
                )

            yield value


def write_json_atomic_cell2(
    path: Path,
    value,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            value,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(temporary_path, path)


def write_jsonl_atomic_cell2(
    path: Path,
    records,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        for record in records:
            handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                    allow_nan=False,
                )
                + "\n"
            )

        handle.flush()
        os.fsync(handle.fileno())

    os.replace(temporary_path, path)


def locate_expanded_payload_cell2(
    relative_path: str,
) -> tuple[Path, bool]:
    root = STAGE_02_PROJECT_ROOT.resolve()
    expected = root / relative_path
    if not expected.resolve().is_relative_to(root):
        raise RuntimeError(f"Unsafe Stage 02 prompt path: {relative_path}")

    plain = expected.with_suffix("")
    filenames = {expected.name, expected.name + ".tmp", plain.name, plain.name + ".tmp"}
    if expected.is_file():
        found = [expected]
    else:
        found = sorted(
            path for filename in filenames
            for path in payload_files_by_name_cell2.get(filename, ())
            if path.resolve().is_relative_to(root)
        )
    if len(found) != 1 or found[0].is_symlink():
        matches = ", ".join(str(path.relative_to(root)) for path in found) or "none"
        raise RuntimeError(
            f"Expected one Stage 02 prompt payload for {relative_path}; found {matches}. "
            "Check the expanded files under the Stage 02 provenance input."
        )

    path = found[0]
    with path.open("rb") as handle:
        gzip_encoded = handle.read(2) == b"\x1f\x8b"
    return path, gzip_encoded


def read_payload_records_cell2(
    path: Path,
    gzip_encoded: bool,
):
    opener = (
        gzip.open
        if gzip_encoded
        else open
    )

    with opener(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                value = json.loads(line)
            except Exception as error:
                raise RuntimeError(
                    "Invalid expanded prompt JSONL at "
                    f"{path}:{line_number}: {error!r}"
                ) from error

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Expanded prompt record is not "
                    f"an object at {path}:{line_number}."
                )

            yield value


external_manifest_cell2 = read_json_cell2(
    STAGE_02_EXTERNAL_MANIFEST_PATH
)
internal_manifest_cell2 = read_json_cell2(
    STAGE_02_INTERNAL_MANIFEST_PATH
)

observed_internal_manifest_sha256_cell2 = (
    sha256_file_cell2(
        STAGE_02_INTERNAL_MANIFEST_PATH
    )
)
expected_internal_manifest_sha256_cell2 = (
    external_manifest_cell2
    .get("internal_manifest", {})
    .get("sha256")
)

if (
    observed_internal_manifest_sha256_cell2
    != expected_internal_manifest_sha256_cell2
    or observed_internal_manifest_sha256_cell2
    != EXPECTED_INTERNAL_MANIFEST_SHA256
):
    raise RuntimeError(
        "Stage 02 internal-manifest hash "
        "does not match the frozen dataset."
    )

artifacts_cell2 = internal_manifest_cell2.get(
    "artifacts",
    [],
)

if len(artifacts_cell2) != EXPECTED_ARTIFACT_COUNT:
    raise RuntimeError(
        f"Expected {EXPECTED_ARTIFACT_COUNT:,} Stage 02 artifacts; "
        f"observed {len(artifacts_cell2)}."
    )

payload_artifacts_cell2 = []
nonchunk_artifacts_cell2 = []

for artifact_cell2 in artifacts_cell2:
    relative_path_cell2 = str(
        artifact_cell2.get("path", "")
    )

    if (
        relative_path_cell2.startswith(
            "data/audits/"
            "rendered_prompt_payload_chunks/"
        )
        and relative_path_cell2.endswith(
            ".jsonl.gz"
        )
    ):
        payload_artifacts_cell2.append(
            artifact_cell2
        )
    else:
        nonchunk_artifacts_cell2.append(
            artifact_cell2
        )

if (
    len(payload_artifacts_cell2)
    != EXPECTED_PROMPT_CHUNK_COUNT
    or len(nonchunk_artifacts_cell2)
    != EXPECTED_NONCHUNK_COUNT
):
    raise RuntimeError(
        "Unexpected Stage 02 artifact split: "
        f"{len(nonchunk_artifacts_cell2)} "
        "ordinary artifacts and "
        f"{len(payload_artifacts_cell2)} "
        "prompt chunks."
    )

# Index the mounted filenames once before hashing the ordinary artifacts.
chunk_directory_cell2 = (
    STAGE_02_PROJECT_ROOT / "data/audits/rendered_prompt_payload_chunks"
)
payload_files_by_name_cell2 = {}
for path_cell2 in chunk_directory_cell2.rglob("*"):
    if path_cell2.is_file() and not path_cell2.is_symlink():
        payload_files_by_name_cell2.setdefault(path_cell2.name, []).append(path_cell2)

payload_locations_cell2 = {
    row["path"]: locate_expanded_payload_cell2(row["path"])
    for row in payload_artifacts_cell2
}


def verify_nonchunk_artifact_cell2(artifact):
    artifact_path = (
        STAGE_02_PROJECT_ROOT
        / artifact["path"]
    )

    if not artifact_path.is_file():
        return {
            "path": artifact["path"],
            "problem": "MISSING_FILE",
        }

    observed_bytes = artifact_path.stat().st_size
    expected_bytes = int(artifact["bytes"])

    if observed_bytes != expected_bytes:
        return {
            "path": artifact["path"],
            "problem": "BYTE_COUNT_MISMATCH",
            "expected": expected_bytes,
            "observed": observed_bytes,
        }

    if (
        sha256_file_cell2(artifact_path)
        != artifact["sha256"]
    ):
        return {
            "path": artifact["path"],
            "problem": "SHA256_MISMATCH",
        }

    return {
        "path": artifact["path"],
        "problem": None,
        "bytes": observed_bytes,
    }


exact_verification_problems_cell2 = []
exact_verified_count_cell2 = 0
exact_verified_bytes_cell2 = 0
hash_workers_cell2 = min(
    16,
    max(4, (os.cpu_count() or 2) * 4),
)

with ThreadPoolExecutor(
    max_workers=hash_workers_cell2
) as executor_cell2:
    futures_cell2 = [
        executor_cell2.submit(
            verify_nonchunk_artifact_cell2,
            artifact_cell2,
        )
        for artifact_cell2
        in nonchunk_artifacts_cell2
    ]

    for checked_cell2, future_cell2 in enumerate(
        as_completed(futures_cell2),
        start=1,
    ):
        result_cell2 = future_cell2.result()

        if result_cell2["problem"] is None:
            exact_verified_count_cell2 += 1
            exact_verified_bytes_cell2 += int(
                result_cell2["bytes"]
            )
        else:
            exact_verification_problems_cell2.append(
                result_cell2
            )

        if (
            checked_cell2 % 1000 == 0
            or checked_cell2
            == len(nonchunk_artifacts_cell2)
        ):
            print(
                json.dumps({
                    "phase": (
                        "exact_nonchunk_verification"
                    ),
                    "checked": checked_cell2,
                    "total": len(
                        nonchunk_artifacts_cell2
                    ),
                    "verified": (
                        exact_verified_count_cell2
                    ),
                    "problem_count": len(
                        exact_verification_problems_cell2
                    ),
                    "workers": hash_workers_cell2,
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - CELL_2_STARTED,
                        1,
                    ),
                })
            )

if exact_verification_problems_cell2:
    raise RuntimeError(
        "Stage 02 ordinary-artifact verification "
        "failed:\n"
        + json.dumps(
            exact_verification_problems_cell2[:20],
            indent=2,
        )
    )

prompt_index_records_cell2 = list(
    read_jsonl_cell2(PROMPT_INDEX_PATH)
)
measurement_records_cell2 = list(
    read_jsonl_cell2(
        PROMPT_MEASUREMENTS_PATH
    )
)

if (
    len(prompt_index_records_cell2)
    != EXPECTED_PROMPT_CHUNK_COUNT
):
    raise RuntimeError(
        "Expected 235 prompt-index records."
    )

if (
    len(measurement_records_cell2)
    != EXPECTED_MEASUREMENT_COUNT
):
    raise RuntimeError(
        f"Expected {EXPECTED_MEASUREMENT_COUNT:,} prompt-measurement "
        "records."
    )

measurement_order_cell2 = []
measurement_by_op_cell2 = {}

for measurement_cell2 in measurement_records_cell2:
    op_id_cell2 = measurement_cell2.get(
        "op_id"
    )

    if (
        not isinstance(op_id_cell2, str)
        or op_id_cell2 in measurement_by_op_cell2
    ):
        raise RuntimeError(
            "Invalid or duplicate operation ID in "
            "the prompt-measurement ledger."
        )

    measurement_order_cell2.append(
        op_id_cell2
    )
    measurement_by_op_cell2[
        op_id_cell2
    ] = measurement_cell2

payload_artifact_by_name_cell2 = {
    Path(record["path"]).name: record
    for record in payload_artifacts_cell2
}

chunk_ledger_cell2 = []
logical_problems_cell2 = []
measurement_cursor_cell2 = 0
assigned_operation_total_cell2 = 0
expected_payload_total_cell2 = 0
observed_payload_total_cell2 = 0
invalid_assigned_operations_cell2 = []

for position_cell2, index_record_cell2 in enumerate(
    prompt_index_records_cell2,
    start=1,
):
    chunk_number_cell2 = int(
        index_record_cell2["chunk_number"]
    )
    chunk_filename_cell2 = (
        f"chunk_{chunk_number_cell2:05d}.jsonl.gz"
    )
    artifact_cell2 = (
        payload_artifact_by_name_cell2.get(
            chunk_filename_cell2
        )
    )

    if artifact_cell2 is None:
        logical_problems_cell2.append({
            "chunk": chunk_number_cell2,
            "problem": "MISSING_MANIFEST_CHUNK",
        })
        continue

    assigned_count_cell2 = int(
        index_record_cell2["operation_count"]
    )
    assigned_ids_cell2 = measurement_order_cell2[
        measurement_cursor_cell2:
        measurement_cursor_cell2
        + assigned_count_cell2
    ]
    measurement_cursor_cell2 += (
        assigned_count_cell2
    )
    assigned_operation_total_cell2 += len(
        assigned_ids_cell2
    )

    expected_payload_ids_cell2 = []
    invalid_ids_cell2 = []

    for op_id_cell2 in assigned_ids_cell2:
        measurement_cell2 = (
            measurement_by_op_cell2[op_id_cell2]
        )
        payload_expected_cell2 = (
            measurement_cell2.get("slice_status")
            == "PASS"
            and measurement_cell2.get(
                "rendered_prompt_sha256"
            )
            is not None
            and measurement_cell2.get(
                "token_ids_sha256"
            )
            is not None
        )

        if payload_expected_cell2:
            expected_payload_ids_cell2.append(
                op_id_cell2
            )
        else:
            invalid_ids_cell2.append(
                op_id_cell2
            )
            invalid_assigned_operations_cell2.append({
                "chunk_number": chunk_number_cell2,
                "op_id": op_id_cell2,
                "slice_status": measurement_cell2.get(
                    "slice_status"
                ),
            })

    expected_payload_total_cell2 += len(
        expected_payload_ids_cell2
    )

    payload_path_cell2, gzip_encoded_cell2 = payload_locations_cell2[artifact_cell2["path"]]
    if gzip_encoded_cell2 and (
        payload_path_cell2.stat().st_size != artifact_cell2["bytes"]
        or sha256_file_cell2(payload_path_cell2) != artifact_cell2["sha256"]
    ):
        raise RuntimeError(f"Stage 02 compressed prompt hash or size mismatch: {artifact_cell2['path']}")
    payload_records_cell2 = list(
        read_payload_records_cell2(
            payload_path_cell2,
            gzip_encoded_cell2,
        )
    )
    observed_payload_total_cell2 += len(
        payload_records_cell2
    )
    observed_payload_ids_cell2 = []
    record_problems_cell2 = []

    for record_number_cell2, payload_cell2 in enumerate(
        payload_records_cell2,
        start=1,
    ):
        op_id_cell2 = payload_cell2.get("op_id")
        observed_payload_ids_cell2.append(
            op_id_cell2
        )
        measurement_cell2 = (
            measurement_by_op_cell2.get(
                op_id_cell2
            )
        )

        if measurement_cell2 is None:
            record_problems_cell2.append({
                "record": record_number_cell2,
                "op_id": op_id_cell2,
                "problem": "UNKNOWN_OPERATION_ID",
            })
            continue

        rendered_prompt_cell2 = payload_cell2.get(
            "rendered_prompt"
        )
        token_ids_cell2 = payload_cell2.get(
            "token_ids"
        )

        if not isinstance(rendered_prompt_cell2, str):
            record_problems_cell2.append({
                "record": record_number_cell2,
                "op_id": op_id_cell2,
                "problem": "INVALID_RENDERED_PROMPT",
            })
            continue

        if not isinstance(token_ids_cell2, list):
            record_problems_cell2.append({
                "record": record_number_cell2,
                "op_id": op_id_cell2,
                "problem": "INVALID_TOKEN_IDS",
            })
            continue

        recomputed_prompt_sha256_cell2 = (
            sha256_text_cell2(
                rendered_prompt_cell2
            )
        )
        recomputed_token_sha256_cell2 = (
            sha256_value_cell2(
                token_ids_cell2
            )
        )

        record_checks_cell2 = {
            "prompt_hash_internal": (
                recomputed_prompt_sha256_cell2
                == payload_cell2.get(
                    "rendered_prompt_sha256"
                )
            ),
            "token_hash_internal": (
                recomputed_token_sha256_cell2
                == payload_cell2.get(
                    "token_ids_sha256"
                )
            ),
            "token_count_internal": (
                len(token_ids_cell2)
                == int(
                    payload_cell2.get(
                        "token_count",
                        -1,
                    )
                )
            ),
            "prompt_hash_matches_measurement": (
                recomputed_prompt_sha256_cell2
                == measurement_cell2.get(
                    "rendered_prompt_sha256"
                )
            ),
            "token_hash_matches_measurement": (
                recomputed_token_sha256_cell2
                == measurement_cell2.get(
                    "token_ids_sha256"
                )
            ),
            "token_count_matches_measurement": (
                len(token_ids_cell2)
                == int(
                    measurement_cell2.get(
                        "rendered_prompt_tokens",
                        -1,
                    )
                )
            ),
            "boundary_matches_measurement": (
                payload_cell2.get("boundary")
                == measurement_cell2.get(
                    "boundary"
                )
            ),
            "renderer_matches_measurement": (
                measurement_cell2.get(
                    "renderer_fingerprint"
                )
                == index_record_cell2.get(
                    "renderer_fingerprint"
                )
            ),
        }

        failed_checks_cell2 = [
            check_name_cell2
            for check_name_cell2, passed_cell2
            in record_checks_cell2.items()
            if not passed_cell2
        ]

        if failed_checks_cell2:
            record_problems_cell2.append({
                "record": record_number_cell2,
                "op_id": op_id_cell2,
                "problem": "PAYLOAD_CONTENT_MISMATCH",
                "failed_checks": failed_checks_cell2,
            })

    expected_payload_count_cell2 = int(
        index_record_cell2[
            "payload_record_count"
        ]
    )
    chunk_checks_cell2 = {
        "sequential_chunk_number": (
            chunk_number_cell2
            == position_cell2 - 1
        ),
        "assigned_count_matches_index": (
            len(assigned_ids_cell2)
            == assigned_count_cell2
        ),
        "assigned_ids_hash_matches_index": (
            sha256_value_cell2(
                assigned_ids_cell2
            )
            == index_record_cell2[
                "operation_ids_sha256"
            ]
        ),
        "expected_payload_count_matches_index": (
            len(expected_payload_ids_cell2)
            == expected_payload_count_cell2
        ),
        "observed_payload_count_matches_index": (
            len(payload_records_cell2)
            == expected_payload_count_cell2
        ),
        "payload_ids_match_valid_subset": (
            observed_payload_ids_cell2
            == expected_payload_ids_cell2
        ),
        "payload_ids_hash_matches_valid_subset": (
            sha256_value_cell2(
                observed_payload_ids_cell2
            )
            == sha256_value_cell2(
                expected_payload_ids_cell2
            )
        ),
        "no_record_problems": (
            not record_problems_cell2
        ),
    }
    chunk_status_cell2 = (
        "PASS"
        if all(chunk_checks_cell2.values())
        else "BLOCKED"
    )

    if chunk_status_cell2 != "PASS":
        logical_problems_cell2.append({
            "chunk": chunk_number_cell2,
            "problem": "CHUNK_VERIFICATION_FAILURE",
            "failed_checks": [
                name_cell2
                for name_cell2, passed_cell2
                in chunk_checks_cell2.items()
                if not passed_cell2
            ],
            "record_problems": (
                record_problems_cell2[:20]
            ),
        })

    chunk_ledger_cell2.append({
        "chunk_number": chunk_number_cell2,
        "source_artifact_path": (
            artifact_cell2["path"]
        ),
        "observed_payload_path": str(
            payload_path_cell2
        ),
        "representation": (
            "MANIFEST_VERIFIED_GZIP"
            if gzip_encoded_cell2
            else "KAGGLE_EXPANDED_JSONL"
        ),
        "assigned_operation_count": len(
            assigned_ids_cell2
        ),
        "payload_record_count": len(
            payload_records_cell2
        ),
        "invalid_assigned_operation_ids": (
            invalid_ids_cell2
        ),
        "assigned_operation_ids_sha256": (
            sha256_value_cell2(
                assigned_ids_cell2
            )
        ),
        "payload_operation_ids_sha256": (
            sha256_value_cell2(
                observed_payload_ids_cell2
            )
        ),
        "checks": chunk_checks_cell2,
        "status": chunk_status_cell2,
    })

    if (
        position_cell2 % 10 == 0
        or position_cell2
        == EXPECTED_PROMPT_CHUNK_COUNT
    ):
        print(
            json.dumps({
                "phase": (
                    "expanded_payload_logical_"
                    "verification"
                ),
                "chunks_checked": position_cell2,
                "chunk_total": (
                    EXPECTED_PROMPT_CHUNK_COUNT
                ),
                "chunks_passed": sum(
                    record["status"] == "PASS"
                    for record in chunk_ledger_cell2
                ),
                "payload_records_checked": (
                    observed_payload_total_cell2
                ),
                "problem_count": len(
                    logical_problems_cell2
                ),
                "elapsed_seconds": round(
                    time.perf_counter()
                    - CELL_2_STARTED,
                    1,
                ),
            })
        )

write_jsonl_atomic_cell2(
    CHUNK_LEDGER_PATH,
    chunk_ledger_cell2,
)

global_checks_cell2 = {
    "internal_manifest_hash_matches": (
        observed_internal_manifest_sha256_cell2
        == EXPECTED_INTERNAL_MANIFEST_SHA256
    ),
    "all_nonchunk_artifacts_exactly_verified": (
        exact_verified_count_cell2
        == EXPECTED_NONCHUNK_COUNT
    ),
    "all_measurements_consumed": (
        measurement_cursor_cell2
        == EXPECTED_MEASUREMENT_COUNT
    ),
    "assigned_operation_total_matches_manifest": (
        assigned_operation_total_cell2
        == EXPECTED_MEASUREMENT_COUNT
    ),
    "expected_payload_total_matches_manifest": (
        expected_payload_total_cell2
        == EXPECTED_PAYLOAD_COUNT
    ),
    "observed_payload_total_matches_manifest": (
        observed_payload_total_cell2
        == EXPECTED_PAYLOAD_COUNT
    ),
    "exactly_one_invalid_assigned_operation": (
        len(invalid_assigned_operations_cell2)
        == 1
    ),
    "invalid_operation_identity_matches": (
        len(invalid_assigned_operations_cell2)
        == 1
        and invalid_assigned_operations_cell2[0][
            "op_id"
        ]
        == EXPECTED_INVALID_OPERATION_ID
    ),
    "invalid_operation_is_in_chunk_83": (
        len(invalid_assigned_operations_cell2)
        == 1
        and invalid_assigned_operations_cell2[0][
            "chunk_number"
        ]
        == 83
    ),
    "all_235_chunks_pass": (
        len(chunk_ledger_cell2)
        == EXPECTED_PROMPT_CHUNK_COUNT
        and all(
            record["status"] == "PASS"
            for record in chunk_ledger_cell2
        )
    ),
    "no_logical_problems": (
        not logical_problems_cell2
    ),
    "all_manifest_artifacts_accounted_for": (
        exact_verified_count_cell2
        + len(chunk_ledger_cell2)
        == EXPECTED_ARTIFACT_COUNT
    ),
}

cell_2_status = (
    "PASS"
    if all(global_checks_cell2.values())
    else "BLOCKED"
)

cell_2_report = {
    "stage": "02",
    "verification_mode": (
        "fresh_exact_nonchunk_plus_record_level_"
        "logical_verification_of_kaggle_expanded_"
        "gzip_payloads_with_assigned_vs_emitted_"
        "chunk_semantics"
    ),
    "internal_manifest_path": str(
        STAGE_02_INTERNAL_MANIFEST_PATH
    ),
    "internal_manifest_sha256": (
        observed_internal_manifest_sha256_cell2
    ),
    "expected_artifact_count": (
        EXPECTED_ARTIFACT_COUNT
    ),
    "exact_byte_verified_artifact_count": (
        exact_verified_count_cell2
    ),
    "exact_byte_verified_artifact_bytes": (
        exact_verified_bytes_cell2
    ),
    "logically_verified_expanded_chunk_count": (
        sum(
            record["status"] == "PASS"
            for record in chunk_ledger_cell2
        )
    ),
    "accounted_artifact_count": (
        exact_verified_count_cell2
        + len(chunk_ledger_cell2)
    ),
    "measurement_operation_count": (
        len(measurement_records_cell2)
    ),
    "assigned_operation_count": (
        assigned_operation_total_cell2
    ),
    "payload_operation_count": (
        observed_payload_total_cell2
    ),
    "invalid_assigned_operations": (
        invalid_assigned_operations_cell2
    ),
    "representation_adjustment": {
        "type": (
            "KAGGLE_AUTOMATIC_GZIP_EXPANSION"
        ),
        "scientific_content_changed": False,
        "verification_basis": (
            "Every expanded payload record was "
            "parsed and its prompt hash, token-ID "
            "hash, token count, boundary, renderer "
            "fingerprint, operation identity, and "
            "measurement-ledger entry were "
            "independently reconciled."
        ),
    },
    "global_checks": global_checks_cell2,
    "problem_count": len(
        logical_problems_cell2
    ),
    "problem_examples": (
        logical_problems_cell2[:20]
    ),
    "corrected_chunk_ledger_path": str(
        CHUNK_LEDGER_PATH
    ),
    "corrected_chunk_ledger_sha256": (
        sha256_file_cell2(
            CHUNK_LEDGER_PATH
        )
    ),
    "duration_seconds": round(
        time.perf_counter()
        - CELL_2_STARTED,
        3,
    ),
    "status": cell_2_status,
}

write_json_atomic_cell2(
    REPORT_PATH,
    cell_2_report,
)

cell_2_report["report_path"] = str(
    REPORT_PATH
)
cell_2_report["report_sha256"] = (
    sha256_file_cell2(REPORT_PATH)
)

print(
    json.dumps(
        cell_2_report,
        indent=2,
        ensure_ascii=False,
    )
)

if cell_2_status != "PASS":
    raise RuntimeError(
        "Stage 02 fresh-session verification "
        "failed. Review the reported problems."
    )

print(
    "\nNOTEBOOK 03 — CELL 2: PASS"
)


{"phase": "exact_nonchunk_verification", "checked": 1000, "total": 58646, "verified": 1000, "problem_count": 0, "workers": 16, "elapsed_seconds": 21.1}
{"phase": "exact_nonchunk_verification", "checked": 2000, "total": 58646, "verified": 2000, "problem_count": 0, "workers": 16, "elapsed_seconds": 22.7}
{"phase": "exact_nonchunk_verification", "checked": 3000, "total": 58646, "verified": 3000, "problem_count": 0, "workers": 16, "elapsed_seconds": 24.3}
{"phase": "exact_nonchunk_verification", "checked": 4000, "total": 58646, "verified": 4000, "problem_count": 0, "workers": 16, "elapsed_seconds": 25.9}
{"phase": "exact_nonchunk_verification", "checked": 5000, "total": 58646, "verified": 5000, "problem_count": 0, "workers": 16, "elapsed_seconds": 27.4}
{"phase": "exact_nonchunk_verification", "checked": 6000, "total": 58646, "verified": 6000, "problem_count": 0, "workers": 16, "elapsed_seconds": 28.8}
{"phase": "exact_nonchunk_verification", "checked": 7000, "total": 58646, "verified": 70

In [3]:
import importlib
import importlib.metadata
import json
import sys


DEPENDENCIES_CELL3 = [
    {
        "import_name": "yaml",
        "distribution": "PyYAML",
        "required_version": None,
    },
    {
        "import_name": "pandas",
        "distribution": "pandas",
        "required_version": None,
    },
    {
        "import_name": "numpy",
        "distribution": "numpy",
        "required_version": None,
    },
    {
        "import_name": "jsonschema",
        "distribution": "jsonschema",
        "required_version": None,
    },
    {
        "import_name": "referencing",
        "distribution": "referencing",
        "required_version": None,
    },
    {
        "import_name": "openapi_spec_validator",
        "distribution": "openapi-spec-validator",
        "required_version": "0.9.0",
    },
    {
        "import_name": "transformers",
        "distribution": "transformers",
        "required_version": None,
    },
    {
        "import_name": "huggingface_hub",
        "distribution": "huggingface-hub",
        "required_version": None,
    },
    {
        "import_name": "tqdm",
        "distribution": "tqdm",
        "required_version": None,
    },
]


dependency_results_cell3 = []
missing_packages_cell3 = []
version_mismatches_cell3 = []

for dependency_cell3 in DEPENDENCIES_CELL3:
    import_name_cell3 = (
        dependency_cell3["import_name"]
    )

    distribution_cell3 = (
        dependency_cell3["distribution"]
    )

    required_version_cell3 = (
        dependency_cell3["required_version"]
    )

    available_cell3 = False
    installed_version_cell3 = None
    import_path_cell3 = None
    diagnostic_cell3 = None

    try:
        module_cell3 = importlib.import_module(
            import_name_cell3
        )

        available_cell3 = True

        import_path_cell3 = getattr(
            module_cell3,
            "__file__",
            None,
        )

        installed_version_cell3 = (
            importlib.metadata.version(
                distribution_cell3
            )
        )

    except Exception as error:
        diagnostic_cell3 = repr(error)

    version_matches_cell3 = (
        available_cell3
        and (
            required_version_cell3 is None
            or installed_version_cell3
            == required_version_cell3
        )
    )

    if not available_cell3:
        package_spec_cell3 = (
            distribution_cell3
            if required_version_cell3 is None
            else (
                f"{distribution_cell3}"
                f"=={required_version_cell3}"
            )
        )

        missing_packages_cell3.append(
            package_spec_cell3
        )

    elif not version_matches_cell3:
        version_mismatches_cell3.append({
            "distribution": (
                distribution_cell3
            ),
            "required_version": (
                required_version_cell3
            ),
            "installed_version": (
                installed_version_cell3
            ),
        })

    dependency_results_cell3.append({
        "import_name": import_name_cell3,
        "distribution": distribution_cell3,
        "available": available_cell3,
        "installed_version": (
            installed_version_cell3
        ),
        "required_version": (
            required_version_cell3
        ),
        "version_matches": (
            version_matches_cell3
        ),
        "import_path": import_path_cell3,
        "diagnostic": diagnostic_cell3,
    })


if missing_packages_cell3:
    status_cell3 = "NEEDS_INSTALL"
elif version_mismatches_cell3:
    status_cell3 = "NEEDS_VERSION_REPAIR"
else:
    status_cell3 = "PASS"


cell_3_report = {
    "python_version": sys.version,
    "dependencies": (
        dependency_results_cell3
    ),
    "missing_packages": (
        missing_packages_cell3
    ),
    "version_mismatches": (
        version_mismatches_cell3
    ),
    "full_model_required": False,
    "tokenizer_required_later": True,
    "accelerator_required": False,
    "status": status_cell3,
}

print(
    json.dumps(
        cell_3_report,
        indent=2,
        ensure_ascii=False,
    )
)

if missing_packages_cell3:
    print(
        "\nPackages requiring installation:"
    )

    for package_cell3 in (
        missing_packages_cell3
    ):
        print(f"- {package_cell3}")

if version_mismatches_cell3:
    print(
        "\nPackages requiring version repair:"
    )

    for mismatch_cell3 in (
        version_mismatches_cell3
    ):
        print(
            "- "
            f"{mismatch_cell3['distribution']}: "
            f"installed "
            f"{mismatch_cell3['installed_version']}, "
            f"required "
            f"{mismatch_cell3['required_version']}"
        )

if status_cell3 == "PASS":
    print(
        "\nNOTEBOOK 03 — CELL 3: PASS"
    )
else:
    print(
        "\nNOTEBOOK 03 — CELL 3: "
        f"{status_cell3}"
    )

{
  "python_version": "3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]",
  "dependencies": [
    {
      "import_name": "yaml",
      "distribution": "PyYAML",
      "available": true,
      "installed_version": "6.0.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/dist-packages/yaml/__init__.py",
      "diagnostic": null
    },
    {
      "import_name": "pandas",
      "distribution": "pandas",
      "available": true,
      "installed_version": "2.3.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/dist-packages/pandas/__init__.py",
      "diagnostic": null
    },
    {
      "import_name": "numpy",
      "distribution": "numpy",
      "available": true,
      "installed_version": "2.1.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/dist-packages/numpy/__init__.py",
      "diagnostic":

In [4]:
import importlib
import importlib.metadata
import json
import subprocess
import sys
import time


CELL_4_PACKAGE = (
    "openapi-spec-validator"
)

CELL_4_REQUIRED_VERSION = "0.9.0"

CELL_4_PACKAGE_SPEC = (
    f"{CELL_4_PACKAGE}"
    f"=={CELL_4_REQUIRED_VERSION}"
)

cell_4_started = time.perf_counter()


install_command_cell4 = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    CELL_4_PACKAGE_SPEC,
]

install_result_cell4 = subprocess.run(
    install_command_cell4,
    check=False,
    capture_output=True,
    text=True,
)


if install_result_cell4.stdout:
    print(
        install_result_cell4.stdout
    )

if install_result_cell4.stderr:
    print(
        install_result_cell4.stderr
    )


importlib.invalidate_caches()

module_cell4 = importlib.import_module(
    "openapi_spec_validator"
)

installed_version_cell4 = (
    importlib.metadata.version(
        CELL_4_PACKAGE
    )
)

import_path_cell4 = getattr(
    module_cell4,
    "__file__",
    None,
)

version_matches_cell4 = (
    installed_version_cell4
    == CELL_4_REQUIRED_VERSION
)

status_cell4 = (
    "PASS"
    if (
        install_result_cell4.returncode == 0
        and version_matches_cell4
    )
    else "BLOCKED"
)


cell_4_report = {
    "package": CELL_4_PACKAGE,
    "required_version": (
        CELL_4_REQUIRED_VERSION
    ),
    "installed_version": (
        installed_version_cell4
    ),
    "import_path": import_path_cell4,
    "pip_return_code": (
        install_result_cell4.returncode
    ),
    "version_matches": (
        version_matches_cell4
    ),
    "duration_seconds": round(
        time.perf_counter()
        - cell_4_started,
        3,
    ),
    "status": status_cell4,
}

print(
    json.dumps(
        cell_4_report,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell4 != "PASS":
    raise RuntimeError(
        "Failed to install and verify "
        f"{CELL_4_PACKAGE_SPEC}."
    )

print(
    "\nNOTEBOOK 03 — CELL 4: PASS"
)



{
  "package": "openapi-spec-validator",
  "required_version": "0.9.0",
  "installed_version": "0.9.0",
  "import_path": "/usr/local/lib/python3.13/dist-packages/openapi_spec_validator/__init__.py",
  "pip_return_code": 0,
  "version_matches": true,
  "duration_seconds": 5.18,
  "status": "PASS"
}

NOTEBOOK 03 — CELL 4: PASS


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import yaml


PREFERRED_STAGE_00_DATASET_ROOT = Path(
    "/kaggle/input/datasets/anonuser01/"
    "stage-00"
)


def sha256_file_cell5(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_json_cell5(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def write_text_atomic_cell5(
    path: Path,
    text: str,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    temporary_path.replace(path)


def write_json_atomic_cell5(
    path: Path,
    value,
) -> None:
    write_text_atomic_cell5(
        path,
        json.dumps(
            value,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
    )


def write_yaml_atomic_cell5(
    path: Path,
    value,
) -> None:
    write_text_atomic_cell5(
        path,
        yaml.safe_dump(
            value,
            sort_keys=False,
            allow_unicode=True,
            default_flow_style=False,
        ),
    )


def locate_stage_00_dataset_root_cell5():
    preferred_manifest_cell5 = (
        PREFERRED_STAGE_00_DATASET_ROOT
        / "stage_00_manifest.json"
    )

    if preferred_manifest_cell5.is_file():
        return PREFERRED_STAGE_00_DATASET_ROOT

    matches_cell5 = sorted(
        Path("/kaggle/input").rglob(
            "stage_00_manifest.json"
        ),
        key=lambda path: str(path),
    )

    if len(matches_cell5) != 1:
        raise RuntimeError(
            "Could not uniquely locate the "
            "Stage 00 dataset. Found:\n"
            + "\n".join(
                str(path)
                for path in matches_cell5
            )
        )

    return matches_cell5[0].parent


def locate_stage_00_project_root_cell5(
    dataset_root: Path,
):
    candidates_cell5 = [
        (
            dataset_root
            / "stage_00_protocol_runtime_v1"
            / "prose_operation_experiment"
        ),
        dataset_root / "prose_operation_experiment",
    ]

    for candidate_cell5 in candidates_cell5:
        if (
            candidate_cell5
            / "registries"
            / "00_manifest.json"
        ).is_file():
            return candidate_cell5

    matches_cell5 = sorted(
        dataset_root.rglob(
            "prose_operation_experiment/"
            "registries/00_manifest.json"
        ),
        key=lambda path: str(path),
    )

    if len(matches_cell5) != 1:
        raise RuntimeError(
            "Could not uniquely locate the "
            "expanded Stage 00 project."
        )

    return matches_cell5[0].parents[1]


def normalize_stage_00_artifact_cell5(
    record,
):
    relative_path_cell5 = (
        record.get("path")
        or record.get("artifact_path")
    )

    return {
        "path": relative_path_cell5,
        "bytes": int(record["bytes"]),
        "sha256": record["sha256"],
    }


STAGE_00_DATASET_ROOT = (
    locate_stage_00_dataset_root_cell5()
)

STAGE_00_PROJECT_ROOT = (
    locate_stage_00_project_root_cell5(
        STAGE_00_DATASET_ROOT
    )
)

STAGE_00_EXTERNAL_MANIFEST_PATH = (
    STAGE_00_DATASET_ROOT
    / "stage_00_manifest.json"
)

STAGE_00_INTERNAL_MANIFEST_PATH = (
    STAGE_00_PROJECT_ROOT
    / "registries"
    / "00_manifest.json"
)


external_manifest_cell5 = read_json_cell5(
    STAGE_00_EXTERNAL_MANIFEST_PATH
)

internal_manifest_cell5 = read_json_cell5(
    STAGE_00_INTERNAL_MANIFEST_PATH
)

external_status_cell5 = (
    external_manifest_cell5.get(
        "stage_status"
    )
    or external_manifest_cell5.get(
        "status"
    )
)

internal_status_cell5 = (
    internal_manifest_cell5.get(
        "stage_status"
    )
    or internal_manifest_cell5.get(
        "status"
    )
)

expected_internal_manifest_sha256_cell5 = (
    external_manifest_cell5.get(
        "internal_manifest_sha256"
    )
    or external_manifest_cell5.get(
        "internal_manifest",
        {},
    ).get("sha256")
)

observed_internal_manifest_sha256_cell5 = (
    sha256_file_cell5(
        STAGE_00_INTERNAL_MANIFEST_PATH
    )
)


raw_artifact_records_cell5 = (
    internal_manifest_cell5.get(
        "artifacts",
        [],
    )
)

if not isinstance(
    raw_artifact_records_cell5,
    list,
):
    raise RuntimeError(
        "Stage 00 artifact records are not a list."
    )

normalized_artifact_records_cell5 = [
    normalize_stage_00_artifact_cell5(
        record_cell5
    )
    for record_cell5
    in raw_artifact_records_cell5
]

expected_artifact_count_cell5 = int(
    internal_manifest_cell5.get(
        "artifact_count",
        len(normalized_artifact_records_cell5),
    )
)

# The legacy Stage 00 internal manifest has no
# top-level artifact_bytes field.
expected_artifact_bytes_cell5 = sum(
    record_cell5["bytes"]
    for record_cell5
    in normalized_artifact_records_cell5
)


verification_problems_cell5 = []

if external_manifest_cell5.get("stage") != "00":
    verification_problems_cell5.append(
        "External Stage 00 manifest has "
        "the wrong stage identifier."
    )

if internal_manifest_cell5.get("stage") != "00":
    verification_problems_cell5.append(
        "Internal Stage 00 manifest has "
        "the wrong stage identifier."
    )

if external_status_cell5 != "PASS":
    verification_problems_cell5.append(
        "Stage 00 external status is not PASS."
    )

if internal_status_cell5 != "PASS":
    verification_problems_cell5.append(
        "Stage 00 internal status is not PASS."
    )

if (
    observed_internal_manifest_sha256_cell5
    != expected_internal_manifest_sha256_cell5
):
    verification_problems_cell5.append(
        "Stage 00 internal manifest SHA-256 "
        "does not match the external manifest."
    )

if (
    len(normalized_artifact_records_cell5)
    != expected_artifact_count_cell5
):
    verification_problems_cell5.append(
        "Stage 00 artifact-record count does "
        "not match the declared count."
    )


normalized_paths_cell5 = [
    record_cell5["path"]
    for record_cell5
    in normalized_artifact_records_cell5
]

if any(
    not isinstance(path_cell5, str)
    or not path_cell5.strip()
    for path_cell5 in normalized_paths_cell5
):
    verification_problems_cell5.append(
        "Stage 00 contains an invalid artifact path."
    )

if (
    len(set(normalized_paths_cell5))
    != len(normalized_paths_cell5)
):
    verification_problems_cell5.append(
        "Stage 00 contains duplicate artifact paths."
    )


verified_artifact_count_cell5 = 0
verified_artifact_bytes_cell5 = 0

for record_cell5 in (
    normalized_artifact_records_cell5
):
    relative_path_text_cell5 = (
        record_cell5["path"]
    )

    if not isinstance(
        relative_path_text_cell5,
        str,
    ):
        continue

    relative_path_cell5 = Path(
        relative_path_text_cell5
    )

    if (
        relative_path_cell5.is_absolute()
        or ".." in relative_path_cell5.parts
    ):
        verification_problems_cell5.append(
            "Unsafe Stage 00 artifact path: "
            f"{relative_path_text_cell5}"
        )
        continue

    artifact_path_cell5 = (
        STAGE_00_PROJECT_ROOT
        / relative_path_cell5
    )

    if not artifact_path_cell5.is_file():
        verification_problems_cell5.append(
            "Missing Stage 00 artifact: "
            f"{relative_path_text_cell5}"
        )
        continue

    observed_bytes_cell5 = (
        artifact_path_cell5.stat().st_size
    )

    if (
        observed_bytes_cell5
        != record_cell5["bytes"]
    ):
        verification_problems_cell5.append(
            "Stage 00 artifact size mismatch: "
            f"{relative_path_text_cell5}"
        )
        continue

    observed_sha256_cell5 = (
        sha256_file_cell5(
            artifact_path_cell5
        )
    )

    if (
        observed_sha256_cell5
        != record_cell5["sha256"]
    ):
        verification_problems_cell5.append(
            "Stage 00 artifact hash mismatch: "
            f"{relative_path_text_cell5}"
        )
        continue

    verified_artifact_count_cell5 += 1
    verified_artifact_bytes_cell5 += (
        observed_bytes_cell5
    )


if (
    verified_artifact_count_cell5
    != expected_artifact_count_cell5
):
    verification_problems_cell5.append(
        "Stage 00 verified artifact count "
        "does not match the manifest."
    )

if (
    verified_artifact_bytes_cell5
    != expected_artifact_bytes_cell5
):
    verification_problems_cell5.append(
        "Stage 00 verified byte total "
        "does not match the artifact records."
    )

if verification_problems_cell5:
    raise RuntimeError(
        "Stage 00 verification failed:\n"
        + "\n".join(
            verification_problems_cell5
        )
    )


stage_03_directories_cell5 = [
    PROJECT_ROOT,
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "config" / "upstream",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "slices",
    PROJECT_ROOT / "data" / "slices" / "L1",
    PROJECT_ROOT / "data" / "audits",
    PROJECT_ROOT / "data" / "inventory",
    PROJECT_ROOT / "data" / "eligibility",
    PROJECT_ROOT / "data" / "checkpoints",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "reports" / "audit",
    PROJECT_ROOT / "reports" / "figures",
    PROJECT_ROOT / "registries",
    STAGE_03_OUTPUT_ROOT,
]

for directory_cell5 in (
    stage_03_directories_cell5
):
    directory_cell5.mkdir(
        parents=True,
        exist_ok=True,
    )


upstream_copy_plan_cell5 = [
    (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "obfuscation.yaml",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_obfuscation.yaml",
    ),
    (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "eligibility.yaml",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_eligibility.yaml",
    ),
    (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "study.yaml",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_study.yaml",
    ),
    (
        STAGE_02_PROJECT_ROOT
        / "config"
        / "stage_02.yaml",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_02_policy.yaml",
    ),
    (
        STAGE_02_PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_prose_counting.yaml",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_prose_counting.yaml",
    ),
    (
        STAGE_02_PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_system_prompt.txt",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_system_prompt.txt",
    ),
    (
        STAGE_02_PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_user_prompt.txt",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_user_prompt.txt",
    ),
    (
        STAGE_02_PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_prompt_reproducibility.json",
        PROJECT_ROOT
        / "config"
        / "upstream"
        / "stage_00_prompt_reproducibility.json",
    ),
]

copied_upstream_files_cell5 = []

for (
    source_path_cell5,
    destination_path_cell5,
) in upstream_copy_plan_cell5:
    if not source_path_cell5.is_file():
        raise RuntimeError(
            "Required upstream configuration "
            f"is missing: {source_path_cell5}"
        )

    shutil.copyfile(
        source_path_cell5,
        destination_path_cell5,
    )

    copied_upstream_files_cell5.append({
        "source_path": str(
            source_path_cell5
        ),
        "destination_path": str(
            destination_path_cell5
        ),
        "bytes": (
            destination_path_cell5
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file_cell5(
                destination_path_cell5
            )
        ),
    })


active_obfuscation_path_cell5 = (
    PROJECT_ROOT
    / "config"
    / "obfuscation.yaml"
)

active_eligibility_path_cell5 = (
    PROJECT_ROOT
    / "config"
    / "eligibility.yaml"
)

shutil.copyfile(
    PROJECT_ROOT
    / "config"
    / "upstream"
    / "stage_00_obfuscation.yaml",
    active_obfuscation_path_cell5,
)

shutil.copyfile(
    PROJECT_ROOT
    / "config"
    / "upstream"
    / "stage_00_eligibility.yaml",
    active_eligibility_path_cell5,
)


with active_obfuscation_path_cell5.open(
    "r",
    encoding="utf-8",
) as file:
    obfuscation_config_cell5 = (
        yaml.safe_load(file)
    )

with active_eligibility_path_cell5.open(
    "r",
    encoding="utf-8",
) as file:
    eligibility_config_cell5 = (
        yaml.safe_load(file)
    )

with (
    PROJECT_ROOT
    / "config"
    / "upstream"
    / "stage_00_study.yaml"
).open(
    "r",
    encoding="utf-8",
) as file:
    study_config_cell5 = (
        yaml.safe_load(file)
    )


policy_problems_cell5 = []

if obfuscation_config_cell5.get(
    "seed"
) != 42:
    policy_problems_cell5.append(
        "Frozen obfuscation seed is not 42."
    )

if (
    obfuscation_config_cell5.get(
        "field_set"
    )
    != [
        "schema_defined_summary",
        "schema_defined_description",
    ]
):
    policy_problems_cell5.append(
        "Frozen obfuscation field set is incorrect."
    )

if study_config_cell5.get(
    "max_rendered_input_tokens"
) != 5120:
    policy_problems_cell5.append(
        "Frozen prompt limit is not 5,120."
    )

expected_model_revision_cell5 = (
    "c03e6d358207e414f1eca0bb1891e29f1"
    "db0e242"
)

if study_config_cell5.get(
    "model_revision"
) != expected_model_revision_cell5:
    policy_problems_cell5.append(
        "Frozen model revision is unexpected."
    )

required_eligibility_flags_cell5 = [
    "require_positive_prose_token_score",
    "require_transformed_ascii_letter",
    "require_applicable_scenario",
    "require_applicable_requirement",
    "require_applicable_response_fault",
    "require_evaluator_support",
]

for flag_cell5 in (
    required_eligibility_flags_cell5
):
    if (
        eligibility_config_cell5.get(
            flag_cell5
        )
        is not True
    ):
        policy_problems_cell5.append(
            "Frozen eligibility flag is not true: "
            f"{flag_cell5}"
        )

if policy_problems_cell5:
    raise RuntimeError(
        "Frozen Stage 00 policy check failed:\n"
        + "\n".join(
            policy_problems_cell5
        )
    )


stage_03_policy_cell5 = {
    "stage": "03",
    "policy_version": "1.0",
    "title": (
        "L0/L1 pair construction and "
        "offline evaluator feasibility"
    ),
    "upstream_requirements": {
        "stage_00_status": "PASS",
        "stage_02_status": "PASS",
        "stage_00_internal_manifest_sha256": (
            observed_internal_manifest_sha256_cell5
        ),
        "stage_02_internal_manifest_sha256": (
            sha256_file_cell5(
                STAGE_02_INTERNAL_MANIFEST_PATH
            )
        ),
        "eligible_operation_count": expected_eligible_operation_count,
        "eligible_specification_count": int(external_counts["eligible_specification_count"]),
        "invalid_slice_operations_reintroduced": 0,
        "over_limit_operations_reintroduced": 0,
    },
    "pair_identity": {
        "source_condition": "L0",
        "treatment_condition": "L1",
        "one_pair_per_operation": True,
        "pair_id_rule": (
            "pair_<first 24 hex characters of "
            "sha256('stage03-pair-v1|<op_id>')>"
        ),
        "preserve_op_id": True,
        "preserve_spec_id": True,
        "preserve_family_identity": True,
    },
    "transformation": {
        "seed": 42,
        "approved_field_set": [
            "schema_defined_summary",
            "schema_defined_description",
        ],
        "source_pointer_authority": (
            "Stage 02 prose-field audit"
        ),
        "deep_copy_required": True,
        "serialized_global_replacement_forbidden": True,
        "ascii_letter_pattern": "[A-Za-z]+",
        "preserve_string_length": True,
        "preserve_whitespace": True,
        "preserve_punctuation": True,
        "preserve_digits": True,
        "preserve_ascii_capitalization_pattern": True,
        "preserve_non_ascii_characters": True,
        "identical_input_strings_map_identically": True,
        "algorithm_version_sha256": None,
        "qualification_status": "PENDING_CELL_6",
    },
    "protected_content": {
        "paths": True,
        "methods": True,
        "operation_identifiers": True,
        "parameter_names": True,
        "schema_property_names": True,
        "types": True,
        "formats": True,
        "required_flags": True,
        "enums": True,
        "defaults": True,
        "patterns": True,
        "constraints": True,
        "references": True,
        "security_contract": True,
        "examples": True,
        "extensions": True,
        "titles": True,
    },
    "validation": {
        "validate_L0": True,
        "validate_L1": True,
        "reference_resolution_required": True,
        "silent_repair_allowed": False,
    },
    "prompt_eligibility": {
        "maximum_rendered_input_tokens": 5120,
        "both_conditions_must_fit": True,
        "truncation_allowed": False,
        "summarization_allowed": False,
        "grouping_value": "prose_tokens_L0",
        "L1_growth_does_not_change_grouping_value": True,
    },
    "treatment_eligibility": {
        "require_positive_prose_token_score": True,
        "require_transformed_ascii_letter": True,
        "english_only": None,
        "placeholder_exclusion_rule": None,
        "short_prose_automatically_excluded": False,
    },
    "evaluator_feasibility": {
        "source": "specification_contract_only",
        "model_outputs_used": False,
        "require_applicable_scenario": True,
        "require_applicable_requirement": True,
        "require_applicable_response_fault": True,
        "require_evaluator_support": True,
        "draft_only": True,
        "final_oracle_created_in_notebook": "05",
    },
    "development_exposure": {
        "exclude_development_records": True,
    },
    "prohibitions": {
        "experimental_generation": True,
        "strata_selection": True,
        "pilot_sampling": True,
        "model_output_based_eligibility": True,
    },
    "checkpointing": {
        "enabled": True,
        "checkpoint_every_operations": 250,
        "atomic_writes": True,
        "resume_only_after_hash_verification": True,
    },
}

STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config"
    / "stage_03.yaml"
)

write_yaml_atomic_cell5(
    STAGE_03_POLICY_PATH,
    stage_03_policy_cell5,
)


requirements_result_cell5 = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
    ],
    check=True,
    capture_output=True,
    text=True,
)

REQUIREMENTS_LOCK_PATH = (
    PROJECT_ROOT
    / "config"
    / "requirements.lock.txt"
)

write_text_atomic_cell5(
    REQUIREMENTS_LOCK_PATH,
    requirements_result_cell5.stdout,
)


tracked_distributions_cell5 = [
    "PyYAML",
    "pandas",
    "numpy",
    "jsonschema",
    "referencing",
    "openapi-spec-validator",
    "transformers",
    "huggingface-hub",
    "tqdm",
]

tracked_versions_cell5 = {
    distribution_cell5: str(
        importlib.metadata.version(
            distribution_cell5
        )
    )
    for distribution_cell5
    in tracked_distributions_cell5
}

environment_cell5 = {
    "captured_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python_version": str(sys.version),
    "platform": str(platform.platform()),
    "processor": str(platform.processor()),
    "cpu_count": os.cpu_count(),
    "kaggle_kernel_run_type": (
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE",
            "UNKNOWN",
        )
    ),
    "accelerator_required": False,
    "full_model_required": False,
    "tracked_package_versions": (
        tracked_versions_cell5
    ),
}

ENVIRONMENT_PATH = (
    PROJECT_ROOT
    / "config"
    / "environment.json"
)

write_json_atomic_cell5(
    ENVIRONMENT_PATH,
    environment_cell5,
)


STAGE_00_VERIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "upstream_stage_00_verification.json"
)

stage_00_verification_report_cell5 = {
    "stage": "00",
    "verification_mode": (
        "legacy_manifest_schema_normalized"
    ),
    "external_manifest_path": str(
        STAGE_00_EXTERNAL_MANIFEST_PATH
    ),
    "external_status_field": (
        "stage_status"
        if "stage_status"
        in external_manifest_cell5
        else "status"
    ),
    "internal_artifact_path_field": (
        "artifact_path"
    ),
    "internal_artifact_bytes_source": (
        "sum_of_artifact_record_bytes"
    ),
    "internal_manifest_path": str(
        STAGE_00_INTERNAL_MANIFEST_PATH
    ),
    "internal_manifest_sha256": (
        observed_internal_manifest_sha256_cell5
    ),
    "expected_artifact_count": (
        expected_artifact_count_cell5
    ),
    "verified_artifact_count": (
        verified_artifact_count_cell5
    ),
    "expected_artifact_bytes": (
        expected_artifact_bytes_cell5
    ),
    "verified_artifact_bytes": (
        verified_artifact_bytes_cell5
    ),
    "verification_problems": [],
    "status": "PASS",
}

write_json_atomic_cell5(
    STAGE_00_VERIFICATION_REPORT_PATH,
    stage_00_verification_report_cell5,
)


STAGE_02_VERIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "upstream_stage_02_verification.json"
)

if not STAGE_02_VERIFICATION_REPORT_PATH.is_file():
    raise RuntimeError(
        "The Stage 02 verification report "
        "from Cell 2 is missing."
    )

stage_02_verification_report_cell5 = (
    read_json_cell5(
        STAGE_02_VERIFICATION_REPORT_PATH
    )
)

if (
    stage_02_verification_report_cell5.get(
        "status"
    )
    != "PASS"
):
    raise RuntimeError(
        "The Stage 02 verification report "
        "from Cell 2 is not PASS."
    )


missing_directories_cell5 = [
    str(path)
    for path in stage_03_directories_cell5
    if not path.is_dir()
]

cell_5_status = (
    "PASS"
    if not missing_directories_cell5
    else "BLOCKED"
)

cell_5_report = {
    "stage_00_dataset_root": str(
        STAGE_00_DATASET_ROOT
    ),
    "stage_00_project_root": str(
        STAGE_00_PROJECT_ROOT
    ),
    "stage_00_external_manifest_schema": (
        "legacy_stage_00_v1"
    ),
    "stage_00_external_status": (
        external_status_cell5
    ),
    "stage_00_internal_status": (
        internal_status_cell5
    ),
    "stage_00_internal_manifest_sha256": (
        observed_internal_manifest_sha256_cell5
    ),
    "stage_00_verified_artifact_count": (
        verified_artifact_count_cell5
    ),
    "stage_00_verified_artifact_bytes": (
        verified_artifact_bytes_cell5
    ),
    "created_directory_count": len(
        stage_03_directories_cell5
    ),
    "missing_directories": (
        missing_directories_cell5
    ),
    "copied_upstream_configuration_count": (
        len(copied_upstream_files_cell5)
    ),
    "copied_upstream_configurations": (
        copied_upstream_files_cell5
    ),
    "stage_03_policy": {
        "path": str(STAGE_03_POLICY_PATH),
        "sha256": sha256_file_cell5(
            STAGE_03_POLICY_PATH
        ),
    },
    "active_obfuscation": {
        "path": str(
            active_obfuscation_path_cell5
        ),
        "sha256": sha256_file_cell5(
            active_obfuscation_path_cell5
        ),
        "status": (
            obfuscation_config_cell5.get(
                "status"
            )
        ),
    },
    "active_eligibility": {
        "path": str(
            active_eligibility_path_cell5
        ),
        "sha256": sha256_file_cell5(
            active_eligibility_path_cell5
        ),
        "status": (
            eligibility_config_cell5.get(
                "status"
            )
        ),
    },
    "environment": {
        "path": str(ENVIRONMENT_PATH),
        "sha256": sha256_file_cell5(
            ENVIRONMENT_PATH
        ),
    },
    "requirements_lock": {
        "path": str(
            REQUIREMENTS_LOCK_PATH
        ),
        "bytes": (
            REQUIREMENTS_LOCK_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file_cell5(
            REQUIREMENTS_LOCK_PATH
        ),
    },
    "upstream_stage_00_verification": {
        "path": str(
            STAGE_00_VERIFICATION_REPORT_PATH
        ),
        "sha256": sha256_file_cell5(
            STAGE_00_VERIFICATION_REPORT_PATH
        ),
    },
    "upstream_stage_02_verification": {
        "path": str(
            STAGE_02_VERIFICATION_REPORT_PATH
        ),
        "sha256": sha256_file_cell5(
            STAGE_02_VERIFICATION_REPORT_PATH
        ),
        "status": (
            stage_02_verification_report_cell5[
                "status"
            ]
        ),
    },
    "status": cell_5_status,
}

print(
    json.dumps(
        cell_5_report,
        indent=2,
        ensure_ascii=False,
    )
)

if cell_5_status != "PASS":
    raise RuntimeError(
        "Stage 03 workspace initialization failed."
    )

print(
    "\nNOTEBOOK 03 — CELL 5: PASS"
)
verify_artifact_provenance(internal_manifest_cell5)
if (internal_manifest.get("upstream_manifest_sha256", {}).get("00")
        != sha256_file_cell5(STAGE_00_INTERNAL_MANIFEST_PATH)):
    raise RuntimeError("Stage 02 belongs to a different Stage 00 release.")


{
  "stage_00_dataset_root": "/kaggle/input/datasets/anonuser01/stage-00-provenance",
  "stage_00_project_root": "/kaggle/input/datasets/anonuser01/stage-00-provenance/stage_00_provenance_v1/prose_operation_experiment",
  "stage_00_external_manifest_schema": "legacy_stage_00_v1",
  "stage_00_external_status": "PASS",
  "stage_00_internal_status": "PASS",
  "stage_00_internal_manifest_sha256": "a064f3bb9814b3545ed81984c5a5bcd5a5ce236ce621f2fd0d039950efafc431",
  "stage_00_verified_artifact_count": 32,
  "stage_00_verified_artifact_bytes": 110218,
  "created_directory_count": 16,
  "missing_directories": [],
  "copied_upstream_configuration_count": 8,
  "copied_upstream_configurations": [
    {
      "source_path": "/kaggle/input/datasets/anonuser01/stage-00-provenance/stage_00_provenance_v1/prose_operation_experiment/config/obfuscation.yaml",
      "destination_path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/config/upstream/stage_00_obfuscation.yaml",
      "bytes": 15

In [6]:
from pathlib import Path
import copy
import hashlib
import importlib.util
import json
import re
import yaml

from openapi_spec_validator import validate


OBFUSCATION_SOURCE_PATH = (
    PROJECT_ROOT
    / "src"
    / "obfuscation.py"
)

OBFUSCATION_QUALIFICATION_ROOT = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "obfuscation_qualification"
)

OBFUSCATION_QUALIFICATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


obfuscation_source_cell6 = '''\
import copy
import hashlib
import re


def obfuscate_annotation(
    text: str,
    seed: int = 42,
) -> str:
    if not isinstance(text, str):
        raise TypeError(
            "Annotation value must be a string."
        )

    def replace_word(match):
        word = match.group(0)
        lower = word.lower()
        chars = []

        for position, char in enumerate(lower):
            key = (
                f"{seed}|{lower}|{position}"
            ).encode("utf-8")

            shift = (
                1
                + int.from_bytes(
                    hashlib.sha256(
                        key
                    ).digest(),
                    "big",
                )
                % 25
            )

            new = chr(
                ord("a")
                + (
                    ord(char)
                    - ord("a")
                    + shift
                )
                % 26
            )

            chars.append(
                new.upper()
                if word[position].isupper()
                else new
            )

        return "".join(chars)

    return re.sub(
        r"[A-Za-z]+",
        replace_word,
        text,
    )


def _decode_pointer_token(token: str) -> str:
    result = []
    position = 0

    while position < len(token):
        character = token[position]

        if character != "~":
            result.append(character)
            position += 1
            continue

        if position + 1 >= len(token):
            raise ValueError(
                "Invalid trailing tilde in JSON pointer."
            )

        escape = token[position + 1]

        if escape == "0":
            result.append("~")
        elif escape == "1":
            result.append("/")
        else:
            raise ValueError(
                "Invalid JSON-pointer escape: "
                f"~{escape}"
            )

        position += 2

    return "".join(result)


def decode_json_pointer(pointer: str):
    if not isinstance(pointer, str):
        raise TypeError(
            "JSON pointer must be a string."
        )

    if pointer == "":
        return []

    if not pointer.startswith("/"):
        raise ValueError(
            "Non-empty JSON pointer must start "
            "with '/'."
        )

    return [
        _decode_pointer_token(token)
        for token in pointer[1:].split("/")
    ]


def resolve_json_pointer(
    document,
    pointer: str,
):
    current = document

    for token in decode_json_pointer(
        pointer
    ):
        if isinstance(current, dict):
            if token not in current:
                raise KeyError(
                    f"Missing pointer token {token!r} "
                    f"in {pointer!r}."
                )

            current = current[token]

        elif isinstance(current, list):
            if not token.isdigit():
                raise TypeError(
                    "List pointer token must be a "
                    f"nonnegative integer: {token!r}"
                )

            index = int(token)

            if index >= len(current):
                raise IndexError(
                    f"List index {index} is outside "
                    f"pointer {pointer!r}."
                )

            current = current[index]

        else:
            raise TypeError(
                "Cannot traverse through value of "
                f"type {type(current).__name__} "
                f"at pointer {pointer!r}."
            )

    return current


def _resolve_parent(
    document,
    pointer: str,
):
    tokens = decode_json_pointer(
        pointer
    )

    if not tokens:
        raise ValueError(
            "The document root cannot be an "
            "annotation target."
        )

    current = document

    for token in tokens[:-1]:
        if isinstance(current, dict):
            if token not in current:
                raise KeyError(
                    f"Missing pointer token {token!r} "
                    f"in {pointer!r}."
                )

            current = current[token]

        elif isinstance(current, list):
            if not token.isdigit():
                raise TypeError(
                    "List pointer token must be a "
                    f"nonnegative integer: {token!r}"
                )

            current = current[int(token)]

        else:
            raise TypeError(
                "Cannot traverse through value of "
                f"type {type(current).__name__}."
            )

    return current, tokens[-1]


def transform_document_at_pointers(
    document,
    pointers,
    seed: int = 42,
):
    pointer_list = list(pointers)

    if len(pointer_list) != len(
        set(pointer_list)
    ):
        raise ValueError(
            "Duplicate annotation pointers are "
            "not allowed."
        )

    transformed_document = copy.deepcopy(
        document
    )

    transformation_log = []

    for pointer in sorted(pointer_list):
        source_value = resolve_json_pointer(
            document,
            pointer,
        )

        if not isinstance(
            source_value,
            str,
        ):
            raise TypeError(
                f"Annotation at {pointer!r} is "
                f"{type(source_value).__name__}, "
                "not str."
            )

        transformed_value = (
            obfuscate_annotation(
                source_value,
                seed=seed,
            )
        )

        parent, final_token = _resolve_parent(
            transformed_document,
            pointer,
        )

        if isinstance(parent, dict):
            parent[final_token] = (
                transformed_value
            )
        elif isinstance(parent, list):
            if not final_token.isdigit():
                raise TypeError(
                    "List pointer token must be a "
                    "nonnegative integer."
                )

            parent[int(final_token)] = (
                transformed_value
            )
        else:
            raise TypeError(
                "Annotation parent is neither "
                "a mapping nor a list."
            )

        ascii_letter_count = sum(
            character.isascii()
            and character.isalpha()
            for character in source_value
        )

        transformation_log.append({
            "pointer": pointer,
            "original_sha256": hashlib.sha256(
                source_value.encode("utf-8")
            ).hexdigest(),
            "transformed_sha256": hashlib.sha256(
                transformed_value.encode("utf-8")
            ).hexdigest(),
            "character_count": len(
                source_value
            ),
            "ascii_letter_count": (
                ascii_letter_count
            ),
            "changed": (
                transformed_value
                != source_value
            ),
            "seed": seed,
        })

    return (
        transformed_document,
        transformation_log,
    )
'''


write_text_atomic_cell5(
    OBFUSCATION_SOURCE_PATH,
    obfuscation_source_cell6,
)

obfuscation_implementation_sha256_cell6 = (
    sha256_file_cell5(
        OBFUSCATION_SOURCE_PATH
    )
)


module_spec_cell6 = (
    importlib.util.spec_from_file_location(
        "stage03_obfuscation",
        OBFUSCATION_SOURCE_PATH,
    )
)

if (
    module_spec_cell6 is None
    or module_spec_cell6.loader is None
):
    raise RuntimeError(
        "Could not load the obfuscation module."
    )

obfuscation_module_cell6 = (
    importlib.util.module_from_spec(
        module_spec_cell6
    )
)

module_spec_cell6.loader.exec_module(
    obfuscation_module_cell6
)

obfuscate_annotation_cell6 = (
    obfuscation_module_cell6
    .obfuscate_annotation
)

transform_document_cell6 = (
    obfuscation_module_cell6
    .transform_document_at_pointers
)


fixture_l0_cell6 = {
    "openapi": "3.0.3",
    "info": {
        "title": "Qualification API",
        "version": "1.0.0",
        "description": (
            "Protected information text."
        ),
    },
    "paths": {
        "/users": {
            "post": {
                "operationId": "createUser",
                "summary": (
                    "Create a new user."
                ),
                "description": (
                    "Creates one user account "
                    "for the supplied profile."
                ),
                "x-protected-note": (
                    "This extension remains readable."
                ),
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "required": [
                                    "name",
                                ],
                                "properties": {
                                    "name": {
                                        "type": "string",
                                        "description": (
                                            "The user's visible name."
                                        ),
                                    },
                                    "description": {
                                        "type": "string",
                                        "example": (
                                            "A property literally named "
                                            "description."
                                        ),
                                    },
                                    "role": {
                                        "type": "string",
                                        "enum": [
                                            "admin",
                                            "member",
                                        ],
                                        "default": "member",
                                    },
                                },
                            }
                        }
                    },
                },
                "responses": {
                    "201": {
                        "description": (
                            "The user was created."
                        ),
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/User"
                                    )
                                }
                            }
                        },
                    }
                },
                "security": [
                    {
                        "ApiKey": [],
                    }
                ],
            }
        }
    },
    "components": {
        "schemas": {
            "User": {
                "type": "object",
                "description": (
                    "A stored user record."
                ),
                "properties": {
                    "id": {
                        "type": "integer",
                        "format": "int64",
                    },
                    "name": {
                        "type": "string",
                    },
                },
            }
        },
        "securitySchemes": {
            "ApiKey": {
                "type": "apiKey",
                "in": "header",
                "name": "X-API-Key",
            }
        },
    },
}

approved_pointers_cell6 = [
    "/paths/~1users/post/summary",
    "/paths/~1users/post/description",
    (
        "/paths/~1users/post/requestBody/"
        "content/application~1json/schema/"
        "properties/name/description"
    ),
    (
        "/paths/~1users/post/responses/"
        "201/description"
    ),
    "/components/schemas/User/description",
]


fixture_l1_cell6, log_first_cell6 = (
    transform_document_cell6(
        fixture_l0_cell6,
        approved_pointers_cell6,
        seed=42,
    )
)

fixture_l1_repeat_cell6, log_repeat_cell6 = (
    transform_document_cell6(
        fixture_l0_cell6,
        approved_pointers_cell6,
        seed=42,
    )
)


def escape_pointer_token_cell6(token):
    return str(token).replace(
        "~",
        "~0",
    ).replace(
        "/",
        "~1",
    )


def collect_leaf_differences_cell6(
    left,
    right,
    pointer="",
):
    differences = []

    if type(left) is not type(right):
        return [pointer]

    if isinstance(left, dict):
        if set(left) != set(right):
            differences.append(pointer)

        for key in sorted(
            set(left) & set(right),
            key=str,
        ):
            child_pointer = (
                pointer
                + "/"
                + escape_pointer_token_cell6(
                    key
                )
            )

            differences.extend(
                collect_leaf_differences_cell6(
                    left[key],
                    right[key],
                    child_pointer,
                )
            )

    elif isinstance(left, list):
        if len(left) != len(right):
            differences.append(pointer)

        for index in range(
            min(len(left), len(right))
        ):
            child_pointer = (
                pointer + f"/{index}"
            )

            differences.extend(
                collect_leaf_differences_cell6(
                    left[index],
                    right[index],
                    child_pointer,
                )
            )

    elif left != right:
        differences.append(pointer)

    return differences


def character_preservation_cell6(
    original,
    transformed,
):
    results = {
        "length_equal": (
            len(original)
            == len(transformed)
        ),
        "every_ascii_letter_changed": True,
        "ascii_case_pattern_preserved": True,
        "all_non_ascii_letter_positions_unchanged": (
            True
        ),
    }

    if len(original) != len(transformed):
        return results

    for (
        original_character,
        transformed_character,
    ) in zip(
        original,
        transformed,
    ):
        is_ascii_letter = (
            original_character.isascii()
            and original_character.isalpha()
        )

        if is_ascii_letter:
            if (
                original_character
                == transformed_character
            ):
                results[
                    "every_ascii_letter_changed"
                ] = False

            if (
                original_character.isupper()
                != transformed_character.isupper()
            ):
                results[
                    "ascii_case_pattern_preserved"
                ] = False

        elif (
            original_character
            != transformed_character
        ):
            results[
                (
                    "all_non_ascii_letter_"
                    "positions_unchanged"
                )
            ] = False

    return results


difference_pointers_cell6 = sorted(
    collect_leaf_differences_cell6(
        fixture_l0_cell6,
        fixture_l1_cell6,
    )
)

expected_difference_pointers_cell6 = sorted(
    approved_pointers_cell6
)

pointer_preservation_results_cell6 = []

for pointer_cell6 in (
    approved_pointers_cell6
):
    original_cell6 = (
        obfuscation_module_cell6
        .resolve_json_pointer(
            fixture_l0_cell6,
            pointer_cell6,
        )
    )

    transformed_cell6 = (
        obfuscation_module_cell6
        .resolve_json_pointer(
            fixture_l1_cell6,
            pointer_cell6,
        )
    )

    pointer_preservation_results_cell6.append({
        "pointer": pointer_cell6,
        **character_preservation_cell6(
            original_cell6,
            transformed_cell6,
        ),
        "changed": (
            original_cell6
            != transformed_cell6
        ),
    })


validate(fixture_l0_cell6)
validate(fixture_l1_cell6)


sample_text_cell6 = (
    "Create USER-42 café Δ now!"
)

sample_transformed_cell6 = (
    obfuscate_annotation_cell6(
        sample_text_cell6,
        seed=42,
    )
)

sample_repeat_cell6 = (
    obfuscate_annotation_cell6(
        sample_text_cell6,
        seed=42,
    )
)

repeated_word_cell6 = (
    obfuscate_annotation_cell6(
        "User User user USER",
        seed=42,
    )
)

repeated_parts_cell6 = (
    repeated_word_cell6.split()
)


checks_cell6 = {
    "render_is_deterministic": (
        fixture_l1_cell6
        == fixture_l1_repeat_cell6
    ),
    "log_is_deterministic": (
        log_first_cell6
        == log_repeat_cell6
    ),
    "only_approved_values_changed": (
        difference_pointers_cell6
        == expected_difference_pointers_cell6
    ),
    "all_approved_values_changed": all(
        result["changed"]
        for result
        in pointer_preservation_results_cell6
    ),
    "all_lengths_preserved": all(
        result["length_equal"]
        for result
        in pointer_preservation_results_cell6
    ),
    "every_ascii_letter_changed": all(
        result[
            "every_ascii_letter_changed"
        ]
        for result
        in pointer_preservation_results_cell6
    ),
    "ascii_case_patterns_preserved": all(
        result[
            "ascii_case_pattern_preserved"
        ]
        for result
        in pointer_preservation_results_cell6
    ),
    "nonletters_and_nonascii_preserved": all(
        result[
            (
                "all_non_ascii_letter_"
                "positions_unchanged"
            )
        ]
        for result
        in pointer_preservation_results_cell6
    ),
    "sample_is_deterministic": (
        sample_transformed_cell6
        == sample_repeat_cell6
    ),
    "sample_length_preserved": (
        len(sample_text_cell6)
        == len(sample_transformed_cell6)
    ),
    "sample_nonascii_preserved": (
        "é" in sample_transformed_cell6
        and "Δ" in sample_transformed_cell6
    ),
    "sample_digits_preserved": (
        "42" in sample_transformed_cell6
    ),
    "identical_words_map_identically": (
        repeated_parts_cell6[0].lower()
        == repeated_parts_cell6[1].lower()
        == repeated_parts_cell6[2].lower()
        == repeated_parts_cell6[3].lower()
    ),
    "empty_string_supported": (
        obfuscate_annotation_cell6(
            "",
            seed=42,
        )
        == ""
    ),
    "property_named_description_preserved": (
        "description"
        in fixture_l1_cell6[
            "paths"
        ]["/users"]["post"][
            "requestBody"
        ]["content"]["application/json"][
            "schema"
        ]["properties"]
    ),
    "protected_info_description_unchanged": (
        fixture_l1_cell6["info"][
            "description"
        ]
        == fixture_l0_cell6["info"][
            "description"
        ]
    ),
    "protected_extension_unchanged": (
        fixture_l1_cell6[
            "paths"
        ]["/users"]["post"][
            "x-protected-note"
        ]
        == fixture_l0_cell6[
            "paths"
        ]["/users"]["post"][
            "x-protected-note"
        ]
    ),
    "L0_validates": True,
    "L1_validates": True,
}


qualification_status_cell6 = (
    "PASS"
    if all(checks_cell6.values())
    else "BLOCKED"
)


FIXTURE_L0_PATH_CELL6 = (
    OBFUSCATION_QUALIFICATION_ROOT
    / "qualification_L0.json"
)

FIXTURE_L1_PATH_CELL6 = (
    OBFUSCATION_QUALIFICATION_ROOT
    / "qualification_L1.json"
)

write_json_atomic_cell5(
    FIXTURE_L0_PATH_CELL6,
    fixture_l0_cell6,
)

write_json_atomic_cell5(
    FIXTURE_L1_PATH_CELL6,
    fixture_l1_cell6,
)


OBFUSCATION_QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "obfuscation_qualification.json"
)

qualification_report_cell6 = {
    "algorithm": (
        "deterministic_nonzero_ascii_"
        "alphabetic_shift"
    ),
    "seed": 42,
    "implementation_path": str(
        OBFUSCATION_SOURCE_PATH
    ),
    "implementation_sha256": (
        obfuscation_implementation_sha256_cell6
    ),
    "approved_pointer_count": len(
        approved_pointers_cell6
    ),
    "observed_difference_pointers": (
        difference_pointers_cell6
    ),
    "expected_difference_pointers": (
        expected_difference_pointers_cell6
    ),
    "pointer_preservation_results": (
        pointer_preservation_results_cell6
    ),
    "transformation_log": (
        log_first_cell6
    ),
    "sample_original": sample_text_cell6,
    "sample_transformed": (
        sample_transformed_cell6
    ),
    "checks": checks_cell6,
    "fixture_L0_path": str(
        FIXTURE_L0_PATH_CELL6
    ),
    "fixture_L0_sha256": (
        sha256_file_cell5(
            FIXTURE_L0_PATH_CELL6
        )
    ),
    "fixture_L1_path": str(
        FIXTURE_L1_PATH_CELL6
    ),
    "fixture_L1_sha256": (
        sha256_file_cell5(
            FIXTURE_L1_PATH_CELL6
        )
    ),
    "status": qualification_status_cell6,
}

write_json_atomic_cell5(
    OBFUSCATION_QUALIFICATION_REPORT_PATH,
    qualification_report_cell6,
)

qualification_report_sha256_cell6 = (
    sha256_file_cell5(
        OBFUSCATION_QUALIFICATION_REPORT_PATH
    )
)


with active_obfuscation_path_cell5.open(
    "r",
    encoding="utf-8",
) as file:
    active_obfuscation_cell6 = (
        yaml.safe_load(file)
    )

active_obfuscation_cell6[
    "algorithm"
] = (
    "deterministic_nonzero_ascii_"
    "alphabetic_shift"
)

active_obfuscation_cell6[
    "algorithm_version_sha256"
] = obfuscation_implementation_sha256_cell6

active_obfuscation_cell6[
    "implementation_path"
] = "src/obfuscation.py"

active_obfuscation_cell6[
    "qualification_report"
] = (
    "reports/audit/"
    "obfuscation_qualification.json"
)

active_obfuscation_cell6[
    "qualification_report_sha256"
] = qualification_report_sha256_cell6

active_obfuscation_cell6[
    "guarantees"
] = {
    "length_preserved": True,
    "whitespace_preserved": True,
    "punctuation_preserved": True,
    "digits_preserved": True,
    "ascii_case_pattern_preserved": True,
    "non_ascii_characters_preserved": True,
    "every_ascii_letter_changed": True,
    "identical_strings_map_identically": True,
    "only_approved_pointers_transformed": True,
}

active_obfuscation_cell6[
    "status"
] = (
    "qualified_notebook_03"
    if qualification_status_cell6 == "PASS"
    else "blocked"
)

write_yaml_atomic_cell5(
    active_obfuscation_path_cell5,
    active_obfuscation_cell6,
)


with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage_03_policy_cell6 = (
        yaml.safe_load(file)
    )

stage_03_policy_cell6[
    "transformation"
][
    "algorithm_version_sha256"
] = obfuscation_implementation_sha256_cell6

stage_03_policy_cell6[
    "transformation"
][
    "qualification_status"
] = (
    "PASS"
    if qualification_status_cell6 == "PASS"
    else "BLOCKED"
)

stage_03_policy_cell6[
    "transformation"
][
    "qualification_report_sha256"
] = qualification_report_sha256_cell6

write_yaml_atomic_cell5(
    STAGE_03_POLICY_PATH,
    stage_03_policy_cell6,
)


cell_6_report = {
    "algorithm": (
        "deterministic_nonzero_ascii_"
        "alphabetic_shift"
    ),
    "seed": 42,
    "implementation": {
        "path": str(
            OBFUSCATION_SOURCE_PATH
        ),
        "sha256": (
            obfuscation_implementation_sha256_cell6
        ),
    },
    "qualification": {
        "path": str(
            OBFUSCATION_QUALIFICATION_REPORT_PATH
        ),
        "sha256": (
            qualification_report_sha256_cell6
        ),
        "check_count": len(checks_cell6),
        "passed_check_count": sum(
            bool(value)
            for value in checks_cell6.values()
        ),
        "failed_checks": [
            name
            for name, passed
            in checks_cell6.items()
            if not passed
        ],
    },
    "active_obfuscation": {
        "path": str(
            active_obfuscation_path_cell5
        ),
        "sha256": (
            sha256_file_cell5(
                active_obfuscation_path_cell5
            )
        ),
        "status": (
            active_obfuscation_cell6[
                "status"
            ]
        ),
    },
    "updated_stage_03_policy": {
        "path": str(
            STAGE_03_POLICY_PATH
        ),
        "sha256": (
            sha256_file_cell5(
                STAGE_03_POLICY_PATH
            )
        ),
    },
    "status": qualification_status_cell6,
}

print(
    json.dumps(
        cell_6_report,
        indent=2,
        ensure_ascii=False,
    )
)

if qualification_status_cell6 != "PASS":
    raise RuntimeError(
        "Obfuscation qualification failed."
    )

print(
    "\nNOTEBOOK 03 — CELL 6: PASS"
)

{
  "algorithm": "deterministic_nonzero_ascii_alphabetic_shift",
  "seed": 42,
  "implementation": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/src/obfuscation.py",
    "sha256": "79eb0c682511ff393845604fb94e9ebe6c261c03208a22e50f79549719b05cbf"
  },
  "qualification": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/reports/audit/obfuscation_qualification.json",
    "sha256": "62b6643316208f0373ab06caa2e7ddb1d3e5730ffa898b641d1f65d08a66f4a1",
    "check_count": 19,
    "passed_check_count": 19,
    "failed_checks": []
  },
  "active_obfuscation": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/config/obfuscation.yaml",
    "sha256": "781a1868197b00be871e8e2ebfed66cdf2f36e633fe97a72898adb6b099da555",
    "status": "qualified_notebook_03"
  },
  "updated_stage_03_policy": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/config/stage_03.yaml",
    "sha256": "e08341ec1062b8e8af7f1176ce9719

In [7]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import importlib.util
import json
import shutil
import time
import yaml


CELL_7_STARTED = time.perf_counter()

PAIRING_SOURCE_PATH = (
    PROJECT_ROOT
    / "src"
    / "pairing.py"
)

FROZEN_ELIGIBLE_REGISTRY_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "stage_02_eligible_operations.jsonl"
)

PAIR_SEED_FRAME_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "pair_frame_seed.jsonl"
)

FAMILY_MAP_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "family_map.csv"
)

PAIRING_QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "pairing_qualification.json"
)


def write_csv_atomic_cell7(
    path: Path,
    rows,
    fieldnames,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
            extrasaction="raise",
        )

        writer.writeheader()
        writer.writerows(rows)

    temporary_path.replace(path)


pairing_source_cell7 = '''\
import hashlib


PAIR_NAMESPACE = "stage03-pair-v1"
FAMILY_NAMESPACE = "api-family-v1"


def _stable_identifier(
    prefix: str,
    namespace: str,
    value: str,
) -> str:
    if not isinstance(value, str):
        raise TypeError(
            "Identifier source value must be a string."
        )

    if not value.strip():
        raise ValueError(
            "Identifier source value cannot be empty."
        )

    digest = hashlib.sha256(
        f"{namespace}|{value}".encode("utf-8")
    ).hexdigest()[:24]

    return f"{prefix}_{digest}"


def make_pair_id(op_id: str) -> str:
    return _stable_identifier(
        "pair",
        PAIR_NAMESPACE,
        op_id,
    )


def build_pair_seed(operation):
    required_fields = [
        "op_id",
        "spec_id",
        "family_id",
        "provider_directory",
        "source_class",
        "method",
        "path",
        "slice_relative_path",
        "slice_sha256",
        "prose_tokens_L0",
        "rendered_prompt_tokens",
        "input_token_limit",
    ]

    missing = [
        field
        for field in required_fields
        if field not in operation
    ]

    if missing:
        raise KeyError(
            "Operation record is missing fields: "
            + ", ".join(missing)
        )

    op_id = operation["op_id"]
    provider_directory = (
        operation["provider_directory"]
    )

    return {
        "pair_id": make_pair_id(op_id),
        "op_id": op_id,
        "spec_id": operation["spec_id"],
        "family_id": operation["family_id"],
        "provider_directory": (
            provider_directory
        ),
        "source_class": (
            operation["source_class"]
        ),
        "method": operation["method"],
        "path": operation["path"],
        "prose_tokens_L0": int(
            operation["prose_tokens_L0"]
        ),
        "rendered_prompt_tokens_L0": int(
            operation[
                "rendered_prompt_tokens"
            ]
        ),
        "input_token_limit": int(
            operation["input_token_limit"]
        ),
        "L0_slice_relative_path": (
            operation["slice_relative_path"]
        ),
        "L0_sha256": (
            operation["slice_sha256"]
        ),
        "L1_slice_relative_path": (
            f"data/slices/L1/{op_id}.json"
        ),
        "pair_status": (
            "PENDING_L1_CONSTRUCTION"
        ),
    }
'''


write_text_atomic_cell5(
    PAIRING_SOURCE_PATH,
    pairing_source_cell7,
)

pairing_source_sha256_cell7 = (
    sha256_file_cell5(
        PAIRING_SOURCE_PATH
    )
)


module_spec_cell7 = (
    importlib.util.spec_from_file_location(
        "stage03_pairing",
        PAIRING_SOURCE_PATH,
    )
)

if (
    module_spec_cell7 is None
    or module_spec_cell7.loader is None
):
    raise RuntimeError(
        "Could not load the Stage 03 "
        "pairing module."
    )

pairing_module_cell7 = (
    importlib.util.module_from_spec(
        module_spec_cell7
    )
)

module_spec_cell7.loader.exec_module(
    pairing_module_cell7
)


pair_fixture_cell7 = {
    "op_id": "op_fixture_001",
    "spec_id": "spec_fixture_001",
    "family_id": "family_" + "a" * 24,
    "provider_directory": "example.com",
    "source_class": "native_openapi",
    "method": "post",
    "path": "/users",
    "slice_relative_path": (
        "data/slices/L0/op_fixture_001.json"
    ),
    "slice_sha256": "a" * 64,
    "prose_tokens_L0": 25,
    "rendered_prompt_tokens": 500,
    "input_token_limit": 5120,
}

pair_fixture_first_cell7 = (
    pairing_module_cell7.build_pair_seed(
        pair_fixture_cell7
    )
)

pair_fixture_second_cell7 = (
    pairing_module_cell7.build_pair_seed(
        pair_fixture_cell7
    )
)

pair_fixture_other_cell7 = (
    pairing_module_cell7.build_pair_seed({
        **pair_fixture_cell7,
        "op_id": "op_fixture_002",
    })
)

pair_fixture_same_family_cell7 = (
    pairing_module_cell7.build_pair_seed({
        **pair_fixture_cell7,
        "op_id": "op_fixture_003",
        "spec_id": "spec_fixture_002",
    })
)

pair_fixture_other_family_cell7 = (
    pairing_module_cell7.build_pair_seed({
        **pair_fixture_cell7,
        "op_id": "op_fixture_004",
        "family_id": "family_" + "b" * 24,
        "provider_directory": (
            "example.com"
        ),
    })
)


fixture_checks_cell7 = {
    "pair_seed_is_deterministic": (
        pair_fixture_first_cell7
        == pair_fixture_second_cell7
    ),
    "pair_id_has_expected_format": (
        pair_fixture_first_cell7[
            "pair_id"
        ].startswith("pair_")
        and len(
            pair_fixture_first_cell7[
                "pair_id"
            ]
        )
        == 29
    ),
    "different_operations_get_different_pairs": (
        pair_fixture_first_cell7[
            "pair_id"
        ]
        != pair_fixture_other_cell7[
            "pair_id"
        ]
    ),
    "same_reviewed_family_is_preserved": (
        pair_fixture_first_cell7[
            "family_id"
        ]
        == pair_fixture_same_family_cell7[
            "family_id"
        ]
    ),
    "distinct_reviewed_family_is_preserved": (
        pair_fixture_first_cell7[
            "family_id"
        ]
        != pair_fixture_other_family_cell7[
            "family_id"
        ]
    ),
    "L0_identity_is_preserved": (
        pair_fixture_first_cell7[
            "op_id"
        ]
        == pair_fixture_cell7["op_id"]
        and pair_fixture_first_cell7[
            "L0_sha256"
        ]
        == pair_fixture_cell7[
            "slice_sha256"
        ]
    ),
    "L1_path_uses_same_operation_identity": (
        pair_fixture_first_cell7[
            "L1_slice_relative_path"
        ]
        == (
            "data/slices/L1/"
            "op_fixture_001.json"
        )
    ),
}


source_registry_sha256_cell7 = (
    sha256_file_cell5(
        STAGE_02_ELIGIBLE_OPERATIONS_PATH
    )
)

expected_source_registry_sha256_cell7 = next(a["sha256"] for a in read_json(STAGE_02_INTERNAL_MANIFEST_PATH)["artifacts"]
     if a["path"] == "data/inventory/final_eligible_operations.jsonl")

if (
    source_registry_sha256_cell7
    != expected_source_registry_sha256_cell7
):
    raise RuntimeError(
        "Stage 02 eligible-operation registry "
        "hash does not match the frozen value."
    )


shutil.copyfile(
    STAGE_02_ELIGIBLE_OPERATIONS_PATH,
    FROZEN_ELIGIBLE_REGISTRY_PATH,
)

frozen_registry_sha256_cell7 = (
    sha256_file_cell5(
        FROZEN_ELIGIBLE_REGISTRY_PATH
    )
)

if (
    frozen_registry_sha256_cell7
    != source_registry_sha256_cell7
):
    raise RuntimeError(
        "Frozen operation registry does not "
        "match its Stage 02 source."
    )


pair_seed_temporary_path_cell7 = (
    PAIR_SEED_FRAME_PATH.with_name(
        PAIR_SEED_FRAME_PATH.name + ".tmp"
    )
)

operation_count_cell7 = 0

operation_ids_cell7 = set()
pair_ids_cell7 = set()
spec_ids_cell7 = set()
family_ids_cell7 = set()

source_class_counts_cell7 = Counter()
method_counts_cell7 = Counter()
family_operation_counts_cell7 = Counter()
family_spec_ids_cell7 = defaultdict(set)

spec_family_map_cell7 = {}
family_provider_map_cell7 = {}

registry_problem_count_cell7 = 0
registry_problem_examples_cell7 = []
registry_problem_limit_cell7 = 100


def record_registry_problem_cell7(
    problem,
):
    global registry_problem_count_cell7

    registry_problem_count_cell7 += 1

    if (
        len(registry_problem_examples_cell7)
        < registry_problem_limit_cell7
    ):
        registry_problem_examples_cell7.append(
            problem
        )


with (
    FROZEN_ELIGIBLE_REGISTRY_PATH.open(
        "r",
        encoding="utf-8",
    )
) as source_file_cell7, (
    pair_seed_temporary_path_cell7.open(
        "w",
        encoding="utf-8",
        newline="\n",
    )
) as pair_file_cell7:
    for line_number_cell7, line_cell7 in enumerate(
        source_file_cell7,
        start=1,
    ):
        if not line_cell7.strip():
            continue

        try:
            operation_cell7 = json.loads(
                line_cell7
            )

            pair_seed_cell7 = (
                pairing_module_cell7
                .build_pair_seed(
                    operation_cell7
                )
            )

        except Exception as error:
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": (
                    "PAIR_SEED_CONSTRUCTION_FAILED"
                ),
                "diagnostic": repr(error),
            })
            continue

        operation_count_cell7 += 1

        op_id_cell7 = pair_seed_cell7[
            "op_id"
        ]

        pair_id_cell7 = pair_seed_cell7[
            "pair_id"
        ]

        spec_id_cell7 = pair_seed_cell7[
            "spec_id"
        ]

        family_id_cell7 = pair_seed_cell7[
            "family_id"
        ]

        provider_cell7 = pair_seed_cell7[
            "provider_directory"
        ]

        if op_id_cell7 in operation_ids_cell7:
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": "DUPLICATE_OP_ID",
                "op_id": op_id_cell7,
            })

        if pair_id_cell7 in pair_ids_cell7:
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": "DUPLICATE_PAIR_ID",
                "pair_id": pair_id_cell7,
            })

        operation_ids_cell7.add(op_id_cell7)
        pair_ids_cell7.add(pair_id_cell7)
        spec_ids_cell7.add(spec_id_cell7)
        family_ids_cell7.add(family_id_cell7)

        if operation_cell7.get(
            "eligible_for_experimental_generation"
        ) is not True:
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": (
                    "UPSTREAM_RECORD_NOT_ELIGIBLE"
                ),
                "op_id": op_id_cell7,
            })

        if operation_cell7.get(
            "slice_status"
        ) != "PASS":
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": (
                    "UPSTREAM_SLICE_STATUS_NOT_PASS"
                ),
                "op_id": op_id_cell7,
            })

        if operation_cell7.get(
            "within_input_limit"
        ) is not True:
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": (
                    "UPSTREAM_PROMPT_OVER_LIMIT"
                ),
                "op_id": op_id_cell7,
            })

        l0_relative_cell7 = Path(
            pair_seed_cell7[
                "L0_slice_relative_path"
            ]
        )

        if (
            l0_relative_cell7.is_absolute()
            or ".." in l0_relative_cell7.parts
        ):
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": "UNSAFE_L0_PATH",
                "op_id": op_id_cell7,
            })

        else:
            l0_path_cell7 = (
                STAGE_02_PROJECT_ROOT
                / l0_relative_cell7
            )

            if not l0_path_cell7.is_file():
                record_registry_problem_cell7({
                    "line": line_number_cell7,
                    "problem": "MISSING_L0_SLICE",
                    "op_id": op_id_cell7,
                })

        previous_spec_family_cell7 = (
            spec_family_map_cell7.get(
                spec_id_cell7
            )
        )

        if (
            previous_spec_family_cell7
            is not None
            and previous_spec_family_cell7
            != family_id_cell7
        ):
            record_registry_problem_cell7({
                "line": line_number_cell7,
                "problem": (
                    "SPEC_ASSIGNED_TO_MULTIPLE_"
                    "FAMILIES"
                ),
                "spec_id": spec_id_cell7,
            })

        spec_family_map_cell7[
            spec_id_cell7
        ] = family_id_cell7

        previous_provider = family_provider_map_cell7.setdefault(family_id_cell7, provider_cell7)
        if previous_provider != provider_cell7:
            raise RuntimeError("A reviewed family cannot span different corpus providers.")

        source_class_counts_cell7[
            pair_seed_cell7["source_class"]
        ] += 1

        method_counts_cell7[
            pair_seed_cell7["method"]
        ] += 1

        family_operation_counts_cell7[
            family_id_cell7
        ] += 1

        family_spec_ids_cell7[
            family_id_cell7
        ].add(spec_id_cell7)

        pair_file_cell7.write(
            json.dumps(
                pair_seed_cell7,
                ensure_ascii=False,
                separators=(",", ":"),
            )
            + "\n"
        )

        if (
            operation_count_cell7 % 5000
            == 0
        ):
            print(
                json.dumps({
                    "operations_processed": (
                        operation_count_cell7
                    ),
                    "unique_pairs": len(
                        pair_ids_cell7
                    ),
                    "unique_families": len(
                        family_ids_cell7
                    ),
                    "problem_count": (
                        registry_problem_count_cell7
                    ),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - CELL_7_STARTED,
                        1,
                    ),
                })
            )


if registry_problem_count_cell7:
    pair_seed_temporary_path_cell7.unlink(
        missing_ok=True
    )

    raise RuntimeError(
        "Pair-seed construction failed:\n"
        + json.dumps(
            registry_problem_examples_cell7,
            indent=2,
        )
    )

pair_seed_temporary_path_cell7.replace(
    PAIR_SEED_FRAME_PATH
)


family_rows_cell7 = []

for family_id_cell7, provider_cell7 in sorted(family_provider_map_cell7.items()):
    family_rows_cell7.append({
        "family_id": family_id_cell7,
        "provider_directory": (
            provider_cell7
        ),
        "specification_count": len(
            family_spec_ids_cell7[
                family_id_cell7
            ]
        ),
        "operation_count": (
            family_operation_counts_cell7[
                family_id_cell7
            ]
        ),
        "family_rule_version": (
            "stage01_verified_family_identity_v1"
        ),
    })

write_csv_atomic_cell7(
    FAMILY_MAP_PATH,
    family_rows_cell7,
    [
        "family_id",
        "provider_directory",
        "specification_count",
        "operation_count",
        "family_rule_version",
    ],
)


population_checks_cell7 = {
    "operation_count_matches_stage02_candidate_count": (
        operation_count_cell7 == expected_eligible_operation_count
    ),
    "unique_operation_ids": (
        len(operation_ids_cell7)
        == operation_count_cell7
    ),
    "unique_pair_ids": (
        len(pair_ids_cell7)
        == operation_count_cell7
    ),
    "eligible_specification_count_matches_stage02": (
        len(spec_ids_cell7) == int(external_counts["eligible_specification_count"])
    ),
    "family_count_is_positive": (
        len(family_ids_cell7) > 0
    ),
    "every_family_has_an_operation": all(
        count_cell7 > 0
        for count_cell7
        in family_operation_counts_cell7.values()
    ),
    "family_map_accounts_for_all_family_ids": (
        len(family_provider_map_cell7)
        == len(family_ids_cell7)
    ),
    "all_specs_have_one_family": (
        len(spec_family_map_cell7)
        == len(spec_ids_cell7)
    ),
    "all_fixture_checks_pass": all(
        fixture_checks_cell7.values()
    ),
    "frozen_registry_matches_source": (
        frozen_registry_sha256_cell7
        == source_registry_sha256_cell7
    ),
    "no_registry_problems": (
        registry_problem_count_cell7 == 0
    ),
}


pairing_status_cell7 = (
    "PASS"
    if all(
        population_checks_cell7.values()
    )
    else "BLOCKED"
)


family_operation_values_cell7 = list(
    family_operation_counts_cell7.values()
)

pairing_report_cell7 = {
    "pairing_policy_version": "1.0",
    "pair_namespace": (
        pairing_module_cell7.PAIR_NAMESPACE
    ),
    "family_namespace": (
        pairing_module_cell7.FAMILY_NAMESPACE
    ),
    "family_rule": (
        "All operations whose exact "
        "provider_directory values match belong "
        "to the same API family."
    ),
    "family_rule_timing": (
        "Prospectively frozen before L1 "
        "construction, strata definition, "
        "sampling, or model generation."
    ),
    "family_rule_reason": (
        "Stage 02 carries the reviewed API family ID from Stage 01. "
        "Provider identity remains separate provenance."
    ),
    "upstream_provider_count_context": {
        "complete_stage_01_02_specification_corpus": (
            549
        ),
        "stage_03_eligible_operation_frame": (
            len(family_ids_cell7)
        ),
        "interpretation": (
            "Providers without an eligible "
            "operation are not represented in "
            "the Stage 03 operation-pair frame."
        ),
    },
    "operation_count": operation_count_cell7,
    "specification_count": len(
        spec_ids_cell7
    ),
    "family_count": len(
        family_ids_cell7
    ),
    "source_class_counts": dict(
        sorted(
            source_class_counts_cell7.items()
        )
    ),
    "method_counts": dict(
        sorted(
            method_counts_cell7.items()
        )
    ),
    "family_operation_distribution": {
        "minimum": min(
            family_operation_values_cell7
        ),
        "maximum": max(
            family_operation_values_cell7
        ),
        "families_with_one_operation": sum(
            count_cell7 == 1
            for count_cell7
            in family_operation_values_cell7
        ),
        "families_with_multiple_operations": sum(
            count_cell7 > 1
            for count_cell7
            in family_operation_values_cell7
        ),
    },
    "fixture_checks": fixture_checks_cell7,
    "population_checks": (
        population_checks_cell7
    ),
    "implementation_path": str(
        PAIRING_SOURCE_PATH
    ),
    "implementation_sha256": (
        pairing_source_sha256_cell7
    ),
    "source_registry": {
        "path": str(
            STAGE_02_ELIGIBLE_OPERATIONS_PATH
        ),
        "sha256": (
            source_registry_sha256_cell7
        ),
    },
    "frozen_registry": {
        "path": str(
            FROZEN_ELIGIBLE_REGISTRY_PATH
        ),
        "bytes": (
            FROZEN_ELIGIBLE_REGISTRY_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            frozen_registry_sha256_cell7
        ),
    },
    "pair_seed_frame": {
        "path": str(
            PAIR_SEED_FRAME_PATH
        ),
        "bytes": (
            PAIR_SEED_FRAME_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file_cell5(
                PAIR_SEED_FRAME_PATH
            )
        ),
    },
    "family_map": {
        "path": str(FAMILY_MAP_PATH),
        "rows": len(family_rows_cell7),
        "bytes": (
            FAMILY_MAP_PATH.stat().st_size
        ),
        "sha256": (
            sha256_file_cell5(
                FAMILY_MAP_PATH
            )
        ),
    },
    "duration_seconds": round(
        time.perf_counter()
        - CELL_7_STARTED,
        3,
    ),
    "status": pairing_status_cell7,
}

write_json_atomic_cell5(
    PAIRING_QUALIFICATION_REPORT_PATH,
    pairing_report_cell7,
)

pairing_report_sha256_cell7 = (
    sha256_file_cell5(
        PAIRING_QUALIFICATION_REPORT_PATH
    )
)


with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage_03_policy_cell7 = (
        yaml.safe_load(file)
    )

stage_03_policy_cell7[
    "pair_identity"
][
    "implementation_sha256"
] = pairing_source_sha256_cell7

stage_03_policy_cell7[
    "pair_identity"
][
    "qualification_report_sha256"
] = pairing_report_sha256_cell7

stage_03_policy_cell7[
    "pair_identity"
][
    "qualification_status"
] = pairing_status_cell7

stage_03_policy_cell7[
    "family_identity"
] = {
    "rule_version": (
        "stage01_verified_family_identity_v1"
    ),
    "source_field": (
        "family_id"
    ),
    "family_id_namespace": (
        pairing_module_cell7.FAMILY_NAMESPACE
    ),
    "exact_match_required": True,
    "normalization_applied": False,
    "complete_corpus_provider_count": 549,
    "eligible_frame_family_count": len(
        family_ids_cell7
    ),
    "timing": (
        "before L1 construction, strata "
        "definition, sampling, or generation"
    ),
}

write_yaml_atomic_cell5(
    STAGE_03_POLICY_PATH,
    stage_03_policy_cell7,
)


cell_7_report = {
    "operation_count": operation_count_cell7,
    "pair_count": len(pair_ids_cell7),
    "specification_count": len(spec_ids_cell7),
    "family_count": len(family_ids_cell7),
    "source_class_counts": dict(
        sorted(
            source_class_counts_cell7.items()
        )
    ),
    "method_counts": dict(
        sorted(
            method_counts_cell7.items()
        )
    ),
    "population_checks": (
        population_checks_cell7
    ),
    "implementation": {
        "path": str(PAIRING_SOURCE_PATH),
        "sha256": (
            pairing_source_sha256_cell7
        ),
    },
    "pair_seed_frame": {
        "path": str(
            PAIR_SEED_FRAME_PATH
        ),
        "sha256": (
            sha256_file_cell5(
                PAIR_SEED_FRAME_PATH
            )
        ),
    },
    "family_map": {
        "path": str(FAMILY_MAP_PATH),
        "sha256": (
            sha256_file_cell5(
                FAMILY_MAP_PATH
            )
        ),
    },
    "qualification_report": {
        "path": str(
            PAIRING_QUALIFICATION_REPORT_PATH
        ),
        "sha256": (
            pairing_report_sha256_cell7
        ),
    },
    "updated_stage_03_policy_sha256": (
        sha256_file_cell5(
            STAGE_03_POLICY_PATH
        )
    ),
    "duration_seconds": round(
        time.perf_counter()
        - CELL_7_STARTED,
        3,
    ),
    "status": pairing_status_cell7,
}

print(
    json.dumps(
        cell_7_report,
        indent=2,
        ensure_ascii=False,
    )
)

if pairing_status_cell7 != "PASS":
    raise RuntimeError(
        "Pair identity qualification failed."
    )

print(
    "\nNOTEBOOK 03 — CELL 7: PASS"
)

{"operations_processed": 5000, "unique_pairs": 5000, "unique_families": 981, "problem_count": 0, "elapsed_seconds": 4.0}
{"operations_processed": 10000, "unique_pairs": 10000, "unique_families": 1203, "problem_count": 0, "elapsed_seconds": 7.1}
{"operations_processed": 15000, "unique_pairs": 15000, "unique_families": 1325, "problem_count": 0, "elapsed_seconds": 10.2}
{"operations_processed": 20000, "unique_pairs": 20000, "unique_families": 1396, "problem_count": 0, "elapsed_seconds": 13.3}
{"operations_processed": 25000, "unique_pairs": 25000, "unique_families": 1462, "problem_count": 0, "elapsed_seconds": 16.4}
{"operations_processed": 30000, "unique_pairs": 30000, "unique_families": 1496, "problem_count": 0, "elapsed_seconds": 19.4}
{"operations_processed": 35000, "unique_pairs": 35000, "unique_families": 1534, "problem_count": 0, "elapsed_seconds": 22.4}
{"operations_processed": 40000, "unique_pairs": 40000, "unique_families": 1575, "problem_count": 0, "elapsed_seconds": 25.5}
{"ope

In [8]:
from pathlib import Path
import copy
import hashlib
import importlib.util
import json
import yaml

from openapi_spec_validator import validate


FEASIBILITY_SOURCE_PATH = (
    PROJECT_ROOT
    / "src"
    / "feasibility.py"
)

FEASIBILITY_QUALIFICATION_ROOT = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "feasibility_qualification"
)

FEASIBILITY_QUALIFICATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FEASIBILITY_QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "feasibility_builder_qualification.json"
)


feasibility_source_cell8 = '''\
import copy
import re


HTTP_METHODS = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


class FeasibilityError(Exception):
    pass


class UnsupportedSchema(FeasibilityError):
    pass


class UnresolvedReference(FeasibilityError):
    pass


def _decode_pointer_token(token):
    result = []
    position = 0

    while position < len(token):
        character = token[position]

        if character != "~":
            result.append(character)
            position += 1
            continue

        if position + 1 >= len(token):
            raise UnresolvedReference(
                "Invalid trailing tilde in reference."
            )

        escape = token[position + 1]

        if escape == "0":
            result.append("~")
        elif escape == "1":
            result.append("/")
        else:
            raise UnresolvedReference(
                f"Invalid JSON-pointer escape ~{escape}."
            )

        position += 2

    return "".join(result)


def resolve_local_reference(
    document,
    reference,
):
    if not isinstance(reference, str):
        raise UnresolvedReference(
            "Reference is not a string."
        )

    if not reference.startswith("#/"):
        raise UnresolvedReference(
            "Only local JSON references are supported."
        )

    current = document

    for raw_token in reference[2:].split("/"):
        token = _decode_pointer_token(
            raw_token
        )

        if isinstance(current, dict):
            if token not in current:
                raise UnresolvedReference(
                    f"Missing reference token {token!r} "
                    f"in {reference!r}."
                )

            current = current[token]

        elif isinstance(current, list):
            if not token.isdigit():
                raise UnresolvedReference(
                    "List reference token is not "
                    "an integer."
                )

            index = int(token)

            if index >= len(current):
                raise UnresolvedReference(
                    "List reference index is outside "
                    "the document."
                )

            current = current[index]

        else:
            raise UnresolvedReference(
                "Reference traverses through a "
                "non-container value."
            )

    return current


def dereference(
    document,
    value,
    reference_stack=None,
):
    if not isinstance(value, dict):
        return value

    if "$ref" not in value:
        return value

    reference = value["$ref"]

    if reference_stack is None:
        reference_stack = []

    if reference in reference_stack:
        raise UnsupportedSchema(
            "Recursive reference cannot be "
            "synthesized safely."
        )

    resolved = resolve_local_reference(
        document,
        reference,
    )

    if not isinstance(resolved, dict):
        raise UnresolvedReference(
            "Reference target is not an object."
        )

    merged = copy.deepcopy(resolved)

    for key, item in value.items():
        if key != "$ref":
            merged[key] = copy.deepcopy(item)

    if "$ref" in merged:
        return dereference(
            document,
            merged,
            reference_stack + [reference],
        )

    return merged


def select_schema_type(schema):
    schema_type = schema.get("type")

    if isinstance(schema_type, list):
        non_null_types = [
            value
            for value in schema_type
            if value != "null"
        ]

        if not non_null_types:
            return "null"

        return non_null_types[0]

    if isinstance(schema_type, str):
        return schema_type

    if "properties" in schema:
        return "object"

    if "items" in schema:
        return "array"

    return None


def _string_for_format(format_name):
    values = {
        "date": "2000-01-01",
        "date-time": "2000-01-01T00:00:00Z",
        "email": "user@example.com",
        "hostname": "example.com",
        "ipv4": "192.0.2.1",
        "ipv6": "2001:db8::1",
        "uri": "https://example.com/resource",
        "url": "https://example.com/resource",
        "uuid": (
            "00000000-0000-4000-8000-"
            "000000000000"
        ),
        "byte": "dGVzdA==",
        "binary": "test",
        "password": "example-password",
    }

    return values.get(
        format_name,
        "example",
    )


def synthesize_schema(
    document,
    schema,
    context="request",
    depth=0,
    reference_stack=None,
):
    if depth > 40:
        raise UnsupportedSchema(
            "Schema nesting exceeds the "
            "feasibility limit."
        )

    if reference_stack is None:
        reference_stack = []

    if not isinstance(schema, dict):
        raise UnsupportedSchema(
            "Schema is not an object."
        )

    if "$ref" in schema:
        reference = schema["$ref"]

        if reference in reference_stack:
            raise UnsupportedSchema(
                "Recursive schema reference is "
                "not deterministically synthesizable."
            )

        resolved = resolve_local_reference(
            document,
            reference,
        )

        if not isinstance(resolved, dict):
            raise UnresolvedReference(
                "Schema reference target is "
                "not an object."
            )

        merged = copy.deepcopy(resolved)

        for key, value in schema.items():
            if key != "$ref":
                merged[key] = copy.deepcopy(value)

        return synthesize_schema(
            document,
            merged,
            context=context,
            depth=depth + 1,
            reference_stack=(
                reference_stack + [reference]
            ),
        )

    if "const" in schema:
        return copy.deepcopy(
            schema["const"]
        )

    if "example" in schema:
        return copy.deepcopy(
            schema["example"]
        )

    if "default" in schema:
        return copy.deepcopy(
            schema["default"]
        )

    enum_values = schema.get("enum")

    if (
        isinstance(enum_values, list)
        and enum_values
    ):
        non_null_values = [
            value
            for value in enum_values
            if value is not None
        ]

        if non_null_values:
            return copy.deepcopy(
                non_null_values[0]
            )

        return None

    for union_keyword in (
        "oneOf",
        "anyOf",
    ):
        branches = schema.get(
            union_keyword
        )

        if (
            isinstance(branches, list)
            and branches
        ):
            last_error = None

            for branch in branches:
                try:
                    return synthesize_schema(
                        document,
                        branch,
                        context=context,
                        depth=depth + 1,
                        reference_stack=(
                            reference_stack
                        ),
                    )
                except FeasibilityError as error:
                    last_error = error

            raise UnsupportedSchema(
                f"No {union_keyword} branch "
                "could be synthesized."
            ) from last_error

    all_of = schema.get("allOf")

    if isinstance(all_of, list) and all_of:
        combined_object = {}

        for branch in all_of:
            branch_value = synthesize_schema(
                document,
                branch,
                context=context,
                depth=depth + 1,
                reference_stack=reference_stack,
            )

            if isinstance(
                branch_value,
                dict,
            ):
                combined_object.update(
                    branch_value
                )

        if combined_object:
            return combined_object

        raise UnsupportedSchema(
            "allOf did not produce an object."
        )

    schema_type = select_schema_type(
        schema
    )

    if schema_type == "null":
        return None

    if schema_type == "object":
        properties = schema.get(
            "properties",
            {},
        )

        if not isinstance(properties, dict):
            raise UnsupportedSchema(
                "Object properties are not "
                "a mapping."
            )

        required_names = schema.get(
            "required",
            [],
        )

        if not isinstance(
            required_names,
            list,
        ):
            raise UnsupportedSchema(
                "Object required value is not "
                "a list."
            )

        selected_names = list(
            required_names
        )

        if (
            not selected_names
            and properties
        ):
            selected_names = [
                sorted(
                    properties,
                    key=str,
                )[0]
            ]

        result = {}

        for property_name in selected_names:
            if property_name not in properties:
                raise UnsupportedSchema(
                    "Required property has no "
                    f"schema: {property_name}"
                )

            property_schema = properties[
                property_name
            ]

            if not isinstance(
                property_schema,
                dict,
            ):
                raise UnsupportedSchema(
                    "Property schema is not "
                    "an object."
                )

            if (
                context == "request"
                and property_schema.get(
                    "readOnly"
                )
                is True
            ):
                continue

            if (
                context == "response"
                and property_schema.get(
                    "writeOnly"
                )
                is True
            ):
                continue

            result[property_name] = (
                synthesize_schema(
                    document,
                    property_schema,
                    context=context,
                    depth=depth + 1,
                    reference_stack=(
                        reference_stack
                    ),
                )
            )

        return result

    if schema_type == "array":
        items = schema.get("items")

        if not isinstance(items, dict):
            raise UnsupportedSchema(
                "Array has no supported item schema."
            )

        minimum = schema.get(
            "minItems",
            1,
        )

        try:
            item_count = max(
                1,
                int(minimum),
            )
        except Exception as error:
            raise UnsupportedSchema(
                "Invalid minItems value."
            ) from error

        item_count = min(
            item_count,
            3,
        )

        return [
            synthesize_schema(
                document,
                items,
                context=context,
                depth=depth + 1,
                reference_stack=reference_stack,
            )
            for _ in range(item_count)
        ]

    if schema_type == "string":
        if (
            schema.get("pattern")
            and not any(
                key in schema
                for key in (
                    "example",
                    "default",
                    "enum",
                    "const",
                )
            )
        ):
            raise UnsupportedSchema(
                "String pattern has no frozen "
                "example/default/enum/const."
            )

        value = _string_for_format(
            schema.get("format")
        )

        minimum = schema.get(
            "minLength",
            0,
        )

        try:
            minimum = int(minimum)
        except Exception:
            minimum = 0

        if len(value) < minimum:
            value = value + (
                "x"
                * (minimum - len(value))
            )

        maximum = schema.get(
            "maxLength"
        )

        if maximum is not None:
            try:
                maximum = int(maximum)
                value = value[:maximum]
            except Exception:
                pass

        return value

    if schema_type == "integer":
        minimum = schema.get(
            "minimum",
            1,
        )

        try:
            value = int(minimum)
        except Exception:
            value = 1

        if schema.get(
            "exclusiveMinimum"
        ) is True:
            value += 1

        return value

    if schema_type == "number":
        minimum = schema.get(
            "minimum",
            1.0,
        )

        try:
            value = float(minimum)
        except Exception:
            value = 1.0

        if schema.get(
            "exclusiveMinimum"
        ) is True:
            value += 1.0

        return value

    if schema_type == "boolean":
        return True

    if schema_type is None:
        if schema.get("nullable") is True:
            return None

        raise UnsupportedSchema(
            "Schema has no supported type, "
            "enum, example, default, const, "
            "properties, or composition."
        )

    raise UnsupportedSchema(
        f"Unsupported schema type: {schema_type}"
    )


def choose_response(
    document,
    operation,
):
    responses = operation.get(
        "responses"
    )

    if not isinstance(responses, dict):
        raise FeasibilityError(
            "Operation has no response mapping."
        )

    success_codes = sorted(
        str(code)
        for code in responses
        if re.fullmatch(
            r"2[0-9][0-9]",
            str(code),
        )
    )

    if success_codes:
        selected_code = success_codes[0]
    elif "default" in responses:
        selected_code = "default"
    elif responses:
        selected_code = sorted(
            responses,
            key=str,
        )[0]
    else:
        raise FeasibilityError(
            "Operation has no response entries."
        )

    response = dereference(
        document,
        responses[selected_code],
    )

    if not isinstance(response, dict):
        raise FeasibilityError(
            "Selected response is not an object."
        )

    return selected_code, response


def choose_media_schema(content):
    if not isinstance(content, dict):
        return None, None

    media_types = sorted(
        content,
        key=lambda value: (
            0
            if (
                value == "application/json"
                or value.endswith("+json")
            )
            else 1,
            value,
        ),
    )

    for media_type in media_types:
        media_object = content.get(
            media_type
        )

        if not isinstance(
            media_object,
            dict,
        ):
            continue

        schema = media_object.get(
            "schema"
        )

        if isinstance(schema, dict):
            return media_type, schema

    return None, None


def assess_operation_slice(
    document,
    path,
    method,
):
    method = str(method).lower()

    result = {
        "path": path,
        "method": method,
        "feasible": False,
        "request_constructible": False,
        "response_constructible": False,
        "scenario_count": 0,
        "requirement_count": 0,
        "response_fault_count": 0,
        "scenarios": [],
        "requirements": [],
        "response_faults": [],
        "diagnostics": [],
        "model_outputs_used": False,
    }

    try:
        paths = document.get(
            "paths"
        )

        if not isinstance(paths, dict):
            raise FeasibilityError(
                "Document has no paths mapping."
            )

        path_item = paths.get(path)

        if not isinstance(path_item, dict):
            raise FeasibilityError(
                "Target path is missing."
            )

        operation = path_item.get(
            method
        )

        if not isinstance(operation, dict):
            raise FeasibilityError(
                "Target operation is missing."
            )

        parameters = operation.get(
            "parameters",
            [],
        )

        if not isinstance(parameters, list):
            raise FeasibilityError(
                "Operation parameters are not "
                "a list."
            )

        request_values = {
            "path": {},
            "query": {},
            "header": {},
            "cookie": {},
        }

        required_parameter_names = []

        for parameter_value in parameters:
            parameter = dereference(
                document,
                parameter_value,
            )

            if not isinstance(
                parameter,
                dict,
            ):
                raise FeasibilityError(
                    "Parameter is not an object."
                )

            location = parameter.get(
                "in"
            )

            name = parameter.get(
                "name"
            )

            if (
                location
                not in request_values
                or not isinstance(name, str)
            ):
                raise FeasibilityError(
                    "Parameter location or name "
                    "is unsupported."
                )

            schema = parameter.get(
                "schema"
            )

            if not isinstance(schema, dict):
                content = parameter.get(
                    "content"
                )

                _, schema = choose_media_schema(
                    content
                )

            if not isinstance(schema, dict):
                raise FeasibilityError(
                    "Parameter has no supported schema."
                )

            value = synthesize_schema(
                document,
                schema,
                context="request",
            )

            request_values[
                location
            ][name] = value

            if (
                parameter.get(
                    "required"
                )
                is True
                or location == "path"
            ):
                required_parameter_names.append(
                    f"{location}:{name}"
                )

        request_body = operation.get(
            "requestBody"
        )

        request_body_required = False
        request_body_media_type = None
        request_body_value = None
        required_body_properties = []

        if request_body is not None:
            request_body = dereference(
                document,
                request_body,
            )

            if not isinstance(
                request_body,
                dict,
            ):
                raise FeasibilityError(
                    "Request body is not an object."
                )

            request_body_required = (
                request_body.get(
                    "required"
                )
                is True
            )

            (
                request_body_media_type,
                request_body_schema,
            ) = choose_media_schema(
                request_body.get(
                    "content"
                )
            )

            if request_body_schema is None:
                if request_body_required:
                    raise FeasibilityError(
                        "Required request body has "
                        "no supported media schema."
                    )
            else:
                request_body_value = (
                    synthesize_schema(
                        document,
                        request_body_schema,
                        context="request",
                    )
                )

                resolved_request_schema = (
                    dereference(
                        document,
                        request_body_schema,
                    )
                )

                if isinstance(
                    resolved_request_schema,
                    dict,
                ):
                    required_body_properties = list(
                        resolved_request_schema.get(
                            "required",
                            [],
                        )
                        or []
                    )

        result[
            "request_constructible"
        ] = True

        result["scenarios"].append({
            "scenario": (
                "baseline_valid_request"
            ),
            "applicable": True,
        })

        for parameter_name in (
            required_parameter_names
        ):
            result["scenarios"].append({
                "scenario": (
                    "missing_required_parameter"
                ),
                "target": parameter_name,
                "applicable": True,
            })

            result["requirements"].append({
                "requirement": (
                    "required_parameter_present"
                ),
                "target": parameter_name,
            })

        if request_body_required:
            result["scenarios"].append({
                "scenario": (
                    "missing_required_request_body"
                ),
                "applicable": True,
            })

            result["requirements"].append({
                "requirement": (
                    "required_request_body_present"
                ),
            })

        for property_name in (
            required_body_properties
        ):
            result["scenarios"].append({
                "scenario": (
                    "missing_required_body_property"
                ),
                "target": property_name,
                "applicable": True,
            })

            result["requirements"].append({
                "requirement": (
                    "required_body_property_present"
                ),
                "target": property_name,
            })

        (
            response_code,
            response,
        ) = choose_response(
            document,
            operation,
        )

        (
            response_media_type,
            response_schema,
        ) = choose_media_schema(
            response.get("content")
        )

        response_value = None
        response_schema_type = None
        response_required_properties = []

        if response_schema is not None:
            response_value = synthesize_schema(
                document,
                response_schema,
                context="response",
            )

            resolved_response_schema = (
                dereference(
                    document,
                    response_schema,
                )
            )

            if isinstance(
                resolved_response_schema,
                dict,
            ):
                response_schema_type = (
                    select_schema_type(
                        resolved_response_schema
                    )
                )

                response_required_properties = (
                    list(
                        resolved_response_schema.get(
                            "required",
                            [],
                        )
                        or []
                    )
                )

        result[
            "response_constructible"
        ] = True

        result["requirements"].append({
            "requirement": (
                "expected_response_status"
            ),
            "target": response_code,
        })

        result["response_faults"].append({
            "fault": "unexpected_status_code",
            "applicable": True,
        })

        if (
            response_media_type
            and (
                response_media_type
                == "application/json"
                or response_media_type.endswith(
                    "+json"
                )
            )
        ):
            result["response_faults"].append({
                "fault": "malformed_json",
                "applicable": True,
            })

        if response_schema_type is not None:
            result["requirements"].append({
                "requirement": (
                    "response_schema_type"
                ),
                "target": response_schema_type,
            })

            result["response_faults"].append({
                "fault": "wrong_response_type",
                "applicable": True,
            })

        for property_name in (
            response_required_properties
        ):
            result["requirements"].append({
                "requirement": (
                    "required_response_property"
                ),
                "target": property_name,
            })

            result["response_faults"].append({
                "fault": (
                    "missing_required_response_property"
                ),
                "target": property_name,
                "applicable": True,
            })

        result["scenario_count"] = len(
            result["scenarios"]
        )

        result["requirement_count"] = len(
            result["requirements"]
        )

        result["response_fault_count"] = len(
            result["response_faults"]
        )

        result["request_example"] = {
            "parameters": request_values,
            "body_required": (
                request_body_required
            ),
            "body_media_type": (
                request_body_media_type
            ),
            "body": request_body_value,
        }

        result["response_example"] = {
            "status": response_code,
            "media_type": response_media_type,
            "body": response_value,
        }

        result["feasible"] = all([
            result[
                "request_constructible"
            ],
            result[
                "response_constructible"
            ],
            result["scenario_count"] > 0,
            result["requirement_count"] > 0,
            result[
                "response_fault_count"
            ] > 0,
        ])

    except FeasibilityError as error:
        result["diagnostics"].append({
            "type": type(error).__name__,
            "message": str(error),
        })

    except Exception as error:
        result["diagnostics"].append({
            "type": (
                "UNEXPECTED_FEASIBILITY_ERROR"
            ),
            "message": repr(error),
        })

    return result
'''


write_text_atomic_cell5(
    FEASIBILITY_SOURCE_PATH,
    feasibility_source_cell8,
)

feasibility_source_sha256_cell8 = (
    sha256_file_cell5(
        FEASIBILITY_SOURCE_PATH
    )
)


module_spec_cell8 = (
    importlib.util.spec_from_file_location(
        "stage03_feasibility",
        FEASIBILITY_SOURCE_PATH,
    )
)

if (
    module_spec_cell8 is None
    or module_spec_cell8.loader is None
):
    raise RuntimeError(
        "Could not load the feasibility module."
    )

feasibility_module_cell8 = (
    importlib.util.module_from_spec(
        module_spec_cell8
    )
)

module_spec_cell8.loader.exec_module(
    feasibility_module_cell8
)


fixture_get_cell8 = {
    "openapi": "3.0.3",
    "info": {
        "title": "GET fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/users/{user_id}": {
            "get": {
                "parameters": [
                    {
                        "name": "user_id",
                        "in": "path",
                        "required": True,
                        "schema": {
                            "type": "integer",
                            "minimum": 1,
                        },
                    },
                    {
                        "name": "verbose",
                        "in": "query",
                        "required": False,
                        "schema": {
                            "type": "boolean",
                        },
                    },
                ],
                "responses": {
                    "200": {
                        "description": "User",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/User"
                                    )
                                }
                            }
                        },
                    }
                },
            }
        }
    },
    "components": {
        "schemas": {
            "User": {
                "type": "object",
                "required": [
                    "id",
                    "name",
                ],
                "properties": {
                    "id": {
                        "type": "integer",
                    },
                    "name": {
                        "type": "string",
                    },
                },
            }
        }
    },
}

fixture_post_cell8 = {
    "openapi": "3.1.0",
    "info": {
        "title": "POST fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/users": {
            "post": {
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "required": [
                                    "name",
                                ],
                                "properties": {
                                    "name": {
                                        "type": "string",
                                        "minLength": 3,
                                    },
                                    "nickname": {
                                        "type": [
                                            "string",
                                            "null",
                                        ]
                                    },
                                },
                            }
                        }
                    },
                },
                "responses": {
                    "201": {
                        "description": "Created",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "type": "object",
                                    "required": [
                                        "id",
                                    ],
                                    "properties": {
                                        "id": {
                                            "type": "integer",
                                        }
                                    },
                                }
                            }
                        },
                    }
                },
            }
        }
    },
}

fixture_unresolved_cell8 = {
    "openapi": "3.0.3",
    "info": {
        "title": "Unresolved fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/broken": {
            "post": {
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "$ref": (
                                    "#/components/"
                                    "schemas/Missing"
                                )
                            }
                        }
                    },
                },
                "responses": {
                    "200": {
                        "description": "Response",
                    }
                },
            }
        }
    },
}

fixture_pattern_cell8 = {
    "openapi": "3.0.3",
    "info": {
        "title": "Pattern fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/codes/{code}": {
            "get": {
                "parameters": [
                    {
                        "name": "code",
                        "in": "path",
                        "required": True,
                        "schema": {
                            "type": "string",
                            "pattern": "^[A-Z]{3}$",
                        },
                    }
                ],
                "responses": {
                    "200": {
                        "description": "Response",
                    }
                },
            }
        }
    },
}


validate(fixture_get_cell8)
validate(fixture_post_cell8)


fixture_results_cell8 = {
    "supported_get": (
        feasibility_module_cell8
        .assess_operation_slice(
            fixture_get_cell8,
            "/users/{user_id}",
            "get",
        )
    ),
    "supported_post": (
        feasibility_module_cell8
        .assess_operation_slice(
            fixture_post_cell8,
            "/users",
            "post",
        )
    ),
    "unresolved_reference": (
        feasibility_module_cell8
        .assess_operation_slice(
            fixture_unresolved_cell8,
            "/broken",
            "post",
        )
    ),
    "unsupported_pattern": (
        feasibility_module_cell8
        .assess_operation_slice(
            fixture_pattern_cell8,
            "/codes/{code}",
            "get",
        )
    ),
}

fixture_repeat_cell8 = (
    feasibility_module_cell8
    .assess_operation_slice(
        fixture_get_cell8,
        "/users/{user_id}",
        "get",
    )
)


checks_cell8 = {
    "supported_get_is_feasible": (
        fixture_results_cell8[
            "supported_get"
        ]["feasible"]
        is True
    ),
    "supported_get_request_constructible": (
        fixture_results_cell8[
            "supported_get"
        ]["request_constructible"]
        is True
    ),
    "supported_get_response_constructible": (
        fixture_results_cell8[
            "supported_get"
        ]["response_constructible"]
        is True
    ),
    "supported_get_has_scenario": (
        fixture_results_cell8[
            "supported_get"
        ]["scenario_count"]
        > 0
    ),
    "supported_get_has_requirement": (
        fixture_results_cell8[
            "supported_get"
        ]["requirement_count"]
        > 0
    ),
    "supported_get_has_response_fault": (
        fixture_results_cell8[
            "supported_get"
        ]["response_fault_count"]
        > 0
    ),
    "supported_post_is_feasible": (
        fixture_results_cell8[
            "supported_post"
        ]["feasible"]
        is True
    ),
    "supported_post_has_required_body_scenario": (
        any(
            scenario.get("scenario")
            == "missing_required_request_body"
            for scenario in fixture_results_cell8[
                "supported_post"
            ]["scenarios"]
        )
    ),
    "unresolved_reference_is_infeasible": (
        fixture_results_cell8[
            "unresolved_reference"
        ]["feasible"]
        is False
    ),
    "unresolved_reference_is_diagnosed": (
        any(
            diagnostic.get("type")
            == "UnresolvedReference"
            for diagnostic
            in fixture_results_cell8[
                "unresolved_reference"
            ]["diagnostics"]
        )
    ),
    "unsupported_pattern_is_infeasible": (
        fixture_results_cell8[
            "unsupported_pattern"
        ]["feasible"]
        is False
    ),
    "unsupported_pattern_is_diagnosed": (
        any(
            diagnostic.get("type")
            == "UnsupportedSchema"
            for diagnostic
            in fixture_results_cell8[
                "unsupported_pattern"
            ]["diagnostics"]
        )
    ),
    "assessment_is_deterministic": (
        fixture_results_cell8[
            "supported_get"
        ]
        == fixture_repeat_cell8
    ),
    "no_model_outputs_used": all(
        result["model_outputs_used"]
        is False
        for result in fixture_results_cell8.values()
    ),
}


qualification_status_cell8 = (
    "PASS"
    if all(checks_cell8.values())
    else "BLOCKED"
)


fixture_paths_cell8 = {}

for (
    fixture_name_cell8,
    fixture_document_cell8,
) in {
    "supported_get": fixture_get_cell8,
    "supported_post": fixture_post_cell8,
    "unresolved_reference": (
        fixture_unresolved_cell8
    ),
    "unsupported_pattern": (
        fixture_pattern_cell8
    ),
}.items():
    fixture_path_cell8 = (
        FEASIBILITY_QUALIFICATION_ROOT
        / f"{fixture_name_cell8}.json"
    )

    write_json_atomic_cell5(
        fixture_path_cell8,
        fixture_document_cell8,
    )

    fixture_paths_cell8[
        fixture_name_cell8
    ] = {
        "path": str(
            fixture_path_cell8
        ),
        "sha256": (
            sha256_file_cell5(
                fixture_path_cell8
            )
        ),
    }


qualification_report_cell8 = {
    "policy_version": "1.0",
    "purpose": (
        "Deterministic draft feasibility "
        "assessment from specification content "
        "only."
    ),
    "implementation_path": str(
        FEASIBILITY_SOURCE_PATH
    ),
    "implementation_sha256": (
        feasibility_source_sha256_cell8
    ),
    "fixture_count": len(
        fixture_results_cell8
    ),
    "check_count": len(checks_cell8),
    "passed_check_count": sum(
        bool(value)
        for value in checks_cell8.values()
    ),
    "failed_checks": [
        name
        for name, passed
        in checks_cell8.items()
        if not passed
    ],
    "checks": checks_cell8,
    "fixture_results": (
        fixture_results_cell8
    ),
    "fixture_files": fixture_paths_cell8,
    "limitations": [
        (
            "This is an automated draft "
            "feasibility screen, not a reviewed "
            "operation oracle."
        ),
        (
            "Unsupported or ambiguous schema "
            "constructs are retained as explicit "
            "infeasibility diagnostics."
        ),
        (
            "No model output is inspected or "
            "executed."
        ),
    ],
    "status": qualification_status_cell8,
}

write_json_atomic_cell5(
    FEASIBILITY_QUALIFICATION_REPORT_PATH,
    qualification_report_cell8,
)

qualification_report_sha256_cell8 = (
    sha256_file_cell5(
        FEASIBILITY_QUALIFICATION_REPORT_PATH
    )
)


with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage_03_policy_cell8 = (
        yaml.safe_load(file)
    )

stage_03_policy_cell8[
    "evaluator_feasibility"
][
    "implementation_path"
] = "src/feasibility.py"

stage_03_policy_cell8[
    "evaluator_feasibility"
][
    "implementation_sha256"
] = feasibility_source_sha256_cell8

stage_03_policy_cell8[
    "evaluator_feasibility"
][
    "qualification_report_sha256"
] = qualification_report_sha256_cell8

stage_03_policy_cell8[
    "evaluator_feasibility"
][
    "qualification_status"
] = qualification_status_cell8

write_yaml_atomic_cell5(
    STAGE_03_POLICY_PATH,
    stage_03_policy_cell8,
)


cell_8_report = {
    "implementation": {
        "path": str(
            FEASIBILITY_SOURCE_PATH
        ),
        "sha256": (
            feasibility_source_sha256_cell8
        ),
    },
    "qualification": {
        "path": str(
            FEASIBILITY_QUALIFICATION_REPORT_PATH
        ),
        "sha256": (
            qualification_report_sha256_cell8
        ),
        "fixture_count": len(
            fixture_results_cell8
        ),
        "check_count": len(checks_cell8),
        "passed_check_count": sum(
            bool(value)
            for value in checks_cell8.values()
        ),
        "failed_checks": [
            name
            for name, passed
            in checks_cell8.items()
            if not passed
        ],
    },
    "updated_stage_03_policy_sha256": (
        sha256_file_cell5(
            STAGE_03_POLICY_PATH
        )
    ),
    "model_outputs_used": False,
    "status": qualification_status_cell8,
}

print(
    json.dumps(
        cell_8_report,
        indent=2,
        ensure_ascii=False,
    )
)

if qualification_status_cell8 != "PASS":
    raise RuntimeError(
        "Evaluator-feasibility builder "
        "qualification failed."
    )

print(
    "\nNOTEBOOK 03 — CELL 8: PASS"
)

{
  "implementation": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/src/feasibility.py",
    "sha256": "e468f755d6d0e494b247e9e1bf0ebf3a8f0c3a1a2b8a67ef881d997be98f7789"
  },
  "qualification": {
    "path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/reports/audit/feasibility_builder_qualification.json",
    "sha256": "bb98c14a6f860e57d442f4a6055b350ad4b44cec08105f2c0f433860a645ce83",
    "fixture_count": 4,
    "check_count": 14,
    "passed_check_count": 14,
    "failed_checks": []
  },
  "updated_stage_03_policy_sha256": "657d237b71a6e01d841b3a31ff93ab5838a172fa7f5cd9689dd2a74de2ead64f",
  "model_outputs_used": false,
  "status": "PASS"
}

NOTEBOOK 03 — CELL 8: PASS


In [9]:
# Notebook 03 — Cell 9 (corrected)
# Construct L1 only from Notebook 02's authoritative prose-walker pointers.

from __future__ import annotations

import copy
import csv
import gzip
import hashlib
import importlib.util
import json
import os
import shutil
import sqlite3
import time
from collections import Counter
from pathlib import Path

import yaml


CELL9_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

SOURCE_REGISTRY = (
    STAGE_02_PROJECT_ROOT
    / "data/inventory/final_eligible_operations.jsonl"
)
PROSE_FIELDS_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data/audits/prose_fields.csv"
)
PAIR_FRAME_PATH = (
    PROJECT_ROOT
    / "data/inventory/pair_frame_seed.jsonl"
)
OBFUSCATION_MODULE_PATH = (
    PROJECT_ROOT
    / "src/obfuscation.py"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)

L0_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/slices/L0"
)
L1_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/slices/L1"
)
CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "data/checkpoints/l1_construction"
)
POINTER_DATABASE_PATH = (
    PROJECT_ROOT
    / "data/checkpoints/"
    "cell09_authoritative_pointers.sqlite"
)
AUDIT_ROOT = (
    PROJECT_ROOT
    / "data/audits"
)
REPORT_ROOT = (
    PROJECT_ROOT
    / "reports/audit"
)

CONSTRUCTION_RECORDS_PATH = (
    AUDIT_ROOT
    / "l1_construction_records.jsonl"
)
REPORT_PATH = (
    REPORT_ROOT
    / "l1_construction_summary.json"
)

EXPECTED_OPERATION_COUNT = expected_eligible_operation_count

EXPECTED_SOURCE_REGISTRY_SHA256 = next(a["sha256"] for a in read_json(STAGE_02_INTERNAL_MANIFEST_PATH)["artifacts"] if a["path"] == "data/inventory/final_eligible_operations.jsonl")
EXPECTED_PROSE_FIELDS_SHA256 = next(a["sha256"] for a in read_json(STAGE_02_INTERNAL_MANIFEST_PATH)["artifacts"] if a["path"] == "data/audits/prose_fields.csv")
EXPECTED_PAIR_FRAME_SHA256 = json.loads((PROJECT_ROOT / "reports/audit/pairing_qualification.json").read_text())["pair_seed_frame"]["sha256"]
EXPECTED_OBFUSCATION_SHA256 = (
    "79eb0c682511ff393845604fb94e9ebe6"
    "c261c03208a22e50f79549719b05cbf"
)

OBFUSCATION_SEED = 42
CHECKPOINT_CHUNK_SIZE = 250
CONSTRUCTION_VERSION = (
    "authoritative-prose-walker-v1"
)

for directory in [
    L0_OUTPUT_ROOT,
    L1_OUTPUT_ROOT,
    CHECKPOINT_ROOT,
    POINTER_DATABASE_PATH.parent,
    AUDIT_ROOT,
    REPORT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            1024 * 1024
        ):
            digest.update(block)

    return digest.hexdigest()


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(
        value
    ).hexdigest()


def sha256_text(value: str) -> str:
    return sha256_bytes(
        value.encode("utf-8")
    )


def canonical_json_bytes(value) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")


def atomic_write_bytes(
    path: Path,
    content: bytes,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open("wb") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def atomic_write_json(
    path: Path,
    value,
) -> None:
    content = (
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")

    atomic_write_bytes(
        path,
        content,
    )


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}"
                )

            records.append(value)

    return records


def select_column(
    fieldnames: list[str],
    candidates: list[str],
    required: bool = False,
) -> str | None:
    normalized = {
        field.strip().lower(): field
        for field in fieldnames
    }

    for candidate in candidates:
        match = normalized.get(
            candidate.lower()
        )

        if match is not None:
            return match

    if required:
        raise RuntimeError(
            "Required column was not found. "
            f"Expected one of {candidates}; "
            f"observed {fieldnames}."
        )

    return None


def status_is_usable(value) -> bool:
    if value is None:
        return True

    normalized = str(
        value
    ).strip().upper()

    if not normalized:
        return True

    rejected_markers = [
        "NONSTRING",
        "INVALID",
        "EXCLUDED",
        "BLOCKED",
        "ERROR",
        "FAILED",
        "FAILURE",
    ]

    return not any(
        marker in normalized
        for marker in rejected_markers
    )


def decode_pointer_part(
    value: str,
) -> str:
    return (
        value
        .replace("~1", "/")
        .replace("~0", "~")
    )


def pointer_parts(
    pointer: str,
) -> tuple[str, ...]:
    if pointer == "":
        return ()

    if not pointer.startswith("/"):
        raise ValueError(
            f"Invalid JSON pointer: {pointer}"
        )

    return tuple(
        decode_pointer_part(part)
        for part in pointer[1:].split("/")
    )


def pointer_field_name(
    pointer: str,
) -> str | None:
    parts = pointer_parts(
        pointer
    )

    if not parts:
        return None

    return parts[-1]


def pointer_value(
    document,
    pointer: str,
):
    current = document

    for part in pointer_parts(pointer):
        if isinstance(current, list):
            current = current[int(part)]
        elif isinstance(current, dict):
            current = current[part]
        else:
            raise KeyError(pointer)

    return current


def assign_pointer_value(
    document,
    pointer: str,
    new_value,
) -> None:
    parts = pointer_parts(
        pointer
    )

    if not parts:
        raise ValueError(
            "The document root cannot be "
            "an annotation target."
        )

    current = document

    for part in parts[:-1]:
        if isinstance(current, list):
            current = current[int(part)]
        else:
            current = current[part]

    final_part = parts[-1]

    if isinstance(current, list):
        current[int(final_part)] = (
            new_value
        )
    else:
        current[final_part] = (
            new_value
        )


def count_ascii_letters(
    value: str,
) -> int:
    return sum(
        character.isascii()
        and character.isalpha()
        for character in value
    )


def count_non_ascii_characters(
    value: str,
) -> int:
    return sum(
        not character.isascii()
        for character in value
    )


def validate_transformation(
    original: str,
    transformed: str,
) -> dict:
    same_length = (
        len(original)
        == len(transformed)
    )
    protected_positions_unchanged = True
    every_ascii_letter_changed = True
    capitalization_pattern_preserved = True

    if same_length:
        for (
            original_character,
            transformed_character,
        ) in zip(
            original,
            transformed,
        ):
            is_ascii_letter = (
                original_character.isascii()
                and original_character.isalpha()
            )

            if is_ascii_letter:
                if (
                    original_character
                    == transformed_character
                ):
                    every_ascii_letter_changed = (
                        False
                    )

                if (
                    original_character.isupper()
                    != transformed_character.isupper()
                ):
                    capitalization_pattern_preserved = (
                        False
                    )

                if (
                    original_character.islower()
                    != transformed_character.islower()
                ):
                    capitalization_pattern_preserved = (
                        False
                    )

            elif (
                original_character
                != transformed_character
            ):
                protected_positions_unchanged = (
                    False
                )
    else:
        protected_positions_unchanged = False
        every_ascii_letter_changed = False
        capitalization_pattern_preserved = False

    ascii_letter_count = (
        count_ascii_letters(original)
    )

    return {
        "same_length": same_length,
        "protected_positions_unchanged": (
            protected_positions_unchanged
        ),
        "every_ascii_letter_changed": (
            every_ascii_letter_changed
            if ascii_letter_count > 0
            else True
        ),
        "capitalization_pattern_preserved": (
            capitalization_pattern_preserved
        ),
        "ascii_letter_count": (
            ascii_letter_count
        ),
        "non_ascii_character_count": (
            count_non_ascii_characters(
                original
            )
        ),
    }


def output_path_for(
    root: Path,
    op_id: str,
) -> Path:
    shard = op_id.removeprefix(
        "op_"
    )[:2]

    return (
        root
        / shard
        / f"{op_id}.json"
    )


def resolve_record_path(
    raw_path,
) -> Path | None:
    if not raw_path:
        return None

    path_text = str(raw_path)
    direct_path = Path(path_text)

    if direct_path.exists():
        return direct_path

    marker = (
        "prose_operation_experiment/"
    )

    if marker in path_text:
        relative_text = path_text.split(
            marker,
            maxsplit=1,
        )[1]

        candidate = (
            STAGE_02_PROJECT_ROOT
            / relative_text
        )

        if candidate.exists():
            return candidate

    candidate = (
        STAGE_02_PROJECT_ROOT
        / path_text.lstrip("/")
    )

    if candidate.exists():
        return candidate

    return None


def build_stage02_l0_index() -> dict[str, Path]:
    candidate_roots = [
        STAGE_02_PROJECT_ROOT
        / "data/slices/L0",
        STAGE_02_PROJECT_ROOT
        / "data/slices/l0",
        STAGE_02_PROJECT_ROOT
        / "data/operation_slices",
        STAGE_02_PROJECT_ROOT
        / "data/slices",
    ]

    roots = [
        root
        for root in candidate_roots
        if root.exists()
    ]

    if not roots:
        raise RuntimeError(
            "No Stage 02 L0 slice root "
            "was found."
        )

    index = {}

    for root in roots:
        for path in root.rglob(
            "*.json"
        ):
            if not path.name.startswith(
                "op_"
            ):
                continue

            op_id = path.name.split(
                ".",
                maxsplit=1,
            )[0]

            if op_id not in index:
                index[op_id] = path

    return index


def checkpoint_path_for(
    chunk_number: int,
) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"chunk_{chunk_number:05d}.jsonl.gz"
    )


def read_checkpoint(
    path: Path,
) -> list[dict]:
    records = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            if line.strip():
                records.append(
                    json.loads(line)
                )

    return records


def write_checkpoint(
    path: Path,
    records: list[dict],
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with gzip.GzipFile(
        filename=str(temporary_path),
        mode="wb",
        compresslevel=6,
        mtime=0,
    ) as handle:
        for record in records:
            serialized = (
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                    allow_nan=False,
                )
                + "\n"
            ).encode("utf-8")

            handle.write(serialized)

    os.replace(
        temporary_path,
        path,
    )


def checkpoint_is_reusable(
    records: list[dict],
    expected_op_ids: list[str],
    authority_sha256: str,
) -> bool:
    if [
        record.get("op_id")
        for record in records
    ] != expected_op_ids:
        return False

    for record in records:
        if (
            record.get(
                "construction_version"
            )
            != CONSTRUCTION_VERSION
        ):
            return False

        if (
            record.get(
                "pointer_authority_sha256"
            )
            != authority_sha256
        ):
            return False

        l0_path = (
            PROJECT_ROOT
            / record[
                "l0_relative_path"
            ]
        )
        l1_path = (
            PROJECT_ROOT
            / record[
                "l1_relative_path"
            ]
        )

        if (
            not l0_path.exists()
            or not l1_path.exists()
        ):
            return False

        if (
            sha256_file(l0_path)
            != record["l0_sha256"]
        ):
            return False

        if (
            sha256_file(l1_path)
            != record["l1_sha256"]
        ):
            return False

    return True


required_paths = [
    SOURCE_REGISTRY,
    PROSE_FIELDS_PATH,
    PAIR_FRAME_PATH,
    OBFUSCATION_MODULE_PATH,
    STAGE_03_POLICY_PATH,
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)

observed_source_registry_sha256 = (
    sha256_file(SOURCE_REGISTRY)
)
observed_prose_fields_sha256 = (
    sha256_file(PROSE_FIELDS_PATH)
)
observed_pair_frame_sha256 = (
    sha256_file(PAIR_FRAME_PATH)
)
observed_obfuscation_sha256 = (
    sha256_file(
        OBFUSCATION_MODULE_PATH
    )
)

frozen_hash_checks = {
    "source_registry": (
        observed_source_registry_sha256
        == EXPECTED_SOURCE_REGISTRY_SHA256
    ),
    "prose_fields": (
        observed_prose_fields_sha256
        == EXPECTED_PROSE_FIELDS_SHA256
    ),
    "pair_frame": (
        observed_pair_frame_sha256
        == EXPECTED_PAIR_FRAME_SHA256
    ),
    "obfuscation_implementation": (
        observed_obfuscation_sha256
        == EXPECTED_OBFUSCATION_SHA256
    ),
}

if not all(
    frozen_hash_checks.values()
):
    raise RuntimeError(
        "A frozen Cell 9 input hash changed:\n"
        + json.dumps(
            frozen_hash_checks,
            indent=2,
        )
    )

module_spec = (
    importlib.util.spec_from_file_location(
        "stage03_obfuscation",
        OBFUSCATION_MODULE_PATH,
    )
)

if (
    module_spec is None
    or module_spec.loader is None
):
    raise RuntimeError(
        "Could not load the frozen "
        "obfuscation implementation."
    )

obfuscation_module = (
    importlib.util.module_from_spec(
        module_spec
    )
)
module_spec.loader.exec_module(
    obfuscation_module
)

obfuscate_annotation = getattr(
    obfuscation_module,
    "obfuscate_annotation",
    None,
)

if not callable(
    obfuscate_annotation
):
    raise RuntimeError(
        "The frozen implementation does not "
        "expose obfuscate_annotation()."
    )

source_records = load_jsonl(
    SOURCE_REGISTRY
)
pair_records = load_jsonl(
    PAIR_FRAME_PATH
)

if (
    len(source_records)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected source-operation count: "
        f"{len(source_records)}"
    )

if (
    len(pair_records)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected pair-frame count: "
        f"{len(pair_records)}"
    )

source_by_op_id = {}

for record in source_records:
    op_id = record.get("op_id")

    if not isinstance(op_id, str):
        raise RuntimeError(
            "Source record lacks op_id."
        )

    if op_id in source_by_op_id:
        raise RuntimeError(
            f"Duplicate source op_id: {op_id}"
        )

    source_by_op_id[op_id] = record

pair_by_op_id = {}

for record in pair_records:
    op_id = record.get("op_id")

    if not isinstance(op_id, str):
        raise RuntimeError(
            "Pair-frame record lacks op_id."
        )

    if op_id in pair_by_op_id:
        raise RuntimeError(
            f"Duplicate pair-frame op_id: {op_id}"
        )

    pair_by_op_id[op_id] = record

if (
    set(source_by_op_id)
    != set(pair_by_op_id)
):
    raise RuntimeError(
        "Source registry and pair frame have "
        "different operation-ID sets."
    )

ordered_op_ids = sorted(
    source_by_op_id
)
eligible_op_ids = set(
    ordered_op_ids
)

# Build the authoritative treatment-pointer database.

connection = sqlite3.connect(
    POINTER_DATABASE_PATH
)
cursor = connection.cursor()

cursor.execute(
    "PRAGMA journal_mode=WAL"
)
cursor.execute(
    "PRAGMA synchronous=NORMAL"
)
cursor.execute(
    "PRAGMA temp_store=FILE"
)
cursor.execute(
    "PRAGMA cache_size=-200000"
)

cursor.executescript(
    """
    DROP TABLE IF EXISTS approved;

    CREATE TABLE approved (
        op_id TEXT NOT NULL,
        pointer TEXT NOT NULL,
        original_sha256 TEXT NOT NULL,
        occurrence_count INTEGER NOT NULL,
        hash_conflict INTEGER NOT NULL,
        PRIMARY KEY (op_id, pointer)
    );

    CREATE INDEX approved_op_id_index
        ON approved(op_id);
    """
)

with PROSE_FIELDS_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as handle:
    reader = csv.DictReader(handle)

    if reader.fieldnames is None:
        raise RuntimeError(
            "Notebook 02 prose ledger "
            "has no header."
        )

    fieldnames = list(
        reader.fieldnames
    )

    op_id_column = select_column(
        fieldnames,
        [
            "op_id",
            "operation_id",
        ],
        required=True,
    )
    pointer_column = select_column(
        fieldnames,
        [
            "slice_pointer",
            "json_pointer",
            "pointer",
        ],
        required=True,
    )
    field_name_column = select_column(
        fieldnames,
        [
            "field_name",
            "annotation_field",
            "prose_field",
        ],
    )
    status_column = select_column(
        fieldnames,
        [
            "status",
            "annotation_status",
            "measurement_status",
            "prose_status",
        ],
    )
    type_column = select_column(
        fieldnames,
        [
            "observed_type",
            "value_type",
            "text_type",
        ],
    )
    hash_column = select_column(
        fieldnames,
        [
            "text_sha256",
            "value_sha256",
            "prose_sha256",
            "annotation_sha256",
        ],
    )
    text_column = select_column(
        fieldnames,
        [
            "text",
            "prose_text",
            "annotation_text",
            "value",
        ],
    )

    if (
        hash_column is None
        and text_column is None
    ):
        raise RuntimeError(
            "The authoritative prose ledger "
            "provides neither text nor text hash."
        )

    insert_batch = []
    prose_rows_scanned = 0
    authoritative_rows_selected = 0
    unusable_rows_skipped = 0

    for row in reader:
        prose_rows_scanned += 1

        op_id = str(
            row.get(
                op_id_column,
                "",
            )
        ).strip()

        if op_id not in eligible_op_ids:
            continue

        if (
            status_column is not None
            and not status_is_usable(
                row.get(status_column)
            )
        ):
            unusable_rows_skipped += 1
            continue

        if type_column is not None:
            observed_type = str(
                row.get(
                    type_column,
                    "",
                )
            ).strip().lower()

            if (
                observed_type
                and observed_type
                not in {
                    "str",
                    "string",
                    "text",
                }
            ):
                unusable_rows_skipped += 1
                continue

        pointer = str(
            row.get(
                pointer_column,
                "",
            )
        ).strip()

        if not pointer:
            continue

        if field_name_column is not None:
            field_name = str(
                row.get(
                    field_name_column,
                    "",
                )
            ).strip()
        else:
            field_name = (
                pointer_field_name(pointer)
                or ""
            )

        if field_name not in {
            "summary",
            "description",
        }:
            continue

        original_sha256 = ""

        if hash_column is not None:
            original_sha256 = str(
                row.get(
                    hash_column,
                    "",
                )
            ).strip()

        if (
            not original_sha256
            and text_column is not None
        ):
            original_sha256 = sha256_text(
                str(
                    row.get(
                        text_column,
                        "",
                    )
                )
            )

        if not original_sha256:
            raise RuntimeError(
                "Authoritative pointer lacks "
                f"a text hash: {op_id} "
                f"{pointer}"
            )

        insert_batch.append(
            (
                op_id,
                pointer,
                original_sha256,
                1,
                0,
            )
        )
        authoritative_rows_selected += 1

        if len(insert_batch) >= 10000:
            cursor.executemany(
                """
                INSERT INTO approved (
                    op_id,
                    pointer,
                    original_sha256,
                    occurrence_count,
                    hash_conflict
                )
                VALUES (?, ?, ?, ?, ?)
                ON CONFLICT (op_id, pointer)
                DO UPDATE SET
                    occurrence_count =
                        approved.occurrence_count + 1,
                    hash_conflict =
                        CASE
                            WHEN
                                approved.original_sha256
                                != excluded.original_sha256
                            THEN 1
                            ELSE approved.hash_conflict
                        END
                """,
                insert_batch,
            )
            insert_batch.clear()

        if (
            prose_rows_scanned
            % 500000
            == 0
        ):
            connection.commit()

            print(
                json.dumps(
                    {
                        "phase": (
                            "authoritative_pointer_scan"
                        ),
                        "prose_rows_scanned": (
                            prose_rows_scanned
                        ),
                        "authoritative_rows_selected": (
                            authoritative_rows_selected
                        ),
                        "elapsed_seconds": round(
                            time.time()
                            - CELL9_STARTED,
                            1,
                        ),
                    }
                )
            )

    if insert_batch:
        cursor.executemany(
            """
            INSERT INTO approved (
                op_id,
                pointer,
                original_sha256,
                occurrence_count,
                hash_conflict
            )
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT (op_id, pointer)
            DO UPDATE SET
                occurrence_count =
                    approved.occurrence_count + 1,
                hash_conflict =
                    CASE
                        WHEN
                            approved.original_sha256
                            != excluded.original_sha256
                        THEN 1
                        ELSE approved.hash_conflict
                    END
            """,
            insert_batch,
        )

connection.commit()

approved_pointer_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM approved
    """
).fetchone()[0]

duplicate_pointer_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM approved
    WHERE occurrence_count != 1
    """
).fetchone()[0]

pointer_hash_conflict_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM approved
    WHERE hash_conflict != 0
    """
).fetchone()[0]

if (
    approved_pointer_count
    != authoritative_rows_selected
):
    raise RuntimeError(
        "Unexpected authoritative-pointer "
        f"count: {approved_pointer_count}; "
        "expected "
        f"{authoritative_rows_selected} source rows."
    )

if duplicate_pointer_count:
    raise RuntimeError(
        "Duplicate authoritative pointers "
        f"detected: {duplicate_pointer_count}"
    )

if pointer_hash_conflict_count:
    raise RuntimeError(
        "Conflicting authoritative text hashes "
        f"detected: {pointer_hash_conflict_count}"
    )

# Resolve exact Stage 02 L0 sources.

direct_slice_fields = [
    "l0_slice_path",
    "slice_path",
    "slice_file",
    "slice_relative_path",
    "l0_path",
    "operation_slice_path",
]

resolved_source_paths = {}
unresolved_op_ids = []

for op_id in ordered_op_ids:
    existing_l0_path = output_path_for(
        L0_OUTPUT_ROOT,
        op_id,
    )

    if existing_l0_path.exists():
        resolved_source_paths[
            op_id
        ] = existing_l0_path
        continue

    source_record = source_by_op_id[
        op_id
    ]
    resolved_path = None

    for field_name in direct_slice_fields:
        resolved_path = resolve_record_path(
            source_record.get(
                field_name
            )
        )

        if resolved_path is not None:
            break

    if resolved_path is None:
        unresolved_op_ids.append(op_id)
    else:
        resolved_source_paths[
            op_id
        ] = resolved_path

if unresolved_op_ids:
    print(
        json.dumps(
            {
                "phase": "l0_path_resolution",
                "direct_paths_resolved": (
                    len(
                        resolved_source_paths
                    )
                ),
                "fallback_index_required": (
                    len(unresolved_op_ids)
                ),
            }
        )
    )

    l0_index = (
        build_stage02_l0_index()
    )

    for op_id in unresolved_op_ids:
        path = l0_index.get(op_id)

        if path is None:
            raise RuntimeError(
                "Could not locate the "
                f"Stage 02 L0 slice for "
                f"{op_id}."
            )

        resolved_source_paths[
            op_id
        ] = path

if (
    len(resolved_source_paths)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Not all L0 source paths "
        "were resolved."
    )

all_summary_records = []

reused_operation_count = 0
newly_constructed_operation_count = 0

annotation_field_count = 0
changed_field_count = 0
no_ascii_field_count = 0
ascii_letter_count = 0
non_ascii_character_count = 0

l0_bytes_total = 0
l1_bytes_total = 0

status_counts = Counter()
problem_examples = []

total_chunks = (
    EXPECTED_OPERATION_COUNT
    + CHECKPOINT_CHUNK_SIZE
    - 1
) // CHECKPOINT_CHUNK_SIZE

for chunk_number in range(
    total_chunks
):
    start_index = (
        chunk_number
        * CHECKPOINT_CHUNK_SIZE
    )
    end_index = min(
        start_index
        + CHECKPOINT_CHUNK_SIZE,
        EXPECTED_OPERATION_COUNT,
    )

    chunk_op_ids = ordered_op_ids[
        start_index:end_index
    ]
    checkpoint_path = (
        checkpoint_path_for(
            chunk_number
        )
    )

    chunk_records = None

    if checkpoint_path.exists():
        try:
            candidate_records = (
                read_checkpoint(
                    checkpoint_path
                )
            )

            if checkpoint_is_reusable(
                candidate_records,
                chunk_op_ids,
                observed_prose_fields_sha256,
            ):
                chunk_records = (
                    candidate_records
                )
        except Exception:
            chunk_records = None

    if chunk_records is not None:
        reused_operation_count += len(
            chunk_records
        )

    else:
        chunk_records = []

        for op_id in chunk_op_ids:
            source_record = (
                source_by_op_id[op_id]
            )
            pair_record = (
                pair_by_op_id[op_id]
            )
            source_l0_path = (
                resolved_source_paths[
                    op_id
                ]
            )

            source_bytes = (
                source_l0_path.read_bytes()
            )
            source_sha256 = (
                sha256_bytes(source_bytes)
            )

            declared_l0_sha256 = None

            for hash_field in [
                "slice_sha256",
                "l0_sha256",
                "slice_hash",
            ]:
                candidate_hash = (
                    source_record.get(
                        hash_field
                    )
                )

                if isinstance(
                    candidate_hash,
                    str,
                ):
                    declared_l0_sha256 = (
                        candidate_hash
                    )
                    break

            if (
                declared_l0_sha256
                is not None
                and declared_l0_sha256
                != source_sha256
            ):
                raise RuntimeError(
                    "Stage 02 L0 hash "
                    f"mismatch for {op_id}."
                )

            try:
                l0_document = json.loads(
                    source_bytes
                )
            except Exception as error:
                raise RuntimeError(
                    "Invalid L0 JSON for "
                    f"{op_id}: {error!r}"
                ) from error

            if not isinstance(
                l0_document,
                dict,
            ):
                raise RuntimeError(
                    f"L0 slice for {op_id} "
                    "is not a JSON object."
                )

            approved_rows = cursor.execute(
                """
                SELECT
                    pointer,
                    original_sha256
                FROM approved
                WHERE op_id = ?
                ORDER BY pointer
                """,
                (op_id,),
            ).fetchall()

            l1_document = copy.deepcopy(
                l0_document
            )
            field_records = []
            operation_problems = []

            for (
                pointer,
                authoritative_sha256,
            ) in approved_rows:
                try:
                    original_text = (
                        pointer_value(
                            l0_document,
                            pointer,
                        )
                    )
                except Exception as error:
                    operation_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "AUTHORITATIVE_POINTER_"
                                "RESOLUTION_FAILED"
                            ),
                            "diagnostic": repr(
                                error
                            ),
                        }
                    )
                    continue

                if not isinstance(
                    original_text,
                    str,
                ):
                    operation_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "AUTHORITATIVE_POINTER_"
                                "IS_NOT_STRING"
                            ),
                            "observed_type": (
                                type(
                                    original_text
                                ).__name__
                            ),
                        }
                    )
                    continue

                observed_original_sha256 = (
                    sha256_text(
                        original_text
                    )
                )

                if (
                    observed_original_sha256
                    != authoritative_sha256
                ):
                    operation_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "AUTHORITATIVE_TEXT_"
                                "HASH_MISMATCH"
                            ),
                            "expected_sha256": (
                                authoritative_sha256
                            ),
                            "observed_sha256": (
                                observed_original_sha256
                            ),
                        }
                    )
                    continue

                transformed_text = (
                    obfuscate_annotation(
                        original_text,
                        seed=(
                            OBFUSCATION_SEED
                        ),
                    )
                )

                checks = (
                    validate_transformation(
                        original_text,
                        transformed_text,
                    )
                )

                preservation_pass = all(
                    [
                        checks[
                            "same_length"
                        ],
                        checks[
                            "protected_positions_unchanged"
                        ],
                        checks[
                            "every_ascii_letter_changed"
                        ],
                        checks[
                            "capitalization_pattern_preserved"
                        ],
                    ]
                )

                if not preservation_pass:
                    operation_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "TRANSFORMATION_"
                                "PRESERVATION_FAILED"
                            ),
                            "checks": checks,
                        }
                    )
                    continue

                assign_pointer_value(
                    l1_document,
                    pointer,
                    transformed_text,
                )

                changed = (
                    transformed_text
                    != original_text
                )
                treatment_class = (
                    "TRANSFORMED_ASCII"
                    if checks[
                        "ascii_letter_count"
                    ] > 0
                    else "NO_ASCII_LETTERS"
                )

                field_records.append(
                    {
                        "slice_pointer": (
                            pointer
                        ),
                        "field_name": (
                            pointer_field_name(
                                pointer
                            )
                        ),
                        "original_sha256": (
                            observed_original_sha256
                        ),
                        "transformed_sha256": (
                            sha256_text(
                                transformed_text
                            )
                        ),
                        "original_length": (
                            len(original_text)
                        ),
                        "transformed_length": (
                            len(
                                transformed_text
                            )
                        ),
                        "ascii_letter_count": (
                            checks[
                                "ascii_letter_count"
                            ]
                        ),
                        "non_ascii_character_count": (
                            checks[
                                "non_ascii_character_count"
                            ]
                        ),
                        "changed": changed,
                        "treatment_class": (
                            treatment_class
                        ),
                        "preservation_checks": {
                            "same_length": (
                                checks[
                                    "same_length"
                                ]
                            ),
                            "protected_positions_unchanged": (
                                checks[
                                    "protected_positions_unchanged"
                                ]
                            ),
                            "every_ascii_letter_changed": (
                                checks[
                                    "every_ascii_letter_changed"
                                ]
                            ),
                            "capitalization_pattern_preserved": (
                                checks[
                                    "capitalization_pattern_preserved"
                                ]
                            ),
                        },
                    }
                )

            if (
                len(field_records)
                != len(approved_rows)
                and not operation_problems
            ):
                operation_problems.append(
                    {
                        "problem": (
                            "AUTHORITATIVE_POINTER_"
                            "ACCOUNTING_MISMATCH"
                        ),
                        "expected": (
                            len(approved_rows)
                        ),
                        "observed": (
                            len(field_records)
                        ),
                    }
                )

            l0_output_path = (
                output_path_for(
                    L0_OUTPUT_ROOT,
                    op_id,
                )
            )
            l1_output_path = (
                output_path_for(
                    L1_OUTPUT_ROOT,
                    op_id,
                )
            )

            l0_output_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            l1_output_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            if (
                not l0_output_path.exists()
                or sha256_file(
                    l0_output_path
                )
                != source_sha256
            ):
                temporary_l0_path = (
                    l0_output_path.with_name(
                        l0_output_path.name
                        + ".tmp"
                    )
                )

                shutil.copyfile(
                    source_l0_path,
                    temporary_l0_path,
                )
                os.replace(
                    temporary_l0_path,
                    l0_output_path,
                )

            l1_bytes = (
                canonical_json_bytes(
                    l1_document
                )
            )
            l1_sha256 = (
                sha256_bytes(l1_bytes)
            )

            atomic_write_bytes(
                l1_output_path,
                l1_bytes,
            )

            operation_status = (
                "PASS"
                if not operation_problems
                else "BLOCKED"
            )

            chunk_records.append(
                {
                    "construction_version": (
                        CONSTRUCTION_VERSION
                    ),
                    "pointer_authority_sha256": (
                        observed_prose_fields_sha256
                    ),
                    "op_id": op_id,
                    "pair_id": (
                        pair_record.get(
                            "pair_id"
                        )
                    ),
                    "spec_id": (
                        source_record.get(
                            "spec_id"
                        )
                    ),
                    "family_id": (
                        pair_record.get(
                            "family_id"
                        )
                    ),
                    "provider_directory": (
                        source_record.get(
                            "provider_directory"
                        )
                    ),
                    "source_class": (
                        source_record.get(
                            "source_class"
                        )
                    ),
                    "method": (
                        source_record.get(
                            "method"
                        )
                    ),
                    "path": (
                        source_record.get(
                            "path"
                        )
                    ),
                    "l0_relative_path": str(
                        l0_output_path.relative_to(
                            PROJECT_ROOT
                        )
                    ).replace("\\", "/"),
                    "l1_relative_path": str(
                        l1_output_path.relative_to(
                            PROJECT_ROOT
                        )
                    ).replace("\\", "/"),
                    "l0_sha256": (
                        source_sha256
                    ),
                    "l1_sha256": (
                        l1_sha256
                    ),
                    "l0_bytes": (
                        len(source_bytes)
                    ),
                    "l1_bytes": (
                        len(l1_bytes)
                    ),
                    "annotation_field_count": (
                        len(field_records)
                    ),
                    "changed_field_count": sum(
                        int(
                            field_record[
                                "changed"
                            ]
                        )
                        for field_record
                        in field_records
                    ),
                    "no_ascii_field_count": sum(
                        int(
                            field_record[
                                "treatment_class"
                            ]
                            == "NO_ASCII_LETTERS"
                        )
                        for field_record
                        in field_records
                    ),
                    "ascii_letter_count": sum(
                        field_record[
                            "ascii_letter_count"
                        ]
                        for field_record
                        in field_records
                    ),
                    "non_ascii_character_count": sum(
                        field_record[
                            "non_ascii_character_count"
                        ]
                        for field_record
                        in field_records
                    ),
                    "field_records": (
                        field_records
                    ),
                    "problems": (
                        operation_problems
                    ),
                    "status": (
                        operation_status
                    ),
                }
            )

            newly_constructed_operation_count += (
                1
            )

        write_checkpoint(
            checkpoint_path,
            chunk_records,
        )

    for record in chunk_records:
        annotation_field_count += int(
            record[
                "annotation_field_count"
            ]
        )
        changed_field_count += int(
            record[
                "changed_field_count"
            ]
        )
        no_ascii_field_count += int(
            record[
                "no_ascii_field_count"
            ]
        )
        ascii_letter_count += int(
            record[
                "ascii_letter_count"
            ]
        )
        non_ascii_character_count += int(
            record[
                "non_ascii_character_count"
            ]
        )
        l0_bytes_total += int(
            record["l0_bytes"]
        )
        l1_bytes_total += int(
            record["l1_bytes"]
        )

        status_counts[
            record["status"]
        ] += 1

        if (
            record["problems"]
            and len(problem_examples) < 20
        ):
            problem_examples.append(
                {
                    "op_id": (
                        record["op_id"]
                    ),
                    "problems": (
                        record[
                            "problems"
                        ][:10]
                    ),
                }
            )

        all_summary_records.append(
            {
                key: value
                for key, value
                in record.items()
                if key
                not in {
                    "field_records",
                    "problems",
                }
            }
        )

    processed = end_index

    if (
        processed % 1000 == 0
        or processed
        == EXPECTED_OPERATION_COUNT
    ):
        print(
            json.dumps(
                {
                    "processed": (
                        processed
                    ),
                    "total": (
                        EXPECTED_OPERATION_COUNT
                    ),
                    "checkpoint_reused": (
                        reused_operation_count
                    ),
                    "newly_constructed": (
                        newly_constructed_operation_count
                    ),
                    "authoritative_fields": (
                        annotation_field_count
                    ),
                    "changed_fields": (
                        changed_field_count
                    ),
                    "problem_count": (
                        status_counts[
                            "BLOCKED"
                        ]
                    ),
                    "elapsed_seconds": round(
                        time.time()
                        - CELL9_STARTED,
                        1,
                    ),
                }
            )
        )

connection.close()

all_summary_records.sort(
    key=lambda record: record["op_id"]
)

construction_content = "".join(
    json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )
    + "\n"
    for record in all_summary_records
).encode("utf-8")

atomic_write_bytes(
    CONSTRUCTION_RECORDS_PATH,
    construction_content,
)

construction_records_sha256 = (
    sha256_file(
        CONSTRUCTION_RECORDS_PATH
    )
)

checks = {
    "operation_count_matches_stage02_candidate_count": (
        len(all_summary_records)
        == EXPECTED_OPERATION_COUNT
    ),
    "unique_operation_ids": (
        len(
            {
                record["op_id"]
                for record
                in all_summary_records
            }
        )
        == EXPECTED_OPERATION_COUNT
    ),
    "unique_pair_ids": (
        len(
            {
                record["pair_id"]
                for record
                in all_summary_records
            }
        )
        == EXPECTED_OPERATION_COUNT
    ),
    "authoritative_pointer_count_matches_source_rows": (
        approved_pointer_count
        == authoritative_rows_selected
    ),
    "constructed_field_count_matches_authority": (
        annotation_field_count
        == approved_pointer_count
    ),
    "no_duplicate_authoritative_pointers": (
        duplicate_pointer_count == 0
    ),
    "no_authoritative_hash_conflicts": (
        pointer_hash_conflict_count
        == 0
    ),
    "all_l0_files_exist": all(
        (
            PROJECT_ROOT
            / record[
                "l0_relative_path"
            ]
        ).exists()
        for record in all_summary_records
    ),
    "all_l1_files_exist": all(
        (
            PROJECT_ROOT
            / record[
                "l1_relative_path"
            ]
        ).exists()
        for record in all_summary_records
    ),
    "all_transformations_pass": (
        status_counts["BLOCKED"]
        == 0
    ),
    "positive_changed_field_population": (
        changed_field_count > 0
    ),
    "all_frozen_hashes_match": all(
        frozen_hash_checks.values()
    ),
}

status = (
    "PASS"
    if all(checks.values())
    else "BLOCKED"
)

report = {
    "status": status,
    "stage": (
        "AUTHORITATIVE_L0_L1_"
        "SLICE_CONSTRUCTION"
    ),
    "construction_version": (
        CONSTRUCTION_VERSION
    ),
    "operation_count": (
        len(all_summary_records)
    ),
    "pair_count": (
        len(
            {
                record["pair_id"]
                for record
                in all_summary_records
            }
        )
    ),
    "prose_rows_scanned": (
        prose_rows_scanned
    ),
    "authoritative_pointer_count": (
        approved_pointer_count
    ),
    "authoritative_rows_selected": (
        authoritative_rows_selected
    ),
    "unusable_rows_skipped": (
        unusable_rows_skipped
    ),
    "annotation_field_count": (
        annotation_field_count
    ),
    "changed_field_count": (
        changed_field_count
    ),
    "no_ascii_field_count": (
        no_ascii_field_count
    ),
    "ascii_letter_count": (
        ascii_letter_count
    ),
    "non_ascii_character_count": (
        non_ascii_character_count
    ),
    "duplicate_authoritative_pointer_count": (
        duplicate_pointer_count
    ),
    "authoritative_hash_conflict_count": (
        pointer_hash_conflict_count
    ),
    "operation_status_counts": dict(
        sorted(
            status_counts.items()
        )
    ),
    "checkpoint_chunk_size": (
        CHECKPOINT_CHUNK_SIZE
    ),
    "checkpoint_count": (
        total_chunks
    ),
    "checkpoint_reused_operation_count": (
        reused_operation_count
    ),
    "newly_constructed_operation_count": (
        newly_constructed_operation_count
    ),
    "l0_total_bytes": (
        l0_bytes_total
    ),
    "l1_total_bytes": (
        l1_bytes_total
    ),
    "pointer_authority": {
        "path": str(
            PROSE_FIELDS_PATH
        ),
        "sha256": (
            observed_prose_fields_sha256
        ),
        "selection_rule": (
            "Eligible-operation rows whose "
            "Notebook 02 field_name is summary "
            "or description and whose annotation "
            "value is a valid string."
        ),
        "arbitrary_example_payload_values_excluded": (
            True
        ),
        "vendor_extension_payload_values_excluded": (
            True
        ),
    },
    "obfuscation": {
        "algorithm": (
            "deterministic_nonzero_"
            "ascii_alphabetic_shift"
        ),
        "seed": (
            OBFUSCATION_SEED
        ),
        "implementation_path": str(
            OBFUSCATION_MODULE_PATH
        ),
        "implementation_sha256": (
            observed_obfuscation_sha256
        ),
    },
    "construction_records": {
        "path": str(
            CONSTRUCTION_RECORDS_PATH
        ),
        "bytes": (
            CONSTRUCTION_RECORDS_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            construction_records_sha256
        ),
    },
    "pointer_database": {
        "path": str(
            POINTER_DATABASE_PATH
        ),
        "bytes": (
            POINTER_DATABASE_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                POINTER_DATABASE_PATH
            )
        ),
    },
    "checks": checks,
    "problem_examples": (
        problem_examples
    ),
    "duration_seconds": round(
        time.time()
        - CELL9_STARTED,
        3,
    ),
    "interpretation": (
        "L1 was generated from a deep copy of "
        "L0 using only the exact annotation "
        "positions selected by Notebook 02. "
        "A mapping key named description, an "
        "example payload property named "
        "description, or description content "
        "inside an arbitrary extension is not "
        "treated unless Notebook 02 explicitly "
        "classified that position as prose."
    ),
    "supersedes": {
        "construction_version": (
            "generic-summary-description-"
            "traversal"
        ),
        "reason": (
            "Generic traversal included 3,082 "
            "positions outside the authoritative "
            "Notebook 02 prose-walker set."
        ),
    },
    "report_path": str(
        REPORT_PATH
    ),
}

atomic_write_json(
    REPORT_PATH,
    report,
)

report["report_sha256"] = (
    sha256_file(REPORT_PATH)
)

with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

stage_03_policy.setdefault(
    "l1_construction",
    {},
).update(
    {
        "status": status,
        "construction_version": (
            CONSTRUCTION_VERSION
        ),
        "operation_count": (
            len(all_summary_records)
        ),
        "pair_count": (
            len(all_summary_records)
        ),
        "seed": (
            OBFUSCATION_SEED
        ),
        "implementation_sha256": (
            observed_obfuscation_sha256
        ),
        "pointer_authority_path": str(
            PROSE_FIELDS_PATH.relative_to(
                STAGE_02_PROJECT_ROOT
            )
        ).replace("\\", "/"),
        "pointer_authority_sha256": (
            observed_prose_fields_sha256
        ),
        "pointer_authority_count": (
            approved_pointer_count
        ),
        "generic_recursive_key_traversal_allowed": (
            False
        ),
        "construction_records_path": str(
            CONSTRUCTION_RECORDS_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/"),
        "construction_records_sha256": (
            construction_records_sha256
        ),
        "report_path": str(
            REPORT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/"),
        "report_sha256": (
            report["report_sha256"]
        ),
        "authoritative_pointer_reconciliation": (
            "PENDING_CELL_10_RERUN"
        ),
        "contract_preservation_audit": (
            "PENDING_CELL_10_RERUN"
        ),
        "openapi_validation": (
            "PENDING_CELL_11"
        ),
    }
)

temporary_policy_path = (
    STAGE_03_POLICY_PATH.with_name(
        STAGE_03_POLICY_PATH.name
        + ".tmp"
    )
)

with temporary_policy_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    yaml.safe_dump(
        stage_03_policy,
        handle,
        sort_keys=False,
        allow_unicode=True,
    )
    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_policy_path,
    STAGE_03_POLICY_PATH,
)

report[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

if status != "PASS":
    raise RuntimeError(
        "Authoritative L0/L1 construction "
        "failed. Review the checks and "
        "problem examples above."
    )

print(
    "\nNOTEBOOK 03 — CELL 9: PASS"
)

{"processed": 1000, "total": 52922, "checkpoint_reused": 0, "newly_constructed": 1000, "authoritative_fields": 23794, "changed_fields": 23175, "problem_count": 0, "elapsed_seconds": 84.7}
{"processed": 2000, "total": 52922, "checkpoint_reused": 0, "newly_constructed": 2000, "authoritative_fields": 47124, "changed_fields": 45817, "problem_count": 0, "elapsed_seconds": 99.9}
{"processed": 3000, "total": 52922, "checkpoint_reused": 0, "newly_constructed": 3000, "authoritative_fields": 70757, "changed_fields": 68851, "problem_count": 0, "elapsed_seconds": 115.5}
{"processed": 4000, "total": 52922, "checkpoint_reused": 0, "newly_constructed": 4000, "authoritative_fields": 94185, "changed_fields": 91829, "problem_count": 0, "elapsed_seconds": 141.6}
{"processed": 5000, "total": 52922, "checkpoint_reused": 0, "newly_constructed": 5000, "authoritative_fields": 117041, "changed_fields": 114243, "problem_count": 0, "elapsed_seconds": 178.3}
{"processed": 6000, "total": 52922, "checkpoint_reused"

In [10]:
# Notebook 03 — Cell 10
# Reconcile authoritative prose pointers and audit complete L0/L1 preservation.

from __future__ import annotations

import csv
import gzip
import hashlib
import json
import os
import sqlite3
import time
from collections import Counter, defaultdict
from pathlib import Path

import yaml


CELL10_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "data/checkpoints/l1_construction"
)
CONSTRUCTION_RECORDS_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_construction_records.jsonl"
)
PROSE_FIELDS_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data/audits/prose_fields.csv"
)
OBFUSCATION_LOG_PATH = (
    PROJECT_ROOT
    / "data/audits/obfuscation_log.jsonl"
)
PAIR_AUDIT_PATH = (
    PROJECT_ROOT
    / "data/audits/pair_audit.csv"
)
SQLITE_PATH = (
    PROJECT_ROOT
    / "data/checkpoints/"
    "cell10_pointer_reconciliation.sqlite"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "l0_l1_contract_preservation.json"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)

EXPECTED_OPERATION_COUNT = expected_eligible_operation_count
EXPECTED_CHECKPOINT_COUNT = (EXPECTED_OPERATION_COUNT + 249) // 250
EXPECTED_OBFUSCATION_SHA256 = (
    "79eb0c682511ff393845604fb94e9ebe6"
    "c261c03208a22e50f79549719b05cbf"
)

for directory in [
    OBFUSCATION_LOG_PATH.parent,
    PAIR_AUDIT_PATH.parent,
    SQLITE_PATH.parent,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            1024 * 1024
        ):
            digest.update(block)

    return digest.hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def atomic_write_json(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    content = (
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def load_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}"
                )

            records.append(value)

    return records


def read_checkpoint(
    path: Path,
) -> list[dict]:
    records = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Invalid checkpoint record at "
                    f"{path}:{line_number}"
                )

            records.append(value)

    return records


def decode_pointer_part(
    value: str,
) -> str:
    return (
        value
        .replace("~1", "/")
        .replace("~0", "~")
    )


def pointer_parts(
    pointer: str,
) -> tuple[str, ...]:
    if pointer == "":
        return ()

    if not pointer.startswith("/"):
        raise ValueError(
            f"Invalid JSON pointer: {pointer}"
        )

    return tuple(
        decode_pointer_part(part)
        for part in pointer[1:].split("/")
    )


def pointer_value(
    document,
    pointer: str,
):
    current = document

    for part in pointer_parts(pointer):
        if isinstance(current, list):
            current = current[int(part)]
        elif isinstance(current, dict):
            current = current[part]
        else:
            raise KeyError(pointer)

    return current


def escape_pointer_part(
    value: str,
) -> str:
    return (
        str(value)
        .replace("~", "~0")
        .replace("/", "~1")
    )


def child_pointer(
    parent: str,
    child,
) -> str:
    encoded = escape_pointer_part(
        str(child)
    )

    if parent:
        return f"{parent}/{encoded}"

    return f"/{encoded}"


def collect_difference_pointers(
    left,
    right,
    pointer: str = "",
) -> set[str]:
    differences = set()

    if type(left) is not type(right):
        differences.add(pointer)
        return differences

    if isinstance(left, dict):
        left_keys = set(left)
        right_keys = set(right)

        if left_keys != right_keys:
            for key in sorted(
                left_keys ^ right_keys,
                key=str,
            ):
                differences.add(
                    child_pointer(
                        pointer,
                        key,
                    )
                )

        for key in sorted(
            left_keys & right_keys,
            key=str,
        ):
            differences.update(
                collect_difference_pointers(
                    left[key],
                    right[key],
                    child_pointer(
                        pointer,
                        key,
                    ),
                )
            )

        return differences

    if isinstance(left, list):
        if len(left) != len(right):
            differences.add(pointer)
            return differences

        for index, (
            left_value,
            right_value,
        ) in enumerate(
            zip(left, right)
        ):
            differences.update(
                collect_difference_pointers(
                    left_value,
                    right_value,
                    child_pointer(
                        pointer,
                        index,
                    ),
                )
            )

        return differences

    if left != right:
        differences.add(pointer)

    return differences


def select_column(
    fieldnames: list[str],
    candidates: list[str],
    required: bool = False,
) -> str | None:
    normalized = {
        field.strip().lower(): field
        for field in fieldnames
    }

    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[
                candidate.lower()
            ]

    if required:
        raise RuntimeError(
            "Required column not found. "
            f"Expected one of {candidates}; "
            f"observed {fieldnames}."
        )

    return None


def status_is_usable(
    value,
) -> bool:
    if value is None:
        return True

    normalized = str(value).strip().upper()

    if not normalized:
        return True

    rejected_markers = [
        "NONSTRING",
        "INVALID",
        "EXCLUDED",
        "BLOCKED",
        "ERROR",
        "FAILED",
        "FAILURE",
    ]

    return not any(
        marker in normalized
        for marker in rejected_markers
    )


def pointer_field_name(
    pointer: str,
) -> str | None:
    try:
        parts = pointer_parts(pointer)
    except Exception:
        return None

    if not parts:
        return None

    return parts[-1]


required_paths = [
    CHECKPOINT_ROOT,
    CONSTRUCTION_RECORDS_PATH,
    PROSE_FIELDS_PATH,
    STAGE_03_POLICY_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

checkpoint_paths = sorted(
    CHECKPOINT_ROOT.glob(
        "chunk_*.jsonl.gz"
    )
)

if (
    len(checkpoint_paths)
    != EXPECTED_CHECKPOINT_COUNT
):
    raise RuntimeError(
        "Expected "
        f"{EXPECTED_CHECKPOINT_COUNT} "
        "Cell 9 checkpoints, found "
        f"{len(checkpoint_paths)}."
    )

construction_records = load_jsonl(
    CONSTRUCTION_RECORDS_PATH
)

if (
    len(construction_records)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected Cell 9 construction-record "
        f"count: {len(construction_records)}"
    )

construction_by_op_id = {
    record["op_id"]: record
    for record in construction_records
}

if (
    len(construction_by_op_id)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Cell 9 construction records contain "
        "duplicate operation IDs."
    )

eligible_op_ids = set(
    construction_by_op_id
)

connection = sqlite3.connect(
    SQLITE_PATH
)
cursor = connection.cursor()

cursor.execute(
    "PRAGMA journal_mode=WAL"
)
cursor.execute(
    "PRAGMA synchronous=NORMAL"
)
cursor.execute(
    "PRAGMA temp_store=FILE"
)
cursor.execute(
    "PRAGMA cache_size=-200000"
)

cursor.executescript(
    """
    DROP TABLE IF EXISTS expected;
    DROP TABLE IF EXISTS walker;

    CREATE TABLE expected (
        op_id TEXT NOT NULL,
        pointer TEXT NOT NULL,
        original_sha256 TEXT NOT NULL,
        transformed_sha256 TEXT NOT NULL,
        changed INTEGER NOT NULL,
        PRIMARY KEY (op_id, pointer)
    );

    CREATE TABLE walker (
        op_id TEXT NOT NULL,
        pointer TEXT NOT NULL,
        observed_sha256 TEXT NOT NULL,
        occurrence_count INTEGER NOT NULL,
        PRIMARY KEY (
            op_id,
            pointer,
            observed_sha256
        )
    );

    CREATE INDEX expected_op_id_index
        ON expected(op_id);

    CREATE INDEX walker_op_id_index
        ON walker(op_id);
    """
)

temporary_obfuscation_log = (
    OBFUSCATION_LOG_PATH.with_name(
        OBFUSCATION_LOG_PATH.name
        + ".tmp"
    )
)

expected_insert_batch = []
expected_pointer_count = 0
duplicate_checkpoint_pointer_count = 0
checkpoint_operation_count = 0

with temporary_obfuscation_log.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as log_handle:
    for checkpoint_number, checkpoint_path in enumerate(
        checkpoint_paths,
        start=1,
    ):
        checkpoint_records = (
            read_checkpoint(
                checkpoint_path
            )
        )

        for operation_record in checkpoint_records:
            op_id = operation_record["op_id"]
            checkpoint_operation_count += 1

            for field_record in operation_record[
                "field_records"
            ]:
                pointer = field_record[
                    "slice_pointer"
                ]

                expected_insert_batch.append(
                    (
                        op_id,
                        pointer,
                        field_record[
                            "original_sha256"
                        ],
                        field_record[
                            "transformed_sha256"
                        ],
                        int(
                            field_record[
                                "changed"
                            ]
                        ),
                    )
                )

                log_record = {
                    "op_id": op_id,
                    "pair_id": (
                        operation_record.get(
                            "pair_id"
                        )
                    ),
                    "spec_id": (
                        operation_record.get(
                            "spec_id"
                        )
                    ),
                    "family_id": (
                        operation_record.get(
                            "family_id"
                        )
                    ),
                    "slice_pointer": pointer,
                    "field_name": (
                        field_record.get(
                            "field_name"
                        )
                    ),
                    "original_sha256": (
                        field_record[
                            "original_sha256"
                        ]
                    ),
                    "transformed_sha256": (
                        field_record[
                            "transformed_sha256"
                        ]
                    ),
                    "original_length": (
                        field_record[
                            "original_length"
                        ]
                    ),
                    "transformed_length": (
                        field_record[
                            "transformed_length"
                        ]
                    ),
                    "ascii_letter_count": (
                        field_record[
                            "ascii_letter_count"
                        ]
                    ),
                    "non_ascii_character_count": (
                        field_record[
                            "non_ascii_character_count"
                        ]
                    ),
                    "changed": (
                        field_record["changed"]
                    ),
                    "treatment_class": (
                        field_record[
                            "treatment_class"
                        ]
                    ),
                    "preservation_checks": (
                        field_record[
                            "preservation_checks"
                        ]
                    ),
                    "algorithm": (
                        "deterministic_nonzero_"
                        "ascii_alphabetic_shift"
                    ),
                    "algorithm_sha256": (
                        EXPECTED_OBFUSCATION_SHA256
                    ),
                    "seed": 42,
                }

                log_handle.write(
                    json.dumps(
                        log_record,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(",", ":"),
                        allow_nan=False,
                    )
                )
                log_handle.write("\n")

                expected_pointer_count += 1

                if (
                    len(expected_insert_batch)
                    >= 10000
                ):
                    try:
                        cursor.executemany(
                            """
                            INSERT INTO expected (
                                op_id,
                                pointer,
                                original_sha256,
                                transformed_sha256,
                                changed
                            )
                            VALUES (?, ?, ?, ?, ?)
                            """,
                            expected_insert_batch,
                        )
                    except sqlite3.IntegrityError:
                        for row in expected_insert_batch:
                            try:
                                cursor.execute(
                                    """
                                    INSERT INTO expected (
                                        op_id,
                                        pointer,
                                        original_sha256,
                                        transformed_sha256,
                                        changed
                                    )
                                    VALUES (?, ?, ?, ?, ?)
                                    """,
                                    row,
                                )
                            except sqlite3.IntegrityError:
                                duplicate_checkpoint_pointer_count += (
                                    1
                                )

                    expected_insert_batch.clear()

        if checkpoint_number % 25 == 0:
            connection.commit()

            print(
                json.dumps(
                    {
                        "checkpoint_files_loaded": (
                            checkpoint_number
                        ),
                        "checkpoint_file_total": (
                            len(checkpoint_paths)
                        ),
                        "operations_loaded": (
                            checkpoint_operation_count
                        ),
                        "expected_pointers_loaded": (
                            expected_pointer_count
                        ),
                        "elapsed_seconds": round(
                            time.time()
                            - CELL10_STARTED,
                            1,
                        ),
                    }
                )
            )

    if expected_insert_batch:
        try:
            cursor.executemany(
                """
                INSERT INTO expected (
                    op_id,
                    pointer,
                    original_sha256,
                    transformed_sha256,
                    changed
                )
                VALUES (?, ?, ?, ?, ?)
                """,
                expected_insert_batch,
            )
        except sqlite3.IntegrityError:
            for row in expected_insert_batch:
                try:
                    cursor.execute(
                        """
                        INSERT INTO expected (
                            op_id,
                            pointer,
                            original_sha256,
                            transformed_sha256,
                            changed
                        )
                        VALUES (?, ?, ?, ?, ?)
                        """,
                        row,
                    )
                except sqlite3.IntegrityError:
                    duplicate_checkpoint_pointer_count += (
                        1
                    )

        expected_insert_batch.clear()

    log_handle.flush()
    os.fsync(log_handle.fileno())

connection.commit()

os.replace(
    temporary_obfuscation_log,
    OBFUSCATION_LOG_PATH,
)

if (
    checkpoint_operation_count
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected checkpoint operation count: "
        f"{checkpoint_operation_count}"
    )

with PROSE_FIELDS_PATH.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as prose_handle:
    reader = csv.DictReader(
        prose_handle
    )

    if reader.fieldnames is None:
        raise RuntimeError(
            "Notebook 02 prose ledger has "
            "no header."
        )

    fieldnames = list(
        reader.fieldnames
    )

    op_id_column = select_column(
        fieldnames,
        [
            "op_id",
            "operation_id",
        ],
        required=True,
    )
    pointer_column = select_column(
        fieldnames,
        [
            "slice_pointer",
            "json_pointer",
            "pointer",
        ],
        required=True,
    )
    field_name_column = select_column(
        fieldnames,
        [
            "field_name",
            "annotation_field",
            "prose_field",
        ],
        required=False,
    )
    status_column = select_column(
        fieldnames,
        [
            "status",
            "annotation_status",
            "measurement_status",
            "prose_status",
        ],
        required=False,
    )
    hash_column = select_column(
        fieldnames,
        [
            "text_sha256",
            "value_sha256",
            "prose_sha256",
            "annotation_sha256",
        ],
        required=False,
    )
    text_column = select_column(
        fieldnames,
        [
            "text",
            "prose_text",
            "annotation_text",
            "value",
        ],
        required=False,
    )
    type_column = select_column(
        fieldnames,
        [
            "observed_type",
            "value_type",
            "text_type",
        ],
        required=False,
    )

    walker_insert_batch = []
    prose_rows_scanned = 0
    eligible_target_rows = 0
    unusable_rows_skipped = 0

    for row in reader:
        prose_rows_scanned += 1

        op_id = str(
            row.get(
                op_id_column,
                "",
            )
        ).strip()

        if op_id not in eligible_op_ids:
            continue

        if (
            status_column is not None
            and not status_is_usable(
                row.get(status_column)
            )
        ):
            unusable_rows_skipped += 1
            continue

        if type_column is not None:
            observed_type = str(
                row.get(
                    type_column,
                    "",
                )
            ).strip().lower()

            if (
                observed_type
                and observed_type
                not in {
                    "str",
                    "string",
                    "text",
                }
            ):
                unusable_rows_skipped += 1
                continue

        pointer = str(
            row.get(
                pointer_column,
                "",
            )
        ).strip()

        if not pointer:
            continue

        if field_name_column is not None:
            field_name = str(
                row.get(
                    field_name_column,
                    "",
                )
            ).strip()
        else:
            field_name = (
                pointer_field_name(
                    pointer
                )
                or ""
            )

        if field_name not in {
            "summary",
            "description",
        }:
            continue

        observed_sha256 = ""

        if hash_column is not None:
            observed_sha256 = str(
                row.get(
                    hash_column,
                    "",
                )
            ).strip()

        if (
            not observed_sha256
            and text_column is not None
        ):
            observed_sha256 = (
                sha256_text(
                    str(
                        row.get(
                            text_column,
                            "",
                        )
                    )
                )
            )

        walker_insert_batch.append(
            (
                op_id,
                pointer,
                observed_sha256,
                1,
            )
        )
        eligible_target_rows += 1

        if (
            len(walker_insert_batch)
            >= 10000
        ):
            cursor.executemany(
                """
                INSERT INTO walker (
                    op_id,
                    pointer,
                    observed_sha256,
                    occurrence_count
                )
                VALUES (?, ?, ?, ?)
                ON CONFLICT (
                    op_id,
                    pointer,
                    observed_sha256
                )
                DO UPDATE SET
                    occurrence_count =
                        occurrence_count + 1
                """,
                walker_insert_batch,
            )
            walker_insert_batch.clear()

        if (
            prose_rows_scanned
            % 500000
            == 0
        ):
            connection.commit()

            print(
                json.dumps(
                    {
                        "prose_rows_scanned": (
                            prose_rows_scanned
                        ),
                        "eligible_target_rows": (
                            eligible_target_rows
                        ),
                        "elapsed_seconds": round(
                            time.time()
                            - CELL10_STARTED,
                            1,
                        ),
                    }
                )
            )

    if walker_insert_batch:
        cursor.executemany(
            """
            INSERT INTO walker (
                op_id,
                pointer,
                observed_sha256,
                occurrence_count
            )
            VALUES (?, ?, ?, ?)
            ON CONFLICT (
                op_id,
                pointer,
                observed_sha256
            )
            DO UPDATE SET
                occurrence_count =
                    occurrence_count + 1
            """,
            walker_insert_batch,
        )

connection.commit()

database_expected_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM expected
    """
).fetchone()[0]

database_walker_key_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM walker
    """
).fetchone()[0]

missing_pointer_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM expected AS e
    WHERE NOT EXISTS (
        SELECT 1
        FROM walker AS w
        WHERE
            w.op_id = e.op_id
            AND w.pointer = e.pointer
    )
    """
).fetchone()[0]

unexpected_pointer_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM walker AS w
    WHERE NOT EXISTS (
        SELECT 1
        FROM expected AS e
        WHERE
            e.op_id = w.op_id
            AND e.pointer = w.pointer
    )
    """
).fetchone()[0]

duplicate_walker_pointer_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT
            op_id,
            pointer,
            SUM(occurrence_count) AS n
        FROM walker
        GROUP BY
            op_id,
            pointer
        HAVING n != 1
    )
    """
).fetchone()[0]

hash_mismatch_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM expected AS e
    JOIN walker AS w
        ON w.op_id = e.op_id
        AND w.pointer = e.pointer
    WHERE
        w.observed_sha256 != ''
        AND w.observed_sha256
            != e.original_sha256
    """
).fetchone()[0]

missing_by_op_id = {
    op_id: count
    for op_id, count in cursor.execute(
        """
        SELECT
            e.op_id,
            COUNT(*)
        FROM expected AS e
        WHERE NOT EXISTS (
            SELECT 1
            FROM walker AS w
            WHERE
                w.op_id = e.op_id
                AND w.pointer = e.pointer
        )
        GROUP BY e.op_id
        """
    )
}

unexpected_by_op_id = {
    op_id: count
    for op_id, count in cursor.execute(
        """
        SELECT
            w.op_id,
            COUNT(*)
        FROM walker AS w
        WHERE NOT EXISTS (
            SELECT 1
            FROM expected AS e
            WHERE
                e.op_id = w.op_id
                AND e.pointer = w.pointer
        )
        GROUP BY w.op_id
        """
    )
}

duplicate_by_op_id = {
    op_id: count
    for op_id, count in cursor.execute(
        """
        SELECT
            op_id,
            COUNT(*)
        FROM (
            SELECT
                op_id,
                pointer,
                SUM(occurrence_count) AS n
            FROM walker
            GROUP BY
                op_id,
                pointer
            HAVING n != 1
        )
        GROUP BY op_id
        """
    )
}

hash_mismatch_by_op_id = {
    op_id: count
    for op_id, count in cursor.execute(
        """
        SELECT
            e.op_id,
            COUNT(*)
        FROM expected AS e
        JOIN walker AS w
            ON w.op_id = e.op_id
            AND w.pointer = e.pointer
        WHERE
            w.observed_sha256 != ''
            AND w.observed_sha256
                != e.original_sha256
        GROUP BY e.op_id
        """
    )
}

pointer_problem_examples = {
    "missing": [
        {
            "op_id": op_id,
            "pointer": pointer,
        }
        for op_id, pointer in cursor.execute(
            """
            SELECT
                e.op_id,
                e.pointer
            FROM expected AS e
            WHERE NOT EXISTS (
                SELECT 1
                FROM walker AS w
                WHERE
                    w.op_id = e.op_id
                    AND w.pointer = e.pointer
            )
            LIMIT 20
            """
        )
    ],
    "unexpected": [
        {
            "op_id": op_id,
            "pointer": pointer,
        }
        for op_id, pointer in cursor.execute(
            """
            SELECT
                w.op_id,
                w.pointer
            FROM walker AS w
            WHERE NOT EXISTS (
                SELECT 1
                FROM expected AS e
                WHERE
                    e.op_id = w.op_id
                    AND e.pointer = w.pointer
            )
            LIMIT 20
            """
        )
    ],
    "hash_mismatch": [
        {
            "op_id": op_id,
            "pointer": pointer,
            "expected_sha256": expected_hash,
            "observed_sha256": observed_hash,
        }
        for (
            op_id,
            pointer,
            expected_hash,
            observed_hash,
        ) in cursor.execute(
            """
            SELECT
                e.op_id,
                e.pointer,
                e.original_sha256,
                w.observed_sha256
            FROM expected AS e
            JOIN walker AS w
                ON w.op_id = e.op_id
                AND w.pointer = e.pointer
            WHERE
                w.observed_sha256 != ''
                AND w.observed_sha256
                    != e.original_sha256
            LIMIT 20
            """
        )
    ],
}

connection.commit()

temporary_pair_audit = (
    PAIR_AUDIT_PATH.with_name(
        PAIR_AUDIT_PATH.name
        + ".tmp"
    )
)

pair_audit_rows = []
tree_status_counts = Counter()
unexpected_tree_difference_count = 0
missing_expected_difference_count = 0
field_hash_problem_count = 0
tree_problem_examples = []

processed_operations = 0

with temporary_pair_audit.open(
    "w",
    encoding="utf-8",
    newline="",
) as audit_handle:
    fieldnames = [
        "op_id",
        "pair_id",
        "spec_id",
        "family_id",
        "l0_sha256",
        "l1_sha256",
        "annotation_field_count",
        "changed_field_count",
        "observed_difference_count",
        "missing_authoritative_pointer_count",
        "unexpected_authoritative_pointer_count",
        "duplicate_authoritative_pointer_count",
        "authoritative_hash_mismatch_count",
        "unexpected_tree_difference_count",
        "missing_expected_difference_count",
        "field_hash_problem_count",
        "pointer_reconciliation_status",
        "contract_preservation_status",
        "status",
    ]

    writer = csv.DictWriter(
        audit_handle,
        fieldnames=fieldnames,
    )
    writer.writeheader()

    for checkpoint_path in checkpoint_paths:
        checkpoint_records = (
            read_checkpoint(
                checkpoint_path
            )
        )

        for record in checkpoint_records:
            op_id = record["op_id"]

            l0_path = (
                PROJECT_ROOT
                / record[
                    "l0_relative_path"
                ]
            )
            l1_path = (
                PROJECT_ROOT
                / record[
                    "l1_relative_path"
                ]
            )

            with l0_path.open(
                "r",
                encoding="utf-8",
            ) as handle:
                l0_document = json.load(
                    handle
                )

            with l1_path.open(
                "r",
                encoding="utf-8",
            ) as handle:
                l1_document = json.load(
                    handle
                )

            field_records = (
                record["field_records"]
            )
            field_by_pointer = {
                field_record[
                    "slice_pointer"
                ]: field_record
                for field_record
                in field_records
            }

            if (
                len(field_by_pointer)
                != len(field_records)
            ):
                raise RuntimeError(
                    "Duplicate transformed pointer "
                    f"inside checkpoint for {op_id}."
                )

            expected_changed_pointers = {
                pointer
                for pointer, field_record
                in field_by_pointer.items()
                if field_record["changed"]
            }

            observed_differences = (
                collect_difference_pointers(
                    l0_document,
                    l1_document,
                )
            )

            unexpected_tree_differences = (
                observed_differences
                - expected_changed_pointers
            )
            missing_expected_differences = (
                expected_changed_pointers
                - observed_differences
            )

            operation_field_hash_problems = []

            for (
                pointer,
                field_record,
            ) in field_by_pointer.items():
                try:
                    l0_value = pointer_value(
                        l0_document,
                        pointer,
                    )
                    l1_value = pointer_value(
                        l1_document,
                        pointer,
                    )
                except Exception as error:
                    operation_field_hash_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "POINTER_RESOLUTION_FAILED"
                            ),
                            "diagnostic": repr(
                                error
                            ),
                        }
                    )
                    continue

                if not isinstance(
                    l0_value,
                    str,
                ) or not isinstance(
                    l1_value,
                    str,
                ):
                    operation_field_hash_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "NONSTRING_TARGET"
                            ),
                        }
                    )
                    continue

                if (
                    sha256_text(l0_value)
                    != field_record[
                        "original_sha256"
                    ]
                ):
                    operation_field_hash_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "L0_HASH_MISMATCH"
                            ),
                        }
                    )

                if (
                    sha256_text(l1_value)
                    != field_record[
                        "transformed_sha256"
                    ]
                ):
                    operation_field_hash_problems.append(
                        {
                            "pointer": pointer,
                            "problem": (
                                "L1_HASH_MISMATCH"
                            ),
                        }
                    )

            operation_missing_pointer_count = (
                int(
                    missing_by_op_id.get(
                        op_id,
                        0,
                    )
                )
            )
            operation_unexpected_pointer_count = (
                int(
                    unexpected_by_op_id.get(
                        op_id,
                        0,
                    )
                )
            )
            operation_duplicate_pointer_count = (
                int(
                    duplicate_by_op_id.get(
                        op_id,
                        0,
                    )
                )
            )
            operation_hash_mismatch_count = (
                int(
                    hash_mismatch_by_op_id.get(
                        op_id,
                        0,
                    )
                )
            )

            pointer_reconciliation_pass = (
                operation_missing_pointer_count
                == 0
                and operation_unexpected_pointer_count
                == 0
                and operation_duplicate_pointer_count
                == 0
                and operation_hash_mismatch_count
                == 0
            )

            contract_preservation_pass = (
                not unexpected_tree_differences
                and not missing_expected_differences
                and not operation_field_hash_problems
            )

            operation_status = (
                "PASS"
                if (
                    pointer_reconciliation_pass
                    and contract_preservation_pass
                )
                else "BLOCKED"
            )

            tree_status_counts[
                operation_status
            ] += 1

            unexpected_tree_difference_count += (
                len(
                    unexpected_tree_differences
                )
            )
            missing_expected_difference_count += (
                len(
                    missing_expected_differences
                )
            )
            field_hash_problem_count += len(
                operation_field_hash_problems
            )

            if (
                operation_status != "PASS"
                and len(
                    tree_problem_examples
                ) < 20
            ):
                tree_problem_examples.append(
                    {
                        "op_id": op_id,
                        "unexpected_tree_differences": (
                            sorted(
                                unexpected_tree_differences
                            )[:20]
                        ),
                        "missing_expected_differences": (
                            sorted(
                                missing_expected_differences
                            )[:20]
                        ),
                        "field_hash_problems": (
                            operation_field_hash_problems[
                                :20
                            ]
                        ),
                        "pointer_counts": {
                            "missing": (
                                operation_missing_pointer_count
                            ),
                            "unexpected": (
                                operation_unexpected_pointer_count
                            ),
                            "duplicate": (
                                operation_duplicate_pointer_count
                            ),
                            "hash_mismatch": (
                                operation_hash_mismatch_count
                            ),
                        },
                    }
                )

            audit_row = {
                "op_id": op_id,
                "pair_id": (
                    record.get(
                        "pair_id"
                    )
                ),
                "spec_id": (
                    record.get(
                        "spec_id"
                    )
                ),
                "family_id": (
                    record.get(
                        "family_id"
                    )
                ),
                "l0_sha256": (
                    record["l0_sha256"]
                ),
                "l1_sha256": (
                    record["l1_sha256"]
                ),
                "annotation_field_count": (
                    record[
                        "annotation_field_count"
                    ]
                ),
                "changed_field_count": (
                    record[
                        "changed_field_count"
                    ]
                ),
                "observed_difference_count": (
                    len(
                        observed_differences
                    )
                ),
                "missing_authoritative_pointer_count": (
                    operation_missing_pointer_count
                ),
                "unexpected_authoritative_pointer_count": (
                    operation_unexpected_pointer_count
                ),
                "duplicate_authoritative_pointer_count": (
                    operation_duplicate_pointer_count
                ),
                "authoritative_hash_mismatch_count": (
                    operation_hash_mismatch_count
                ),
                "unexpected_tree_difference_count": (
                    len(
                        unexpected_tree_differences
                    )
                ),
                "missing_expected_difference_count": (
                    len(
                        missing_expected_differences
                    )
                ),
                "field_hash_problem_count": (
                    len(
                        operation_field_hash_problems
                    )
                ),
                "pointer_reconciliation_status": (
                    "PASS"
                    if pointer_reconciliation_pass
                    else "BLOCKED"
                ),
                "contract_preservation_status": (
                    "PASS"
                    if contract_preservation_pass
                    else "BLOCKED"
                ),
                "status": (
                    operation_status
                ),
            }

            writer.writerow(
                audit_row
            )
            pair_audit_rows.append(
                audit_row
            )

            processed_operations += 1

            if (
                processed_operations
                % 1000
                == 0
                or processed_operations
                == EXPECTED_OPERATION_COUNT
            ):
                print(
                    json.dumps(
                        {
                            "pairs_audited": (
                                processed_operations
                            ),
                            "pair_total": (
                                EXPECTED_OPERATION_COUNT
                            ),
                            "pair_pass_count": (
                                tree_status_counts[
                                    "PASS"
                                ]
                            ),
                            "pair_problem_count": (
                                tree_status_counts[
                                    "BLOCKED"
                                ]
                            ),
                            "elapsed_seconds": round(
                                time.time()
                                - CELL10_STARTED,
                                1,
                            ),
                        }
                    )
                )

    audit_handle.flush()
    os.fsync(
        audit_handle.fileno()
    )

os.replace(
    temporary_pair_audit,
    PAIR_AUDIT_PATH,
)

connection.close()

checks = {
    "checkpoint_operation_count_matches_stage02_candidate_count": (
        checkpoint_operation_count
        == EXPECTED_OPERATION_COUNT
    ),
    "expected_pointer_count_matches_database": (
        expected_pointer_count
        == database_expected_count
    ),
    "no_duplicate_checkpoint_pointers": (
        duplicate_checkpoint_pointer_count
        == 0
    ),
    "authoritative_pointer_count_matches": (
        database_walker_key_count
        == database_expected_count
    ),
    "no_missing_authoritative_pointers": (
        missing_pointer_count == 0
    ),
    "no_unexpected_authoritative_pointers": (
        unexpected_pointer_count == 0
    ),
    "no_duplicate_authoritative_pointers": (
        duplicate_walker_pointer_count
        == 0
    ),
    "no_authoritative_hash_mismatches": (
        hash_mismatch_count == 0
    ),
    "pair_audit_count_matches_stage02_candidate_count": (
        len(pair_audit_rows)
        == EXPECTED_OPERATION_COUNT
    ),
    "all_pairs_pass_contract_audit": (
        tree_status_counts["PASS"]
        == EXPECTED_OPERATION_COUNT
        and tree_status_counts[
            "BLOCKED"
        ]
        == 0
    ),
    "no_unapproved_tree_differences": (
        unexpected_tree_difference_count
        == 0
    ),
    "all_expected_changes_observed": (
        missing_expected_difference_count
        == 0
    ),
    "all_field_hashes_match": (
        field_hash_problem_count == 0
    ),
}

status = (
    "PASS"
    if all(checks.values())
    else "BLOCKED"
)

report = {
    "status": status,
    "stage": (
        "AUTHORITATIVE_POINTER_AND_"
        "CONTRACT_PRESERVATION_AUDIT"
    ),
    "operation_count": (
        processed_operations
    ),
    "expected_annotation_pointer_count": (
        database_expected_count
    ),
    "authoritative_walker_pointer_count": (
        database_walker_key_count
    ),
    "authoritative_target_row_count": (
        eligible_target_rows
    ),
    "prose_rows_scanned": (
        prose_rows_scanned
    ),
    "unusable_prose_rows_skipped": (
        unusable_rows_skipped
    ),
    "pointer_reconciliation": {
        "missing_pointer_count": (
            missing_pointer_count
        ),
        "unexpected_pointer_count": (
            unexpected_pointer_count
        ),
        "duplicate_walker_pointer_count": (
            duplicate_walker_pointer_count
        ),
        "hash_mismatch_count": (
            hash_mismatch_count
        ),
        "duplicate_checkpoint_pointer_count": (
            duplicate_checkpoint_pointer_count
        ),
        "hash_comparison_available": (
            hash_column is not None
            or text_column is not None
        ),
        "problem_examples": (
            pointer_problem_examples
        ),
    },
    "contract_preservation": {
        "status_counts": dict(
            sorted(
                tree_status_counts.items()
            )
        ),
        "unexpected_tree_difference_count": (
            unexpected_tree_difference_count
        ),
        "missing_expected_difference_count": (
            missing_expected_difference_count
        ),
        "field_hash_problem_count": (
            field_hash_problem_count
        ),
        "problem_examples": (
            tree_problem_examples
        ),
    },
    "obfuscation_log": {
        "path": str(
            OBFUSCATION_LOG_PATH
        ),
        "bytes": (
            OBFUSCATION_LOG_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                OBFUSCATION_LOG_PATH
            )
        ),
        "record_count": (
            expected_pointer_count
        ),
    },
    "pair_audit": {
        "path": str(
            PAIR_AUDIT_PATH
        ),
        "bytes": (
            PAIR_AUDIT_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                PAIR_AUDIT_PATH
            )
        ),
        "record_count": (
            len(pair_audit_rows)
        ),
    },
    "reconciliation_database": {
        "path": str(
            SQLITE_PATH
        ),
        "bytes": (
            SQLITE_PATH.stat().st_size
        ),
        "sha256": (
            sha256_file(
                SQLITE_PATH
            )
        ),
    },
    "checks": checks,
    "duration_seconds": round(
        time.time()
        - CELL10_STARTED,
        3,
    ),
    "interpretation": (
        "Every transformed position must match "
        "a Notebook 02 prose-walker position, "
        "and every observed L0/L1 difference "
        "must be an approved transformed "
        "summary or description value. "
        "OpenAPI validation is performed in "
        "Cell 11."
    ),
    "report_path": str(
        REPORT_PATH
    ),
}

atomic_write_json(
    REPORT_PATH,
    report,
)

report["report_sha256"] = (
    sha256_file(
        REPORT_PATH
    )
)

with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

stage_03_policy.setdefault(
    "l1_construction",
    {},
).update(
    {
        "authoritative_pointer_reconciliation": (
            status
        ),
        "contract_preservation_audit": (
            status
        ),
        "obfuscation_log_path": str(
            OBFUSCATION_LOG_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/"),
        "obfuscation_log_sha256": (
            report[
                "obfuscation_log"
            ]["sha256"]
        ),
        "pair_audit_path": str(
            PAIR_AUDIT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/"),
        "pair_audit_sha256": (
            report[
                "pair_audit"
            ]["sha256"]
        ),
        "preservation_report_path": str(
            REPORT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/"),
        "preservation_report_sha256": (
            report["report_sha256"]
        ),
    }
)

temporary_policy_path = (
    STAGE_03_POLICY_PATH.with_name(
        STAGE_03_POLICY_PATH.name
        + ".tmp"
    )
)

with temporary_policy_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    yaml.safe_dump(
        stage_03_policy,
        handle,
        sort_keys=False,
        allow_unicode=True,
    )
    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_policy_path,
    STAGE_03_POLICY_PATH,
)

report[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

if status != "PASS":
    raise RuntimeError(
        "Authoritative pointer or contract "
        "preservation audit failed. Review "
        "the reported mismatch examples."
    )

print(
    "\nNOTEBOOK 03 — CELL 10: PASS"
)

{"checkpoint_files_loaded": 25, "checkpoint_file_total": 212, "operations_loaded": 6250, "expected_pointers_loaded": 147433, "elapsed_seconds": 8.0}
{"checkpoint_files_loaded": 50, "checkpoint_file_total": 212, "operations_loaded": 12500, "expected_pointers_loaded": 292254, "elapsed_seconds": 14.8}
{"checkpoint_files_loaded": 75, "checkpoint_file_total": 212, "operations_loaded": 18750, "expected_pointers_loaded": 439159, "elapsed_seconds": 21.3}
{"checkpoint_files_loaded": 100, "checkpoint_file_total": 212, "operations_loaded": 25000, "expected_pointers_loaded": 583062, "elapsed_seconds": 28.0}
{"checkpoint_files_loaded": 125, "checkpoint_file_total": 212, "operations_loaded": 31250, "expected_pointers_loaded": 727298, "elapsed_seconds": 36.8}
{"checkpoint_files_loaded": 150, "checkpoint_file_total": 212, "operations_loaded": 37500, "expected_pointers_loaded": 868397, "elapsed_seconds": 43.3}
{"checkpoint_files_loaded": 175, "checkpoint_file_total": 212, "operations_loaded": 43750, "e

In [11]:
# Notebook 03 — Cell 11
# Directly validate every L0 and L1 slice as OpenAPI documents.

from __future__ import annotations

import csv
import gzip
import hashlib
import json
import os
import time
import warnings
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from importlib.metadata import version
from pathlib import Path

import yaml


CELL11_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)
CONSTRUCTION_RECORDS_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_construction_records.jsonl"
)
PAIR_AUDIT_PATH = (
    PROJECT_ROOT
    / "data/audits/pair_audit.csv"
)
CELL10_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "l0_l1_contract_preservation.json"
)
CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "data/checkpoints/pair_openapi_validation"
)
VALIDATION_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/audits/pair_openapi_validation.csv"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "pair_openapi_validation.json"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)

EXPECTED_OPERATION_COUNT = expected_eligible_operation_count
VALIDATION_VERSION = (
    "openapi-spec-validator-0.9.0-"
    "direct-pair-validation-v1"
)
CHECKPOINT_CHUNK_SIZE = 250
MAX_WORKERS = max(
    1,
    min(
        4,
        os.cpu_count() or 1,
    ),
)

for directory in [
    CHECKPOINT_ROOT,
    VALIDATION_LEDGER_PATH.parent,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            1024 * 1024
        ):
            digest.update(block)

    return digest.hexdigest()


def atomic_write_json(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    content = (
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def load_jsonl(
    path: Path,
) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}"
                )

            records.append(value)

    return records


def checkpoint_path_for(
    chunk_number: int,
) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"chunk_{chunk_number:05d}.jsonl.gz"
    )


def read_checkpoint(
    path: Path,
) -> list[dict]:
    records = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            if line.strip():
                records.append(
                    json.loads(line)
                )

    return records


def write_checkpoint(
    path: Path,
    records: list[dict],
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with gzip.GzipFile(
        filename=str(temporary_path),
        mode="wb",
        compresslevel=6,
        mtime=0,
    ) as handle:
        for record in records:
            serialized = (
                json.dumps(
                    record,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                    allow_nan=False,
                )
                + "\n"
            ).encode("utf-8")

            handle.write(serialized)

    os.replace(
        temporary_path,
        path,
    )


def checkpoint_is_reusable(
    records: list[dict],
    expected_records: list[dict],
) -> bool:
    if len(records) != len(
        expected_records
    ):
        return False

    for observed, expected in zip(
        records,
        expected_records,
    ):
        if (
            observed.get(
                "validation_version"
            )
            != VALIDATION_VERSION
        ):
            return False

        if (
            observed.get("op_id")
            != expected["op_id"]
        ):
            return False

        if (
            observed.get("l0_sha256")
            != expected["l0_sha256"]
        ):
            return False

        if (
            observed.get("l1_sha256")
            != expected["l1_sha256"]
        ):
            return False

    return True


def validate_pair_worker(
    task: dict,
) -> dict:
    from openapi_spec_validator import (
        validate,
    )

    result = {
        "validation_version": (
            VALIDATION_VERSION
        ),
        "op_id": task["op_id"],
        "pair_id": task["pair_id"],
        "spec_id": task["spec_id"],
        "family_id": task["family_id"],
        "l0_sha256": task["l0_sha256"],
        "l1_sha256": task["l1_sha256"],
        "l0_status": "PENDING",
        "l1_status": "PENDING",
        "l0_diagnostic": None,
        "l1_diagnostic": None,
        "l0_warning_count": 0,
        "l1_warning_count": 0,
        "warning_examples": [],
        "status": "PENDING",
    }

    for condition in [
        "l0",
        "l1",
    ]:
        path = Path(
            task[f"{condition}_path"]
        )
        expected_sha256 = task[
            f"{condition}_sha256"
        ]

        try:
            observed_sha256 = (
                sha256_file(path)
            )

            if (
                observed_sha256
                != expected_sha256
            ):
                raise RuntimeError(
                    "File hash mismatch: "
                    f"expected {expected_sha256}, "
                    f"observed {observed_sha256}"
                )

            with path.open(
                "r",
                encoding="utf-8",
            ) as handle:
                document = json.load(handle)

            if not isinstance(
                document,
                dict,
            ):
                raise TypeError(
                    "OpenAPI document is not "
                    "a JSON object."
                )

            with warnings.catch_warnings(
                record=True
            ) as captured_warnings:
                warnings.simplefilter(
                    "always"
                )
                validate(document)

            result[
                f"{condition}_warning_count"
            ] = len(captured_warnings)

            for warning_record in (
                captured_warnings[:3]
            ):
                warning_text = str(
                    warning_record.message
                )

                if (
                    warning_text
                    not in result[
                        "warning_examples"
                    ]
                ):
                    result[
                        "warning_examples"
                    ].append(
                        warning_text
                    )

            result[
                f"{condition}_status"
            ] = "PASS"

        except Exception as error:
            result[
                f"{condition}_status"
            ] = "FAIL"
            result[
                f"{condition}_diagnostic"
            ] = (
                f"{type(error).__name__}: "
                f"{error}"
            )[:4000]

    result["status"] = (
        "PASS"
        if (
            result["l0_status"]
            == "PASS"
            and result["l1_status"]
            == "PASS"
        )
        else "BLOCKED"
    )

    return result


required_paths = [
    CONSTRUCTION_RECORDS_PATH,
    PAIR_AUDIT_PATH,
    CELL10_REPORT_PATH,
    STAGE_03_POLICY_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            required_path
        )

# Resolve the Cell 10 hash from the current run,
# rather than using a previous session's hash.
with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    current_stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

expected_cell10_report_sha256 = (
    current_stage_03_policy
    .get("l1_construction", {})
    .get("preservation_report_sha256")
)

if not isinstance(
    expected_cell10_report_sha256,
    str,
) or len(
    expected_cell10_report_sha256
) != 64:
    raise RuntimeError(
        "The current Cell 10 preservation-"
        "report hash is missing or malformed "
        "in stage_03.yaml."
    )

observed_cell10_report_sha256 = (
    sha256_file(
        CELL10_REPORT_PATH
    )
)

if (
    observed_cell10_report_sha256
    != expected_cell10_report_sha256
):
    raise RuntimeError(
        "The Cell 10 report does not match "
        "the hash recorded by the current "
        "stage_03.yaml."
    )

with CELL10_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cell10_report = json.load(handle)

if (
    cell10_report.get("status")
    != "PASS"
):
    raise RuntimeError(
        "Cell 10 did not report PASS."
    )

if (
    cell10_report.get("stage")
    != (
        "AUTHORITATIVE_POINTER_AND_"
        "CONTRACT_PRESERVATION_AUDIT"
    )
):
    raise RuntimeError(
        "Cell 10 report has an unexpected "
        "stage identifier."
    )

construction_records = load_jsonl(
    CONSTRUCTION_RECORDS_PATH
)

if (
    len(construction_records)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected construction-record count: "
        f"{len(construction_records)}"
    )

construction_records.sort(
    key=lambda record: record["op_id"]
)

if (
    len(
        {
            record["op_id"]
            for record
            in construction_records
        }
    )
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Construction records contain "
        "duplicate operation IDs."
    )

pair_audit_status_by_op_id = {}

with PAIR_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle:
    reader = csv.DictReader(handle)

    for row in reader:
        pair_audit_status_by_op_id[
            row["op_id"]
        ] = row["status"]

if (
    len(pair_audit_status_by_op_id)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Unexpected pair-audit row count."
    )

nonpassing_pair_audits = [
    op_id
    for op_id, audit_status
    in pair_audit_status_by_op_id.items()
    if audit_status != "PASS"
]

if nonpassing_pair_audits:
    raise RuntimeError(
        "Pair audit contains non-PASS rows: "
        + ", ".join(
            nonpassing_pair_audits[:20]
        )
    )

tasks = []

for record in construction_records:
    l0_path = (
        PROJECT_ROOT
        / record[
            "l0_relative_path"
        ]
    )
    l1_path = (
        PROJECT_ROOT
        / record[
            "l1_relative_path"
        ]
    )

    if (
        not l0_path.exists()
        or not l1_path.exists()
    ):
        raise RuntimeError(
            "Missing pair file for "
            f"{record['op_id']}."
        )

    tasks.append(
        {
            "op_id": (
                record["op_id"]
            ),
            "pair_id": (
                record["pair_id"]
            ),
            "spec_id": (
                record["spec_id"]
            ),
            "family_id": (
                record["family_id"]
            ),
            "l0_path": str(l0_path),
            "l1_path": str(l1_path),
            "l0_sha256": (
                record["l0_sha256"]
            ),
            "l1_sha256": (
                record["l1_sha256"]
            ),
        }
    )

total_chunks = (
    EXPECTED_OPERATION_COUNT
    + CHECKPOINT_CHUNK_SIZE
    - 1
) // CHECKPOINT_CHUNK_SIZE

all_validation_records = []

reused_operation_count = 0
newly_validated_operation_count = 0

status_counts = Counter()
l0_status_counts = Counter()
l1_status_counts = Counter()

l0_warning_total = 0
l1_warning_total = 0

warning_examples = []
failure_examples = []

executor = None

try:
    for chunk_number in range(
        total_chunks
    ):
        start_index = (
            chunk_number
            * CHECKPOINT_CHUNK_SIZE
        )
        end_index = min(
            start_index
            + CHECKPOINT_CHUNK_SIZE,
            EXPECTED_OPERATION_COUNT,
        )

        chunk_tasks = tasks[
            start_index:end_index
        ]
        checkpoint_path = (
            checkpoint_path_for(
                chunk_number
            )
        )

        chunk_records = None

        if checkpoint_path.exists():
            try:
                candidate_records = (
                    read_checkpoint(
                        checkpoint_path
                    )
                )

                if checkpoint_is_reusable(
                    candidate_records,
                    chunk_tasks,
                ):
                    chunk_records = (
                        candidate_records
                    )
            except Exception:
                chunk_records = None

        if chunk_records is not None:
            reused_operation_count += len(
                chunk_records
            )

        else:
            if executor is None:
                executor = (
                    ProcessPoolExecutor(
                        max_workers=(
                            MAX_WORKERS
                        )
                    )
                )

            chunk_records = list(
                executor.map(
                    validate_pair_worker,
                    chunk_tasks,
                    chunksize=4,
                )
            )

            write_checkpoint(
                checkpoint_path,
                chunk_records,
            )

            newly_validated_operation_count += (
                len(chunk_records)
            )

        for record in chunk_records:
            status_counts[
                record["status"]
            ] += 1
            l0_status_counts[
                record["l0_status"]
            ] += 1
            l1_status_counts[
                record["l1_status"]
            ] += 1

            l0_warning_total += int(
                record[
                    "l0_warning_count"
                ]
            )
            l1_warning_total += int(
                record[
                    "l1_warning_count"
                ]
            )

            for warning_text in record[
                "warning_examples"
            ]:
                if (
                    warning_text
                    not in warning_examples
                    and len(
                        warning_examples
                    ) < 20
                ):
                    warning_examples.append(
                        warning_text
                    )

            if (
                record["status"]
                != "PASS"
                and len(
                    failure_examples
                ) < 20
            ):
                failure_examples.append(
                    {
                        "op_id": (
                            record["op_id"]
                        ),
                        "l0_status": (
                            record[
                                "l0_status"
                            ]
                        ),
                        "l1_status": (
                            record[
                                "l1_status"
                            ]
                        ),
                        "l0_diagnostic": (
                            record[
                                "l0_diagnostic"
                            ]
                        ),
                        "l1_diagnostic": (
                            record[
                                "l1_diagnostic"
                            ]
                        ),
                    }
                )

            all_validation_records.append(
                record
            )

        processed = end_index

        if (
            processed % 1000 == 0
            or processed
            == EXPECTED_OPERATION_COUNT
        ):
            print(
                json.dumps(
                    {
                        "validated_pairs": (
                            processed
                        ),
                        "pair_total": (
                            EXPECTED_OPERATION_COUNT
                        ),
                        "checkpoint_reused": (
                            reused_operation_count
                        ),
                        "newly_validated": (
                            newly_validated_operation_count
                        ),
                        "pair_pass_count": (
                            status_counts[
                                "PASS"
                            ]
                        ),
                        "pair_problem_count": (
                            status_counts[
                                "BLOCKED"
                            ]
                        ),
                        "workers": (
                            MAX_WORKERS
                        ),
                        "elapsed_seconds": round(
                            time.time()
                            - CELL11_STARTED,
                            1,
                        ),
                    }
                )
            )

finally:
    if executor is not None:
        executor.shutdown(
            wait=True,
            cancel_futures=False,
        )

all_validation_records.sort(
    key=lambda record: record["op_id"]
)

temporary_ledger_path = (
    VALIDATION_LEDGER_PATH.with_name(
        VALIDATION_LEDGER_PATH.name
        + ".tmp"
    )
)

with temporary_ledger_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    fieldnames = [
        "op_id",
        "pair_id",
        "spec_id",
        "family_id",
        "l0_sha256",
        "l1_sha256",
        "l0_status",
        "l1_status",
        "l0_warning_count",
        "l1_warning_count",
        "l0_diagnostic",
        "l1_diagnostic",
        "status",
        "validation_version",
    ]

    writer = csv.DictWriter(
        handle,
        fieldnames=fieldnames,
    )
    writer.writeheader()

    for record in all_validation_records:
        writer.writerow(
            {
                field_name: record.get(
                    field_name
                )
                for field_name
                in fieldnames
            }
        )

    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_ledger_path,
    VALIDATION_LEDGER_PATH,
)

checks = {
    "validation_record_count_matches_stage02_candidate_count": (
        len(all_validation_records)
        == EXPECTED_OPERATION_COUNT
    ),
    "unique_operation_ids": (
        len(
            {
                record["op_id"]
                for record
                in all_validation_records
            }
        )
        == EXPECTED_OPERATION_COUNT
    ),
    "all_l0_documents_validate": (
        l0_status_counts["PASS"]
        == EXPECTED_OPERATION_COUNT
        and l0_status_counts["FAIL"]
        == 0
    ),
    "all_l1_documents_validate": (
        l1_status_counts["PASS"]
        == EXPECTED_OPERATION_COUNT
        and l1_status_counts["FAIL"]
        == 0
    ),
    "all_pairs_validate": (
        status_counts["PASS"]
        == EXPECTED_OPERATION_COUNT
        and status_counts["BLOCKED"]
        == 0
    ),
    "cell10_contract_audit_is_pass": (
        cell10_report.get("status")
        == "PASS"
    ),
    "cell10_hash_matches_current_policy": (
        observed_cell10_report_sha256
        == expected_cell10_report_sha256
    ),
    "validator_version_is_0_9_0": (
        version(
            "openapi-spec-validator"
        )
        == "0.9.0"
    ),
}

status = (
    "PASS"
    if all(checks.values())
    else "BLOCKED"
)

report = {
    "status": status,
    "stage": (
        "DIRECT_L0_L1_OPENAPI_VALIDATION"
    ),
    "validator": (
        "openapi-spec-validator"
    ),
    "validator_version": version(
        "openapi-spec-validator"
    ),
    "validation_version": (
        VALIDATION_VERSION
    ),
    "worker_count": (
        MAX_WORKERS
    ),
    "pair_count": (
        len(all_validation_records)
    ),
    "document_count": (
        2 * len(
            all_validation_records
        )
    ),
    "pair_status_counts": dict(
        sorted(
            status_counts.items()
        )
    ),
    "l0_status_counts": dict(
        sorted(
            l0_status_counts.items()
        )
    ),
    "l1_status_counts": dict(
        sorted(
            l1_status_counts.items()
        )
    ),
    "l0_warning_count": (
        l0_warning_total
    ),
    "l1_warning_count": (
        l1_warning_total
    ),
    "warning_examples": (
        warning_examples
    ),
    "failure_examples": (
        failure_examples
    ),
    "checkpoint_count": (
        total_chunks
    ),
    "checkpoint_chunk_size": (
        CHECKPOINT_CHUNK_SIZE
    ),
    "checkpoint_reused_operation_count": (
        reused_operation_count
    ),
    "newly_validated_operation_count": (
        newly_validated_operation_count
    ),
    "validation_ledger": {
        "path": str(
            VALIDATION_LEDGER_PATH
        ),
        "bytes": (
            VALIDATION_LEDGER_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                VALIDATION_LEDGER_PATH
            )
        ),
        "record_count": (
            len(
                all_validation_records
            )
        ),
    },
    "input_hashes": {
        "construction_records": (
            sha256_file(
                CONSTRUCTION_RECORDS_PATH
            )
        ),
        "pair_audit": (
            sha256_file(
                PAIR_AUDIT_PATH
            )
        ),
        "cell10_report_expected": (
            expected_cell10_report_sha256
        ),
        "cell10_report_observed": (
            observed_cell10_report_sha256
        ),
    },
    "checks": checks,
    "duration_seconds": round(
        time.time()
        - CELL11_STARTED,
        3,
    ),
    "interpretation": (
        "Both members of every L0/L1 pair "
        "were directly validated. A validation "
        "failure in either condition blocks the "
        "pair from later eligibility."
    ),
    "report_path": str(
        REPORT_PATH
    ),
}

atomic_write_json(
    REPORT_PATH,
    report,
)

report["report_sha256"] = (
    sha256_file(REPORT_PATH)
)

stage_03_policy = (
    current_stage_03_policy
)

stage_03_policy.setdefault(
    "l1_construction",
    {},
).update(
    {
        "openapi_validation": (
            status
        ),
        "validation_version": (
            VALIDATION_VERSION
        ),
        "validation_ledger_path": str(
            VALIDATION_LEDGER_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/"),
        "validation_ledger_sha256": (
            report[
                "validation_ledger"
            ]["sha256"]
        ),
        "validation_report_path": str(
            REPORT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/"),
        "validation_report_sha256": (
            report["report_sha256"]
        ),
    }
)

temporary_policy_path = (
    STAGE_03_POLICY_PATH.with_name(
        STAGE_03_POLICY_PATH.name
        + ".tmp"
    )
)

with temporary_policy_path.open(
    "w",
    encoding="utf-8",
    newline="\n",
) as handle:
    yaml.safe_dump(
        stage_03_policy,
        handle,
        sort_keys=False,
        allow_unicode=True,
    )
    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_policy_path,
    STAGE_03_POLICY_PATH,
)

report[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
    )
)

if status != "PASS":
    raise RuntimeError(
        "Direct L0/L1 OpenAPI validation "
        "failed. Review the failure examples."
    )

print(
    "\nNOTEBOOK 03 — CELL 11: PASS"
)

{"validated_pairs": 1000, "pair_total": 52922, "checkpoint_reused": 0, "newly_validated": 1000, "pair_pass_count": 1000, "pair_problem_count": 0, "workers": 4, "elapsed_seconds": 20.8}
{"validated_pairs": 2000, "pair_total": 52922, "checkpoint_reused": 0, "newly_validated": 2000, "pair_pass_count": 2000, "pair_problem_count": 0, "workers": 4, "elapsed_seconds": 37.7}
{"validated_pairs": 3000, "pair_total": 52922, "checkpoint_reused": 0, "newly_validated": 3000, "pair_pass_count": 3000, "pair_problem_count": 0, "workers": 4, "elapsed_seconds": 54.6}
{"validated_pairs": 4000, "pair_total": 52922, "checkpoint_reused": 0, "newly_validated": 4000, "pair_pass_count": 4000, "pair_problem_count": 0, "workers": 4, "elapsed_seconds": 71.5}
{"validated_pairs": 5000, "pair_total": 52922, "checkpoint_reused": 0, "newly_validated": 5000, "pair_pass_count": 5000, "pair_problem_count": 0, "workers": 4, "elapsed_seconds": 88.7}
{"validated_pairs": 6000, "pair_total": 52922, "checkpoint_reused": 0, "new

In [12]:
# Notebook 03 — Cell 12
# Reuse frozen L0 measurements and render/tokenize only the new L1 prompts.

from collections import Counter
from pathlib import Path
import csv
import gzip
import hashlib
import json
import os
import statistics
import time

import yaml
from transformers import AutoTokenizer


CELL12_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

CONSTRUCTION_RECORDS_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_construction_records.jsonl"
)
CELL11_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/pair_openapi_validation.json"
)
STAGE_02_MEASUREMENTS_PATH = (
    STAGE_02_PROJECT_ROOT
    / "data/audits/rendered_prompt_measurements.jsonl"
)
SYSTEM_PROMPT_PATH = (
    PROJECT_ROOT
    / "config/upstream/stage_00_system_prompt.txt"
)
USER_PROMPT_PATH = (
    PROJECT_ROOT
    / "config/upstream/stage_00_user_prompt.txt"
)
STUDY_CONFIG_PATH = (
    PROJECT_ROOT
    / "config/upstream/stage_00_study.yaml"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)

PAYLOAD_ROOT = (
    PROJECT_ROOT
    / "data/audits/l1_rendered_prompt_payload_chunks"
)
PAYLOAD_INDEX_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_rendered_prompt_payload_index.jsonl"
)
TOKEN_AUDIT_PATH = (
    PROJECT_ROOT
    / "data/audits/token_audit.csv"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/pair_prompt_token_measurement.json"
)

EXPECTED_PAIR_COUNT = expected_eligible_operation_count
INPUT_TOKEN_LIMIT = 5120
CHECKPOINT_CHUNK_SIZE = 250

MEASUREMENT_VERSION = (
    "stage03-reuse-l0-measure-l1-v2"
)
renderer_report_path = STAGE_02_PROJECT_ROOT / "reports/audit/rendered_prompt_measurement.json"
renderer_report_digest = next(a["sha256"] for a in read_json(STAGE_02_INTERNAL_MANIFEST_PATH)["artifacts"]
                              if a["path"] == "reports/audit/rendered_prompt_measurement.json")
if sha256_file(renderer_report_path) != renderer_report_digest:
    raise RuntimeError("The Stage 02 renderer report differs from the frozen manifest.")
EXPECTED_RENDERER_FINGERPRINT = read_json(renderer_report_path)["renderer_fingerprint"]
EXPECTED_TOKENIZER_REVISION = (
    "c03e6d358207e414f1eca0bb1891e29f1db0e242"
)

for directory in [
    PAYLOAD_ROOT,
    TOKEN_AUDIT_PATH.parent,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_text(value: str) -> str:
    return sha256_bytes(
        value.encode("utf-8")
    )


def sha256_file(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            chunk_size
        ):
            digest.update(block)

    return digest.hexdigest()


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )


def token_ids_sha256(token_ids) -> str:
    return sha256_text(
        json.dumps(
            [
                int(value)
                for value in token_ids
            ],
            separators=(",", ":"),
        )
    )


def read_jsonl(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}."
                )

            yield value


def write_json_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            value,
            handle,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_text_atomic(
    path: Path,
    text: str,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        handle.write(text)
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_yaml_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        yaml.safe_dump(
            value,
            handle,
            sort_keys=False,
            allow_unicode=True,
        )
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_gzip_jsonl_atomic(
    path: Path,
    records,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "wb"
    ) as raw_handle:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_handle,
            compresslevel=6,
            mtime=0,
        ) as gzip_handle:
            for record in records:
                gzip_handle.write(
                    (
                        canonical_json(record)
                        + "\n"
                    ).encode("utf-8")
                )

        raw_handle.flush()
        os.fsync(raw_handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def read_gzip_jsonl(path: Path):
    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object gzip JSONL record at "
                    f"{path}:{line_number}."
                )

            yield value


required_paths = [
    CONSTRUCTION_RECORDS_PATH,
    CELL11_REPORT_PATH,
    STAGE_02_MEASUREMENTS_PATH,
    SYSTEM_PROMPT_PATH,
    USER_PROMPT_PATH,
    STUDY_CONFIG_PATH,
    STAGE_03_POLICY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Cell 12 required files are missing:\n"
        + "\n".join(missing_paths)
    )

with STUDY_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    study_config = (
        yaml.safe_load(handle)
        or {}
    )

with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

with CELL11_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cell11_report = json.load(handle)

study_id = str(
    study_config.get(
        "study_id",
        "",
    )
)
model_id = str(
    study_config.get(
        "model_id",
        "",
    )
)
tokenizer_revision = str(
    study_config.get(
        "tokenizer_revision"
    )
    or ""
)
configured_limit = int(
    study_config.get(
        "max_rendered_input_tokens",
        -1,
    )
)

if study_id != "prose-operation-study":
    raise RuntimeError(
        f"Unexpected study_id: {study_id!r}."
    )

if (
    model_id
    != "Qwen/Qwen2.5-Coder-7B-Instruct"
):
    raise RuntimeError(
        f"Unexpected model_id: {model_id!r}."
    )

if (
    tokenizer_revision
    != EXPECTED_TOKENIZER_REVISION
):
    raise RuntimeError(
        "Unexpected tokenizer revision: "
        f"{tokenizer_revision!r}."
    )

if configured_limit != INPUT_TOKEN_LIMIT:
    raise RuntimeError(
        "Frozen input-token limit is not "
        "5,120."
    )

expected_cell11_hash = (
    stage_03_policy
    .get(
        "l1_construction",
        {},
    )
    .get(
        "validation_report_sha256"
    )
)
observed_cell11_hash = sha256_file(
    CELL11_REPORT_PATH
)

if (
    not isinstance(
        expected_cell11_hash,
        str,
    )
    or len(expected_cell11_hash) != 64
    or (
        expected_cell11_hash
        != observed_cell11_hash
    )
):
    raise RuntimeError(
        "Cell 11 report does not match the "
        "hash recorded in stage_03.yaml."
    )

if cell11_report.get("status") != "PASS":
    raise RuntimeError(
        "Cell 11 did not report PASS."
    )

if int(
    cell11_report.get(
        "pair_count",
        -1,
    )
) != EXPECTED_PAIR_COUNT:
    raise RuntimeError(
        "Cell 11 pair count is not "
        "the Stage 02 candidate count."
    )

if int(
    cell11_report
    .get(
        "pair_status_counts",
        {},
    )
    .get(
        "PASS",
        0,
    )
) != EXPECTED_PAIR_COUNT:
    raise RuntimeError(
        "Cell 11 did not validate every pair."
    )

system_prompt = (
    SYSTEM_PROMPT_PATH.read_text(
        encoding="utf-8"
    )
)
user_prompt_template = (
    USER_PROMPT_PATH.read_text(
        encoding="utf-8"
    )
)

if (
    "{BOUNDARY}"
    not in user_prompt_template
    or (
        "{CANONICAL_OPENAPI_JSON}"
        not in user_prompt_template
    )
):
    raise RuntimeError(
        "The frozen user prompt is missing "
        "its required placeholders."
    )

construction_records = sorted(
    read_jsonl(
        CONSTRUCTION_RECORDS_PATH
    ),
    key=lambda record: str(
        record["op_id"]
    ),
)

if (
    len(construction_records)
    != EXPECTED_PAIR_COUNT
):
    raise RuntimeError(
        "Unexpected construction-record "
        f"count: {len(construction_records)}."
    )

operation_ids = [
    str(record["op_id"])
    for record in construction_records
]
required_operation_ids = set(
    operation_ids
)

if (
    len(required_operation_ids)
    != EXPECTED_PAIR_COUNT
):
    raise RuntimeError(
        "Construction operation IDs are "
        "not unique."
    )

if any(
    record.get("status") != "PASS"
    for record in construction_records
):
    raise RuntimeError(
        "Cell 9 contains a non-PASS "
        "construction record."
    )

l0_measurements = {}
renderer_fingerprints = set()

for record in read_jsonl(
    STAGE_02_MEASUREMENTS_PATH
):
    op_id = str(
        record.get(
            "op_id",
            "",
        )
    )

    if op_id not in required_operation_ids:
        continue

    if op_id in l0_measurements:
        raise RuntimeError(
            "Duplicate Stage 02 prompt "
            f"measurement for {op_id}."
        )

    l0_measurements[op_id] = record

    renderer_fingerprints.add(
        str(
            record.get(
                "renderer_fingerprint"
            )
        )
    )

if (
    set(l0_measurements)
    != required_operation_ids
):
    missing_measurements = sorted(
        required_operation_ids
        - set(l0_measurements)
    )

    raise RuntimeError(
        "Missing Stage 02 L0 prompt "
        "measurements: "
        + ", ".join(
            missing_measurements[:20]
        )
    )

if renderer_fingerprints != {
    EXPECTED_RENDERER_FINGERPRINT
}:
    raise RuntimeError(
        "Unexpected Stage 02 renderer "
        "fingerprints: "
        f"{sorted(renderer_fingerprints)}"
    )

l0_linkage_problems = []

for source_record in construction_records:
    op_id = str(
        source_record["op_id"]
    )
    prior = l0_measurements[op_id]

    expected_boundary = hashlib.sha256(
        (
            f"{study_id}|{op_id}"
        ).encode("utf-8")
    ).hexdigest()[:24]

    if str(
        prior.get("slice_sha256")
    ) != str(
        source_record["l0_sha256"]
    ):
        l0_linkage_problems.append(
            {
                "op_id": op_id,
                "problem": (
                    "L0_SLICE_HASH_MISMATCH"
                ),
            }
        )

    if str(
        prior.get("boundary")
    ) != expected_boundary:
        l0_linkage_problems.append(
            {
                "op_id": op_id,
                "problem": (
                    "L0_BOUNDARY_MISMATCH"
                ),
            }
        )

    if (
        not bool(
            prior.get(
                "within_input_limit"
            )
        )
        or int(
            prior.get(
                "rendered_prompt_tokens",
                -1,
            )
        ) > INPUT_TOKEN_LIMIT
    ):
        l0_linkage_problems.append(
            {
                "op_id": op_id,
                "problem": (
                    "L0_NOT_WITHIN_LIMIT"
                ),
            }
        )

if l0_linkage_problems:
    raise RuntimeError(
        "Frozen L0 measurement linkage "
        "failed: "
        + json.dumps(
            l0_linkage_problems[:20],
            ensure_ascii=False,
        )
    )

hf_token = os.environ.get(
    "HF_TOKEN"
)

if not hf_token:
    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )

        hf_token = (
            UserSecretsClient()
            .get_secret("HF_TOKEN")
        )

    except Exception:
        hf_token = None

tokenizer_load_started = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=tokenizer_revision,
    token=hf_token,
    trust_remote_code=False,
)

tokenizer_load_duration = (
    time.time()
    - tokenizer_load_started
)

if not tokenizer.chat_template:
    raise RuntimeError(
        "The frozen tokenizer has no "
        "chat template."
    )


def render_prompt(
    op_id: str,
    canonical_document_json: str,
):
    boundary = hashlib.sha256(
        (
            f"{study_id}|{op_id}"
        ).encode("utf-8")
    ).hexdigest()[:24]

    if boundary in canonical_document_json:
        raise RuntimeError(
            "Prompt boundary occurs inside "
            "canonical JSON for "
            f"{op_id}."
        )

    user_prompt = (
        user_prompt_template
        .replace(
            "{BOUNDARY}",
            boundary,
        )
        .replace(
            "{CANONICAL_OPENAPI_JSON}",
            canonical_document_json,
        )
    )

    rendered_prompt = (
        tokenizer.apply_chat_template(
            [
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    if not isinstance(
        rendered_prompt,
        str,
    ):
        raise RuntimeError(
            "Tokenizer chat rendering did "
            "not return a string."
        )

    return (
        boundary,
        rendered_prompt,
    )


qualification_source = (
    construction_records[0]
)
qualification_op_id = str(
    qualification_source["op_id"]
)
qualification_l0_path = (
    PROJECT_ROOT
    / qualification_source[
        "l0_relative_path"
    ]
)
qualification_l0_bytes = (
    qualification_l0_path.read_bytes()
)

if sha256_bytes(
    qualification_l0_bytes
) != str(
    qualification_source["l0_sha256"]
):
    raise RuntimeError(
        "Renderer qualification L0 hash "
        "mismatch."
    )

qualification_l0_document = json.loads(
    qualification_l0_bytes.decode(
        "utf-8"
    )
)

(
    qualification_boundary,
    qualification_prompt,
) = render_prompt(
    qualification_op_id,
    canonical_json(
        qualification_l0_document
    ),
)

qualification_token_ids = [
    int(value)
    for value in tokenizer(
        qualification_prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]
]

qualification_prior = (
    l0_measurements[
        qualification_op_id
    ]
)

renderer_qualification_checks = {
    "boundary": (
        qualification_boundary
        == qualification_prior.get(
            "boundary"
        )
    ),
    "rendered_prompt_sha256": (
        sha256_text(
            qualification_prompt
        )
        == qualification_prior.get(
            "rendered_prompt_sha256"
        )
    ),
    "token_ids_sha256": (
        token_ids_sha256(
            qualification_token_ids
        )
        == qualification_prior.get(
            "token_ids_sha256"
        )
    ),
    "token_count": (
        len(qualification_token_ids)
        == int(
            qualification_prior.get(
                "rendered_prompt_tokens",
                -1,
            )
        )
    ),
}

if not all(
    renderer_qualification_checks.values()
):
    raise RuntimeError(
        "Current tokenizer renderer does "
        "not reproduce the frozen Stage 02 "
        "prompt: "
        + json.dumps(
            renderer_qualification_checks,
            sort_keys=True,
        )
    )

measurement_context = {
    "measurement_version": (
        MEASUREMENT_VERSION
    ),
    "study_id": study_id,
    "model_id": model_id,
    "tokenizer_revision": (
        tokenizer_revision
    ),
    "input_token_limit": (
        INPUT_TOKEN_LIMIT
    ),
    "renderer_fingerprint": (
        EXPECTED_RENDERER_FINGERPRINT
    ),
    "system_prompt_sha256": (
        sha256_file(
            SYSTEM_PROMPT_PATH
        )
    ),
    "user_prompt_sha256": (
        sha256_file(
            USER_PROMPT_PATH
        )
    ),
    "construction_records_sha256": (
        sha256_file(
            CONSTRUCTION_RECORDS_PATH
        )
    ),
    "cell11_report_sha256": (
        observed_cell11_hash
    ),
    "stage_02_measurements_sha256": (
        sha256_file(
            STAGE_02_MEASUREMENTS_PATH
        )
    ),
}

measurement_context_sha256 = (
    sha256_text(
        canonical_json(
            measurement_context
        )
    )
)


def checkpoint_is_reusable(
    records,
    expected_records,
) -> bool:
    if len(records) != len(
        expected_records
    ):
        return False

    expected_by_op_id = {
        str(record["op_id"]): record
        for record in expected_records
    }

    if {
        str(record.get("op_id"))
        for record in records
    } != set(expected_by_op_id):
        return False

    for record in records:
        op_id = str(
            record.get("op_id")
        )
        expected = (
            expected_by_op_id[op_id]
        )

        if record.get(
            "measurement_context_sha256"
        ) != measurement_context_sha256:
            return False

        if record.get(
            "l0_slice_sha256"
        ) != str(
            expected["l0_sha256"]
        ):
            return False

        if record.get(
            "l1_slice_sha256"
        ) != str(
            expected["l1_sha256"]
        ):
            return False

        if sha256_text(
            str(
                record.get(
                    "l1_rendered_prompt",
                    "",
                )
            )
        ) != record.get(
            "l1_rendered_prompt_sha256"
        ):
            return False

        if token_ids_sha256(
            record.get(
                "l1_token_ids",
                [],
            )
        ) != record.get(
            "l1_token_ids_sha256"
        ):
            return False

        if len(
            record.get(
                "l1_token_ids",
                [],
            )
        ) != int(
            record.get(
                "l1_input_tokens",
                -1,
            )
        ):
            return False

    return True


chunk_count = (
    EXPECTED_PAIR_COUNT
    + CHECKPOINT_CHUNK_SIZE
    - 1
) // CHECKPOINT_CHUNK_SIZE

checkpoint_reused = 0
newly_measured = 0
payload_index_records = []

for chunk_number in range(
    chunk_count
):
    start_index = (
        chunk_number
        * CHECKPOINT_CHUNK_SIZE
    )
    end_index = min(
        start_index
        + CHECKPOINT_CHUNK_SIZE,
        EXPECTED_PAIR_COUNT,
    )

    source_chunk = (
        construction_records[
            start_index:end_index
        ]
    )

    payload_path = (
        PAYLOAD_ROOT
        / (
            f"chunk_"
            f"{chunk_number:05d}"
            f".jsonl.gz"
        )
    )

    chunk_records = None

    if payload_path.is_file():
        try:
            candidate_records = list(
                read_gzip_jsonl(
                    payload_path
                )
            )

            if checkpoint_is_reusable(
                candidate_records,
                source_chunk,
            ):
                chunk_records = (
                    candidate_records
                )

        except Exception:
            chunk_records = None

    if chunk_records is not None:
        checkpoint_reused += len(
            chunk_records
        )

    else:
        prepared = []
        rendered_l1_batch = []

        for source_record in source_chunk:
            op_id = str(
                source_record["op_id"]
            )
            l1_path = (
                PROJECT_ROOT
                / source_record[
                    "l1_relative_path"
                ]
            )

            if not l1_path.is_file():
                raise RuntimeError(
                    "Missing L1 slice for "
                    f"{op_id}."
                )

            l1_bytes = (
                l1_path.read_bytes()
            )

            if sha256_bytes(
                l1_bytes
            ) != str(
                source_record[
                    "l1_sha256"
                ]
            ):
                raise RuntimeError(
                    "L1 slice hash mismatch "
                    f"for {op_id}."
                )

            l1_document = json.loads(
                l1_bytes.decode("utf-8")
            )

            (
                boundary,
                rendered_l1,
            ) = render_prompt(
                op_id,
                canonical_json(
                    l1_document
                ),
            )

            prior = (
                l0_measurements[op_id]
            )

            if boundary != str(
                prior["boundary"]
            ):
                raise RuntimeError(
                    "L0/L1 boundaries differ "
                    f"for {op_id}."
                )

            prepared.append(
                {
                    "source_record": (
                        source_record
                    ),
                    "prior": prior,
                    "boundary": boundary,
                    "rendered_l1": (
                        rendered_l1
                    ),
                }
            )
            rendered_l1_batch.append(
                rendered_l1
            )

        batch_l1_token_ids = tokenizer(
            rendered_l1_batch,
            add_special_tokens=False,
            padding=False,
            truncation=False,
        )["input_ids"]

        if (
            len(batch_l1_token_ids)
            != len(prepared)
        ):
            raise RuntimeError(
                "Tokenizer batch size "
                "mismatch."
            )

        chunk_records = []

        for (
            pair_index,
            prepared_record,
        ) in enumerate(prepared):
            source_record = (
                prepared_record[
                    "source_record"
                ]
            )
            prior = (
                prepared_record["prior"]
            )
            op_id = str(
                source_record["op_id"]
            )
            rendered_l1 = (
                prepared_record[
                    "rendered_l1"
                ]
            )

            l1_token_ids = [
                int(value)
                for value in (
                    batch_l1_token_ids[
                        pair_index
                    ]
                )
            ]

            l0_tokens = int(
                prior[
                    "rendered_prompt_tokens"
                ]
            )
            l1_tokens = len(
                l1_token_ids
            )
            l0_within_limit = bool(
                prior[
                    "within_input_limit"
                ]
            )
            l1_within_limit = (
                l1_tokens
                <= INPUT_TOKEN_LIMIT
            )
            pair_within_limit = (
                l0_within_limit
                and l1_within_limit
            )

            chunk_records.append(
                {
                    "measurement_version": (
                        MEASUREMENT_VERSION
                    ),
                    "measurement_context_sha256": (
                        measurement_context_sha256
                    ),
                    "renderer_fingerprint": (
                        EXPECTED_RENDERER_FINGERPRINT
                    ),
                    "op_id": op_id,
                    "pair_id": (
                        source_record[
                            "pair_id"
                        ]
                    ),
                    "spec_id": (
                        source_record.get(
                            "spec_id"
                        )
                    ),
                    "family_id": (
                        source_record.get(
                            "family_id"
                        )
                    ),
                    "source_class": (
                        source_record.get(
                            "source_class"
                        )
                    ),
                    "method": (
                        source_record.get(
                            "method"
                        )
                    ),
                    "path": (
                        source_record.get(
                            "path"
                        )
                    ),
                    "boundary": (
                        prepared_record[
                            "boundary"
                        ]
                    ),
                    "l0_measurement_source": (
                        "verified_stage_02_"
                        "measurement"
                    ),
                    "l0_slice_sha256": (
                        source_record[
                            "l0_sha256"
                        ]
                    ),
                    "l1_slice_sha256": (
                        source_record[
                            "l1_sha256"
                        ]
                    ),
                    "l0_rendered_prompt_bytes": (
                        int(
                            prior[
                                "rendered_prompt_bytes"
                            ]
                        )
                    ),
                    "l1_rendered_prompt_bytes": (
                        len(
                            rendered_l1.encode(
                                "utf-8"
                            )
                        )
                    ),
                    "l0_rendered_prompt_sha256": (
                        prior[
                            "rendered_prompt_sha256"
                        ]
                    ),
                    "l1_rendered_prompt_sha256": (
                        sha256_text(
                            rendered_l1
                        )
                    ),
                    "l0_token_ids_sha256": (
                        prior[
                            "token_ids_sha256"
                        ]
                    ),
                    "l1_token_ids_sha256": (
                        token_ids_sha256(
                            l1_token_ids
                        )
                    ),
                    "l0_input_tokens": (
                        l0_tokens
                    ),
                    "l1_input_tokens": (
                        l1_tokens
                    ),
                    "token_delta_l1_minus_l0": (
                        l1_tokens
                        - l0_tokens
                    ),
                    "input_token_limit": (
                        INPUT_TOKEN_LIMIT
                    ),
                    "l0_within_limit": (
                        l0_within_limit
                    ),
                    "l1_within_limit": (
                        l1_within_limit
                    ),
                    "pair_within_limit": (
                        pair_within_limit
                    ),
                    "status": (
                        "PASS_WITHIN_INPUT_LIMIT"
                        if pair_within_limit
                        else (
                            "EXCLUDE_PROMPT_LIMIT"
                        )
                    ),
                    "l1_encoding": "utf-8",
                    "l1_rendered_prompt": (
                        rendered_l1
                    ),
                    "l1_token_ids": (
                        l1_token_ids
                    ),
                }
            )

        write_gzip_jsonl_atomic(
            payload_path,
            chunk_records,
        )

        newly_measured += len(
            chunk_records
        )

    payload_index_records.append(
        {
            "chunk_number": (
                chunk_number
            ),
            "measurement_context_sha256": (
                measurement_context_sha256
            ),
            "operation_count": len(
                chunk_records
            ),
            "operation_ids_sha256": (
                sha256_text(
                    canonical_json(
                        [
                            str(
                                record[
                                    "op_id"
                                ]
                            )
                            for record
                            in chunk_records
                        ]
                    )
                )
            ),
            "payload_relative_path": (
                str(
                    payload_path.relative_to(
                        PROJECT_ROOT
                    )
                ).replace("\\", "/")
            ),
            "payload_bytes": (
                payload_path
                .stat()
                .st_size
            ),
            "payload_sha256": (
                sha256_file(
                    payload_path
                )
            ),
        }
    )

    processed = end_index

    if (
        processed % 1000 == 0
        or processed
        == EXPECTED_PAIR_COUNT
    ):
        print(
            json.dumps(
                {
                    "measured_pairs": (
                        processed
                    ),
                    "pair_total": (
                        EXPECTED_PAIR_COUNT
                    ),
                    "l0_measurements_reused": (
                        processed
                    ),
                    "l1_checkpoint_reused": (
                        checkpoint_reused
                    ),
                    "l1_newly_measured": (
                        newly_measured
                    ),
                    "elapsed_seconds": (
                        round(
                            time.time()
                            - CELL12_STARTED,
                            1,
                        )
                    ),
                }
            )
        )

payload_index_text = "".join(
    canonical_json(record) + "\n"
    for record
    in payload_index_records
)

write_text_atomic(
    PAYLOAD_INDEX_PATH,
    payload_index_text,
)

audit_fieldnames = [
    "op_id",
    "pair_id",
    "spec_id",
    "family_id",
    "source_class",
    "method",
    "path",
    "boundary",
    "l0_measurement_source",
    "l0_slice_sha256",
    "l1_slice_sha256",
    "l0_rendered_prompt_bytes",
    "l1_rendered_prompt_bytes",
    "l0_rendered_prompt_sha256",
    "l1_rendered_prompt_sha256",
    "l0_token_ids_sha256",
    "l1_token_ids_sha256",
    "l0_input_tokens",
    "l1_input_tokens",
    "token_delta_l1_minus_l0",
    "input_token_limit",
    "l0_within_limit",
    "l1_within_limit",
    "pair_within_limit",
    "status",
]

temporary_audit_path = (
    TOKEN_AUDIT_PATH.with_name(
        TOKEN_AUDIT_PATH.name
        + ".tmp"
    )
)

status_counts = Counter()
l0_over_limit_count = 0
l1_over_limit_count = 0
token_deltas = []
audit_row_count = 0
observed_operation_ids = set()

with temporary_audit_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=audit_fieldnames,
        lineterminator="\n",
    )
    writer.writeheader()

    for index_record in (
        payload_index_records
    ):
        payload_path = (
            PROJECT_ROOT
            / index_record[
                "payload_relative_path"
            ]
        )

        if sha256_file(
            payload_path
        ) != str(
            index_record[
                "payload_sha256"
            ]
        ):
            raise RuntimeError(
                "Payload hash changed before "
                "audit materialization: "
                f"{payload_path}."
            )

        for record in read_gzip_jsonl(
            payload_path
        ):
            op_id = str(
                record["op_id"]
            )

            if (
                op_id
                in observed_operation_ids
            ):
                raise RuntimeError(
                    "Duplicate token-audit "
                    f"op_id: {op_id}."
                )

            observed_operation_ids.add(
                op_id
            )
            audit_row_count += 1

            status_counts[
                str(record["status"])
            ] += 1

            l0_over_limit_count += int(
                not bool(
                    record[
                        "l0_within_limit"
                    ]
                )
            )
            l1_over_limit_count += int(
                not bool(
                    record[
                        "l1_within_limit"
                    ]
                )
            )

            token_deltas.append(
                int(
                    record[
                        "token_delta_l1_minus_l0"
                    ]
                )
            )

            writer.writerow(
                {
                    field: record.get(
                        field
                    )
                    for field
                    in audit_fieldnames
                }
            )

    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_audit_path,
    TOKEN_AUDIT_PATH,
)

pair_within_limit_count = (
    status_counts[
        "PASS_WITHIN_INPUT_LIMIT"
    ]
)
pair_excluded_count = (
    status_counts[
        "EXCLUDE_PROMPT_LIMIT"
    ]
)

checks = {
    "cell11_report_hash_matches_policy": (
        expected_cell11_hash
        == observed_cell11_hash
    ),
    "renderer_qualification_passes": (
        all(
            renderer_qualification_checks
            .values()
        )
    ),
    "all_l0_measurements_linked_by_slice_hash": (
        len(l0_linkage_problems)
        == 0
    ),
    "audit_row_count_matches_stage02_candidate_count": (
        audit_row_count
        == EXPECTED_PAIR_COUNT
    ),
    "operation_ids_match_construction": (
        observed_operation_ids
        == required_operation_ids
    ),
    "all_l0_prompts_remain_within_limit": (
        l0_over_limit_count
        == 0
    ),
    "pair_statuses_account_for_population": (
        pair_within_limit_count
        + pair_excluded_count
        == EXPECTED_PAIR_COUNT
    ),
    "payload_chunk_count_matches": (
        len(payload_index_records)
        == chunk_count
    ),
    "only_l1_was_newly_rendered_and_tokenized": (
        True
    ),
    "no_truncation_or_summarization": (
        True
    ),
}

if not all(checks.values()):
    raise RuntimeError(
        "Cell 12 prompt-measurement checks "
        "failed: "
        + json.dumps(
            checks,
            sort_keys=True,
        )
    )

report = {
    "status": "PASS",
    "stage": (
        "REUSED_L0_AND_MEASURED_L1_PROMPTS"
    ),
    "measurement_version": (
        MEASUREMENT_VERSION
    ),
    "model_id": model_id,
    "tokenizer_revision": (
        tokenizer_revision
    ),
    "tokenizer_class": (
        type(tokenizer).__name__
    ),
    "renderer_fingerprint": (
        EXPECTED_RENDERER_FINGERPRINT
    ),
    "renderer_qualification": {
        "op_id": (
            qualification_op_id
        ),
        "checks": (
            renderer_qualification_checks
        ),
    },
    "measurement_context_sha256": (
        measurement_context_sha256
    ),
    "pair_count": (
        audit_row_count
    ),
    "condition_prompt_count": (
        2 * audit_row_count
    ),
    "l0_measurement_reused_count": (
        audit_row_count
    ),
    "l1_measurement_created_count": (
        audit_row_count
    ),
    "input_token_limit": (
        INPUT_TOKEN_LIMIT
    ),
    "pair_status_counts": dict(
        sorted(
            status_counts.items()
        )
    ),
    "pair_within_limit_count": (
        pair_within_limit_count
    ),
    "pair_excluded_prompt_limit_count": (
        pair_excluded_count
    ),
    "l0_over_limit_count": (
        l0_over_limit_count
    ),
    "l1_over_limit_count": (
        l1_over_limit_count
    ),
    "token_delta_l1_minus_l0": {
        "minimum": min(
            token_deltas
        ),
        "median": (
            statistics.median(
                token_deltas
            )
        ),
        "maximum": max(
            token_deltas
        ),
        "mean": round(
            statistics.fmean(
                token_deltas
            ),
            6,
        ),
    },
    "checkpoint_chunk_size": (
        CHECKPOINT_CHUNK_SIZE
    ),
    "checkpoint_count": len(
        payload_index_records
    ),
    "checkpoint_reused_pair_count": (
        checkpoint_reused
    ),
    "newly_measured_l1_pair_count": (
        newly_measured
    ),
    "tokenizer_load_seconds": round(
        tokenizer_load_duration,
        3,
    ),
    "token_audit": {
        "path": str(
            TOKEN_AUDIT_PATH
        ),
        "bytes": (
            TOKEN_AUDIT_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                TOKEN_AUDIT_PATH
            )
        ),
        "record_count": (
            audit_row_count
        ),
    },
    "l1_prompt_payload_index": {
        "path": str(
            PAYLOAD_INDEX_PATH
        ),
        "bytes": (
            PAYLOAD_INDEX_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            sha256_file(
                PAYLOAD_INDEX_PATH
            )
        ),
        "chunk_count": len(
            payload_index_records
        ),
    },
    "input_hashes": (
        measurement_context
    ),
    "checks": checks,
    "duration_seconds": round(
        time.time()
        - CELL12_STARTED,
        3,
    ),
    "interpretation": (
        "The already verified Stage 02 L0 "
        "token measurements were joined by "
        "operation and slice hash. Only L1 "
        "prompts were newly rendered and "
        "tokenized. A pair remains prompt-"
        "feasible only when both conditions "
        "contain at most 5,120 tokens."
    ),
    "report_path": str(
        REPORT_PATH
    ),
}

write_json_atomic(
    REPORT_PATH,
    report,
)

report_sha256 = sha256_file(
    REPORT_PATH
)

stage_03_policy.setdefault(
    "prompt_eligibility",
    {},
).update(
    {
        "status": "PASS",
        "measurement_version": (
            MEASUREMENT_VERSION
        ),
        "l0_measurement_source": (
            "verified_stage_02_measurements"
        ),
        "l1_measurement_source": (
            "rendered_in_notebook_03_cell_12"
        ),
        "model_id": model_id,
        "tokenizer_revision": (
            tokenizer_revision
        ),
        "renderer_fingerprint": (
            EXPECTED_RENDERER_FINGERPRINT
        ),
        "maximum_rendered_input_tokens": (
            INPUT_TOKEN_LIMIT
        ),
        "both_conditions_must_fit": (
            True
        ),
        "truncation_allowed": (
            False
        ),
        "summarization_allowed": (
            False
        ),
        "pair_count": (
            audit_row_count
        ),
        "pair_within_limit_count": (
            pair_within_limit_count
        ),
        "pair_excluded_prompt_limit_count": (
            pair_excluded_count
        ),
        "token_audit_path": (
            str(
                TOKEN_AUDIT_PATH
                .relative_to(
                    PROJECT_ROOT
                )
            ).replace("\\", "/")
        ),
        "token_audit_sha256": (
            sha256_file(
                TOKEN_AUDIT_PATH
            )
        ),
        "l1_payload_index_path": (
            str(
                PAYLOAD_INDEX_PATH
                .relative_to(
                    PROJECT_ROOT
                )
            ).replace("\\", "/")
        ),
        "l1_payload_index_sha256": (
            sha256_file(
                PAYLOAD_INDEX_PATH
            )
        ),
        "report_path": (
            str(
                REPORT_PATH.relative_to(
                    PROJECT_ROOT
                )
            ).replace("\\", "/")
        ),
        "report_sha256": (
            report_sha256
        ),
    }
)

write_yaml_atomic(
    STAGE_03_POLICY_PATH,
    stage_03_policy,
)

output = dict(report)
output["report_sha256"] = (
    report_sha256
)
output[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

print(
    "\nNOTEBOOK 03 — CELL 12: PASS"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{"measured_pairs": 1000, "pair_total": 52922, "l0_measurements_reused": 1000, "l1_checkpoint_reused": 0, "l1_newly_measured": 1000, "elapsed_seconds": 11.3}
{"measured_pairs": 2000, "pair_total": 52922, "l0_measurements_reused": 2000, "l1_checkpoint_reused": 0, "l1_newly_measured": 2000, "elapsed_seconds": 15.8}
{"measured_pairs": 3000, "pair_total": 52922, "l0_measurements_reused": 3000, "l1_checkpoint_reused": 0, "l1_newly_measured": 3000, "elapsed_seconds": 20.3}
{"measured_pairs": 4000, "pair_total": 52922, "l0_measurements_reused": 4000, "l1_checkpoint_reused": 0, "l1_newly_measured": 4000, "elapsed_seconds": 25.0}
{"measured_pairs": 5000, "pair_total": 52922, "l0_measurements_reused": 5000, "l1_checkpoint_reused": 0, "l1_newly_measured": 5000, "elapsed_seconds": 29.5}
{"measured_pairs": 6000, "pair_total": 52922, "l0_measurements_reused": 6000, "l1_checkpoint_reused": 0, "l1_newly_measured": 6000, "elapsed_seconds": 34.3}
{"measured_pairs": 7000, "pair_total": 52922, "l0_measurem

In [13]:
# Notebook 03 — Cell 13
# Apply operation-level prose eligibility and audit placeholder/language flags.

from collections import Counter
from itertools import groupby
from pathlib import Path
import csv
import hashlib
import json
import os
import re
import time

import yaml


CELL13_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

PAIR_FRAME_PATH = (
    PROJECT_ROOT
    / "data/inventory/pair_frame_seed.jsonl"
)
CONSTRUCTION_RECORDS_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_construction_records.jsonl"
)
OBFUSCATION_LOG_PATH = (
    PROJECT_ROOT
    / "data/audits/obfuscation_log.jsonl"
)
TOKEN_AUDIT_PATH = (
    PROJECT_ROOT
    / "data/audits/token_audit.csv"
)
CELL12_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/pair_prompt_token_measurement.json"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)
ELIGIBILITY_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "operation_language_prose_eligibility.csv"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "operation_language_prose_eligibility.json"
)

EXPECTED_OPERATION_COUNT = expected_eligible_operation_count

PLACEHOLDER_AUDIT_VERSION = (
    "exact-placeholder-audit-v1"
)
ELIGIBILITY_VERSION = (
    "stage03-operation-prose-eligibility-v1"
)

# Audit-only list. Placeholder flags do not cause
# exclusion because no exclusion rule was frozen.
PLACEHOLDER_EXACT_VALUES = {
    "description",
    "summary",
    "placeholder",
    "todo",
    "tbd",
    "n/a",
    "na",
    "none",
    "null",
    "not available",
    "not applicable",
    "not documented",
    "undocumented",
    "coming soon",
    "to be defined",
    "to be determined",
    "to be documented",
    "insert description",
    "insert summary",
    "add description",
    "add summary",
}

for directory in [
    ELIGIBILITY_LEDGER_PATH.parent,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            chunk_size
        ):
            digest.update(block)

    return digest.hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def read_jsonl(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}."
                )

            yield value


def write_json_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            value,
            handle,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_yaml_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        yaml.safe_dump(
            value,
            handle,
            sort_keys=False,
            allow_unicode=True,
        )
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def parse_boolean(value) -> bool:
    if isinstance(value, bool):
        return value

    normalized = str(
        value
    ).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
    }:
        return False

    raise ValueError(
        "Cannot parse boolean value: "
        f"{value!r}"
    )


def decode_json_pointer_token(
    token: str,
) -> str:
    return (
        token
        .replace("~1", "/")
        .replace("~0", "~")
    )


def resolve_json_pointer(
    document,
    pointer: str,
):
    if pointer == "":
        return document

    if not pointer.startswith("/"):
        raise ValueError(
            "JSON Pointer must start with '/': "
            f"{pointer!r}"
        )

    current = document

    for encoded_token in pointer.split("/")[1:]:
        token = decode_json_pointer_token(
            encoded_token
        )

        if isinstance(current, dict):
            if token not in current:
                raise KeyError(
                    "Missing mapping token "
                    f"{token!r} while resolving "
                    f"{pointer!r}."
                )

            current = current[token]

        elif isinstance(current, list):
            if not token.isdigit():
                raise TypeError(
                    "List JSON Pointer token is "
                    f"not an integer: {token!r}."
                )

            index = int(token)

            if (
                index < 0
                or index >= len(current)
            ):
                raise IndexError(
                    "List JSON Pointer index is "
                    f"out of range: {index}."
                )

            current = current[index]

        else:
            raise TypeError(
                "JSON Pointer traversed through "
                f"{type(current).__name__}."
            )

    return current


def normalize_placeholder_text(
    value: str,
) -> str:
    normalized = (
        value.casefold().strip()
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    return normalized.strip(
        " \t\r\n"
        ".,:;!?-_"
        "()[]{}<>"
        "\"'`"
    )


def is_exact_placeholder(
    value: str,
) -> bool:
    return (
        normalize_placeholder_text(value)
        in PLACEHOLDER_EXACT_VALUES
    )


required_paths = [
    PAIR_FRAME_PATH,
    CONSTRUCTION_RECORDS_PATH,
    OBFUSCATION_LOG_PATH,
    TOKEN_AUDIT_PATH,
    CELL12_REPORT_PATH,
    STAGE_03_POLICY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Cell 13 required files are missing:\n"
        + "\n".join(missing_paths)
    )

with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

with CELL12_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cell12_report = json.load(handle)

expected_cell12_hash = (
    stage_03_policy
    .get(
        "prompt_eligibility",
        {},
    )
    .get("report_sha256")
)
observed_cell12_hash = sha256_file(
    CELL12_REPORT_PATH
)

if (
    not isinstance(
        expected_cell12_hash,
        str,
    )
    or len(expected_cell12_hash) != 64
    or (
        expected_cell12_hash
        != observed_cell12_hash
    )
):
    raise RuntimeError(
        "Cell 12 report does not match the "
        "hash recorded in stage_03.yaml."
    )

if cell12_report.get("status") != "PASS":
    raise RuntimeError(
        "Cell 12 did not report PASS."
    )

if int(
    cell12_report.get(
        "pair_count",
        -1,
    )
) != EXPECTED_OPERATION_COUNT:
    raise RuntimeError(
        "Cell 12 pair count is not the Stage 02 candidate count."
    )

pair_frame_by_op_id = {}

for record in read_jsonl(
    PAIR_FRAME_PATH
):
    op_id = str(
        record.get(
            "op_id",
            "",
        )
    )

    if not op_id:
        raise RuntimeError(
            "Pair-frame record has no op_id."
        )

    if op_id in pair_frame_by_op_id:
        raise RuntimeError(
            "Duplicate pair-frame op_id: "
            f"{op_id}."
        )

    pair_frame_by_op_id[op_id] = (
        record
    )

construction_by_op_id = {}

for record in read_jsonl(
    CONSTRUCTION_RECORDS_PATH
):
    op_id = str(
        record.get(
            "op_id",
            "",
        )
    )

    if not op_id:
        raise RuntimeError(
            "Construction record has no op_id."
        )

    if op_id in construction_by_op_id:
        raise RuntimeError(
            "Duplicate construction op_id: "
            f"{op_id}."
        )

    if record.get("status") != "PASS":
        raise RuntimeError(
            "Construction record is not PASS "
            f"for {op_id}."
        )

    construction_by_op_id[op_id] = (
        record
    )

token_audit_by_op_id = {}

with TOKEN_AUDIT_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle:
    for row in csv.DictReader(handle):
        op_id = str(
            row.get(
                "op_id",
                "",
            )
        )

        if not op_id:
            raise RuntimeError(
                "Token-audit row has no op_id."
            )

        if op_id in token_audit_by_op_id:
            raise RuntimeError(
                "Duplicate token-audit op_id: "
                f"{op_id}."
            )

        token_audit_by_op_id[op_id] = (
            row
        )

operation_id_sets = {
    "pair_frame": set(
        pair_frame_by_op_id
    ),
    "construction": set(
        construction_by_op_id
    ),
    "token_audit": set(
        token_audit_by_op_id
    ),
}

for name, operation_ids in (
    operation_id_sets.items()
):
    if (
        len(operation_ids)
        != EXPECTED_OPERATION_COUNT
    ):
        raise RuntimeError(
            f"{name} contains "
            f"{len(operation_ids)} operations; "
            "expected the Stage 02 candidate count."
        )

if not (
    operation_id_sets["pair_frame"]
    == operation_id_sets["construction"]
    == operation_id_sets["token_audit"]
):
    raise RuntimeError(
        "Cell 13 input operation populations "
        "do not match."
    )

required_operation_ids = (
    operation_id_sets["pair_frame"]
)

# Audit trusted L0 annotation values. The obfuscation
# log stores the annotation location as slice_pointer.
text_audit_by_op_id = {}
completed_log_operation_ids = set()

obfuscation_records = read_jsonl(
    OBFUSCATION_LOG_PATH
)

for (
    operation_number,
    (
        op_id,
        grouped_records,
    ),
) in enumerate(
    groupby(
        obfuscation_records,
        key=lambda record: str(
            record.get(
                "op_id",
                "",
            )
        ),
    ),
    start=1,
):
    if not op_id:
        raise RuntimeError(
            "Obfuscation-log record has no op_id."
        )

    if op_id in completed_log_operation_ids:
        raise RuntimeError(
            "Obfuscation log is not grouped "
            "contiguously by op_id: "
            f"{op_id}."
        )

    if op_id not in required_operation_ids:
        raise RuntimeError(
            "Obfuscation log contains an "
            f"unexpected op_id: {op_id}."
        )

    completed_log_operation_ids.add(
        op_id
    )

    construction_record = (
        construction_by_op_id[op_id]
    )
    l0_path = (
        PROJECT_ROOT
        / construction_record[
            "l0_relative_path"
        ]
    )

    if not l0_path.is_file():
        raise FileNotFoundError(
            l0_path
        )

    l0_bytes = l0_path.read_bytes()

    if hashlib.sha256(
        l0_bytes
    ).hexdigest() != str(
        construction_record["l0_sha256"]
    ):
        raise RuntimeError(
            "L0 hash mismatch during prose "
            f"audit for {op_id}."
        )

    l0_document = json.loads(
        l0_bytes.decode("utf-8")
    )

    annotation_count = 0
    placeholder_field_count = 0
    ascii_letter_count = 0
    non_ascii_letter_count = 0
    character_count = 0
    field_hash_problem_count = 0

    for log_record in grouped_records:
        pointer_value = (
            log_record.get(
                "slice_pointer"
            )
        )

        if not isinstance(
            pointer_value,
            str,
        ) or not pointer_value.startswith("/"):
            raise RuntimeError(
                "Obfuscation-log record has an "
                "invalid slice_pointer for "
                f"{op_id}: {pointer_value!r}."
            )

        pointer = pointer_value

        annotation_value = (
            resolve_json_pointer(
                l0_document,
                pointer,
            )
        )

        if not isinstance(
            annotation_value,
            str,
        ):
            raise RuntimeError(
                "Approved prose annotation is "
                "not a string for "
                f"{op_id} at {pointer}."
            )

        observed_text_hash = (
            sha256_text(
                annotation_value
            )
        )
        expected_text_hash = str(
            log_record.get(
                "original_sha256",
                "",
            )
        )

        if (
            observed_text_hash
            != expected_text_hash
        ):
            field_hash_problem_count += 1

        annotation_count += 1
        character_count += len(
            annotation_value
        )

        placeholder_field_count += int(
            is_exact_placeholder(
                annotation_value
            )
        )

        for character in annotation_value:
            if character.isalpha():
                if character.isascii():
                    ascii_letter_count += 1
                else:
                    non_ascii_letter_count += 1

    if field_hash_problem_count:
        raise RuntimeError(
            "Original annotation hashes did "
            "not reconcile for "
            f"{op_id}: "
            f"{field_hash_problem_count} "
            "problem(s)."
        )

    expected_annotation_count = int(
        construction_record[
            "annotation_field_count"
        ]
    )

    if (
        annotation_count
        != expected_annotation_count
    ):
        raise RuntimeError(
            "Annotation count mismatch for "
            f"{op_id}: expected "
            f"{expected_annotation_count}, "
            f"observed {annotation_count}."
        )

    placeholder_only = (
        annotation_count > 0
        and (
            placeholder_field_count
            == annotation_count
        )
    )

    text_audit_by_op_id[op_id] = {
        "annotation_count": (
            annotation_count
        ),
        "character_count": (
            character_count
        ),
        "placeholder_field_count": (
            placeholder_field_count
        ),
        "placeholder_only_flag": (
            placeholder_only
        ),
        "ascii_letter_count_observed": (
            ascii_letter_count
        ),
        "non_ascii_letter_count": (
            non_ascii_letter_count
        ),
        "language_review_flag": (
            non_ascii_letter_count > 0
        ),
    }

    if (
        operation_number % 5000 == 0
        or operation_number
        == EXPECTED_OPERATION_COUNT
    ):
        print(
            json.dumps(
                {
                    "operations_audited": (
                        operation_number
                    ),
                    "operation_total": (
                        EXPECTED_OPERATION_COUNT
                    ),
                    "placeholder_only_found": (
                        sum(
                            int(
                                result[
                                    "placeholder_only_flag"
                                ]
                            )
                            for result
                            in text_audit_by_op_id
                            .values()
                        )
                    ),
                    "language_review_flagged": (
                        sum(
                            int(
                                result[
                                    "language_review_flag"
                                ]
                            )
                            for result
                            in text_audit_by_op_id
                            .values()
                        )
                    ),
                    "elapsed_seconds": round(
                        time.time()
                        - CELL13_STARTED,
                        1,
                    ),
                }
            )
        )

if (
    completed_log_operation_ids
    != required_operation_ids
):
    missing_log_operations = sorted(
        required_operation_ids
        - completed_log_operation_ids
    )

    raise RuntimeError(
        "Obfuscation log is missing operations: "
        + ", ".join(
            missing_log_operations[:20]
        )
    )

ledger_fieldnames = [
    "op_id",
    "pair_id",
    "spec_id",
    "family_id",
    "provider_directory",
    "source_class",
    "method",
    "path",
    "prose_tokens_L0",
    "annotation_field_count",
    "changed_field_count",
    "transformed_ascii_letter_count",
    "observed_ascii_letter_count",
    "non_ascii_letter_count",
    "placeholder_field_count",
    "placeholder_only_flag",
    "placeholder_rule_applied_for_exclusion",
    "language_review_flag",
    "english_only_rule_applied",
    "l0_input_tokens",
    "l1_input_tokens",
    "prompt_pair_within_limit",
    "positive_prose_score",
    "has_transformed_ascii_letter",
    "prose_eligibility_pass",
    "exclusion_reasons",
    "status",
]

temporary_ledger_path = (
    ELIGIBILITY_LEDGER_PATH.with_name(
        ELIGIBILITY_LEDGER_PATH.name
        + ".tmp"
    )
)

status_counts = Counter()
exclusion_reason_counts = Counter()
source_class_status_counts = Counter()
method_status_counts = Counter()

placeholder_only_count = 0
language_review_flag_count = 0
positive_prose_count = 0
transformed_ascii_count = 0
prompt_feasible_count = 0
eligible_for_feasibility_count = 0

with temporary_ledger_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=ledger_fieldnames,
        lineterminator="\n",
    )
    writer.writeheader()

    for op_id in sorted(
        required_operation_ids
    ):
        pair_record = (
            pair_frame_by_op_id[op_id]
        )
        construction_record = (
            construction_by_op_id[op_id]
        )
        token_record = (
            token_audit_by_op_id[op_id]
        )
        text_audit = (
            text_audit_by_op_id[op_id]
        )

        if str(
            pair_record["pair_id"]
        ) != str(
            construction_record["pair_id"]
        ):
            raise RuntimeError(
                "Pair ID mismatch for "
                f"{op_id}."
            )

        if str(
            pair_record["spec_id"]
        ) != str(
            construction_record["spec_id"]
        ):
            raise RuntimeError(
                "Specification ID mismatch for "
                f"{op_id}."
            )

        if str(
            pair_record["family_id"]
        ) != str(
            construction_record["family_id"]
        ):
            raise RuntimeError(
                "Family ID mismatch for "
                f"{op_id}."
            )

        if str(
            token_record["pair_id"]
        ) != str(
            pair_record["pair_id"]
        ):
            raise RuntimeError(
                "Token-audit pair ID mismatch "
                f"for {op_id}."
            )

        prose_tokens_l0 = int(
            pair_record[
                "prose_tokens_L0"
            ]
        )
        changed_field_count = int(
            construction_record[
                "changed_field_count"
            ]
        )
        transformed_ascii_letters = int(
            construction_record[
                "ascii_letter_count"
            ]
        )

        prompt_pair_within_limit = (
            parse_boolean(
                token_record[
                    "pair_within_limit"
                ]
            )
        )

        positive_prose_score = (
            prose_tokens_l0 > 0
        )
        has_transformed_ascii_letter = (
            transformed_ascii_letters > 0
            and changed_field_count > 0
        )

        exclusion_reasons = []

        if not prompt_pair_within_limit:
            exclusion_reasons.append(
                "EXCLUDED_PROMPT_LIMIT"
            )

        if not positive_prose_score:
            exclusion_reasons.append(
                "EXCLUDED_NONPOSITIVE_PROSE_SCORE"
            )

        if not has_transformed_ascii_letter:
            exclusion_reasons.append(
                "EXCLUDED_NO_TRANSFORMED_ASCII_LETTER"
            )

        prose_eligibility_pass = (
            len(exclusion_reasons) == 0
        )

        status = (
            "ELIGIBLE_FOR_FEASIBILITY_PROBE"
            if prose_eligibility_pass
            else "INELIGIBLE"
        )

        status_counts[status] += 1

        source_class_status_counts[
            (
                f"{pair_record['source_class']}"
                f"::{status}"
            )
        ] += 1

        method_status_counts[
            (
                f"{pair_record['method']}"
                f"::{status}"
            )
        ] += 1

        for reason in exclusion_reasons:
            exclusion_reason_counts[
                reason
            ] += 1

        placeholder_only_count += int(
            text_audit[
                "placeholder_only_flag"
            ]
        )
        language_review_flag_count += int(
            text_audit[
                "language_review_flag"
            ]
        )
        positive_prose_count += int(
            positive_prose_score
        )
        transformed_ascii_count += int(
            has_transformed_ascii_letter
        )
        prompt_feasible_count += int(
            prompt_pair_within_limit
        )
        eligible_for_feasibility_count += int(
            prose_eligibility_pass
        )

        writer.writerow(
            {
                "op_id": op_id,
                "pair_id": (
                    pair_record["pair_id"]
                ),
                "spec_id": (
                    pair_record["spec_id"]
                ),
                "family_id": (
                    pair_record["family_id"]
                ),
                "provider_directory": (
                    pair_record[
                        "provider_directory"
                    ]
                ),
                "source_class": (
                    pair_record[
                        "source_class"
                    ]
                ),
                "method": (
                    pair_record["method"]
                ),
                "path": (
                    pair_record["path"]
                ),
                "prose_tokens_L0": (
                    prose_tokens_l0
                ),
                "annotation_field_count": (
                    construction_record[
                        "annotation_field_count"
                    ]
                ),
                "changed_field_count": (
                    changed_field_count
                ),
                "transformed_ascii_letter_count": (
                    transformed_ascii_letters
                ),
                "observed_ascii_letter_count": (
                    text_audit[
                        "ascii_letter_count_observed"
                    ]
                ),
                "non_ascii_letter_count": (
                    text_audit[
                        "non_ascii_letter_count"
                    ]
                ),
                "placeholder_field_count": (
                    text_audit[
                        "placeholder_field_count"
                    ]
                ),
                "placeholder_only_flag": (
                    text_audit[
                        "placeholder_only_flag"
                    ]
                ),
                "placeholder_rule_applied_for_exclusion": (
                    False
                ),
                "language_review_flag": (
                    text_audit[
                        "language_review_flag"
                    ]
                ),
                "english_only_rule_applied": (
                    False
                ),
                "l0_input_tokens": int(
                    token_record[
                        "l0_input_tokens"
                    ]
                ),
                "l1_input_tokens": int(
                    token_record[
                        "l1_input_tokens"
                    ]
                ),
                "prompt_pair_within_limit": (
                    prompt_pair_within_limit
                ),
                "positive_prose_score": (
                    positive_prose_score
                ),
                "has_transformed_ascii_letter": (
                    has_transformed_ascii_letter
                ),
                "prose_eligibility_pass": (
                    prose_eligibility_pass
                ),
                "exclusion_reasons": (
                    ";".join(
                        exclusion_reasons
                    )
                ),
                "status": status,
            }
        )

    handle.flush()
    os.fsync(handle.fileno())

os.replace(
    temporary_ledger_path,
    ELIGIBILITY_LEDGER_PATH,
)

checks = {
    "operation_count_matches_stage02_candidate_count": (
        sum(status_counts.values())
        == EXPECTED_OPERATION_COUNT
    ),
    "operation_populations_match": (
        operation_id_sets["pair_frame"]
        == operation_id_sets["construction"]
        == operation_id_sets["token_audit"]
        == completed_log_operation_ids
    ),
    "cell12_hash_matches_policy": (
        expected_cell12_hash
        == observed_cell12_hash
    ),
    "prompt_feasible_count_matches_cell12": (
        prompt_feasible_count
        == int(
            cell12_report[
                "pair_within_limit_count"
            ]
        )
    ),
    "prompt_exclusion_count_matches_cell12": (
        exclusion_reason_counts[
            "EXCLUDED_PROMPT_LIMIT"
        ]
        == int(
            cell12_report[
                "pair_excluded_prompt_limit_count"
            ]
        )
    ),
    "placeholder_rule_not_used_for_exclusion": (
        True
    ),
    "english_only_rule_not_used": (
        True
    ),
    "eligible_count_is_positive": (
        eligible_for_feasibility_count
        > 0
    ),
    "ledger_exists_and_is_nonempty": (
        ELIGIBILITY_LEDGER_PATH.is_file()
        and (
            ELIGIBILITY_LEDGER_PATH
            .stat()
            .st_size > 0
        )
    ),
}

status = (
    "PASS"
    if all(checks.values())
    else "BLOCKED"
)

report = {
    "status": status,
    "stage": (
        "OPERATION_LANGUAGE_PROSE_ELIGIBILITY"
    ),
    "eligibility_version": (
        ELIGIBILITY_VERSION
    ),
    "operation_count": (
        EXPECTED_OPERATION_COUNT
    ),
    "prompt_feasible_operation_count": (
        prompt_feasible_count
    ),
    "positive_prose_operation_count": (
        positive_prose_count
    ),
    "transformed_ascii_operation_count": (
        transformed_ascii_count
    ),
    "eligible_for_feasibility_probe_count": (
        eligible_for_feasibility_count
    ),
    "ineligible_before_feasibility_count": (
        status_counts["INELIGIBLE"]
    ),
    "status_counts": dict(
        sorted(
            status_counts.items()
        )
    ),
    "exclusion_reason_counts": dict(
        sorted(
            exclusion_reason_counts.items()
        )
    ),
    "source_class_status_counts": dict(
        sorted(
            source_class_status_counts.items()
        )
    ),
    "method_status_counts": dict(
        sorted(
            method_status_counts.items()
        )
    ),
    "placeholder_audit": {
        "version": (
            PLACEHOLDER_AUDIT_VERSION
        ),
        "pointer_field": (
            "slice_pointer"
        ),
        "exact_values": sorted(
            PLACEHOLDER_EXACT_VALUES
        ),
        "placeholder_only_operation_count": (
            placeholder_only_count
        ),
        "used_for_exclusion": False,
        "reason": (
            "The frozen policy contains no "
            "placeholder-exclusion rule. Exact "
            "placeholder-only cases are flagged "
            "but retained unless another rule "
            "excludes them."
        ),
    },
    "language_audit": {
        "language_review_flag_count": (
            language_review_flag_count
        ),
        "english_only_rule_applied": False,
        "english_inference_performed": False,
        "interpretation": (
            "Non-ASCII alphabetic content is "
            "flagged for possible review. ASCII "
            "letters are not treated as proof "
            "of English."
        ),
    },
    "eligibility_rules": {
        "require_pair_within_prompt_limit": (
            True
        ),
        "require_positive_prose_token_score": (
            True
        ),
        "require_transformed_ascii_letter": (
            True
        ),
        "short_prose_automatically_excluded": (
            False
        ),
        "placeholder_only_automatically_excluded": (
            False
        ),
        "english_only": None,
        "evaluator_feasibility": (
            "PENDING_CELL_14"
        ),
    },
    "eligibility_ledger": {
        "path": str(
            ELIGIBILITY_LEDGER_PATH
        ),
        "bytes": (
            ELIGIBILITY_LEDGER_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            ELIGIBILITY_LEDGER_PATH
        ),
        "record_count": (
            EXPECTED_OPERATION_COUNT
        ),
    },
    "input_hashes": {
        "pair_frame": sha256_file(
            PAIR_FRAME_PATH
        ),
        "construction_records": (
            sha256_file(
                CONSTRUCTION_RECORDS_PATH
            )
        ),
        "obfuscation_log": (
            sha256_file(
                OBFUSCATION_LOG_PATH
            )
        ),
        "token_audit": (
            sha256_file(
                TOKEN_AUDIT_PATH
            )
        ),
        "cell12_report": (
            observed_cell12_hash
        ),
    },
    "checks": checks,
    "duration_seconds": round(
        time.time()
        - CELL13_STARTED,
        3,
    ),
    "interpretation": (
        "Operations are retained for the next "
        "feasibility probe only when both prompts "
        "fit the frozen input limit, the L0 prose "
        "score is positive, and at least one ASCII "
        "letter was transformed. Placeholder and "
        "language indicators are audit flags only."
    ),
    "report_path": str(
        REPORT_PATH
    ),
}

write_json_atomic(
    REPORT_PATH,
    report,
)

report_sha256 = sha256_file(
    REPORT_PATH
)

stage_03_policy[
    "operation_level_eligibility"
] = {
    "status": status,
    "eligibility_version": (
        ELIGIBILITY_VERSION
    ),
    "operation_count": (
        EXPECTED_OPERATION_COUNT
    ),
    "eligible_for_feasibility_probe_count": (
        eligible_for_feasibility_count
    ),
    "ineligible_before_feasibility_count": (
        status_counts["INELIGIBLE"]
    ),
    "require_pair_within_prompt_limit": (
        True
    ),
    "require_positive_prose_token_score": (
        True
    ),
    "require_transformed_ascii_letter": (
        True
    ),
    "placeholder_audit_version": (
        PLACEHOLDER_AUDIT_VERSION
    ),
    "placeholder_pointer_field": (
        "slice_pointer"
    ),
    "placeholder_exclusion_rule": None,
    "placeholder_only_operation_count": (
        placeholder_only_count
    ),
    "english_only": None,
    "language_review_flag_count": (
        language_review_flag_count
    ),
    "eligibility_ledger_path": (
        str(
            ELIGIBILITY_LEDGER_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/")
    ),
    "eligibility_ledger_sha256": (
        sha256_file(
            ELIGIBILITY_LEDGER_PATH
        )
    ),
    "report_path": (
        str(
            REPORT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/")
    ),
    "report_sha256": (
        report_sha256
    ),
    "evaluator_feasibility": (
        "PENDING_CELL_14"
    ),
}

write_yaml_atomic(
    STAGE_03_POLICY_PATH,
    stage_03_policy,
)

output = dict(report)
output["report_sha256"] = (
    report_sha256
)
output[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if status != "PASS":
    raise RuntimeError(
        "Operation-level prose eligibility "
        "failed. Review the checks above."
    )

print(
    "\nNOTEBOOK 03 — CELL 13: PASS"
)

{"operations_audited": 5000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 38, "elapsed_seconds": 8.5}
{"operations_audited": 10000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 69, "elapsed_seconds": 14.3}
{"operations_audited": 15000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 107, "elapsed_seconds": 20.1}
{"operations_audited": 20000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 142, "elapsed_seconds": 25.8}
{"operations_audited": 25000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 175, "elapsed_seconds": 32.8}
{"operations_audited": 30000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 217, "elapsed_seconds": 38.7}
{"operations_audited": 35000, "operation_total": 52922, "placeholder_only_found": 0, "language_review_flagged": 248, "elapsed_seconds": 44.4}
{"operatio

In [14]:
import csv
import hashlib
import io
import json
import os
import re
import zipfile
from pathlib import Path, PurePosixPath

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
DESIGN = "openapi-fresh-pilot-v1"


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def encode(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                       allow_nan=False) + "\n").encode()


def load(path):
    return json.loads(path.read_bytes())


def records(path):
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def csv_records(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def atomic(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)


def jsonl(path, rows):
    atomic(path, b"".join(map(encode, rows)))


def csv_write(path, fields, rows):
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    atomic(path, output.getvalue().encode())


def safe_path(name):
    p = PurePosixPath(name)
    if p.is_absolute() or ".." in p.parts or "\\" in name or p.as_posix() != name:
        raise ValueError(f"Unsafe manifest path: {name}")
    return Path(*p.parts)


import hashlib
import json
import re
from pathlib import PurePosixPath


ARTIFACT_SCHEMA = "openapi-artifact-v1"
PRODUCER_NOTEBOOKS = {
    "00": "00_protocol_and_runtime",
    "01": "01_corpus_preparation_and_eligibility",
    "02": "02_operation_slicing_and_prose_measurement",
    "03": "03_pair_and_evaluator_feasibility",
    "04": "04_strata_and_pilot_selection",
    "05": "05_oracle_build_and_review",
    "06": "06_build_corrected_evaluator_and_generation_tasks",
    "07": "07_pilot_generation",
    "08": "08_pilot_execution_and_review",
}


def provenance_digest(value):
    data = json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(data).hexdigest()


def artifact_path_for(record):
    paths = {record[key] for key in ("artifact_path", "relative_path", "path")
             if key in record}
    if len(paths) != 1:
        raise RuntimeError("Missing or inconsistent artifact paths")
    name = paths.pop()
    path = PurePosixPath(name)
    if not name or name == "." or path.is_absolute() or ".." in path.parts or "\\" in name or path.as_posix() != name:
        raise RuntimeError(f"Invalid artifact path: {name!r}")
    return name


def manifest_digest_valid(value):
    return isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) is not None


def register_artifact_provenance(manifest):
    stage = manifest["stage"]
    artifacts = manifest["artifacts"]
    if "upstream_manifest_sha256" not in manifest:
        if stage != "00":
            raise RuntimeError(f"Stage {stage} has no upstream identity")
        manifest["upstream_manifest_sha256"] = {}
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest.get("configuration_hash")
    if config_hash is None:
        config_hash = provenance_digest({artifact_path_for(row): row["sha256"]
                                         for row in artifacts
                                         if artifact_path_for(row).startswith("config/")})
    manifest["configuration_hash"] = config_hash
    manifest["artifact_record_schema"] = ARTIFACT_SCHEMA
    upstream_hash = provenance_digest(links)
    for row in artifacts:
        row.update(artifact_path=artifact_path_for(row), schema_version=ARTIFACT_SCHEMA,
                   producer_notebook=PRODUCER_NOTEBOOKS[stage], config_hash=config_hash,
                   upstream_manifest_hash=upstream_hash)
    verify_artifact_provenance(manifest)


def verify_artifact_provenance(manifest):
    stage = manifest["stage"]
    if manifest.get("artifact_record_schema") != ARTIFACT_SCHEMA:
        raise RuntimeError(f"Stage {stage} requires a corrected provenance export")
    links = manifest["upstream_manifest_sha256"]
    config_hash = manifest["configuration_hash"]
    if (not isinstance(links, dict) or bool(links) != (stage != "00")
            or any(key not in PRODUCER_NOTEBOOKS or key >= stage or not manifest_digest_valid(value)
                   for key, value in links.items()) or not manifest_digest_valid(config_hash)):
        raise RuntimeError(f"Invalid Stage {stage} provenance")
    expected = (ARTIFACT_SCHEMA, PRODUCER_NOTEBOOKS[stage], config_hash, provenance_digest(links))
    seen = set()
    artifacts = manifest["artifacts"]
    if not artifacts or len(artifacts) != manifest["artifact_count"]:
        raise RuntimeError(f"Invalid Stage {stage} artifact count")
    for row in artifacts:
        name = artifact_path_for(row)
        actual = tuple(row.get(key) for key in
                       ("schema_version", "producer_notebook", "config_hash", "upstream_manifest_hash"))
        if (row.get("artifact_path") != name or name in seen or actual != expected
                or not manifest_digest_valid(row.get("sha256"))
                or type(row.get("bytes")) is not int or row["bytes"] < 0):
            raise RuntimeError(f"Invalid Stage {stage} artifact provenance: {name}")
        seen.add(name)


def stage_input(stage, *, fresh=True):
    matches = []
    for external in sorted(INPUT.rglob(f"stage_{stage}_manifest.json")):
        sidecar = load(external)
        if fresh and sidecar.get("design_id") != DESIGN:
            continue
        for manifest in external.parent.rglob(f"registries/{stage}_manifest.json"):
            matches.append((external, manifest))
    if len(matches) != 1:
        raise RuntimeError(f"Attach one expanded Stage {stage} release. Found {len(matches)}.")
    external, internal = matches[0]
    sidecar, manifest = load(external), load(internal)
    expected = sidecar.get("internal_manifest_sha256") or sidecar.get("internal_manifest", {}).get("sha256")
    if expected != file_sha(internal):
        raise RuntimeError(f"Stage {stage} manifest digest mismatch.")
    if (manifest.get("stage_status", manifest.get("status")) != "PASS"
            or sidecar.get("stage_status", sidecar.get("status")) != "PASS"):
        raise RuntimeError(f"Stage {stage} has no passing handoff.")
    verify_artifact_provenance(manifest)
    root = internal.parent.parent
    paths = {}
    for item in manifest["artifacts"]:
        name = item.get("relative_path") or item.get("artifact_path") or item.get("path")
        path = root / safe_path(name)
        if name in paths or path.is_symlink() or not path.is_file() or path.stat().st_size != item["bytes"] or file_sha(path) != item["sha256"]:
            raise RuntimeError(f"Stage {stage} artifact mismatch: {name}")
        paths[name] = item["sha256"]
    if len(paths) != manifest["artifact_count"]:
        raise RuntimeError(f"Stage {stage} artifact count mismatch.")
    return root, manifest, expected, paths


def pack(stage, root, metadata):
    output = WORK / f"stage_{stage}_fresh_dataset"
    output.mkdir(parents=True, exist_ok=True)
    manifest_path = root / f"registries/{stage}_manifest.json"
    files = sorted(p for p in root.rglob("*") if p.is_file() and p != manifest_path
                   and "__pycache__" not in p.parts and not p.name.endswith(".tmp"))
    artifacts = [{"relative_path": p.relative_to(root).as_posix(), "bytes": p.stat().st_size,
                  "sha256": file_sha(p)} for p in files]
    manifest = {"stage": stage, "status": "PASS", "design_id": DESIGN,
                "artifact_count": len(artifacts), "artifacts": artifacts, **metadata}
    register_artifact_provenance(manifest)
    atomic(manifest_path, encode(manifest))
    archive = output / f"stage_{stage}_fresh.zip"
    tmp = archive.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=4) as z:
        for p in [*files, manifest_path]:
            z.write(p, "prose_operation_experiment/" + p.relative_to(root).as_posix())
    tmp.replace(archive)
    external = {k: v for k, v in manifest.items() if k != "artifacts"}
    external.update(archive=archive.name, archive_sha256=file_sha(archive),
                    archive_bytes=archive.stat().st_size, internal_manifest_sha256=file_sha(manifest_path))
    atomic(output / f"stage_{stage}_manifest.json", encode(external))
    atomic(output / f"stage_{stage}_checksums.sha256",
           f'{external["archive_sha256"]}  {archive.name}\n'.encode())
    atomic(output / "README.md", f"# Stage {stage} fresh pilot\n\nDesign {DESIGN}.\n".encode())
    print(f"Stage {stage} PASS. Publish {output}")
    return external


import yaml

SHARED_ORACLE_SOURCES = {'oracle.py': 'from collections import Counter\n\nimport jsonschema\n\nimport oracle_baselines\nimport oracle_faults\nimport oracle_requirements\nimport oracle_scenarios\nimport oracle_variations\nimport routing\n\nORACLE_VERSION = "shared-pilot-oracle-v3"\nFAULT_ERRORS = {"FROZEN_RESPONSE_SCHEMA_MISMATCH", "UNDOCUMENTED_FROZEN_RESPONSE_BODY",\n                "MISSING_REQUIRED_FROZEN_RESPONSE_HEADER", "UNDOCUMENTED_FROZEN_RESPONSE_STATUS"}\nVALIDATORS = {}\n\n\nclass UnsupportedOracle(ValueError):\n    pass\n\n\ndef validate_records(contracts, package):\n    if not VALIDATORS:\n        VALIDATORS.update({key: jsonschema.Draft202012Validator(schema)\n                           for key, schema in contracts["records"].items()})\n    for kind, rows in package.items():\n        id_field = {"requirements": "requirement_id", "interface_items": "interface_item_id",\n                    "scenarios": "scenario_id", "baselines": "baseline_id",\n                    "legitimate_variations": "variation_id", "response_faults": "fault_id"}[kind]\n        ids = [row[id_field] for row in rows]\n        if len(ids) != len(set(ids)):\n            raise UnsupportedOracle("duplicate records: " + kind)\n        for row in rows:\n            error = next(VALIDATORS[kind].iter_errors(row), None)\n            if error:\n                raise UnsupportedOracle(f"oracle schema {kind}/{row[id_field]}: {error.message}")\n\n\ndef audit_response(document, scenario, response, baseline, fault=False):\n    try:\n        routing.audit_frozen_response(document, scenario, response, baseline)\n        routing.prepare_frozen_response(response, baseline)\n    except routing.EvaluatorDefect as error:\n        if not fault or error.code not in FAULT_ERRORS:\n            raise\n        routing.prepare_frozen_response(response, baseline)\n    else:\n        if fault:\n            raise UnsupportedOracle("fault is accepted by the documented response contract")\n\n\ndef json_media(media):\n    media = media.split(";", 1)[0].strip().lower()\n    return media == "application/json" or media.endswith("+json")\n\n\ndef build_oracle(document, op_id, method, path, contracts):\n    operation = document["paths"][path][method.lower()]\n    if operation.get("callbacks") or document.get("webhooks"):\n        raise UnsupportedOracle("callbacks and webhooks are outside the supported profile")\n    request = routing._object(document, operation.get("requestBody", {}))\n    if request.get("content") and not any(map(json_media, request["content"])):\n        raise UnsupportedOracle("request has no supported JSON representation")\n    req = oracle_requirements.extract_operation_requirements(document, op_id, path, method)\n    requirements, interface = req["requirements"], req["interface_items"]\n    sc = oracle_scenarios.build_scenarios(document, op_id, method, path, requirements, interface)\n    scenarios = sc["scenarios"]\n    bl = oracle_baselines.build_baselines(document, op_id, method, path, scenarios)\n    baselines = bl["baselines"]\n    if req.get("blocking") or sc.get("blocking") or bl.get("blocking"):\n        codes = sorted({d["code"] for r in (req, sc, bl) for d in r.get("diagnostics", [])\n                        if d.get("severity") == "BLOCKING"})\n        raise UnsupportedOracle("blocked oracle builder: " + ", ".join(codes))\n    if not requirements or not any(s.get("applicability") == "APPLICABLE" for s in scenarios) or not baselines:\n        raise UnsupportedOracle("missing requirements, scenarios or baselines")\n    mapped = [s for s in scenarios if s.get("documented_response_mapping") is not None]\n    if Counter(b["scenario_id"] for b in baselines) != Counter(s["scenario_id"] for s in mapped):\n        raise UnsupportedOracle("mapped scenarios require exactly one baseline")\n    scenarios_by_id = {s["scenario_id"]: s for s in scenarios}\n    baselines_by_id = {b["baseline_id"]: b for b in baselines}\n    for baseline in baselines:\n        scenario = scenarios_by_id[baseline["scenario_id"]]\n        prepared = routing.prepare_frozen_fixture(document, scenario)\n        if prepared.get("media_type") and not json_media(prepared["media_type"]):\n            raise UnsupportedOracle("non-JSON request fixture")\n        selected = routing.select_frozen_response(routing.classify_request(prepared, scenarios, document), baselines)\n        if selected["baseline_id"] != baseline["baseline_id"]:\n            raise UnsupportedOracle("baseline round trip changed scenario")\n        audit_response(document, scenario, baseline, baseline)\n    va = oracle_variations.build_variations(document, op_id, method, path, baselines)\n    fa = oracle_faults.build_faults(document, op_id, method, path, baselines, requirements)\n    variations, faults = va["variations"], fa["faults"]\n    if fa["status"] != "PASS" or not faults:\n        raise UnsupportedOracle("missing qualified response faults")\n    applicable = {r["requirement_id"] for r in requirements if r.get("applicability") == "APPLICABLE"}\n    for row, is_fault in [(v, False) for v in variations] + [(f, True) for f in faults]:\n        baseline = baselines_by_id[row["baseline_id"]]\n        if row.get("scenario_id") not in (None, baseline["scenario_id"]):\n            raise UnsupportedOracle("mutation scenario linkage mismatch")\n        if is_fault and (row["requirement_id"] not in applicable or row.get("non_equivalent") is not True):\n            raise UnsupportedOracle("fault lacks applicable requirement and non-equivalence evidence")\n        audit_response(document, scenarios_by_id[baseline["scenario_id"]], row, baseline, is_fault)\n    package = {"requirements": requirements, "interface_items": interface, "scenarios": scenarios,\n               "baselines": baselines, "legitimate_variations": variations, "response_faults": faults}\n    validate_records(contracts, package)\n    return package\n', 'oracle_baselines.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nfrom copy import deepcopy\nfrom urllib.parse import unquote\n\nfrom jsonschema import (\n    Draft4Validator,\n    Draft202012Validator,\n    FormatChecker,\n)\nfrom referencing import Registry\n\n\nBUILDER_VERSION = "stage05-deterministic-baseline-builder-v2-offline-refs"\nREFERENCE_NORMALIZATION_EVENTS = []\n\n\ndef canonical_json(value) -> str:\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    )\n\n\ndef pointer_escape(value: str) -> str:\n    return str(value).replace("~", "~0").replace("/", "~1")\n\n\ndef stable_id(prefix: str, value) -> str:\n    digest = hashlib.sha256(\n        canonical_json(value).encode("utf-8")\n    ).hexdigest()[:24]\n    return f"{prefix}_{digest}"\n\n\ndef pointer_token(value: str) -> str:\n    return value.replace("~1", "/").replace("~0", "~")\n\n\ndef resolve_pointer(document: dict, pointer: str):\n    if pointer in {"", "#"}:\n        return document\n\n    if not pointer.startswith("#/"):\n        raise ValueError(f"Unsupported reference: {pointer}")\n\n    node = document\n\n    for token in pointer[2:].split("/"):\n        key = pointer_token(token)\n        if isinstance(node, dict) and key not in node:\n            # One verified canonical document uses %7B/%7D in an\n            # internal path pointer while its actual path key has {}.\n            # Normalize only if the literal key does not exist and the\n            # decoded key does. Record the exception for human audit.\n            decoded = unquote(key)\n            if decoded != key and decoded in node:\n                REFERENCE_NORMALIZATION_EVENTS.append({\n                    "pointer": pointer,\n                    "encoded_token": key,\n                    "resolved_token": decoded,\n                })\n                key = decoded\n        if isinstance(node, list):\n            node = node[int(key)]\n        else:\n            node = node[key]\n\n    return node\n\n\ndef dereference(document: dict, value, depth: int = 0):\n    if depth > 30:\n        raise ValueError("Reference depth exceeded.")\n\n    if not isinstance(value, dict):\n        return value\n\n    reference = value.get("$ref")\n\n    if not isinstance(reference, str):\n        return value\n\n    target = deepcopy(\n        resolve_pointer(document, reference)\n    )\n    siblings = {\n        key: deepcopy(child)\n        for key, child in value.items()\n        if key != "$ref"\n    }\n\n    if isinstance(target, dict):\n        target.update(siblings)\n\n    return dereference(\n        document,\n        target,\n        depth + 1,\n    )\n\n\ndef schema_types(schema: dict) -> list[str]:\n    schema_type = schema.get("type")\n\n    if isinstance(schema_type, str):\n        result = [schema_type]\n    elif isinstance(schema_type, list):\n        result = [\n            value\n            for value in schema_type\n            if isinstance(value, str)\n        ]\n    elif isinstance(schema.get("properties"), dict):\n        result = ["object"]\n    elif "items" in schema:\n        result = ["array"]\n    else:\n        result = []\n\n    if schema.get("nullable") is True and "null" not in result:\n        result.append("null")\n\n    return result\n\n\ndef string_value(schema: dict) -> str:\n    for key in ["example", "default", "const"]:\n        value = schema.get(key)\n        if isinstance(value, str):\n            return value\n\n    enum_values = schema.get("enum")\n\n    if isinstance(enum_values, list):\n        for value in enum_values:\n            if isinstance(value, str):\n                return value\n\n    format_values = {\n        "date-time": "2026-01-01T00:00:00Z",\n        "date": "2026-01-01",\n        "time": "12:00:00Z",\n        "email": "user@example.com",\n        "hostname": "example.com",\n        "ipv4": "192.0.2.1",\n        "ipv6": "2001:db8::1",\n        "uri": "https://example.com",\n        "uuid": "123e4567-e89b-12d3-a456-426614174000",\n        "byte": "dGVzdA==",\n        "binary": "test",\n    }\n\n    if schema.get("format") in format_values:\n        return format_values[schema["format"]]\n\n    minimum = max(int(schema.get("minLength", 1) or 1), 1)\n    maximum = schema.get("maxLength")\n    candidates = [\n        "a",\n        "test",\n        "value",\n        "date",\n        "Bearer token",\n        "abc-123",\n    ]\n\n    for candidate in candidates:\n        if len(candidate) < minimum:\n            candidate += "a" * (minimum - len(candidate))\n\n        if isinstance(maximum, int):\n            candidate = candidate[:maximum]\n\n        return candidate\n\n    return "a" * minimum\n\n\ndef numeric_value(schema: dict, integer: bool):\n    minimum = schema.get("minimum")\n    maximum = schema.get("maximum")\n    exclusive_minimum = schema.get("exclusiveMinimum")\n    exclusive_maximum = schema.get("exclusiveMaximum")\n    multiple = schema.get("multipleOf")\n    step = 1 if integer else 0.5\n\n    lower = minimum if isinstance(minimum, (int, float)) else 0\n\n    if exclusive_minimum is True:\n        lower += step\n    elif (\n        isinstance(exclusive_minimum, (int, float))\n        and not isinstance(exclusive_minimum, bool)\n    ):\n        lower = exclusive_minimum + step\n\n    value = lower\n\n    if isinstance(multiple, (int, float)) and multiple > 0:\n        value = ((value + multiple - 1) // multiple) * multiple\n\n    if isinstance(maximum, (int, float)) and value > maximum:\n        value = maximum\n\n    if exclusive_maximum is True and value >= maximum:\n        value = maximum - step\n    elif (\n        isinstance(exclusive_maximum, (int, float))\n        and not isinstance(exclusive_maximum, bool)\n        and value >= exclusive_maximum\n    ):\n        value = exclusive_maximum - step\n\n    return int(value) if integer else float(value)\n\n\ndef valid_value(document: dict, schema, depth: int = 0):\n    if depth > 20:\n        return None\n\n    schema = dereference(document, schema)\n\n    if schema is True:\n        return None\n\n    if schema is False:\n        raise ValueError("False schema has no valid instance.")\n\n    if not isinstance(schema, dict):\n        return None\n\n    if "const" in schema:\n        return deepcopy(schema["const"])\n\n    enum_values = schema.get("enum")\n\n    if isinstance(enum_values, list) and enum_values:\n        return deepcopy(enum_values[0])\n\n    for keyword in ["oneOf", "anyOf"]:\n        branches = schema.get(keyword)\n\n        if isinstance(branches, list):\n            for branch in branches:\n                candidate = valid_value(\n                    document,\n                    branch,\n                    depth + 1,\n                )\n\n                if validate_instance(document, schema, candidate):\n                    return candidate\n\n    all_of = schema.get("allOf")\n\n    if isinstance(all_of, list):\n        result = {}\n\n        for branch in all_of:\n            candidate = valid_value(\n                document,\n                branch,\n                depth + 1,\n            )\n\n            if isinstance(candidate, dict):\n                result.update(candidate)\n\n        if validate_instance(document, schema, result):\n            return result\n\n    types = schema_types(schema)\n    selected_type = next(\n        (value for value in types if value != "null"),\n        "string",\n    )\n\n    if selected_type == "object":\n        properties = schema.get("properties", {})\n        required = schema.get("required", [])\n        result = {}\n\n        if not isinstance(properties, dict):\n            properties = {}\n\n        if not isinstance(required, list):\n            required = []\n\n        for name in required:\n            result[name] = valid_value(\n                document,\n                properties.get(name, {}),\n                depth + 1,\n            )\n\n        return result\n\n    if selected_type == "array":\n        count = max(int(schema.get("minItems", 0) or 0), 1)\n        item_schema = schema.get("items", {})\n        result = []\n\n        for index in range(count):\n            item = valid_value(\n                document,\n                item_schema,\n                depth + 1,\n            )\n\n            if schema.get("uniqueItems") and index > 0:\n                item_types = schema_types(\n                    dereference(document, item_schema)\n                )\n\n                if "integer" in item_types:\n                    item = index + 1\n                elif "number" in item_types:\n                    item = float(index + 1)\n                elif "string" in item_types:\n                    item = f"value{index + 1}"\n\n            result.append(item)\n\n        return result\n\n    if selected_type == "integer":\n        return numeric_value(schema, True)\n\n    if selected_type == "number":\n        return numeric_value(schema, False)\n\n    if selected_type == "boolean":\n        return True\n\n    if selected_type == "null":\n        return None\n\n    return string_value(schema)\n\n\ndef validate_instance(document: dict, schema, instance) -> bool:\n    if isinstance(schema, dict) and "$ref" in schema:\n        reference = schema["$ref"]\n        if not isinstance(reference, str) or not reference.startswith("#/"):\n            raise ValueError(\n                "External schema retrieval prohibited by Stage 05: "\n                + str(reference)\n            )\n    schema = dereference(document, schema)\n\n    if schema is True:\n        return True\n\n    if schema is False:\n        return False\n\n    if not isinstance(schema, dict):\n        return True\n\n    # A response fragment is not its own reference root. Nested\n    # \'#/components/...\' references must resolve against the verified\n    # *full canonical document*, never against this fragment.\n    is_openapi_31 = str(document.get("openapi", "")).startswith("3.1")\n    validator_class = (\n        Draft202012Validator if is_openapi_31 else Draft4Validator\n    )\n    validation_root = {\n        "$schema": (\n            "https://json-schema.org/draft/2020-12/schema"\n            if is_openapi_31\n            else "http://json-schema.org/draft-04/schema#"\n        ),\n        "$ref": "#/$defs/stage05_selected_response_schema",\n        "$defs": {"stage05_selected_response_schema": schema},\n        "components": document.get("components", {}),\n        "definitions": document.get("definitions", {}),\n    }\n\n    def block_external_reference(uri):\n        raise ValueError(\n            "External schema retrieval prohibited by Stage 05: "\n            + str(uri)\n        )\n\n    validator = validator_class(\n        validation_root,\n        format_checker=FormatChecker(),\n        registry=Registry(retrieve=block_external_reference),\n    )\n\n    return not list(validator.iter_errors(instance))\n\n\ndef response_object(\n    document: dict,\n    method: str,\n    path_template: str,\n    status: int,\n):\n    path_item = document.get("paths", {}).get(path_template)\n    operation = (\n        path_item.get(method.lower())\n        if isinstance(path_item, dict)\n        else None\n    )\n\n    if not isinstance(operation, dict):\n        raise KeyError(\n            f"Operation not found: {method} {path_template}"\n        )\n\n    responses = operation.get("responses", {})\n\n    if not isinstance(responses, dict):\n        raise TypeError("Operation responses are not an object.")\n\n    response_key = str(status)\n\n    if response_key not in responses:\n        return None, None\n\n    return (\n        response_key,\n        dereference(document, responses[response_key]),\n    )\n\n\ndef choose_content_schema(response: dict):\n    content = response.get("content")\n\n    if not isinstance(content, dict):\n        return None, None\n\n    media_types = sorted(content)\n\n    if "application/json" in content:\n        media_types.remove("application/json")\n        media_types.insert(0, "application/json")\n\n    for media_type in media_types:\n        media = content.get(media_type)\n\n        if (\n            isinstance(media, dict)\n            and "schema" in media\n        ):\n            return media_type, media["schema"]\n\n    return None, None\n\n\ndef build_baselines(\n    document: dict,\n    op_id: str,\n    method: str,\n    path_template: str,\n    scenarios: list[dict],\n) -> dict:\n    baselines = []\n    diagnostics = []\n\n    for scenario in sorted(\n        scenarios,\n        key=lambda value: value["scenario_id"],\n    ):\n        mapping = scenario.get(\n            "documented_response_mapping"\n        )\n\n        if mapping is None:\n            continue\n\n        status = mapping.get("status")\n\n        if not isinstance(status, int):\n            diagnostics.append(\n                {\n                    "severity": "BLOCKING",\n                    "code": "INVALID_RESPONSE_MAPPING",\n                    "scenario_id": scenario["scenario_id"],\n                }\n            )\n            continue\n\n        response_key, response = response_object(\n            document,\n            method,\n            path_template,\n            status,\n        )\n\n        if response is None:\n            diagnostics.append(\n                {\n                    "severity": "BLOCKING",\n                    "code": "UNDOCUMENTED_RESPONSE_STATUS",\n                    "scenario_id": scenario["scenario_id"],\n                    "status": status,\n                }\n            )\n            continue\n\n        media_type, schema = choose_content_schema(response)\n        body = None\n\n        if schema is not None:\n            body = valid_value(document, schema)\n\n            if not validate_instance(document, schema, body):\n                diagnostics.append(\n                    {\n                        "severity": "BLOCKING",\n                        "code": "BASELINE_BODY_SCHEMA_FAILURE",\n                        "scenario_id": scenario["scenario_id"],\n                        "status": status,\n                    }\n                )\n                continue\n\n        status_pointer = (\n            "/paths/" + pointer_escape(path_template)\n            + "/" + method.lower() + "/responses/"\n            + pointer_escape(response_key)\n        )\n        raw_response = (\n            document["paths"][path_template]\n            [method.lower()]["responses"][response_key]\n        )\n        response_is_reference = (\n            isinstance(raw_response, dict) and "$ref" in raw_response\n        )\n\n        evidence = [\n            {\n                "kind": "documented_response_status",\n                "source_pointer": status_pointer,\n                "status": status,\n            }\n        ]\n\n        if schema is not None:\n            evidence.append(\n                {\n                    "kind": "response_schema",\n                    "source_pointer": (\n                        status_pointer\n                        if response_is_reference\n                        else status_pointer + "/content/"\n                        + pointer_escape(media_type) + "/schema"\n                    ),\n                    "media_type": media_type,\n                    "resolved_via_response_ref": response_is_reference,\n                }\n            )\n\n        identity = {\n            "builder_version": BUILDER_VERSION,\n            "op_id": op_id,\n            "scenario_id": scenario["scenario_id"],\n            "status": status,\n            "headers": {},\n            "body": body,\n        }\n\n        baselines.append(\n            {\n                "baseline_id": stable_id("baseline", identity),\n                "scenario_id": scenario["scenario_id"],\n                "status": status,\n                "headers": {},\n                "body": body,\n                "source_evidence": evidence,\n                "validation_status": "PASS",\n            }\n        )\n\n    baseline_ids = [\n        baseline["baseline_id"]\n        for baseline in baselines\n    ]\n\n    if len(baseline_ids) != len(set(baseline_ids)):\n        diagnostics.append(\n            {\n                "severity": "BLOCKING",\n                "code": "DUPLICATE_BASELINE_ID",\n            }\n        )\n\n    return {\n        "builder_version": BUILDER_VERSION,\n        "baselines": baselines,\n        "diagnostics": diagnostics,\n        "blocking": any(\n            diagnostic.get("severity") == "BLOCKING"\n            for diagnostic in diagnostics\n        ),\n    }\n', 'oracle_faults.py': '"""Deterministic Stage 05 response fault builder; embedded by Notebook 05 Cell 16."""\n\nfrom collections import defaultdict\nfrom copy import deepcopy\nimport hashlib\nimport json\nimport re\nfrom urllib.parse import unquote\n\nimport jsonschema\nfrom referencing import Registry\n\n\nBUILDER_VERSION = "stage05-deterministic-response-fault-builder-v2-offline"\n\n\ndef canonical_json(value):\n    return json.dumps(value, sort_keys=True, ensure_ascii=False,\n                      separators=(",", ":"), allow_nan=False)\n\n\ndef stable_id(prefix, value):\n    digest = hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()\n    return f"{prefix}_{digest[:24]}"\n\n\ndef escape_token(value):\n    return str(value).replace("~", "~0").replace("/", "~1")\n\n\ndef join_pointer(pointer, *tokens):\n    return pointer + "".join("/" + escape_token(token) for token in tokens)\n\n\ndef resolve_local_pointer(document, reference):\n    if not isinstance(reference, str) or not reference.startswith("#/"):\n        raise ValueError(f"Nonlocal or unsupported reference: {reference!r}")\n    pointer = unquote(reference[1:])\n    node = document\n    for raw_token in pointer.split("/")[1:]:\n        token = raw_token.replace("~1", "/").replace("~0", "~")\n        if isinstance(node, dict):\n            node = node[token]\n        elif isinstance(node, list):\n            node = node[int(token)]\n        else:\n            raise KeyError(f"Nontraversable canonical pointer: {reference}")\n    return node, pointer\n\n\ndef schema_view(document, schema, pointer, maximum_depth=16):\n    visited = set()\n    node = schema\n    current = pointer\n    for _ in range(maximum_depth):\n        if not isinstance(node, dict):\n            return {}, current\n        reference = node.get("$ref")\n        if reference is None:\n            return node, current\n        if reference in visited:\n            raise ValueError(f"Recursive response schema reference: {reference}")\n        visited.add(reference)\n        node, current = resolve_local_pointer(document, reference)\n    raise ValueError("Local response schema reference depth exceeded")\n\n\ndef path_shape(path):\n    return re.sub(r"\\{[^{}]+\\}", "{}", path)\n\n\ndef locate_operation(document, method, path_template):\n    method = str(method).lower()\n    paths = document.get("paths", {})\n    if isinstance(paths.get(path_template), dict) and isinstance(\n        paths[path_template].get(method), dict\n    ):\n        return path_template, paths[path_template][method]\n    options = [\n        (path, item[method])\n        for path, item in paths.items()\n        if isinstance(item, dict) and isinstance(item.get(method), dict)\n        and path_shape(path) == path_shape(path_template)\n    ]\n    if len(options) != 1:\n        raise KeyError(f"Expected one canonical {method} {path_template}; found {len(options)}")\n    return options[0]\n\n\ndef response_object(document, operation, status):\n    responses = operation.get("responses", {})\n    requested = str(status)\n    if requested in responses:\n        key = requested\n    elif len(requested) == 3 and requested.isdigit() and requested[0] + "XX" in responses:\n        key = requested[0] + "XX"\n    elif "default" in responses:\n        key = "default"\n    else:\n        raise KeyError(f"Undocumented response status {requested}")\n    response, _ = schema_view(document, responses[key], "")\n    return key, response\n\n\ndef response_schema(document, matched_path, method, response_key, response):\n    root = join_pointer("/paths", matched_path, method, "responses", response_key)\n    content = response.get("content", {})\n    if isinstance(content, dict):\n        candidates = sorted(content, key=lambda item: (\n            0 if item == "application/json" else\n            1 if item.endswith("+json") else 2, item\n        ))\n        for media_type in candidates:\n            if media_type != "application/json" and not media_type.endswith("+json"):\n                continue  # Stage 05\'s qualified response-body scope is JSON.\n            media = content[media_type]\n            if isinstance(media, dict) and isinstance(media.get("schema"), dict):\n                return (media_type, media["schema"],\n                        join_pointer(root, "content", media_type, "schema"))\n    if isinstance(response.get("schema"), dict):\n        return None, response["schema"], join_pointer(root, "schema")\n    return None, None, root\n\n\ndef validator_for(document, schema):\n    is_31 = str(document.get("openapi", "3.0.0")).startswith("3.1")\n    validator_class = (\n        jsonschema.Draft202012Validator if is_31 else jsonschema.Draft4Validator\n    )\n    root = {\n        "$schema": (\n            "https://json-schema.org/draft/2020-12/schema" if is_31\n            else "http://json-schema.org/draft-04/schema#"\n        ),\n        "$ref": "#/$defs/stage05_response",\n        "$defs": {"stage05_response": schema},\n        "components": document.get("components", {}),\n        "definitions": document.get("definitions", {}),\n    }\n\n    def block_external(reference):\n        raise ValueError("External response schema retrieval prohibited: " + str(reference))\n\n    return validator_class(\n        root, registry=Registry(retrieve=block_external),\n        format_checker=jsonschema.FormatChecker(),\n    )\n\n\ndef validation_errors(document, schema, body):\n    if schema is None:\n        return [] if body is None else ["Body present without documented schema"]\n    errors = list(validator_for(document, schema).iter_errors(body))\n    return sorted(\n        {f"/{\'/\'.join(str(x) for x in error.path)}: {error.message}" for error in errors}\n    )\n\n\ndef requirement_ids_for(requirements, kinds, pointer, property_name=None):\n    results = []\n    for record in requirements:\n        if (\n            record.get("kind") not in kinds\n            or record.get("source_pointer") != pointer\n            or record.get("review_status") != "AUTO_VERIFIED"\n            or record.get("applicability") != "APPLICABLE"\n        ):\n            continue\n        if property_name is not None and record.get("expected_rule", {}).get("property") != property_name:\n            continue\n        results.append(record["requirement_id"])\n    return sorted(set(results))\n\n\ndef wrong_type_value(schema):\n    observed_type = schema.get("type")\n    if isinstance(observed_type, list):\n        observed_type = next((x for x in observed_type if x != "null"), observed_type[0])\n    choices = {\n        "object": [], "array": {}, "string": 12345,\n        "integer": "stage05_wrong_integer", "number": "stage05_wrong_number",\n        "boolean": "stage05_wrong_boolean", "null": 12345,\n    }\n    return choices.get(observed_type)\n\n\ndef invalid_enum_value(options):\n    index = 0\n    while True:\n        candidate = f"__stage05_invalid_enum_{index}__"\n        if all(canonical_json(candidate) != canonical_json(value) for value in options):\n            return candidate\n        index += 1\n\n\ndef _replace_at(root, path, value, delete=False):\n    result = deepcopy(root)\n    if not path:\n        if delete:\n            raise ValueError("Cannot delete a whole response")\n        return deepcopy(value)\n    target = result\n    for key in path[:-1]:\n        target = target[key]\n    if delete:\n        if isinstance(target, dict):\n            del target[path[-1]]\n        else:\n            target.pop(path[-1])\n    else:\n        target[path[-1]] = deepcopy(value)\n    return result\n\n\ndef enumerate_candidates(document, body, schema, schema_pointer, requirements,\n                         maximum_depth=7, maximum_candidates=512):\n    """Propose one-target mutations with an exact auto-verified rule pointer."""\n    proposals = []\n    emitted = set()\n\n    def propose(kind, path, replacement, pointer, rule_kind, mutation,\n                delete=False, property_name=None):\n        ids = requirement_ids_for(requirements, {rule_kind}, pointer, property_name)\n        if not ids:\n            return\n        candidate = _replace_at(body, path, replacement, delete=delete)\n        if canonical_json(candidate) == canonical_json(body):\n            return\n        identity = canonical_json({"body": candidate, "kind": kind, "pointer": pointer})\n        if identity in emitted:\n            return\n        emitted.add(identity)\n        proposals.append({\n            "fault_kind": kind, "body": candidate, "requirement_ids": ids,\n            "mutation": {"operation": kind, "target_path": list(path),\n                         "source_pointer": pointer, **mutation},\n        })\n\n    def visit(value, node, pointer, path, depth):\n        if depth > maximum_depth or len(proposals) >= maximum_candidates:\n            return\n        node, pointer = schema_view(document, node, pointer)\n\n        enum = node.get("enum")\n        if isinstance(enum, list) and enum:\n            propose("invalid_enum", path, invalid_enum_value(enum),\n                    join_pointer(pointer, "enum"), "schema_enum", {})\n\n        if node.get("type") is not None:\n            candidate = wrong_type_value(node)\n            if candidate is not None:\n                propose("wrong_type", path, candidate,\n                        join_pointer(pointer, "type"), "schema_type", {})\n\n        if isinstance(value, (int, float)) and not isinstance(value, bool):\n            for keyword, kind, sign in (\n                ("minimum", "below_minimum", -1),\n                ("maximum", "above_maximum", 1),\n                ("exclusiveMinimum", "below_exclusive_minimum", -1),\n                ("exclusiveMaximum", "above_exclusive_maximum", 1),\n            ):\n                bound = node.get(keyword)\n                if isinstance(bound, (int, float)) and not isinstance(bound, bool):\n                    candidate = bound + sign\n                    propose(kind, path, candidate, join_pointer(pointer, keyword),\n                            {"minimum": "numeric_minimum", "maximum": "numeric_maximum",\n                             "exclusiveMinimum": "numeric_exclusive_minimum",\n                             "exclusiveMaximum": "numeric_exclusive_maximum"}[keyword],\n                            {"bound": bound})\n\n        if isinstance(value, str):\n            minimum = node.get("minLength")\n            if isinstance(minimum, int) and minimum > 0:\n                propose("string_too_short", path, "x" * (minimum - 1),\n                        join_pointer(pointer, "minLength"), "string_min_length",\n                        {"bound": minimum})\n            maximum = node.get("maxLength")\n            if isinstance(maximum, int) and maximum >= 0:\n                propose("string_too_long", path, "x" * (maximum + 1),\n                        join_pointer(pointer, "maxLength"), "string_max_length",\n                        {"bound": maximum})\n            if isinstance(node.get("pattern"), str):\n                propose("pattern_violation", path, "stage05_pattern_violation!",\n                        join_pointer(pointer, "pattern"), "string_pattern",\n                        {"pattern": node["pattern"]})\n\n        if isinstance(value, dict):\n            required = node.get("required", [])\n            if isinstance(required, list):\n                for name in sorted(required):\n                    if name in value:\n                        propose("missing_required_property", path + (name,), None,\n                                join_pointer(pointer, "required"),\n                                "object_required_property", {"property": name},\n                                delete=True, property_name=name)\n            if node.get("additionalProperties") is False:\n                unused = "stage05_forbidden_extra"\n                while unused in value or unused in node.get("properties", {}):\n                    unused += "_"\n                propose("forbidden_additional_property", path + (unused,), True,\n                        join_pointer(pointer, "additionalProperties"),\n                        "additional_properties", {"property": unused})\n            properties = node.get("properties", {})\n            if isinstance(properties, dict):\n                for name in sorted(properties):\n                    child_pointer = join_pointer(pointer, "properties", name)\n                    if name in value:\n                        visit(value[name], properties[name], child_pointer,\n                              path + (name,), depth + 1)\n                    else:\n                        child_schema, child_pointer = schema_view(\n                            document, properties[name], child_pointer\n                        )\n                        enum = child_schema.get("enum")\n                        if isinstance(enum, list) and enum:\n                            propose("invalid_enum", path + (name,),\n                                    invalid_enum_value(enum),\n                                    join_pointer(child_pointer, "enum"),\n                                    "schema_enum", {"property": name,\n                                                    "optional_property_injected": True})\n                        elif child_schema.get("type") is not None:\n                            wrong = wrong_type_value(child_schema)\n                            if wrong is not None:\n                                propose("wrong_type", path + (name,), wrong,\n                                        join_pointer(child_pointer, "type"),\n                                        "schema_type", {"property": name,\n                                                        "optional_property_injected": True})\n\n        if isinstance(value, list):\n            minimum = node.get("minItems")\n            if isinstance(minimum, int) and minimum > 0 and value:\n                propose("array_too_short", path, value[:minimum - 1],\n                        join_pointer(pointer, "minItems"), "array_min_items",\n                        {"bound": minimum})\n            maximum = node.get("maxItems")\n            if isinstance(maximum, int):\n                proposed = list(value) + [value[-1] if value else None]\n                while len(proposed) <= maximum:\n                    proposed.append(proposed[-1] if proposed else None)\n                propose("array_too_long", path, proposed,\n                        join_pointer(pointer, "maxItems"), "array_max_items",\n                        {"bound": maximum})\n            if node.get("uniqueItems") is True and value:\n                propose("array_duplicate_item", path, value + [deepcopy(value[0])],\n                        join_pointer(pointer, "uniqueItems"), "array_unique_items", {})\n            items = node.get("items")\n            if isinstance(items, dict):\n                for index, item in enumerate(value[:3]):\n                    visit(item, items, join_pointer(pointer, "items"),\n                          path + (index,), depth + 1)\n\n        for composition in ("allOf", "anyOf", "oneOf"):\n            branches = node.get(composition)\n            if isinstance(branches, list):\n                for index, branch in enumerate(branches[:8]):\n                    visit(value, branch, join_pointer(pointer, composition, index),\n                          path, depth + 1)\n\n    visit(body, schema, schema_pointer, (), 0)\n    return proposals\n\n\ndef build_faults(document, op_id, method, path_template, baselines,\n                 requirements, maximum_per_baseline=32):\n    if maximum_per_baseline < 1:\n        raise ValueError("Fault cap must be positive")\n    matched_path, operation = locate_operation(document, method, path_template)\n    linked_requirements = [\n        row for row in requirements\n        if row.get("op_id") == op_id or "op_id" not in row\n    ]\n    faults = []\n    diagnostics = []\n\n    for baseline in sorted(baselines, key=lambda row: row["baseline_id"]):\n        baseline_id = baseline["baseline_id"]\n        if baseline.get("validation_status") != "PASS":\n            diagnostics.append({"severity": "BLOCKING", "code": "INVALID_BASELINE_STATUS",\n                                "op_id": op_id, "baseline_id": baseline_id})\n            continue\n        response_key, response = response_object(document, operation, baseline["status"])\n        media_type, schema, schema_pointer = response_schema(\n            document, matched_path, method, response_key, response\n        )\n        body = baseline.get("body")\n        baseline_errors = validation_errors(document, schema, body)\n        if baseline_errors:\n            diagnostics.append({"severity": "BLOCKING", "code": "INVALID_BASELINE_BODY",\n                                "op_id": op_id, "baseline_id": baseline_id,\n                                "errors": baseline_errors[:5]})\n            continue\n        candidates = []\n        if schema is not None:\n            candidates = enumerate_candidates(\n                document, body, schema, schema_pointer, linked_requirements\n            )\n        distinct_candidates = defaultdict(list)\n        seen_payloads = set()\n        for candidate in candidates:\n            candidate_body = candidate["body"]\n            payload_key = canonical_json(candidate_body)\n            if payload_key in seen_payloads or payload_key == canonical_json(body):\n                continue\n            errors = validation_errors(document, schema, candidate_body)\n            if not errors:\n                continue\n            seen_payloads.add(payload_key)\n            candidate["errors"] = errors\n            distinct_candidates[candidate["fault_kind"]].append(candidate)\n\n        # Fair, stable round-robin over fault kinds. A large number of wrong-type\n        # candidates must not starve a later required-property or enum fault.\n        prioritized = []\n        for offset in range(max((len(v) for v in distinct_candidates.values()), default=0)):\n            for kind in sorted(distinct_candidates):\n                group = distinct_candidates[kind]\n                if offset < len(group):\n                    prioritized.append(group[offset])\n                    if len(prioritized) >= maximum_per_baseline:\n                        break\n            if len(prioritized) >= maximum_per_baseline:\n                break\n\n        for candidate in prioritized:\n            identity = {\n                "builder": BUILDER_VERSION, "op_id": op_id,\n                "baseline_id": baseline_id, "mutation": candidate["mutation"],\n                "body": candidate["body"], "requirement_ids": candidate["requirement_ids"],\n            }\n            faults.append({\n                "fault_id": stable_id("fault", identity),\n                "op_id": op_id,\n                "baseline_id": baseline_id,\n                "scenario_id": baseline["scenario_id"],\n                "fault_kind": candidate["fault_kind"],\n                "mutation": candidate["mutation"],\n                "status": str(baseline["status"]),\n                "headers": deepcopy(baseline.get("headers", {})),\n                "body": candidate["body"],\n                "requirement_id": candidate["requirement_ids"][0],\n                "violated_requirement_ids": candidate["requirement_ids"],\n                "expected_detectable": True,\n                "applicability": "APPLICABLE",\n                "applicable": True,\n                "non_equivalent": True,\n                "validation_evidence": {\n                    "baseline_valid": True,\n                    "mutated_response_valid": False,\n                    "schema_pointer": schema_pointer,\n                    "rule_pointer": candidate["mutation"]["source_pointer"],\n                    "response_key": response_key,\n                    "media_type": media_type,\n                    "validation_basis": "offline_canonical_openapi_response_schema",\n                    "error_examples": candidate["errors"][:5],\n                },\n                "source_evidence": {\n                    "matched_path": matched_path,\n                    "response_key": response_key,\n                    "source_pointer": candidate["mutation"]["source_pointer"],\n                },\n                "validation_status": "PASS",\n            })\n\n        if not prioritized:\n            diagnostics.append({\n                "severity": "INFO", "code": "NO_EVIDENCE_LINKED_SCHEMA_INVALID_MUTATION",\n                "op_id": op_id, "baseline_id": baseline_id,\n                "response_key": response_key,\n                "explanation": "No automatically justified response fault; preserve for later review.",\n            })\n\n    return {\n        "builder_version": BUILDER_VERSION,\n        "op_id": op_id,\n        "matched_path": matched_path,\n        "faults": sorted(faults, key=lambda row: row["fault_id"]),\n        "diagnostics": sorted(diagnostics, key=lambda row: (\n            row.get("op_id", ""), row.get("baseline_id", ""), row.get("code", "")\n        )),\n        "status": "BLOCKED" if any(x["severity"] == "BLOCKING" for x in diagnostics) else "PASS",\n    }\n', 'oracle_requirements.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom copy import deepcopy\n\n\nHTTP_METHODS = {\n    "get",\n    "put",\n    "post",\n    "delete",\n    "options",\n    "head",\n    "patch",\n    "trace",\n}\n\nNORMATIVE_PATTERN = re.compile(\n    r"\\b("\n    r"must|shall|required|only|cannot|"\n    r"may\\s+not|should|at\\s+least|"\n    r"at\\s+most|between|invalid|valid"\n    r")\\b",\n    re.IGNORECASE,\n)\n\nSCHEMA_CONSTRAINTS = {\n    "type": "schema_type",\n    "enum": "schema_enum",\n    "const": "schema_const",\n    "minimum": "numeric_minimum",\n    "maximum": "numeric_maximum",\n    "exclusiveMinimum": (\n        "numeric_exclusive_minimum"\n    ),\n    "exclusiveMaximum": (\n        "numeric_exclusive_maximum"\n    ),\n    "multipleOf": "numeric_multiple_of",\n    "minLength": "string_min_length",\n    "maxLength": "string_max_length",\n    "pattern": "string_pattern",\n    "format": "string_format",\n    "minItems": "array_min_items",\n    "maxItems": "array_max_items",\n    "uniqueItems": "array_unique_items",\n    "minProperties": (\n        "object_min_properties"\n    ),\n    "maxProperties": (\n        "object_max_properties"\n    ),\n    "nullable": "nullable",\n    "readOnly": "read_only",\n    "writeOnly": "write_only",\n}\n\nREVIEW_EXTENSION_NAMES = {\n    "x-ms-enum",\n    "x-ms-mutability",\n    "x-nullable",\n    "x-required",\n    "x-oneOf",\n    "x-anyOf",\n}\n\n\nclass ExternalReferenceError(\n    RuntimeError\n):\n    pass\n\n\ndef canonical_bytes(value) -> bytes:\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    ).encode("utf-8")\n\n\ndef stable_identifier(\n    prefix: str,\n    seed,\n) -> str:\n    digest = hashlib.sha256(\n        canonical_bytes(seed)\n    ).hexdigest()[:24]\n\n    return f"{prefix}_{digest}"\n\n\ndef pointer_escape(value: str) -> str:\n    return (\n        value.replace("~", "~0")\n        .replace("/", "~1")\n    )\n\n\ndef pointer_unescape(value: str) -> str:\n    return (\n        value.replace("~1", "/")\n        .replace("~0", "~")\n    )\n\n\ndef join_pointer(\n    base: str,\n    *tokens,\n) -> str:\n    result = base.rstrip("/")\n\n    for token in tokens:\n        result += (\n            "/"\n            + pointer_escape(str(token))\n        )\n\n    return result or "/"\n\n\nLOCAL_REFERENCE_RESOLVER_VERSION = "local-json-pointer-and-anchor-v1"\n\n\ndef resolve_local_reference(\n    document: dict,\n    reference: str,\n):\n    from urllib.parse import unquote\n\n    if not isinstance(reference, str):\n        raise TypeError(\n            "$ref must be a string."\n        )\n\n    if reference == "#":\n        return document, "/"\n\n    if not reference.startswith("#"):\n        raise ExternalReferenceError(\n            "External reference is prohibited: "\n            f"{reference}"\n        )\n\n    fragment = unquote(\n        reference[1:]\n    )\n\n    if fragment.startswith("/"):\n        node = document\n\n        for encoded_token in (\n            fragment[1:].split("/")\n        ):\n            token = pointer_unescape(\n                unquote(encoded_token)\n            )\n\n            if isinstance(node, dict):\n                if token not in node:\n                    raise KeyError(\n                        "Unresolved local reference "\n                        f"token {token!r} in "\n                        f"{reference}"\n                    )\n\n                node = node[token]\n\n            elif isinstance(node, list):\n                node = node[int(token)]\n\n            else:\n                raise TypeError(\n                    "Reference traversed into a "\n                    "non-container value: "\n                    f"{reference}"\n                )\n\n        return node, fragment\n\n    if not fragment:\n        return document, "/"\n\n    anchor_matches = []\n\n    def visit(\n        value,\n        pointer: str,\n    ):\n        if isinstance(value, dict):\n            if (\n                value.get("$anchor")\n                == fragment\n                or value.get(\n                    "$dynamicAnchor"\n                )\n                == fragment\n            ):\n                anchor_matches.append(\n                    (value, pointer)\n                )\n\n            for key, nested_value in (\n                value.items()\n            ):\n                visit(\n                    nested_value,\n                    join_pointer(\n                        pointer,\n                        key,\n                    ),\n                )\n\n        elif isinstance(value, list):\n            for index, nested_value in (\n                enumerate(value)\n            ):\n                visit(\n                    nested_value,\n                    join_pointer(\n                        pointer,\n                        index,\n                    ),\n                )\n\n    visit(document, "")\n\n    if len(anchor_matches) != 1:\n        raise KeyError(\n            "Local anchor reference must resolve "\n            "to exactly one object: "\n            f"{reference}; matches="\n            f"{len(anchor_matches)}"\n        )\n\n    return anchor_matches[0]\n\ndef resolve_object(\n    document: dict,\n    value,\n    pointer: str,\n):\n    if (\n        not isinstance(value, dict)\n        or "$ref" not in value\n    ):\n        return value, pointer\n\n    reference = value["$ref"]\n\n    if not isinstance(reference, str):\n        raise TypeError(\n            "$ref must be a string."\n        )\n\n    target, target_pointer = (\n        resolve_local_reference(\n            document,\n            reference,\n        )\n    )\n\n    if not isinstance(target, dict):\n        raise TypeError(\n            "Referenced object is not a "\n            "mapping."\n        )\n\n    siblings = {\n        key: deepcopy(nested_value)\n        for key, nested_value\n        in value.items()\n        if key != "$ref"\n    }\n\n    if siblings:\n        merged = deepcopy(target)\n        merged.update(siblings)\n        return merged, target_pointer\n\n    return deepcopy(target), target_pointer\n\n\ndef extract_operation_requirements(\n    document: dict,\n    op_id: str,\n    path_template: str,\n    method: str,\n) -> dict:\n    if not isinstance(document, dict):\n        raise TypeError(\n            "OpenAPI document must be "\n            "an object."\n        )\n\n    normalized_method = method.lower()\n\n    if normalized_method not in (\n        HTTP_METHODS\n    ):\n        raise ValueError(\n            f"Unsupported method: {method}"\n        )\n\n    paths_object = document.get("paths")\n\n    if not isinstance(\n        paths_object,\n        dict,\n    ):\n        raise ValueError(\n            "Document has no paths object."\n        )\n\n    path_item = paths_object.get(\n        path_template\n    )\n\n    if not isinstance(path_item, dict):\n        raise KeyError(\n            f"Path not found: {path_template}"\n        )\n\n    operation = path_item.get(\n        normalized_method\n    )\n\n    if not isinstance(operation, dict):\n        raise KeyError(\n            "Operation not found: "\n            f"{normalized_method} "\n            f"{path_template}"\n        )\n\n    operation_pointer = (\n        "/paths/"\n        + pointer_escape(path_template)\n        + "/"\n        + normalized_method\n    )\n    path_pointer = (\n        "/paths/"\n        + pointer_escape(path_template)\n    )\n\n    requirements = []\n    interface_items = []\n    diagnostics = []\n\n    seen_requirements = set()\n    seen_interface_items = set()\n\n    def add_diagnostic(\n        severity: str,\n        code: str,\n        pointer: str,\n        message: str,\n    ):\n        diagnostics.append(\n            {\n                "severity": severity,\n                "code": code,\n                "source_pointer": pointer,\n                "message": message,\n            }\n        )\n\n    def add_requirement(\n        pointer: str,\n        kind: str,\n        expected_rule: dict,\n        applicability: str = (\n            "APPLICABLE"\n        ),\n        evidence_type: str = (\n            "canonical_structured_contract"\n        ),\n        review_status: str = (\n            "AUTO_VERIFIED"\n        ),\n        condition=None,\n        diagnostic=None,\n    ):\n        seed = {\n            "op_id": op_id,\n            "source_pointer": pointer,\n            "kind": kind,\n            "expected_rule": expected_rule,\n            "applicability": applicability,\n            "evidence_type": (\n                evidence_type\n            ),\n        }\n\n        deduplication_key = (\n            canonical_bytes(seed)\n        )\n\n        if (\n            deduplication_key\n            in seen_requirements\n        ):\n            return\n\n        seen_requirements.add(\n            deduplication_key\n        )\n\n        requirements.append(\n            {\n                "requirement_id": (\n                    stable_identifier(\n                        "req",\n                        seed,\n                    )\n                ),\n                "op_id": op_id,\n                "source_pointer": pointer,\n                "kind": kind,\n                "expected_rule": (\n                    expected_rule\n                ),\n                "applicability": (\n                    applicability\n                ),\n                "evidence_type": (\n                    evidence_type\n                ),\n                "review_status": (\n                    review_status\n                ),\n                "condition": condition,\n                "evidence_sha256": (\n                    hashlib.sha256(\n                        canonical_bytes(\n                            {\n                                "pointer": (\n                                    pointer\n                                ),\n                                "kind": kind,\n                                "rule": (\n                                    expected_rule\n                                ),\n                            }\n                        )\n                    ).hexdigest()\n                ),\n                "diagnostic": diagnostic,\n            }\n        )\n\n    def add_interface_item(\n        pointer: str,\n        item_kind: str,\n        name: str,\n        location: str,\n        required: bool,\n        schema,\n        review_status: str = (\n            "AUTO_VERIFIED"\n        ),\n        diagnostic=None,\n    ):\n        normalized_schema = schema\n\n        if isinstance(schema, bool):\n            normalized_schema = {\n                "boolean_schema": schema,\n            }\n\n        if (\n            normalized_schema is not None\n            and not isinstance(\n                normalized_schema,\n                dict,\n            )\n        ):\n            normalized_schema = {\n                "observed_schema": (\n                    normalized_schema\n                ),\n            }\n\n        seed = {\n            "op_id": op_id,\n            "source_pointer": pointer,\n            "item_kind": item_kind,\n            "name": name,\n            "location": location,\n        }\n\n        deduplication_key = (\n            canonical_bytes(seed)\n        )\n\n        if (\n            deduplication_key\n            in seen_interface_items\n        ):\n            return\n\n        seen_interface_items.add(\n            deduplication_key\n        )\n\n        interface_items.append(\n            {\n                "interface_item_id": (\n                    stable_identifier(\n                        "iface",\n                        seed,\n                    )\n                ),\n                "op_id": op_id,\n                "item_kind": item_kind,\n                "source_pointer": pointer,\n                "name": name,\n                "location": location,\n                "required": bool(\n                    required\n                ),\n                "schema": (\n                    normalized_schema\n                ),\n                "review_status": (\n                    review_status\n                ),\n                "diagnostic": diagnostic,\n            }\n        )\n\n    def inspect_prose(\n        value,\n        pointer: str,\n        field_name: str,\n    ):\n        if not isinstance(value, str):\n            if value is not None:\n                add_diagnostic(\n                    "REVIEW",\n                    "NONSTRING_PROSE",\n                    pointer,\n                    "Prose annotation is not "\n                    "a string.",\n                )\n            return\n\n        normalized = value.strip()\n\n        if (\n            normalized\n            and NORMATIVE_PATTERN.search(\n                normalized\n            )\n        ):\n            add_requirement(\n                pointer=pointer,\n                kind=(\n                    "normative_prose_candidate"\n                ),\n                expected_rule={\n                    "field_name": (\n                        field_name\n                    ),\n                    "text": normalized,\n                    "interpretation": (\n                        "HUMAN_REVIEW_REQUIRED"\n                    ),\n                },\n                applicability="CONDITIONAL",\n                evidence_type=(\n                    "normative_prose"\n                ),\n                review_status=(\n                    "REVIEW_REQUIRED"\n                ),\n                diagnostic=(\n                    "Normative-looking prose "\n                    "was preserved without "\n                    "automatic interpretation."\n                ),\n            )\n\n    def inspect_extensions(\n        value: dict,\n        pointer: str,\n    ):\n        for key, nested_value in (\n            value.items()\n        ):\n            if key in (\n                REVIEW_EXTENSION_NAMES\n            ):\n                add_requirement(\n                    pointer=join_pointer(\n                        pointer,\n                        key,\n                    ),\n                    kind=(\n                        "extension_semantics_"\n                        "candidate"\n                    ),\n                    expected_rule={\n                        "extension": key,\n                        "value": nested_value,\n                        "interpretation": (\n                            "HUMAN_REVIEW_REQUIRED"\n                        ),\n                    },\n                    applicability=(\n                        "CONDITIONAL"\n                    ),\n                    evidence_type=(\n                        "canonical_structured_"\n                        "contract"\n                    ),\n                    review_status=(\n                        "REVIEW_REQUIRED"\n                    ),\n                    diagnostic=(\n                        "Extension semantics "\n                        "require review."\n                    ),\n                )\n\n    def walk_schema(\n        schema,\n        pointer: str,\n        context: str,\n        applicability: str = (\n            "APPLICABLE"\n        ),\n        reference_stack=(),\n    ):\n        if isinstance(schema, bool):\n            add_requirement(\n                pointer=pointer,\n                kind="schema_boolean",\n                expected_rule={\n                    "value": schema,\n                    "context": context,\n                },\n                applicability=applicability,\n            )\n            return\n\n        if not isinstance(schema, dict):\n            add_diagnostic(\n                "BLOCKING",\n                "NONOBJECT_SCHEMA",\n                pointer,\n                "Schema is neither an "\n                "object nor a boolean.",\n            )\n            return\n\n        if "$ref" in schema:\n            reference = schema["$ref"]\n\n            if not isinstance(\n                reference,\n                str,\n            ):\n                add_diagnostic(\n                    "BLOCKING",\n                    "NONSTRING_REFERENCE",\n                    join_pointer(\n                        pointer,\n                        "$ref",\n                    ),\n                    "$ref is not a string.",\n                )\n                return\n\n            if reference in reference_stack:\n                add_diagnostic(\n                    "INFO",\n                    "RECURSIVE_SCHEMA_REFERENCE",\n                    pointer,\n                    "Recursive schema reference "\n                    "was detected and bounded.",\n                )\n            else:\n                try:\n                    target, target_pointer = (\n                        resolve_local_reference(\n                            document,\n                            reference,\n                        )\n                    )\n\n                    walk_schema(\n                        target,\n                        target_pointer,\n                        context,\n                        applicability,\n                        reference_stack\n                        + (reference,),\n                    )\n\n                except Exception as error:\n                    add_diagnostic(\n                        "BLOCKING",\n                        (\n                            "UNRESOLVED_OR_"\n                            "EXTERNAL_REFERENCE"\n                        ),\n                        pointer,\n                        (\n                            f"{type(error).__name__}: "\n                            f"{error}"\n                        ),\n                    )\n\n            sibling_schema = {\n                key: value\n                for key, value\n                in schema.items()\n                if key != "$ref"\n            }\n\n            if sibling_schema:\n                walk_schema(\n                    sibling_schema,\n                    pointer,\n                    context,\n                    applicability,\n                    reference_stack,\n                )\n\n            return\n\n        for keyword, kind in (\n            SCHEMA_CONSTRAINTS.items()\n        ):\n            if keyword not in schema:\n                continue\n\n            add_requirement(\n                pointer=join_pointer(\n                    pointer,\n                    keyword,\n                ),\n                kind=kind,\n                expected_rule={\n                    "keyword": keyword,\n                    "value": schema[\n                        keyword\n                    ],\n                    "context": context,\n                },\n                applicability=applicability,\n            )\n\n        required_properties = (\n            schema.get("required")\n        )\n\n        if isinstance(\n            required_properties,\n            list,\n        ):\n            for property_name in (\n                required_properties\n            ):\n                if not isinstance(\n                    property_name,\n                    str,\n                ):\n                    add_diagnostic(\n                        "BLOCKING",\n                        (\n                            "NONSTRING_REQUIRED_"\n                            "PROPERTY"\n                        ),\n                        join_pointer(\n                            pointer,\n                            "required",\n                        ),\n                        "Required-property name "\n                        "is not a string.",\n                    )\n                    continue\n\n                add_requirement(\n                    pointer=join_pointer(\n                        pointer,\n                        "required",\n                    ),\n                    kind=(\n                        "object_required_property"\n                    ),\n                    expected_rule={\n                        "property": (\n                            property_name\n                        ),\n                        "context": context,\n                    },\n                    applicability=(\n                        applicability\n                    ),\n                )\n\n        if "additionalProperties" in schema:\n            additional_properties = (\n                schema[\n                    "additionalProperties"\n                ]\n            )\n\n            add_requirement(\n                pointer=join_pointer(\n                    pointer,\n                    "additionalProperties",\n                ),\n                kind=(\n                    "additional_properties"\n                ),\n                expected_rule={\n                    "value": (\n                        additional_properties\n                    ),\n                    "context": context,\n                },\n                applicability=applicability,\n            )\n\n            if isinstance(\n                additional_properties,\n                dict,\n            ):\n                walk_schema(\n                    additional_properties,\n                    join_pointer(\n                        pointer,\n                        "additionalProperties",\n                    ),\n                    context,\n                    applicability,\n                    reference_stack,\n                )\n\n        properties = schema.get(\n            "properties"\n        )\n\n        if isinstance(properties, dict):\n            for property_name, (\n                property_schema\n            ) in properties.items():\n                walk_schema(\n                    property_schema,\n                    join_pointer(\n                        pointer,\n                        "properties",\n                        property_name,\n                    ),\n                    context,\n                    applicability,\n                    reference_stack,\n                )\n\n        if "items" in schema:\n            walk_schema(\n                schema["items"],\n                join_pointer(\n                    pointer,\n                    "items",\n                ),\n                context,\n                applicability,\n                reference_stack,\n            )\n\n        prefix_items = schema.get(\n            "prefixItems"\n        )\n\n        if isinstance(\n            prefix_items,\n            list,\n        ):\n            for index, item_schema in (\n                enumerate(prefix_items)\n            ):\n                walk_schema(\n                    item_schema,\n                    join_pointer(\n                        pointer,\n                        "prefixItems",\n                        index,\n                    ),\n                    context,\n                    applicability,\n                    reference_stack,\n                )\n\n        all_of = schema.get("allOf")\n\n        if isinstance(all_of, list):\n            for index, branch in (\n                enumerate(all_of)\n            ):\n                walk_schema(\n                    branch,\n                    join_pointer(\n                        pointer,\n                        "allOf",\n                        index,\n                    ),\n                    context,\n                    applicability,\n                    reference_stack,\n                )\n\n        for combinator in [\n            "oneOf",\n            "anyOf",\n        ]:\n            branches = schema.get(\n                combinator\n            )\n\n            if not isinstance(\n                branches,\n                list,\n            ):\n                continue\n\n            add_requirement(\n                pointer=join_pointer(\n                    pointer,\n                    combinator,\n                ),\n                kind=(\n                    "schema_composition_review"\n                ),\n                expected_rule={\n                    "combinator": combinator,\n                    "branch_count": len(\n                        branches\n                    ),\n                    "context": context,\n                },\n                applicability="CONDITIONAL",\n                review_status=(\n                    "REVIEW_REQUIRED"\n                ),\n                diagnostic=(\n                    "Branch selection semantics "\n                    "require review."\n                ),\n            )\n\n            for index, branch in (\n                enumerate(branches)\n            ):\n                walk_schema(\n                    branch,\n                    join_pointer(\n                        pointer,\n                        combinator,\n                        index,\n                    ),\n                    context,\n                    "CONDITIONAL",\n                    reference_stack,\n                )\n\n        if "not" in schema:\n            add_requirement(\n                pointer=join_pointer(\n                    pointer,\n                    "not",\n                ),\n                kind=(\n                    "schema_negation_review"\n                ),\n                expected_rule={\n                    "context": context,\n                    "schema": schema["not"],\n                },\n                applicability="CONDITIONAL",\n                review_status=(\n                    "REVIEW_REQUIRED"\n                ),\n                diagnostic=(\n                    "Negated schema semantics "\n                    "require review."\n                ),\n            )\n\n        if "discriminator" in schema:\n            add_requirement(\n                pointer=join_pointer(\n                    pointer,\n                    "discriminator",\n                ),\n                kind=(\n                    "discriminator_review"\n                ),\n                expected_rule={\n                    "context": context,\n                    "discriminator": (\n                        schema[\n                            "discriminator"\n                        ]\n                    ),\n                },\n                applicability="CONDITIONAL",\n                review_status=(\n                    "REVIEW_REQUIRED"\n                ),\n                diagnostic=(\n                    "Discriminator semantics "\n                    "require review."\n                ),\n            )\n\n        inspect_prose(\n            schema.get("description"),\n            join_pointer(\n                pointer,\n                "description",\n            ),\n            "description",\n        )\n        inspect_extensions(\n            schema,\n            pointer,\n        )\n\n    def effective_parameters():\n        combined = {}\n\n        parameter_sources = [\n            (\n                path_item.get(\n                    "parameters",\n                    [],\n                ),\n                join_pointer(\n                    path_pointer,\n                    "parameters",\n                ),\n            ),\n            (\n                operation.get(\n                    "parameters",\n                    [],\n                ),\n                join_pointer(\n                    operation_pointer,\n                    "parameters",\n                ),\n            ),\n        ]\n\n        for parameters, (\n            parameters_pointer\n        ) in parameter_sources:\n            if parameters is None:\n                continue\n\n            if not isinstance(\n                parameters,\n                list,\n            ):\n                add_diagnostic(\n                    "BLOCKING",\n                    "PARAMETERS_NOT_ARRAY",\n                    parameters_pointer,\n                    "Parameters must be an "\n                    "array.",\n                )\n                continue\n\n            for index, raw_parameter in (\n                enumerate(parameters)\n            ):\n                raw_pointer = join_pointer(\n                    parameters_pointer,\n                    index,\n                )\n\n                try:\n                    parameter, (\n                        source_pointer\n                    ) = resolve_object(\n                        document,\n                        raw_parameter,\n                        raw_pointer,\n                    )\n                except Exception as error:\n                    add_diagnostic(\n                        "BLOCKING",\n                        (\n                            "PARAMETER_REFERENCE_"\n                            "FAILURE"\n                        ),\n                        raw_pointer,\n                        (\n                            f"{type(error).__name__}: "\n                            f"{error}"\n                        ),\n                    )\n                    continue\n\n                if not isinstance(\n                    parameter,\n                    dict,\n                ):\n                    add_diagnostic(\n                        "BLOCKING",\n                        (\n                            "PARAMETER_NOT_"\n                            "OBJECT"\n                        ),\n                        source_pointer,\n                        "Parameter is not an "\n                        "object.",\n                    )\n                    continue\n\n                name = parameter.get("name")\n                location = parameter.get(\n                    "in"\n                )\n\n                if (\n                    not isinstance(name, str)\n                    or not isinstance(\n                        location,\n                        str,\n                    )\n                ):\n                    add_diagnostic(\n                        "BLOCKING",\n                        (\n                            "PARAMETER_IDENTITY_"\n                            "MISSING"\n                        ),\n                        source_pointer,\n                        "Parameter name or "\n                        "location is missing.",\n                    )\n                    continue\n\n                combined[\n                    (name, location)\n                ] = (\n                    parameter,\n                    source_pointer,\n                )\n\n        return [\n            (\n                key,\n                value[0],\n                value[1],\n            )\n            for key, value\n            in sorted(\n                combined.items(),\n                key=lambda item: item[0],\n            )\n        ]\n\n    inspect_prose(\n        operation.get("summary"),\n        join_pointer(\n            operation_pointer,\n            "summary",\n        ),\n        "summary",\n    )\n    inspect_prose(\n        operation.get("description"),\n        join_pointer(\n            operation_pointer,\n            "description",\n        ),\n        "description",\n    )\n    inspect_extensions(\n        operation,\n        operation_pointer,\n    )\n\n    for (\n        parameter_identity,\n        parameter,\n        parameter_pointer,\n    ) in effective_parameters():\n        name, location = (\n            parameter_identity\n        )\n\n        required = bool(\n            parameter.get(\n                "required",\n                False,\n            )\n        )\n\n        if location == "path":\n            required = True\n\n        schema = parameter.get(\n            "schema"\n        )\n\n        add_interface_item(\n            pointer=parameter_pointer,\n            item_kind="parameter",\n            name=name,\n            location=location,\n            required=required,\n            schema=(\n                schema\n                if isinstance(\n                    schema,\n                    (dict, bool),\n                )\n                else None\n            ),\n        )\n\n        add_requirement(\n            pointer=parameter_pointer,\n            kind="parameter_presence",\n            expected_rule={\n                "name": name,\n                "location": location,\n            },\n        )\n\n        add_requirement(\n            pointer=join_pointer(\n                parameter_pointer,\n                "in",\n            ),\n            kind="parameter_location",\n            expected_rule={\n                "name": name,\n                "location": location,\n            },\n        )\n\n        if required:\n            add_requirement(\n                pointer=join_pointer(\n                    parameter_pointer,\n                    "required",\n                ),\n                kind="parameter_required",\n                expected_rule={\n                    "name": name,\n                    "location": location,\n                    "required": True,\n                },\n            )\n\n        serialization_rule = {\n            key: parameter[key]\n            for key in [\n                "style",\n                "explode",\n                "allowEmptyValue",\n                "allowReserved",\n            ]\n            if key in parameter\n        }\n\n        if serialization_rule:\n            add_requirement(\n                pointer=parameter_pointer,\n                kind=(\n                    "parameter_serialization"\n                ),\n                expected_rule={\n                    "name": name,\n                    "location": location,\n                    **serialization_rule,\n                },\n            )\n\n        if isinstance(\n            schema,\n            (dict, bool),\n        ):\n            walk_schema(\n                schema,\n                join_pointer(\n                    parameter_pointer,\n                    "schema",\n                ),\n                (\n                    "parameter:"\n                    f"{location}:{name}"\n                ),\n            )\n\n        elif isinstance(\n            parameter.get("content"),\n            dict,\n        ):\n            content = parameter[\n                "content"\n            ]\n\n            if len(content) != 1:\n                add_requirement(\n                    pointer=join_pointer(\n                        parameter_pointer,\n                        "content",\n                    ),\n                    kind=(\n                        "parameter_content_review"\n                    ),\n                    expected_rule={\n                        "name": name,\n                        "media_types": sorted(\n                            content\n                        ),\n                    },\n                    applicability=(\n                        "CONDITIONAL"\n                    ),\n                    review_status=(\n                        "REVIEW_REQUIRED"\n                    ),\n                )\n\n            for media_type, media_object in (\n                content.items()\n            ):\n                if isinstance(\n                    media_object,\n                    dict,\n                ):\n                    media_schema = (\n                        media_object.get(\n                            "schema"\n                        )\n                    )\n\n                    if isinstance(\n                        media_schema,\n                        (dict, bool),\n                    ):\n                        walk_schema(\n                            media_schema,\n                            join_pointer(\n                                parameter_pointer,\n                                "content",\n                                media_type,\n                                "schema",\n                            ),\n                            (\n                                "parameter:"\n                                f"{location}:"\n                                f"{name}"\n                            ),\n                        )\n\n        else:\n            add_diagnostic(\n                "REVIEW",\n                (\n                    "PARAMETER_SCHEMA_"\n                    "MISSING"\n                ),\n                parameter_pointer,\n                "Parameter has neither schema "\n                "nor usable content.",\n            )\n\n        inspect_prose(\n            parameter.get(\n                "description"\n            ),\n            join_pointer(\n                parameter_pointer,\n                "description",\n            ),\n            "description",\n        )\n        inspect_extensions(\n            parameter,\n            parameter_pointer,\n        )\n\n    raw_request_body = operation.get(\n        "requestBody"\n    )\n\n    if raw_request_body is not None:\n        try:\n            request_body, (\n                request_body_pointer\n            ) = resolve_object(\n                document,\n                raw_request_body,\n                join_pointer(\n                    operation_pointer,\n                    "requestBody",\n                ),\n            )\n\n            if not isinstance(\n                request_body,\n                dict,\n            ):\n                raise TypeError(\n                    "Request body is not "\n                    "an object."\n                )\n\n            request_required = bool(\n                request_body.get(\n                    "required",\n                    False,\n                )\n            )\n\n            if request_required:\n                add_requirement(\n                    pointer=join_pointer(\n                        request_body_pointer,\n                        "required",\n                    ),\n                    kind=(\n                        "request_body_required"\n                    ),\n                    expected_rule={\n                        "required": True,\n                    },\n                )\n\n            content = request_body.get(\n                "content"\n            )\n\n            if not isinstance(\n                content,\n                dict,\n            ) or not content:\n                add_diagnostic(\n                    "BLOCKING",\n                    (\n                        "REQUEST_BODY_CONTENT_"\n                        "MISSING"\n                    ),\n                    request_body_pointer,\n                    "Request body has no "\n                    "content mapping.",\n                )\n            else:\n                for (\n                    media_type,\n                    media_object,\n                ) in sorted(\n                    content.items()\n                ):\n                    schema = (\n                        media_object.get(\n                            "schema"\n                        )\n                        if isinstance(\n                            media_object,\n                            dict,\n                        )\n                        else None\n                    )\n\n                    add_interface_item(\n                        pointer=join_pointer(\n                            request_body_pointer,\n                            "content",\n                            media_type,\n                        ),\n                        item_kind=(\n                            "request_body"\n                        ),\n                        name="requestBody",\n                        location=media_type,\n                        required=(\n                            request_required\n                        ),\n                        schema=schema,\n                    )\n\n                    if isinstance(\n                        schema,\n                        (dict, bool),\n                    ):\n                        walk_schema(\n                            schema,\n                            join_pointer(\n                                request_body_pointer,\n                                "content",\n                                media_type,\n                                "schema",\n                            ),\n                            (\n                                "request_body:"\n                                f"{media_type}"\n                            ),\n                        )\n                    else:\n                        add_diagnostic(\n                            "REVIEW",\n                            (\n                                "REQUEST_SCHEMA_"\n                                "MISSING"\n                            ),\n                            join_pointer(\n                                request_body_pointer,\n                                "content",\n                                media_type,\n                            ),\n                            "Request media type "\n                            "has no schema.",\n                        )\n\n            inspect_prose(\n                request_body.get(\n                    "description"\n                ),\n                join_pointer(\n                    request_body_pointer,\n                    "description",\n                ),\n                "description",\n            )\n\n        except Exception as error:\n            add_diagnostic(\n                "BLOCKING",\n                (\n                    "REQUEST_BODY_REFERENCE_"\n                    "FAILURE"\n                ),\n                join_pointer(\n                    operation_pointer,\n                    "requestBody",\n                ),\n                (\n                    f"{type(error).__name__}: "\n                    f"{error}"\n                ),\n            )\n\n    responses = operation.get(\n        "responses"\n    )\n\n    if not isinstance(responses, dict):\n        add_diagnostic(\n            "BLOCKING",\n            "RESPONSES_MISSING",\n            join_pointer(\n                operation_pointer,\n                "responses",\n            ),\n            "Operation responses are missing "\n            "or invalid.",\n        )\n\n    else:\n        for status_key, raw_response in (\n            sorted(responses.items())\n        ):\n            response_pointer = join_pointer(\n                operation_pointer,\n                "responses",\n                status_key,\n            )\n\n            try:\n                response, (\n                    resolved_response_pointer\n                ) = resolve_object(\n                    document,\n                    raw_response,\n                    response_pointer,\n                )\n\n                if not isinstance(\n                    response,\n                    dict,\n                ):\n                    raise TypeError(\n                        "Response is not an "\n                        "object."\n                    )\n\n                add_requirement(\n                    pointer=response_pointer,\n                    kind=(\n                        "documented_response_status"\n                    ),\n                    expected_rule={\n                        "status": status_key,\n                    },\n                )\n\n                inspect_prose(\n                    response.get(\n                        "description"\n                    ),\n                    join_pointer(\n                        resolved_response_pointer,\n                        "description",\n                    ),\n                    "description",\n                )\n\n                response_headers = (\n                    response.get(\n                        "headers",\n                        {},\n                    )\n                )\n\n                if isinstance(\n                    response_headers,\n                    dict,\n                ):\n                    for (\n                        header_name,\n                        raw_header,\n                    ) in sorted(\n                        response_headers.items()\n                    ):\n                        header_pointer = (\n                            join_pointer(\n                                resolved_response_pointer,\n                                "headers",\n                                header_name,\n                            )\n                        )\n\n                        try:\n                            header, (\n                                resolved_header_pointer\n                            ) = resolve_object(\n                                document,\n                                raw_header,\n                                header_pointer,\n                            )\n                        except Exception as error:\n                            add_diagnostic(\n                                "BLOCKING",\n                                (\n                                    "HEADER_REFERENCE_"\n                                    "FAILURE"\n                                ),\n                                header_pointer,\n                                (\n                                    f"{type(error).__name__}: "\n                                    f"{error}"\n                                ),\n                            )\n                            continue\n\n                        if not isinstance(\n                            header,\n                            dict,\n                        ):\n                            continue\n\n                        header_schema = (\n                            header.get(\n                                "schema"\n                            )\n                        )\n\n                        add_interface_item(\n                            pointer=(\n                                resolved_header_pointer\n                            ),\n                            item_kind=(\n                                "response_header"\n                            ),\n                            name=header_name,\n                            location=(\n                                f"response:"\n                                f"{status_key}"\n                            ),\n                            required=False,\n                            schema=(\n                                header_schema\n                            ),\n                        )\n\n                        if isinstance(\n                            header_schema,\n                            (dict, bool),\n                        ):\n                            walk_schema(\n                                header_schema,\n                                join_pointer(\n                                    resolved_header_pointer,\n                                    "schema",\n                                ),\n                                (\n                                    "response_header:"\n                                    f"{status_key}:"\n                                    f"{header_name}"\n                                ),\n                            )\n\n                response_content = (\n                    response.get(\n                        "content",\n                        {},\n                    )\n                )\n\n                if isinstance(\n                    response_content,\n                    dict,\n                ):\n                    for (\n                        media_type,\n                        media_object,\n                    ) in sorted(\n                        response_content.items()\n                    ):\n                        media_pointer = (\n                            join_pointer(\n                                resolved_response_pointer,\n                                "content",\n                                media_type,\n                            )\n                        )\n\n                        add_requirement(\n                            pointer=media_pointer,\n                            kind=(\n                                "response_media_type"\n                            ),\n                            expected_rule={\n                                "status": status_key,\n                                "media_type": (\n                                    media_type\n                                ),\n                            },\n                        )\n\n                        response_schema = (\n                            media_object.get(\n                                "schema"\n                            )\n                            if isinstance(\n                                media_object,\n                                dict,\n                            )\n                            else None\n                        )\n\n                        add_interface_item(\n                            pointer=media_pointer,\n                            item_kind=(\n                                "response_body"\n                            ),\n                            name=(\n                                f"response_"\n                                f"{status_key}"\n                            ),\n                            location=media_type,\n                            required=False,\n                            schema=(\n                                response_schema\n                            ),\n                        )\n\n                        if isinstance(\n                            response_schema,\n                            (dict, bool),\n                        ):\n                            walk_schema(\n                                response_schema,\n                                join_pointer(\n                                    media_pointer,\n                                    "schema",\n                                ),\n                                (\n                                    "response_body:"\n                                    f"{status_key}:"\n                                    f"{media_type}"\n                                ),\n                            )\n\n            except Exception as error:\n                add_diagnostic(\n                    "BLOCKING",\n                    (\n                        "RESPONSE_REFERENCE_"\n                        "FAILURE"\n                    ),\n                    response_pointer,\n                    (\n                        f"{type(error).__name__}: "\n                        f"{error}"\n                    ),\n                )\n\n    if "security" in operation:\n        effective_security = operation[\n            "security"\n        ]\n        security_pointer = join_pointer(\n            operation_pointer,\n            "security",\n        )\n    else:\n        effective_security = (\n            document.get("security")\n        )\n        security_pointer = "/security"\n\n    if effective_security is not None:\n        if not isinstance(\n            effective_security,\n            list,\n        ):\n            add_diagnostic(\n                "BLOCKING",\n                "SECURITY_NOT_ARRAY",\n                security_pointer,\n                "Security requirements must "\n                "be an array.",\n            )\n        else:\n            add_requirement(\n                pointer=security_pointer,\n                kind=(\n                    "security_requirement"\n                ),\n                expected_rule={\n                    "requirements": (\n                        effective_security\n                    ),\n                },\n            )\n\n    security_schemes = (\n        document.get(\n            "components",\n            {},\n        ).get(\n            "securitySchemes",\n            {},\n        )\n    )\n\n    if isinstance(\n        security_schemes,\n        dict,\n    ):\n        referenced_security_names = set()\n\n        if isinstance(\n            effective_security,\n            list,\n        ):\n            for security_requirement in (\n                effective_security\n            ):\n                if isinstance(\n                    security_requirement,\n                    dict,\n                ):\n                    referenced_security_names.update(\n                        security_requirement\n                        .keys()\n                    )\n\n        for scheme_name in sorted(\n            referenced_security_names\n        ):\n            raw_scheme = (\n                security_schemes.get(\n                    scheme_name\n                )\n            )\n\n            if raw_scheme is None:\n                add_diagnostic(\n                    "BLOCKING",\n                    (\n                        "SECURITY_SCHEME_"\n                        "MISSING"\n                    ),\n                    security_pointer,\n                    "Referenced security scheme "\n                    f"is missing: {scheme_name}",\n                )\n                continue\n\n            scheme_pointer = (\n                "/components/securitySchemes/"\n                + pointer_escape(\n                    scheme_name\n                )\n            )\n\n            try:\n                scheme, (\n                    resolved_scheme_pointer\n                ) = resolve_object(\n                    document,\n                    raw_scheme,\n                    scheme_pointer,\n                )\n            except Exception as error:\n                add_diagnostic(\n                    "BLOCKING",\n                    (\n                        "SECURITY_SCHEME_"\n                        "REFERENCE_FAILURE"\n                    ),\n                    scheme_pointer,\n                    (\n                        f"{type(error).__name__}: "\n                        f"{error}"\n                    ),\n                )\n                continue\n\n            if not isinstance(\n                scheme,\n                dict,\n            ):\n                continue\n\n            add_interface_item(\n                pointer=(\n                    resolved_scheme_pointer\n                ),\n                item_kind=(\n                    "security_scheme"\n                ),\n                name=scheme_name,\n                location=str(\n                    scheme.get(\n                        "in",\n                        scheme.get(\n                            "type",\n                            "security",\n                        ),\n                    )\n                ),\n                required=True,\n                schema=scheme,\n            )\n\n            if (\n                scheme.get("type")\n                == "apiKey"\n            ):\n                add_requirement(\n                    pointer=(\n                        resolved_scheme_pointer\n                    ),\n                    kind=(\n                        "security_scheme_location"\n                    ),\n                    expected_rule={\n                        "scheme": scheme_name,\n                        "type": "apiKey",\n                        "name": scheme.get(\n                            "name"\n                        ),\n                        "location": (\n                            scheme.get("in")\n                        ),\n                    },\n                )\n\n    requirements.sort(\n        key=lambda record: (\n            record["source_pointer"],\n            record["kind"],\n            record["requirement_id"],\n        )\n    )\n    interface_items.sort(\n        key=lambda record: (\n            record["source_pointer"],\n            record["item_kind"],\n            record[\n                "interface_item_id"\n            ],\n        )\n    )\n    diagnostics.sort(\n        key=lambda record: (\n            record["severity"],\n            record["source_pointer"],\n            record["code"],\n            record["message"],\n        )\n    )\n\n    blocking = any(\n        diagnostic["severity"]\n        == "BLOCKING"\n        for diagnostic in diagnostics\n    )\n\n    return {\n        "op_id": op_id,\n        "method": normalized_method,\n        "path_template": path_template,\n        "operation_pointer": (\n            operation_pointer\n        ),\n        "requirements": requirements,\n        "interface_items": (\n            interface_items\n        ),\n        "diagnostics": diagnostics,\n        "blocking": blocking,\n    }\n', 'oracle_scenarios.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom copy import deepcopy\n\nfrom routing import EvaluatorUnavailable, success_status_mapping\n\nfrom jsonschema import (\n    Draft4Validator,\n    Draft202012Validator,\n    FormatChecker,\n)\n\n\nHTTP_METHODS = {\n    "get",\n    "put",\n    "post",\n    "delete",\n    "options",\n    "head",\n    "patch",\n    "trace",\n}\n\n\ndef canonical_bytes(value) -> bytes:\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    ).encode("utf-8")\n\n\ndef stable_identifier(\n    prefix: str,\n    seed,\n) -> str:\n    digest = hashlib.sha256(\n        canonical_bytes(seed)\n    ).hexdigest()[:24]\n\n    return f"{prefix}_{digest}"\n\n\ndef pointer_unescape(value: str) -> str:\n    return (\n        value.replace("~1", "/")\n        .replace("~0", "~")\n    )\n\n\ndef pointer_escape(value: str) -> str:\n    return (\n        value.replace("~", "~0")\n        .replace("/", "~1")\n    )\n\n\ndef join_pointer(\n    base: str,\n    *tokens,\n) -> str:\n    result = base.rstrip("/")\n\n    for token in tokens:\n        result += (\n            "/"\n            + pointer_escape(str(token))\n        )\n\n    return result or "/"\n\n\ndef resolve_local_reference(\n    document: dict,\n    reference: str,\n):\n    from urllib.parse import unquote\n\n    if not isinstance(reference, str):\n        raise TypeError(\n            "$ref must be a string."\n        )\n\n    if reference == "#":\n        return document, "/"\n\n    if not reference.startswith("#"):\n        raise RuntimeError(\n            "External reference is not "\n            f"supported: {reference}"\n        )\n\n    fragment = unquote(\n        reference[1:]\n    )\n\n    if fragment.startswith("/"):\n        node = document\n\n        for encoded_token in (\n            fragment[1:].split("/")\n        ):\n            token = pointer_unescape(\n                unquote(encoded_token)\n            )\n\n            if isinstance(node, dict):\n                node = node[token]\n\n            elif isinstance(node, list):\n                node = node[int(token)]\n\n            else:\n                raise TypeError(\n                    "Reference traversed into "\n                    "a scalar value."\n                )\n\n        return node, fragment\n\n    if not fragment:\n        return document, "/"\n\n    matches = []\n\n    def visit(value, pointer):\n        if isinstance(value, dict):\n            if (\n                value.get("$anchor")\n                == fragment\n                or value.get(\n                    "$dynamicAnchor"\n                )\n                == fragment\n            ):\n                matches.append(\n                    (value, pointer)\n                )\n\n            for key, nested_value in (\n                value.items()\n            ):\n                visit(\n                    nested_value,\n                    join_pointer(\n                        pointer,\n                        key,\n                    ),\n                )\n\n        elif isinstance(value, list):\n            for index, nested_value in (\n                enumerate(value)\n            ):\n                visit(\n                    nested_value,\n                    join_pointer(\n                        pointer,\n                        index,\n                    ),\n                )\n\n    visit(document, "")\n\n    if len(matches) != 1:\n        raise KeyError(\n            "Local anchor must resolve to "\n            "exactly one object."\n        )\n\n    return matches[0]\n\n\ndef dereference_schema(\n    document: dict,\n    schema,\n    reference_stack=(),\n    depth: int = 0,\n):\n    if depth > 30:\n        return {}\n\n    if isinstance(schema, bool):\n        return schema\n\n    if not isinstance(schema, dict):\n        return {}\n\n    if "$ref" in schema:\n        reference = schema["$ref"]\n\n        if reference in reference_stack:\n            return {}\n\n        target, _ = (\n            resolve_local_reference(\n                document,\n                reference,\n            )\n        )\n\n        resolved = dereference_schema(\n            document,\n            target,\n            reference_stack\n            + (reference,),\n            depth + 1,\n        )\n\n        siblings = {\n            key: deepcopy(value)\n            for key, value\n            in schema.items()\n            if key != "$ref"\n        }\n\n        if (\n            isinstance(resolved, dict)\n            and siblings\n        ):\n            resolved = {\n                **resolved,\n                **siblings,\n            }\n\n        return resolved\n\n    result = {}\n\n    for key, value in schema.items():\n        if key in {\n            "properties",\n            "patternProperties",\n            "$defs",\n            "definitions",\n        } and isinstance(value, dict):\n            result[key] = {\n                nested_key: (\n                    dereference_schema(\n                        document,\n                        nested_value,\n                        reference_stack,\n                        depth + 1,\n                    )\n                )\n                for nested_key, nested_value\n                in value.items()\n            }\n\n        elif key in {\n            "items",\n            "additionalProperties",\n            "not",\n            "contains",\n            "propertyNames",\n        } and isinstance(\n            value,\n            (dict, bool),\n        ):\n            result[key] = (\n                dereference_schema(\n                    document,\n                    value,\n                    reference_stack,\n                    depth + 1,\n                )\n            )\n\n        elif key in {\n            "allOf",\n            "oneOf",\n            "anyOf",\n            "prefixItems",\n        } and isinstance(value, list):\n            result[key] = [\n                dereference_schema(\n                    document,\n                    nested_value,\n                    reference_stack,\n                    depth + 1,\n                )\n                for nested_value in value\n            ]\n\n        else:\n            result[key] = deepcopy(\n                value\n            )\n\n    if result.get("nullable") is True:\n        observed_type = result.get(\n            "type"\n        )\n\n        if isinstance(observed_type, str):\n            result["type"] = [\n                observed_type,\n                "null",\n            ]\n\n        elif isinstance(\n            observed_type,\n            list,\n        ) and "null" not in observed_type:\n            result["type"] = (\n                list(observed_type)\n                + ["null"]\n            )\n\n    for annotation_key in [\n        "nullable",\n        "example",\n        "examples",\n        "deprecated",\n        "readOnly",\n        "writeOnly",\n        "xml",\n        "externalDocs",\n        "discriminator",\n    ]:\n        result.pop(\n            annotation_key,\n            None,\n        )\n\n    return result\n\n\ndef schema_types(schema) -> list[str]:\n    if not isinstance(schema, dict):\n        return []\n\n    observed = schema.get("type")\n\n    if isinstance(observed, str):\n        return [observed]\n\n    if isinstance(observed, list):\n        return [\n            item\n            for item in observed\n            if isinstance(item, str)\n        ]\n\n    if "properties" in schema:\n        return ["object"]\n\n    if "items" in schema:\n        return ["array"]\n\n    return []\n\n\ndef numeric_interior(schema):\n    minimum = schema.get("minimum")\n    maximum = schema.get("maximum")\n    exclusive_minimum = schema.get(\n        "exclusiveMinimum"\n    )\n    exclusive_maximum = schema.get(\n        "exclusiveMaximum"\n    )\n\n    lower = minimum\n    upper = maximum\n\n    if isinstance(\n        exclusive_minimum,\n        (int, float),\n    ) and not isinstance(\n        exclusive_minimum,\n        bool,\n    ):\n        lower = exclusive_minimum\n\n    if isinstance(\n        exclusive_maximum,\n        (int, float),\n    ) and not isinstance(\n        exclusive_maximum,\n        bool,\n    ):\n        upper = exclusive_maximum\n\n    if (\n        isinstance(lower, (int, float))\n        and isinstance(\n            upper,\n            (int, float),\n        )\n    ):\n        value = (lower + upper) / 2\n\n    elif isinstance(\n        lower,\n        (int, float),\n    ):\n        value = lower + 1\n\n    elif isinstance(\n        upper,\n        (int, float),\n    ):\n        value = upper - 1\n\n    else:\n        value = 1\n\n    if (\n        "integer"\n        in schema_types(schema)\n    ):\n        value = int(\n            math.ceil(value)\n        )\n\n        if (\n            isinstance(\n                exclusive_minimum,\n                (int, float),\n            )\n            and value\n            <= exclusive_minimum\n        ):\n            value = (\n                int(exclusive_minimum)\n                + 1\n            )\n\n        if (\n            isinstance(\n                exclusive_maximum,\n                (int, float),\n            )\n            and value\n            >= exclusive_maximum\n        ):\n            value = (\n                int(exclusive_maximum)\n                - 1\n            )\n\n    return value\n\n\ndef string_example(schema):\n    if "const" in schema:\n        return schema["const"]\n\n    enum_values = schema.get("enum")\n\n    if (\n        isinstance(enum_values, list)\n        and enum_values\n    ):\n        return deepcopy(\n            enum_values[0]\n        )\n\n    for key in [\n        "example",\n        "default",\n    ]:\n        value = schema.get(key)\n\n        if isinstance(value, str):\n            return value\n\n    format_name = schema.get("format")\n\n    format_examples = {\n        "date-time": (\n            "2026-01-01T00:00:00Z"\n        ),\n        "date": "2026-01-01",\n        "time": "12:00:00Z",\n        "email": "user@example.com",\n        "hostname": "example.com",\n        "ipv4": "192.0.2.1",\n        "ipv6": "2001:db8::1",\n        "uri": "https://example.com",\n        "uuid": (\n            "123e4567-e89b-12d3-a456-"\n            "426614174000"\n        ),\n        "byte": "dGVzdA==",\n        "binary": "test",\n    }\n\n    if format_name in format_examples:\n        return format_examples[\n            format_name\n        ]\n\n    minimum_length = int(\n        schema.get("minLength", 1)\n    )\n    maximum_length = schema.get(\n        "maxLength"\n    )\n\n    length = max(\n        minimum_length,\n        1,\n    )\n\n    if isinstance(\n        maximum_length,\n        int,\n    ):\n        length = min(\n            length,\n            maximum_length,\n        )\n\n    pattern = schema.get("pattern")\n\n    candidates = [\n        "A",\n        "a",\n        "1",\n        "test",\n        "Test1",\n        "abc",\n        "ABC",\n        "user-1",\n    ]\n\n    for candidate in candidates:\n        if len(candidate) < length:\n            candidate = (\n                candidate\n                + ("a" * length)\n            )[:length]\n\n        if (\n            isinstance(\n                maximum_length,\n                int,\n            )\n            and len(candidate)\n            > maximum_length\n        ):\n            candidate = candidate[\n                :maximum_length\n            ]\n\n        if not isinstance(pattern, str):\n            return candidate\n\n        try:\n            if re.search(\n                pattern,\n                candidate,\n            ):\n                return candidate\n        except re.error:\n            break\n\n    return "a" * length\n\n\ndef all_of_value(document: dict, schema: dict, depth: int):\n    """Find a witness for the whole intersection, including sibling constraints."""\n    parent = {key: value for key, value in schema.items() if key != "allOf"}\n    branches = [*schema["allOf"]]\n    if parent:\n        branches.append(parent)\n\n    candidates = []\n    merged_object = {}\n    for branch in branches:\n        branch = dereference_schema(document, branch)\n        if branch is False:\n            raise EvaluatorUnavailable("NO_WITNESS_FOR_ALLOF")\n        try:\n            candidate = valid_value(document, branch, depth + 1)\n        except EvaluatorUnavailable as error:\n            if error.code not in {"NO_WITNESS_FOR_ALLOF", "NO_WITNESS_FOR_FALSE_SCHEMA"}:\n                raise\n        else:\n            candidates.append(candidate)\n            if isinstance(candidate, dict):\n                merged_object.update(candidate)\n        if isinstance(branch, dict):\n            enum = branch.get("enum")\n            if isinstance(enum, list):\n                candidates.extend(enum)\n\n    # Preserve object-field composition and consider scalar or array witnesses separately.\n    for candidate in (merged_object, *candidates):\n        if validate_instance(document, schema, candidate):\n            return deepcopy(candidate)\n    raise EvaluatorUnavailable("NO_WITNESS_FOR_ALLOF")\n\n\ndef valid_value(\n    document: dict,\n    schema,\n    depth: int = 0,\n):\n    if depth > 20:\n        return None\n\n    schema = dereference_schema(\n        document,\n        schema,\n    )\n\n    if schema is True:\n        return None\n\n    if schema is False:\n        raise EvaluatorUnavailable("NO_WITNESS_FOR_FALSE_SCHEMA")\n\n    if not isinstance(schema, dict):\n        return None\n\n    if isinstance(schema.get("allOf"), list):\n        return all_of_value(document, schema, depth)\n\n    if "const" in schema:\n        return deepcopy(\n            schema["const"]\n        )\n\n    enum_values = schema.get("enum")\n\n    if (\n        isinstance(enum_values, list)\n        and enum_values\n    ):\n        return deepcopy(\n            enum_values[0]\n        )\n\n    for combinator in [\n        "oneOf",\n        "anyOf",\n    ]:\n        branches = schema.get(\n            combinator\n        )\n\n        if (\n            isinstance(branches, list)\n            and branches\n        ):\n            return valid_value(\n                document,\n                branches[0],\n                depth + 1,\n            )\n\n    types = schema_types(schema)\n    selected_type = next(\n        (\n            item\n            for item in types\n            if item != "null"\n        ),\n        "string",\n    )\n\n    if selected_type == "object":\n        result = {}\n        properties = schema.get(\n            "properties",\n            {},\n        )\n        required_properties = (\n            schema.get(\n                "required",\n                [],\n            )\n        )\n\n        if not isinstance(\n            required_properties,\n            list,\n        ):\n            required_properties = []\n\n        for property_name in (\n            required_properties\n        ):\n            property_schema = (\n                properties.get(\n                    property_name,\n                    {},\n                )\n                if isinstance(\n                    properties,\n                    dict,\n                )\n                else {}\n            )\n\n            result[property_name] = (\n                valid_value(\n                    document,\n                    property_schema,\n                    depth + 1,\n                )\n            )\n\n        return result\n\n    if selected_type == "array":\n        minimum_items = int(\n            schema.get("minItems", 1)\n        )\n        count = max(\n            minimum_items,\n            1,\n        )\n        item_schema = schema.get(\n            "items",\n            {},\n        )\n\n        return [\n            valid_value(\n                document,\n                item_schema,\n                depth + 1,\n            )\n            for _ in range(count)\n        ]\n\n    if selected_type == "integer":\n        return int(\n            numeric_interior(schema)\n        )\n\n    if selected_type == "number":\n        return float(\n            numeric_interior(schema)\n        )\n\n    if selected_type == "boolean":\n        return True\n\n    if selected_type == "null":\n        return None\n\n    return string_example(schema)\n\n\ndef wrong_type_value(value):\n    if isinstance(value, bool):\n        return "not-a-boolean"\n\n    if isinstance(value, int):\n        return "not-an-integer"\n\n    if isinstance(value, float):\n        return "not-a-number"\n\n    if isinstance(value, str):\n        return {\n            "wrong": "type",\n        }\n\n    if isinstance(value, list):\n        return {\n            "wrong": "type",\n        }\n\n    if isinstance(value, dict):\n        return [\n            "wrong",\n            "type",\n        ]\n\n    if value is None:\n        return "not-null"\n\n    return None\n\n\ndef deep_set(\n    value,\n    tokens: list,\n    replacement,\n):\n    result = deepcopy(value)\n\n    if not tokens:\n        return deepcopy(replacement)\n\n    node = result\n\n    for token in tokens[:-1]:\n        node = node[token]\n\n    node[tokens[-1]] = deepcopy(\n        replacement\n    )\n    return result\n\n\ndef deep_remove(\n    value,\n    tokens: list,\n):\n    result = deepcopy(value)\n\n    if not tokens:\n        return None\n\n    node = result\n\n    for token in tokens[:-1]:\n        node = node[token]\n\n    final_token = tokens[-1]\n\n    if isinstance(node, dict):\n        node.pop(\n            final_token,\n            None,\n        )\n\n    elif isinstance(node, list):\n        node.pop(int(final_token))\n\n    return result\n\n\ndef schema_mutations(\n    document: dict,\n    schema,\n    base_value,\n    tokens=None,\n    depth: int = 0,\n) -> list[dict]:\n    if tokens is None:\n        tokens = []\n\n    if depth > 12:\n        return []\n\n    schema = dereference_schema(\n        document,\n        schema,\n    )\n\n    if not isinstance(schema, dict):\n        return []\n\n    mutations = []\n\n    types = schema_types(schema)\n    nonnull_types = [\n        value\n        for value in types\n        if value != "null"\n    ]\n\n    if nonnull_types:\n        mutations.append(\n            {\n                "intent": (\n                    "invalid_wrong_type"\n                ),\n                "tokens": list(tokens),\n                "action": "replace",\n                "value": wrong_type_value(\n                    base_value\n                ),\n                "expected_validity": (\n                    "INVALID"\n                ),\n                "constraint": "type",\n            }\n        )\n\n    enum_values = schema.get("enum")\n\n    if (\n        isinstance(enum_values, list)\n        and enum_values\n    ):\n        invalid_enum_value = (\n            "__not_in_documented_enum__"\n        )\n\n        while (\n            invalid_enum_value\n            in enum_values\n        ):\n            invalid_enum_value += "_x"\n\n        mutations.append(\n            {\n                "intent": (\n                    "invalid_enum_value"\n                ),\n                "tokens": list(tokens),\n                "action": "replace",\n                "value": (\n                    invalid_enum_value\n                ),\n                "expected_validity": (\n                    "INVALID"\n                ),\n                "constraint": "enum",\n            }\n        )\n\n    if isinstance(\n        base_value,\n        (int, float),\n    ) and not isinstance(\n        base_value,\n        bool,\n    ):\n        minimum = schema.get(\n            "minimum"\n        )\n        maximum = schema.get(\n            "maximum"\n        )\n\n        if isinstance(\n            minimum,\n            (int, float),\n        ):\n            boundary_value = minimum\n\n            if (\n                "integer"\n                in nonnull_types\n            ):\n                boundary_value = int(\n                    minimum\n                )\n\n            mutations.extend(\n                [\n                    {\n                        "intent": (\n                            "valid_minimum_"\n                            "boundary"\n                        ),\n                        "tokens": list(\n                            tokens\n                        ),\n                        "action": "replace",\n                        "value": (\n                            boundary_value\n                        ),\n                        "expected_validity": (\n                            "VALID"\n                        ),\n                        "constraint": (\n                            "minimum"\n                        ),\n                    },\n                    {\n                        "intent": (\n                            "invalid_below_"\n                            "minimum"\n                        ),\n                        "tokens": list(\n                            tokens\n                        ),\n                        "action": "replace",\n                        "value": (\n                            boundary_value\n                            - 1\n                        ),\n                        "expected_validity": (\n                            "INVALID"\n                        ),\n                        "constraint": (\n                            "minimum"\n                        ),\n                    },\n                ]\n            )\n\n        if isinstance(\n            maximum,\n            (int, float),\n        ):\n            boundary_value = maximum\n\n            if (\n                "integer"\n                in nonnull_types\n            ):\n                boundary_value = int(\n                    maximum\n                )\n\n            mutations.extend(\n                [\n                    {\n                        "intent": (\n                            "valid_maximum_"\n                            "boundary"\n                        ),\n                        "tokens": list(\n                            tokens\n                        ),\n                        "action": "replace",\n                        "value": (\n                            boundary_value\n                        ),\n                        "expected_validity": (\n                            "VALID"\n                        ),\n                        "constraint": (\n                            "maximum"\n                        ),\n                    },\n                    {\n                        "intent": (\n                            "invalid_above_"\n                            "maximum"\n                        ),\n                        "tokens": list(\n                            tokens\n                        ),\n                        "action": "replace",\n                        "value": (\n                            boundary_value\n                            + 1\n                        ),\n                        "expected_validity": (\n                            "INVALID"\n                        ),\n                        "constraint": (\n                            "maximum"\n                        ),\n                    },\n                ]\n            )\n\n    if isinstance(base_value, str):\n        minimum_length = schema.get(\n            "minLength"\n        )\n        maximum_length = schema.get(\n            "maxLength"\n        )\n\n        if isinstance(\n            minimum_length,\n            int,\n        ):\n            mutations.append(\n                {\n                    "intent": (\n                        "invalid_below_"\n                        "minimum_length"\n                    ),\n                    "tokens": list(\n                        tokens\n                    ),\n                    "action": "replace",\n                    "value": (\n                        "a"\n                        * max(\n                            minimum_length - 1,\n                            0,\n                        )\n                    ),\n                    "expected_validity": (\n                        "INVALID"\n                    ),\n                    "constraint": (\n                        "minLength"\n                    ),\n                }\n            )\n\n        if isinstance(\n            maximum_length,\n            int,\n        ):\n            mutations.append(\n                {\n                    "intent": (\n                        "invalid_above_"\n                        "maximum_length"\n                    ),\n                    "tokens": list(\n                        tokens\n                    ),\n                    "action": "replace",\n                    "value": (\n                        "a"\n                        * (\n                            maximum_length\n                            + 1\n                        )\n                    ),\n                    "expected_validity": (\n                        "INVALID"\n                    ),\n                    "constraint": (\n                        "maxLength"\n                    ),\n                }\n            )\n\n    if isinstance(base_value, list):\n        minimum_items = schema.get(\n            "minItems"\n        )\n        maximum_items = schema.get(\n            "maxItems"\n        )\n\n        if isinstance(\n            minimum_items,\n            int,\n        ) and minimum_items > 0:\n            mutations.append(\n                {\n                    "intent": (\n                        "invalid_below_"\n                        "minimum_items"\n                    ),\n                    "tokens": list(\n                        tokens\n                    ),\n                    "action": "replace",\n                    "value": (\n                        base_value[\n                            :max(\n                                minimum_items\n                                - 1,\n                                0,\n                            )\n                        ]\n                    ),\n                    "expected_validity": (\n                        "INVALID"\n                    ),\n                    "constraint": (\n                        "minItems"\n                    ),\n                }\n            )\n\n        if isinstance(\n            maximum_items,\n            int,\n        ):\n            item_schema = schema.get(\n                "items",\n                {},\n            )\n            item_value = valid_value(\n                document,\n                item_schema,\n                depth + 1,\n            )\n\n            mutations.append(\n                {\n                    "intent": (\n                        "invalid_above_"\n                        "maximum_items"\n                    ),\n                    "tokens": list(\n                        tokens\n                    ),\n                    "action": "replace",\n                    "value": [\n                        deepcopy(item_value)\n                        for _ in range(\n                            maximum_items\n                            + 1\n                        )\n                    ],\n                    "expected_validity": (\n                        "INVALID"\n                    ),\n                    "constraint": (\n                        "maxItems"\n                    ),\n                }\n            )\n\n    if isinstance(base_value, dict):\n        properties = schema.get(\n            "properties",\n            {},\n        )\n        required_properties = (\n            schema.get(\n                "required",\n                [],\n            )\n        )\n\n        if not isinstance(\n            required_properties,\n            list,\n        ):\n            required_properties = []\n\n        for property_name in (\n            required_properties\n        ):\n            if property_name in base_value:\n                mutations.append(\n                    {\n                        "intent": (\n                            "invalid_missing_"\n                            "required_property"\n                        ),\n                        "tokens": (\n                            list(tokens)\n                            + [property_name]\n                        ),\n                        "action": "remove",\n                        "value": None,\n                        "expected_validity": (\n                            "INVALID"\n                        ),\n                        "constraint": (\n                            "required"\n                        ),\n                    }\n                )\n\n        if isinstance(properties, dict):\n            for (\n                property_name,\n                property_schema,\n            ) in properties.items():\n                if (\n                    property_name\n                    not in base_value\n                ):\n                    continue\n\n                mutations.extend(\n                    schema_mutations(\n                        document,\n                        property_schema,\n                        base_value[\n                            property_name\n                        ],\n                        list(tokens)\n                        + [property_name],\n                        depth + 1,\n                    )\n                )\n\n    return mutations\n\n\ndef validate_instance(\n    document: dict,\n    schema,\n    instance,\n) -> bool:\n    normalized_schema = (\n        dereference_schema(\n            document,\n            schema,\n        )\n    )\n\n    if normalized_schema is True:\n        return True\n\n    if normalized_schema is False:\n        return False\n\n    openapi_version = str(\n        document.get(\n            "openapi",\n            "3.0.0",\n        )\n    )\n\n    if openapi_version.startswith(\n        "3.1"\n    ):\n        validator = (\n            Draft202012Validator(\n                normalized_schema,\n                format_checker=(\n                    FormatChecker()\n                ),\n            )\n        )\n    else:\n        validator = Draft4Validator(\n            normalized_schema,\n            format_checker=(\n                FormatChecker()\n            ),\n        )\n\n    return not list(\n        validator.iter_errors(\n            instance\n        )\n    )\n\n\ndef build_scenarios(\n    document: dict,\n    op_id: str,\n    method: str,\n    path_template: str,\n    requirements: list[dict],\n    interface_items: list[dict],\n) -> dict:\n    normalized_method = method.lower()\n\n    if normalized_method not in (\n        HTTP_METHODS\n    ):\n        raise ValueError(\n            f"Unsupported method: {method}"\n        )\n\n    parameter_items = [\n        item\n        for item in interface_items\n        if item["item_kind"]\n        == "parameter"\n    ]\n    request_body_items = [\n        item\n        for item in interface_items\n        if item["item_kind"]\n        == "request_body"\n    ]\n\n    documented_response_mapping = success_status_mapping(\n        document, path_template, normalized_method\n    )\n\n    parameter_values = {\n        "path": {},\n        "query": {},\n        "header": {},\n        "cookie": {},\n    }\n    parameter_schemas = {}\n\n    for item in parameter_items:\n        location = item["location"]\n        name = item["name"]\n        schema = item.get("schema")\n\n        if location not in parameter_values:\n            parameter_values[\n                location\n            ] = {}\n\n        if isinstance(\n            schema,\n            (dict, bool),\n        ):\n            value = valid_value(\n                document,\n                schema,\n            )\n            parameter_schemas[\n                (location, name)\n            ] = schema\n        else:\n            value = "value"\n\n        if item["required"] or schema:\n            parameter_values[\n                location\n            ][name] = value\n\n    selected_request_body_item = (\n        sorted(\n            request_body_items,\n            key=lambda item: (\n                item["location"],\n                item[\n                    "interface_item_id"\n                ],\n            ),\n        )[0]\n        if request_body_items\n        else None\n    )\n\n    body = None\n    body_schema = None\n    request_body_required = False\n\n    if (\n        selected_request_body_item\n        is not None\n    ):\n        body_schema = (\n            selected_request_body_item\n            .get("schema")\n        )\n        request_body_required = bool(\n            selected_request_body_item[\n                "required"\n            ]\n        )\n\n        if isinstance(\n            body_schema,\n            (dict, bool),\n        ):\n            body = valid_value(\n                document,\n                body_schema,\n            )\n\n    request_requirement_ids = sorted(\n        {\n            requirement[\n                "requirement_id"\n            ]\n            for requirement\n            in requirements\n            if (\n                requirement[\n                    "review_status"\n                ]\n                == "AUTO_VERIFIED"\n                and requirement["kind"]\n                not in {\n                    "documented_response_status",\n                    "response_media_type",\n                    "security_requirement",\n                    "security_scheme_location",\n                    "read_only",\n                }\n            )\n        }\n    )\n\n    drafts = [\n        {\n            "intent": (\n                "valid_baseline_request"\n            ),\n            "requirement_ids": (\n                request_requirement_ids\n            ),\n            "parameter_values": (\n                deepcopy(\n                    parameter_values\n                )\n            ),\n            "body": deepcopy(body),\n            "expected_validity": "VALID",\n            "documented_response_mapping": (\n                documented_response_mapping\n            ),\n            "applicability": "APPLICABLE",\n        }\n    ]\n\n    for item in parameter_items:\n        location = item["location"]\n        name = item["name"]\n\n        if (\n            item["required"]\n            and name\n            in parameter_values.get(\n                location,\n                {},\n            )\n        ):\n            mutated_parameters = (\n                deepcopy(\n                    parameter_values\n                )\n            )\n            mutated_parameters[\n                location\n            ].pop(name, None)\n\n            drafts.append(\n                {\n                    "intent": (\n                        "invalid_missing_"\n                        "required_parameter"\n                    ),\n                    "requirement_ids": (\n                        request_requirement_ids\n                    ),\n                    "parameter_values": (\n                        mutated_parameters\n                    ),\n                    "body": deepcopy(\n                        body\n                    ),\n                    "expected_validity": (\n                        "INVALID"\n                    ),\n                    "documented_response_mapping": (\n                        None\n                    ),\n                    "applicability": (\n                        "APPLICABLE"\n                    ),\n                }\n            )\n\n        schema = parameter_schemas.get(\n            (location, name)\n        )\n\n        if (\n            schema is not None\n            and name\n            in parameter_values.get(\n                location,\n                {},\n            )\n        ):\n            base_value = (\n                parameter_values[\n                    location\n                ][name]\n            )\n\n            for mutation in (\n                schema_mutations(\n                    document,\n                    schema,\n                    base_value,\n                )\n            ):\n                mutated_parameters = (\n                    deepcopy(\n                        parameter_values\n                    )\n                )\n\n                if (\n                    mutation["action"]\n                    == "remove"\n                ):\n                    mutated_value = deep_remove(\n                        base_value,\n                        mutation["tokens"],\n                    )\n                else:\n                    mutated_value = deep_set(\n                        base_value,\n                        mutation["tokens"],\n                        mutation["value"],\n                    )\n\n                mutated_parameters[\n                    location\n                ][name] = mutated_value\n\n                drafts.append(\n                    {\n                        "intent": (\n                            "parameter_"\n                            + mutation[\n                                "intent"\n                            ]\n                        ),\n                        "requirement_ids": (\n                            request_requirement_ids\n                        ),\n                        "parameter_values": (\n                            mutated_parameters\n                        ),\n                        "body": deepcopy(\n                            body\n                        ),\n                        "expected_validity": (\n                            mutation[\n                                "expected_validity"\n                            ]\n                        ),\n                        "documented_response_mapping": (\n                            documented_response_mapping\n                            if mutation[\n                                "expected_validity"\n                            ]\n                            == "VALID"\n                            else None\n                        ),\n                        "applicability": (\n                            "APPLICABLE"\n                        ),\n                    }\n                )\n\n    if request_body_required:\n        drafts.append(\n            {\n                "intent": (\n                    "invalid_missing_"\n                    "required_request_body"\n                ),\n                "requirement_ids": (\n                    request_requirement_ids\n                ),\n                "parameter_values": (\n                    deepcopy(\n                        parameter_values\n                    )\n                ),\n                "body": None,\n                "expected_validity": (\n                    "INVALID"\n                ),\n                "documented_response_mapping": (\n                    None\n                ),\n                "applicability": (\n                    "APPLICABLE"\n                ),\n            }\n        )\n\n    if isinstance(\n        body_schema,\n        (dict, bool),\n    ):\n        for mutation in schema_mutations(\n            document,\n            body_schema,\n            body,\n        ):\n            if (\n                mutation["action"]\n                == "remove"\n            ):\n                mutated_body = deep_remove(\n                    body,\n                    mutation["tokens"],\n                )\n            else:\n                mutated_body = deep_set(\n                    body,\n                    mutation["tokens"],\n                    mutation["value"],\n                )\n\n            drafts.append(\n                {\n                    "intent": (\n                        "body_"\n                        + mutation["intent"]\n                    ),\n                    "requirement_ids": (\n                        request_requirement_ids\n                    ),\n                    "parameter_values": (\n                        deepcopy(\n                            parameter_values\n                        )\n                    ),\n                    "body": mutated_body,\n                    "expected_validity": (\n                        mutation[\n                            "expected_validity"\n                        ]\n                    ),\n                    "documented_response_mapping": (\n                        documented_response_mapping\n                        if mutation[\n                            "expected_validity"\n                        ]\n                        == "VALID"\n                        else None\n                    ),\n                    "applicability": (\n                        "APPLICABLE"\n                    ),\n                }\n            )\n\n    deduplicated = {}\n\n    for draft in drafts:\n        request_key = canonical_bytes(\n            {\n                "parameter_values": (\n                    draft[\n                        "parameter_values"\n                    ]\n                ),\n                "body": draft["body"],\n                "expected_validity": (\n                    draft[\n                        "expected_validity"\n                    ]\n                ),\n            }\n        )\n\n        if request_key not in deduplicated:\n            deduplicated[\n                request_key\n            ] = draft\n        else:\n            prior = deduplicated[\n                request_key\n            ]\n            prior[\n                "requirement_ids"\n            ] = sorted(\n                set(\n                    prior[\n                        "requirement_ids"\n                    ]\n                )\n                | set(\n                    draft[\n                        "requirement_ids"\n                    ]\n                )\n            )\n\n    scenarios = []\n    diagnostics = []\n\n    required_parameter_keys = {\n        (\n            item["location"],\n            item["name"],\n        )\n        for item in parameter_items\n        if item["required"]\n    }\n\n    for draft in deduplicated.values():\n        parameter_valid = True\n\n        for location, name in (\n            required_parameter_keys\n        ):\n            if name not in (\n                draft[\n                    "parameter_values"\n                ].get(location, {})\n            ):\n                parameter_valid = False\n\n        for (\n            location,\n            name,\n        ), schema in (\n            parameter_schemas.items()\n        ):\n            location_values = (\n                draft[\n                    "parameter_values"\n                ].get(location, {})\n            )\n\n            if name not in location_values:\n                continue\n\n            if not validate_instance(\n                document,\n                schema,\n                location_values[name],\n            ):\n                parameter_valid = False\n\n        body_valid = True\n\n        if request_body_required and (\n            draft["body"] is None\n        ):\n            body_valid = False\n\n        elif (\n            draft["body"] is not None\n            and isinstance(\n                body_schema,\n                (dict, bool),\n            )\n        ):\n            body_valid = validate_instance(\n                document,\n                body_schema,\n                draft["body"],\n            )\n\n        observed_validity = (\n            "VALID"\n            if (\n                parameter_valid\n                and body_valid\n            )\n            else "INVALID"\n        )\n\n        validation_status = (\n            "PASS"\n            if observed_validity\n            == draft[\n                "expected_validity"\n            ]\n            else "FAIL"\n        )\n\n        if validation_status == "FAIL":\n            diagnostics.append(\n                {\n                    "severity": "BLOCKING",\n                    "code": (\n                        "SCENARIO_VALIDITY_"\n                        "MISMATCH"\n                    ),\n                    "intent": (\n                        draft["intent"]\n                    ),\n                    "expected": (\n                        draft[\n                            "expected_validity"\n                        ]\n                    ),\n                    "observed": (\n                        observed_validity\n                    ),\n                }\n            )\n\n        scenario_seed = {\n            "op_id": op_id,\n            "method": (\n                normalized_method\n            ),\n            "path_template": (\n                path_template\n            ),\n            "parameter_values": (\n                draft[\n                    "parameter_values"\n                ]\n            ),\n            "body": draft["body"],\n            "expected_validity": (\n                draft[\n                    "expected_validity"\n                ]\n            ),\n        }\n\n        scenarios.append(\n            {\n                "scenario_id": (\n                    stable_identifier(\n                        "scenario",\n                        scenario_seed,\n                    )\n                ),\n                "op_id": op_id,\n                "intent": draft["intent"],\n                "requirement_ids": (\n                    sorted(\n                        set(\n                            draft[\n                                "requirement_ids"\n                            ]\n                        )\n                    )\n                ),\n                "method": (\n                    normalized_method\n                ),\n                "path_template": (\n                    path_template\n                ),\n                "parameter_values": (\n                    draft[\n                        "parameter_values"\n                    ]\n                ),\n                "body": draft["body"],\n                "expected_validity": (\n                    draft[\n                        "expected_validity"\n                    ]\n                ),\n                "documented_response_mapping": (\n                    draft[\n                        "documented_response_mapping"\n                    ]\n                ),\n                "applicability": (\n                    draft["applicability"]\n                ),\n                "validation_status": (\n                    validation_status\n                ),\n                "diagnostic": (\n                    None\n                    if validation_status\n                    == "PASS"\n                    else (\n                        "Constructed scenario "\n                        "did not match its "\n                        "expected validity."\n                    )\n                ),\n            }\n        )\n\n    scenarios.sort(\n        key=lambda record: (\n            record["expected_validity"],\n            record["intent"],\n            record["scenario_id"],\n        )\n    )\n\n    diagnostics.sort(\n        key=lambda record: (\n            record["severity"],\n            record["code"],\n            record["intent"],\n        )\n    )\n\n    return {\n        "op_id": op_id,\n        "scenarios": scenarios,\n        "diagnostics": diagnostics,\n        "blocking": any(\n            diagnostic["severity"]\n            == "BLOCKING"\n            for diagnostic in diagnostics\n        ),\n    }\n', 'oracle_variations.py': '\nfrom copy import deepcopy\nimport hashlib\nimport json\nimport re\nimport warnings\nfrom urllib.parse import unquote\nfrom referencing import Registry\n\nwarnings.filterwarnings("ignore", category=DeprecationWarning)\n\nimport jsonschema\n\n\nBUILDER_VERSION = "stage05-deterministic-legitimate-variation-builder-v2-offline-refs"\nHTTP_METHODS = {\n    "get", "put", "post", "delete",\n    "options", "head", "patch", "trace",\n}\n\n\ndef canonical_json(value):\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n        allow_nan=False,\n    )\n\n\ndef stable_id(prefix, value):\n    digest = hashlib.sha256(\n        canonical_json(value).encode("utf-8")\n    ).hexdigest()[:24]\n    return f"{prefix}_{digest}"\n\n\ndef pointer_token(token):\n    value = token\n    for _ in range(4):\n        decoded = unquote(value)\n        if decoded == value:\n            break\n        value = decoded\n    return value.replace("~1", "/").replace("~0", "~")\n\n\ndef path_shape(value):\n    text = unquote(str(value))\n    return re.sub(r"\\{[^{}]+\\}", "{}", text)\n\n\ndef resolve_pointer(document, pointer):\n    if not isinstance(pointer, str) or not pointer.startswith("#/"):\n        raise ValueError(f"Unsupported reference: {pointer!r}")\n\n    node = document\n    for raw_token in pointer[2:].split("/"):\n        token = pointer_token(raw_token)\n\n        if not isinstance(node, dict):\n            raise KeyError(token)\n\n        if token in node:\n            node = node[token]\n            continue\n\n        target_shape = path_shape(token)\n        candidates = [\n            key\n            for key in node\n            if isinstance(key, str)\n            and path_shape(key) == target_shape\n        ]\n\n        if len(candidates) != 1:\n            raise KeyError(token)\n\n        node = node[candidates[0]]\n\n    return node\n\n\ndef dereference(document, value, depth=0, seen=None):\n    if depth > 40:\n        raise RuntimeError("Reference depth exceeded.")\n\n    if not isinstance(value, dict) or "$ref" not in value:\n        return deepcopy(value)\n\n    reference = value["$ref"]\n    if not reference.startswith("#/"):\n        raise ValueError(f"External reference is unsupported: {reference}")\n\n    seen = set() if seen is None else set(seen)\n    if reference in seen:\n        return deepcopy(value)\n\n    seen.add(reference)\n    target = dereference(\n        document,\n        resolve_pointer(document, reference),\n        depth + 1,\n        seen,\n    )\n\n    if not isinstance(target, dict):\n        return target\n\n    for key, sibling in value.items():\n        if key != "$ref":\n            target[key] = deepcopy(sibling)\n\n    return target\n\n\ndef locate_operation(document, method, path_template):\n    paths = document.get("paths", {})\n    method = str(method).lower()\n\n    if (\n        isinstance(paths.get(path_template), dict)\n        and isinstance(paths[path_template].get(method), dict)\n    ):\n        return path_template, paths[path_template][method]\n\n    candidates = []\n    target_shape = path_shape(path_template)\n\n    for candidate_path, path_item in paths.items():\n        if (\n            isinstance(path_item, dict)\n            and method in path_item\n            and path_shape(candidate_path) == target_shape\n        ):\n            candidates.append((candidate_path, path_item[method]))\n\n    if len(candidates) != 1:\n        raise KeyError(\n            f"Expected one operation for {method} {path_template}; "\n            f"found {len(candidates)}."\n        )\n\n    return candidates[0]\n\n\ndef response_object(document, operation, status):\n    responses = operation.get("responses", {})\n    status_text = str(status)\n\n    response_key = None\n\n    if status_text in responses:\n        response_key = status_text\n    else:\n        wildcard = (\n            status_text[0] + "XX"\n            if len(status_text) == 3 and status_text.isdigit()\n            else None\n        )\n        if wildcard and wildcard in responses:\n            response_key = wildcard\n        elif "default" in responses:\n            response_key = "default"\n\n    if response_key is None:\n        raise KeyError(f"No response maps status {status_text}.")\n\n    return response_key, dereference(\n        document,\n        responses[response_key],\n    )\n\n\ndef response_schema(response):\n    content = response.get("content")\n\n    if isinstance(content, dict):\n        preferred = [\n            "application/json",\n            "application/problem+json",\n        ]\n\n        media_types = preferred + sorted(\n            media_type\n            for media_type in content\n            if media_type not in preferred\n        )\n\n        for media_type in media_types:\n            media = content.get(media_type)\n            if (\n                isinstance(media, dict)\n                and isinstance(media.get("schema"), dict)\n            ):\n                return media_type, media["schema"]\n\n    if isinstance(response.get("schema"), dict):\n        return None, response["schema"]\n\n    return None, None\n\n\ndef validator_for(document, schema):\n    version = str(document.get("openapi", "3.0.0"))\n    is_31 = version.startswith("3.1")\n    validator_class = (\n        jsonschema.Draft202012Validator\n        if is_31\n        else jsonschema.Draft4Validator\n    )\n\n    # Resolve all nested \'#/components/...\' pointers against the verified\n    # complete OpenAPI document. No external network retrieval is permitted.\n    root = {\n        "$schema": (\n            "https://json-schema.org/draft/2020-12/schema"\n            if is_31 else "http://json-schema.org/draft-04/schema#"\n        ),\n        "$ref": "#/$defs/stage05_selected_response_schema",\n        "$defs": {"stage05_selected_response_schema": schema},\n        "components": document.get("components", {}),\n        "definitions": document.get("definitions", {}),\n    }\n\n    def block_external(uri):\n        raise ValueError("External schema retrieval prohibited: " + str(uri))\n\n    return validator_class(\n        root,\n        registry=Registry(retrieve=block_external),\n        format_checker=jsonschema.FormatChecker(),\n    )\n\n\ndef instance_is_valid(document, schema, value):\n    if schema is None:\n        return value is None\n\n    return not list(validator_for(document, schema).iter_errors(value))\n\n\ndef schema_view(document, schema):\n    if not isinstance(schema, dict):\n        return {}\n    return dereference(document, schema)\n\n\ndef escaped_token(value):\n    return str(value).replace("~", "~0").replace("/", "~1")\n\n\ndef scalar_alternatives(document, value, schema):\n    schema = schema_view(document, schema)\n\n    if "const" in schema:\n        return []\n\n    enum = schema.get("enum")\n    if isinstance(enum, list):\n        return [\n            deepcopy(candidate)\n            for candidate in enum\n            if canonical_json(candidate) != canonical_json(value)\n        ]\n\n    schema_type = schema.get("type")\n\n    if isinstance(value, bool) or schema_type == "boolean":\n        return [not bool(value)]\n\n    if (\n        isinstance(value, int)\n        and not isinstance(value, bool)\n    ) or schema_type == "integer":\n        candidates = [int(value) + 1, int(value) - 1, 0, 1]\n        return list(dict.fromkeys(candidates))\n\n    if isinstance(value, float) or schema_type == "number":\n        candidates = [float(value) + 1.0, float(value) - 1.0, 0.0]\n        return list(dict.fromkeys(candidates))\n\n    if isinstance(value, str) or schema_type == "string":\n        candidates = [\n            str(value) + "_alt",\n            "alternative",\n            "A",\n            "",\n        ]\n        return list(dict.fromkeys(candidates))\n\n    return []\n\n\ndef mutations(document, value, schema, pointer="", depth=0):\n    if depth > 4:\n        return []\n\n    schema = schema_view(document, schema)\n    results = []\n\n    if isinstance(value, dict):\n        properties = schema.get("properties", {})\n        required = set(schema.get("required", []))\n\n        for key in sorted(value):\n            child_pointer = pointer + "/" + escaped_token(key)\n\n            if key not in required:\n                candidate = deepcopy(value)\n                candidate.pop(key)\n                results.append(\n                    (\n                        candidate,\n                        {\n                            "target": "body",\n                            "operation": "omit_optional_property",\n                            "pointer": child_pointer,\n                        },\n                    )\n                )\n\n            child_schema = (\n                properties.get(key, {})\n                if isinstance(properties, dict)\n                else {}\n            )\n\n            for child_value, transformation in mutations(\n                document,\n                value[key],\n                child_schema,\n                child_pointer,\n                depth + 1,\n            ):\n                candidate = deepcopy(value)\n                candidate[key] = child_value\n                results.append((candidate, transformation))\n\n    elif isinstance(value, list):\n        item_schema = schema.get("items", {})\n\n        if value and len(value) > int(schema.get("minItems", 0)):\n            candidate = deepcopy(value[:-1])\n            results.append(\n                (\n                    candidate,\n                    {\n                        "target": "body",\n                        "operation": "remove_optional_array_item",\n                        "pointer": pointer,\n                    },\n                )\n            )\n\n        for index, item in enumerate(value[:4]):\n            child_pointer = pointer + "/" + str(index)\n            for child_value, transformation in mutations(\n                document,\n                item,\n                item_schema,\n                child_pointer,\n                depth + 1,\n            ):\n                candidate = deepcopy(value)\n                candidate[index] = child_value\n                results.append((candidate, transformation))\n\n    else:\n        for candidate in scalar_alternatives(\n            document,\n            value,\n            schema,\n        ):\n            results.append(\n                (\n                    candidate,\n                    {\n                        "target": "body",\n                        "operation": "replace_with_permitted_value",\n                        "pointer": pointer or "",\n                        "from": deepcopy(value),\n                        "to": deepcopy(candidate),\n                    },\n                )\n            )\n\n    return results\n\n\ndef header_variations(document, response, headers):\n    results = []\n    definitions = response.get("headers", {})\n\n    if not isinstance(definitions, dict):\n        return results\n\n    for header_name in sorted(definitions):\n        header = dereference(document, definitions[header_name])\n        schema = header.get("schema", header)\n        existing_key = next(\n            (\n                key\n                for key in headers\n                if key.lower() == header_name.lower()\n            ),\n            None,\n        )\n\n        if existing_key is None:\n            enum = schema_view(document, schema).get("enum")\n            if isinstance(enum, list) and enum:\n                value = deepcopy(enum[0])\n            else:\n                schema_type = schema_view(document, schema).get(\n                    "type",\n                    "string",\n                )\n                value = {\n                    "integer": 1,\n                    "number": 1.0,\n                    "boolean": True,\n                    "array": [],\n                }.get(schema_type, "value")\n\n            candidate_headers = deepcopy(headers)\n            candidate_headers[header_name] = value\n            results.append(\n                (\n                    candidate_headers,\n                    {\n                        "target": "headers",\n                        "operation": "add_documented_optional_header",\n                        "header": header_name,\n                    },\n                )\n            )\n\n    return results\n\n\ndef headers_are_valid(document, response, headers):\n    definitions = response.get("headers", {})\n    if not isinstance(definitions, dict):\n        return True\n    for name, definition in definitions.items():\n        present = next((key for key in headers if key.lower() == name.lower()), None)\n        header = dereference(document, definition)\n        if not isinstance(header, dict):\n            continue\n        if present is None:\n            if header.get("required") is True:\n                return False\n            continue\n        schema = header.get("schema", {})\n        if not instance_is_valid(document, schema, headers[present]):\n            return False\n    return True\n\n\ndef build_variations(\n    document,\n    op_id,\n    method,\n    path_template,\n    baselines,\n    maximum_per_baseline=12,\n):\n    matched_path, operation = locate_operation(\n        document,\n        method,\n        path_template,\n    )\n\n    variations = []\n    diagnostics = []\n    for baseline in sorted(\n        baselines,\n        key=lambda item: item["baseline_id"],\n    ):\n        # Deduplicate within each baseline; a variation belongs to its own\n        # baseline even when two scenarios start from identical responses.\n        seen_payloads = set()\n        status = str(baseline["status"])\n        headers = deepcopy(baseline.get("headers", {}))\n        body = deepcopy(baseline.get("body"))\n\n        response_key, response = response_object(\n            document,\n            operation,\n            status,\n        )\n        media_type, schema = response_schema(response)\n\n        candidates = []\n\n        if schema is not None:\n            for candidate_body, transformation in mutations(\n                document,\n                body,\n                schema,\n            ):\n                if (\n                    canonical_json(candidate_body)\n                    != canonical_json(body)\n                    and instance_is_valid(\n                        document,\n                        schema,\n                        candidate_body,\n                    )\n                ):\n                    candidates.append(\n                        (\n                            status,\n                            headers,\n                            candidate_body,\n                            transformation,\n                        )\n                    )\n\n        for candidate_headers, transformation in header_variations(\n            document,\n            response,\n            headers,\n        ):\n            candidates.append(\n                (\n                    status,\n                    candidate_headers,\n                    body,\n                    transformation,\n                )\n            )\n\n        responses = operation.get("responses", {})\n        for alternative_status in sorted(responses):\n            if (\n                not re.fullmatch(r"2\\d\\d", str(alternative_status))\n                or str(alternative_status) == status\n            ):\n                continue\n\n            _, alternative_response = response_object(\n                document,\n                operation,\n                alternative_status,\n            )\n            _, alternative_schema = response_schema(\n                alternative_response\n            )\n\n            if instance_is_valid(\n                document,\n                alternative_schema,\n                body,\n            ):\n                candidates.append(\n                    (\n                        str(alternative_status),\n                        headers,\n                        body,\n                        {\n                            "target": "status",\n                            "operation": "use_documented_alternative_success",\n                            "from": status,\n                            "to": str(alternative_status),\n                        },\n                    )\n                )\n\n        emitted_for_baseline = 0\n\n        for (\n            candidate_status,\n            candidate_headers,\n            candidate_body,\n            transformation,\n        ) in candidates:\n            _, candidate_response = response_object(\n                document, operation, candidate_status\n            )\n            _, candidate_schema = response_schema(candidate_response)\n            if not instance_is_valid(document, candidate_schema, candidate_body):\n                continue\n            if not headers_are_valid(document, candidate_response, candidate_headers):\n                continue\n            payload = {\n                "status": candidate_status,\n                "headers": candidate_headers,\n                "body": candidate_body,\n            }\n            payload_key = canonical_json(payload)\n\n            if payload_key in seen_payloads:\n                continue\n\n            baseline_payload = canonical_json(\n                {\n                    "status": status,\n                    "headers": headers,\n                    "body": body,\n                }\n            )\n            if payload_key == baseline_payload:\n                continue\n\n            variation_identity = {\n                "builder_version": BUILDER_VERSION,\n                "op_id": op_id,\n                "baseline_id": baseline["baseline_id"],\n                "payload": payload,\n                "transformation": transformation,\n            }\n\n            variations.append(\n                {\n                    "variation_id": stable_id(\n                        "variation",\n                        variation_identity,\n                    ),\n                    "op_id": op_id,\n                    "baseline_id": baseline["baseline_id"],\n                    "variation_kind": transformation["operation"],\n                    "transformation": transformation,\n                    "status": candidate_status,\n                    "headers": candidate_headers,\n                    "body": candidate_body,\n                    "expected_equivalent": True,\n                    "requirement_ids": sorted(\n                        set(baseline.get("requirement_ids", []))\n                    ),\n                    "source_evidence": {\n                        "matched_path": matched_path,\n                        "response_key": response_key,\n                        "media_type": media_type,\n                    },\n                    "validation_status": "PASS",\n                }\n            )\n\n            seen_payloads.add(payload_key)\n            emitted_for_baseline += 1\n\n            if emitted_for_baseline >= maximum_per_baseline:\n                break\n\n        if emitted_for_baseline == 0:\n            diagnostics.append(\n                {\n                    "severity": "INFO",\n                    "code": "NO_DISTINCT_VALID_VARIATION",\n                    "op_id": op_id,\n                    "baseline_id": baseline["baseline_id"],\n                }\n            )\n\n    variations.sort(key=lambda item: item["variation_id"])\n    diagnostics.sort(\n        key=lambda item: (\n            item["op_id"],\n            item["baseline_id"],\n            item["code"],\n        )\n    )\n\n    return {\n        "builder_version": BUILDER_VERSION,\n        "op_id": op_id,\n        "matched_path": matched_path,\n        "variations": variations,\n        "diagnostics": diagnostics,\n        "status": "PASS",\n    }\n', 'routing.py': '"""Deterministic OpenAPI request evaluation and frozen response delivery."""\nfrom __future__ import annotations\n\nimport base64\nimport hashlib\nimport json\nimport re\nfrom urllib.parse import parse_qs, unquote, urlsplit\n\nfrom jsonschema import Draft4Validator, Draft202012Validator\nfrom jsonschema.exceptions import SchemaError\nfrom referencing import Registry, Resource\nfrom referencing.jsonschema import DRAFT4, DRAFT202012\n\n\nROUTING_VERSION = "stage06-openapi-contract-v6"\nDOCUMENT_URI = "urn:stage06:canonical-operation"\nFORM_MEDIA_TYPE = "application/x-www-form-urlencoded"\nLEGACY_FORM_RULE = {\n    "op_id": "op_42818f46abfce2a2ec0520ea",\n    "method": "PUT",\n    "path_template": "/api/v1/filters/{id}",\n    "document_sha256": "0f23e5b6c3e149d86ba4e3920ad0cb9f3a5feee81ce14b32f839852cb8b56cff",\n    "declared_media_type": "application/form-data",\n    "wire_media_type": FORM_MEDIA_TYPE,\n    "array_fields": {"context": "context[]"},\n    "sources": [\n        "https://docs.joinmastodon.org/methods/filters/#update-legacy",\n        "https://docs.joinmastodon.org/client/intro/#form-data",\n        "https://docs.joinmastodon.org/client/intro/#array",\n    ],\n}\nNEGATIVE_INTENTS = {\n    "invalid_missing_required_parameter": ("parameter", "missing"),\n    "invalid_missing_required_request_body": ("body", "missing"),\n    "parameter_invalid_wrong_type": ("parameter", "type"),\n    "parameter_invalid_enum_value": ("parameter", "enum"),\n    "parameter_invalid_below_minimum": ("parameter", "minimum"),\n    "parameter_invalid_below_minimum_items": ("parameter", "minItems"),\n    "body_invalid_wrong_type": ("body", "type"),\n    "body_invalid_missing_required_property": ("body", "required"),\n    "body_invalid_below_minimum": ("body", "minimum"),\n    "body_invalid_below_minimum_items": ("body", "minItems"),\n    "body_invalid_below_minimum_length": ("body", "minLength"),\n    "body_invalid_above_maximum_length": ("body", "maxLength"),\n}\n\n\nclass UnsupportedRoute(RuntimeError):\n    failure_owner = "GENERATED_OUTPUT"\n\n    def __init__(self, code, detail=None):\n        self.code = code\n        self.detail = detail\n        super().__init__(code)\n\n\nclass EvaluatorUnavailable(UnsupportedRoute):\n    failure_owner = "EVALUATOR_LIMITATION"\n\n\nclass EvaluatorDefect(UnsupportedRoute):\n    failure_owner = "EVALUATOR_FAILURE"\n\n\ndef normalize_prepared_request(method, url, headers=None, body=None):\n    """Keep transport details even for requests the router will reject."""\n    try:\n        parts = urlsplit(url)\n        origin_allowed = (\n            parts.scheme == "http" and parts.netloc.lower() == "openapi.test"\n        )\n    except ValueError as error:\n        raise UnsupportedRoute("MALFORMED_REQUEST_URL") from error\n    headers = {str(k).lower(): str(v) for k, v in (headers or {}).items()}\n    cookies = dict(\n        tuple(part.strip().split("=", 1))\n        for part in headers.get("cookie", "").split(";") if "=" in part\n    )\n    content_type = headers.get("content-type", "").lower().strip()\n    media_type = content_type.split(";", 1)[0].strip()\n    body_present = body is not None and body != b"" and body != ""\n    body_encoding = "none"\n    if body_present:\n        if isinstance(body, bytes):\n            try:\n                body = body.decode("utf-8")\n            except UnicodeDecodeError:\n                body = {"base64": base64.b64encode(body).decode("ascii")}\n                body_encoding = "binary"\n        if isinstance(body, str):\n            body_encoding = "text"\n            if _is_json(media_type):\n                try:\n                    body = json.loads(body, parse_constant=_reject_json_constant)\n                    body_encoding = "json"\n                except ValueError:\n                    body_encoding = "malformed_json"\n            elif media_type == "application/x-www-form-urlencoded":\n                body = _query(body)\n                body_encoding = "form"\n        elif body_encoding != "binary":\n            body_encoding = "json"\n    else:\n        body = None\n    return {\n        "method": method.upper(), "raw_url": url, "origin_allowed": origin_allowed,\n        "path": parts.path or "/", "query": _query(parts.query),\n        "headers": headers, "cookies": cookies, "body": body,\n        "body_present": body_present, "body_encoding": body_encoding,\n        "media_type": content_type,\n    }\n\n\ndef _is_json(media_type):\n    return media_type in {"application/json", "text/json"} or media_type.endswith("+json")\n\n\ndef _reject_json_constant(value):\n    raise ValueError(f"Non-JSON constant: {value}")\n\n\ndef _query(text):\n    return {key: values[0] if len(values) == 1 else values\n            for key, values in parse_qs(text, keep_blank_values=True).items()}\n\n\ndef match_path_template(template, actual_path):\n    """Match path segments before unquoting parameter values."""\n    names = re.findall(r"\\{([^}]+)\\}", template)\n    fragments = re.split(r"\\{[^}]+\\}", template)\n    pattern = "([^/]+)".join(re.escape(fragment) for fragment in fragments)\n    match = re.fullmatch(pattern, actual_path)\n    return None if match is None else dict(zip(names, map(unquote, match.groups())))\n\n\ndef _local_reference(document, reference):\n    if not isinstance(reference, str) or not reference.startswith("#/"):\n        raise EvaluatorUnavailable("UNSUPPORTED_REFERENCE", reference)\n    node = document\n    try:\n        for part in unquote(reference[2:]).split("/"):\n            node = node[part.replace("~1", "/").replace("~0", "~")]\n    except (KeyError, TypeError) as error:\n        raise EvaluatorUnavailable("UNRESOLVED_LOCAL_REFERENCE", reference) from error\n    return node\n\n\ndef _object(document, value):\n    seen = set()\n    while isinstance(value, dict) and "$ref" in value:\n        reference = value["$ref"]\n        if reference in seen:\n            raise EvaluatorUnavailable("CYCLIC_OBJECT_REFERENCE")\n        seen.add(reference)\n        value = _local_reference(document, reference)\n    return value\n\n\ndef _schema_tree(value, openapi30):\n    if isinstance(value, list):\n        return [_schema_tree(item, openapi30) for item in value]\n    if not isinstance(value, dict):\n        return value\n    result = {\n        key: (item if key in {"enum", "const", "default", "example", "examples"}\n              else _schema_tree(item, openapi30))\n        for key, item in value.items()\n    }\n    if openapi30 and result.get("nullable") is True and isinstance(result.get("type"), str):\n        result["type"] = [result["type"], "null"]\n    return result\n\n\ndef _schema_errors(schema, instance, document):\n    if not isinstance(schema, (dict, bool)):\n        raise EvaluatorDefect("MALFORMED_FROZEN_SCHEMA")\n    openapi30 = document.get("openapi", "").startswith("3.0.")\n    spec = DRAFT4 if openapi30 else DRAFT202012\n    validator = Draft4Validator if openapi30 else Draft202012Validator\n    if openapi30 and isinstance(schema, bool):\n        raise EvaluatorUnavailable("UNSUPPORTED_OPENAPI30_BOOLEAN_SCHEMA")\n    # A registry without a retrieval callback only resolves the supplied\n    # canonical document and built-in meta-schemas. It cannot fetch URLs.\n    local_root = _schema_tree(document, openapi30)\n    registry = Registry().with_resource(DOCUMENT_URI, Resource(local_root, spec))\n    schema = _schema_tree(schema, openapi30)\n    try:\n        validator.check_schema(schema)\n        return list(validator(schema, registry=registry,\n                              _resolver=registry.resolver(DOCUMENT_URI),\n                              format_checker=validator.FORMAT_CHECKER).iter_errors(instance))\n    except SchemaError as error:\n        raise EvaluatorUnavailable("UNSUPPORTED_FROZEN_SCHEMA", str(error)) from error\n    except Exception as error:\n        # Referencing exceptions are wrapped by jsonschema.\n        if type(error).__module__.split(".")[0] in {"jsonschema", "referencing"}:\n            raise EvaluatorUnavailable("UNRESOLVED_SCHEMA_REFERENCE", str(error)) from error\n        raise\n\n\ndef _scalar(value, schema):\n    kind = schema.get("type")\n    if isinstance(kind, list):\n        choices = [item for item in kind if item != "null"]\n        if len(choices) != 1:\n            raise EvaluatorUnavailable("AMBIGUOUS_PARAMETER_SERIALIZATION")\n        kind = choices[0]\n    if kind == "integer" and re.fullmatch(r"[+-]?[0-9]+", value):\n        try:\n            return int(value)\n        except ValueError:\n            return value\n    if kind == "number" and re.fullmatch(r"[+-]?(?:[0-9]+(?:\\.[0-9]*)?|\\.[0-9]+)(?:[eE][+-]?[0-9]+)?", value):\n        number = float(value)\n        if number not in (float("inf"), float("-inf")):\n            return number\n    if kind == "boolean" and value in {"true", "false"}:\n        return value == "true"\n    return value\n\n\ndef _parameter_value(raw, parameter, schema, document):\n    location = parameter["in"]\n    style = parameter.get("style", "form" if location in {"query", "cookie"} else "simple")\n    explode = parameter.get("explode", style == "form")\n    if style not in ({"form", "spaceDelimited", "pipeDelimited"} if location == "query"\n                     else {"form"} if location == "cookie" else {"simple"}):\n        raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_STYLE", (location, style))\n    if schema.get("type") == "object":\n        if explode:\n            raise EvaluatorUnavailable("UNSUPPORTED_EXPLODED_OBJECT_PARAMETER", parameter["name"])\n        if isinstance(raw, list):\n            raise EvaluatorUnavailable("AMBIGUOUS_OBJECT_PARAMETER", parameter["name"])\n        parts = raw.split(",")\n        if len(parts) % 2:\n            raise EvaluatorUnavailable("UNSUPPORTED_OBJECT_PARAMETER", parameter["name"])\n        return {key: _scalar(value, schema.get("properties", {}).get(key, {}))\n                for key, value in zip(parts[::2], parts[1::2])}\n    if schema.get("type") != "array":\n        if isinstance(raw, list):\n            return raw\n        return _scalar(raw, schema)\n    items = _object(document, schema.get("items") or {})\n    if style == "form" and explode:\n        values = raw if isinstance(raw, list) else [raw]\n    else:\n        if isinstance(raw, list):\n            raise EvaluatorUnavailable("AMBIGUOUS_ARRAY_SERIALIZATION", parameter["name"])\n        values = raw.split({"spaceDelimited": " ", "pipeDelimited": "|"}.get(style, ","))\n    return [_scalar(value, items) for value in values]\n\n\ndef _select_media(content, received):\n    if not received:\n        raise EvaluatorUnavailable("REQUEST_MEDIA_TYPE_UNSPECIFIED")\n    received, *parameters = [part.strip() for part in received.split(";")]\n    received_params = set(parameters)\n    matches = []\n    for declared, media in content.items():\n        name, *params = [part.strip() for part in declared.lower().split(";")]\n        if not set(params).issubset(received_params):\n            continue\n        if name == received:\n            matches.append(((2, len(params)), media))\n        elif name == received.split("/", 1)[0] + "/*":\n            matches.append(((1, len(params)), media))\n        elif name == "*/*":\n            matches.append(((0, len(params)), media))\n    if not matches:\n        return None\n    best = max(rank for rank, _ in matches)\n    winners = [media for rank, media in matches if rank == best]\n    if len(winners) != 1:\n        raise EvaluatorUnavailable("AMBIGUOUS_REQUEST_MEDIA_TYPE")\n    return winners[0]\n\n\ndef _operation(document, template, method):\n    try:\n        path = _object(document, document["paths"][template])\n        return path, _object(document, path[method.lower()])\n    except (KeyError, TypeError) as error:\n        raise EvaluatorDefect("ORACLE_OPERATION_NOT_IN_CANONICAL_DOCUMENT") from error\n\n\nSUCCESS_STATUS_POLICY = "single-explicit-success-v1"\n\n\ndef success_status_mapping(document, path_template, method):\n    """Return a status only when the operation has one explicit success candidate."""\n    _, operation = _operation(document, path_template, method)\n    responses = operation.get("responses", {})\n    codes = sorted(str(code) for code in responses if re.fullmatch(r"2[0-9]{2}", str(code)))\n    has_range = any(str(code).upper() == "2XX" for code in responses)\n    if len(codes) > 1 or has_range:\n        raise EvaluatorUnavailable("AMBIGUOUS_SUCCESS_STATUS_MAPPING",\n                                   {"statuses": codes, "has_2xx_range": has_range})\n    return {"status": int(codes[0])} if codes else None\n\n\ndef verify_success_status_mapping(document, scenario):\n    if scenario["expected_validity"] != "VALID":\n        return\n    expected = success_status_mapping(document, scenario["path_template"], scenario["method"])\n    if expected is None:\n        raise EvaluatorUnavailable("NO_DOCUMENTED_RESPONSE_MAPPING")\n    if scenario.get("documented_response_mapping") != expected:\n        raise EvaluatorDefect("FROZEN_SUCCESS_STATUS_MAPPING_MISMATCH")\n\n\ndef legacy_form_rule(document, template, method):\n    rule = LEGACY_FORM_RULE\n    if (template, method.upper()) != (rule["path_template"], rule["method"]):\n        return None\n    encoded = json.dumps(document, sort_keys=True, separators=(",", ":"),\n                         ensure_ascii=False).encode()\n    return rule if hashlib.sha256(encoded).hexdigest() == rule["document_sha256"] else None\n\n\ndef _decode_form(body, media, document, rule):\n    schema = _object(document, media.get("schema", {}))\n    if media.get("encoding") or not isinstance(body, dict):\n        raise EvaluatorUnavailable("UNSUPPORTED_FORM_ENCODING")\n    properties = schema.get("properties", {})\n    aliases = {wire: logical for logical, wire in (rule or {}).get("array_fields", {}).items()}\n    decoded = {}\n    for wire, raw in body.items():\n        if wire in (rule or {}).get("array_fields", {}):\n            raise EvaluatorUnavailable("UNSUPPORTED_UNBRACKETED_FORM_ARRAY", wire)\n        logical = aliases.get(wire, wire)\n        property_schema = _object(document, properties.get(logical, {}))\n        decoded[logical] = _parameter_value(\n            raw, {"in": "query", "name": logical}, property_schema, document)\n    return decoded\n\n\ndef _security_observation(document, operation, request):\n    """Record credential presence; never infer authentication success offline."""\n    alternatives = operation.get("security", document.get("security", []))\n    if not alternatives or {} in alternatives:\n        return "NOT_REQUIRED"\n    schemes = (document.get("components") or {}).get("securitySchemes", {})\n    for option in alternatives:\n        supplied = []\n        for name in option:\n            scheme = _object(document, schemes.get(name, {}))\n            if scheme.get("type") == "apiKey":\n                location, key = scheme.get("in"), scheme.get("name", "")\n                source = {"header": request["headers"], "query": request["query"],\n                          "cookie": request["cookies"]}.get(location, {})\n                supplied.append(bool(source.get(key.lower() if location == "header" else key)))\n            else:\n                supplied.append(bool(request["headers"].get("authorization")))\n        if supplied and all(supplied):\n            return "PRESENT_NOT_VERIFIED"\n    return "MISSING_OR_UNVERIFIABLE"\n\n\ndef _assess(request, document, template, path_values):\n    path, operation = _operation(document, template, request["method"])\n    parameters = {}\n    for raw in [*path.get("parameters", []), *operation.get("parameters", [])]:\n        parameter = _object(document, raw)\n        location, name = parameter.get("in"), parameter.get("name")\n        if location not in {"path", "query", "header", "cookie"} or not name:\n            raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_DEFINITION")\n        key = name.lower() if location == "header" else name\n        parameters[location, key] = parameter\n    values = {"path": path_values, "query": request["query"],\n              "header": request["headers"], "cookie": request["cookies"]}\n    violations = set()\n    typed = {}\n    for (location, key), parameter in parameters.items():\n        if key not in values[location]:\n            if parameter.get("required") is True or location == "path":\n                violations.add(("parameter", location, key, "missing"))\n            continue\n        if "content" in parameter:\n            raise EvaluatorUnavailable("UNSUPPORTED_PARAMETER_CONTENT", (location, key))\n        schema = parameter.get("schema") or {}\n        effective_schema = _object(document, schema)\n        value = _parameter_value(values[location][key], parameter, effective_schema, document)\n        typed.setdefault(location, {})[key] = value\n        for error in _schema_errors(schema, value, document):\n            violations.add(("parameter", location, key, str(error.validator)))\n        if (isinstance(value, int) and not isinstance(value, bool)\n                and effective_schema.get("format") in {"int32", "int64"}):\n            bits = 32 if effective_schema["format"] == "int32" else 64\n            if not -(2 ** (bits - 1)) <= value < 2 ** (bits - 1):\n                violations.add(("parameter", location, key, "format"))\n    request_body = _object(document, operation.get("requestBody", {}))\n    if not request["body_present"]:\n        if request_body.get("required") is True:\n            violations.add(("body", "", "", "missing"))\n    elif request_body:\n        content = request_body.get("content", {})\n        rule = legacy_form_rule(document, template, request["method"])\n        wire_type = request["media_type"].split(";", 1)[0].strip()\n        media = (_select_media(content, rule["declared_media_type"])\n                 if rule and wire_type == rule["wire_media_type"]\n                 else _select_media(content, request["media_type"]))\n        if rule and wire_type == rule["declared_media_type"]:\n            raise EvaluatorUnavailable("UNSUPPORTED_LEGACY_FORM_WIRE_TYPE")\n        if media is None:\n            violations.add(("body", "", "", "media_type"))\n        elif request["body_encoding"] == "malformed_json":\n            violations.add(("body", "", "", "malformed_json"))\n        else:\n            if request["body_encoding"] not in {"json", "form"}:\n                raise EvaluatorUnavailable("UNSUPPORTED_REQUEST_BODY_SERIALIZATION")\n            value = request["body"]\n            schema = media.get("schema", {})\n            if request["body_encoding"] == "form":\n                value = _decode_form(value, media, document, rule)\n            for error in _schema_errors(schema, value, document):\n                pointer = "/".join(map(str, error.absolute_path))\n                if error.validator == "required" and isinstance(error.instance, dict):\n                    for missing in set(error.validator_value) - error.instance.keys():\n                        violations.add(("body", "", f"{pointer}/{missing}", "required"))\n                else:\n                    violations.add(("body", "", pointer, str(error.validator)))\n    return typed, violations, _security_observation(document, operation, request)\n\n\ndef _literal(value):\n    if isinstance(value, bool):\n        return str(value).lower()\n    return str(value)\n\n\ndef _fixture_values(scenario):\n    values = scenario.get("parameter_values") or {}\n    return {\n        location: {\n            key.lower() if location == "header" else key:\n            ([_literal(x) for x in value] if isinstance(value, list) and len(value) != 1\n             else _literal(value[0]) if isinstance(value, list) and len(value) == 1\n             else _literal(value))\n            for key, value in (values.get(location) or {}).items()\n        }\n        for location in ("path", "query", "header", "cookie")\n    }\n\n\ndef _exact_fixture(request, path, scenario):\n    values = _fixture_values(scenario)\n    return (\n        path == values["path"] and request["query"] == values["query"]\n        and all(request["headers"].get(k) == v for k, v in values["header"].items())\n        and all(request["cookies"].get(k) == v for k, v in values["cookie"].items())\n        and (scenario.get("body") is not None) == request["body_present"]\n        and json.dumps(request["body"], sort_keys=True)\n        == json.dumps(scenario.get("body"), sort_keys=True)\n    )\n\n\ndef _negative_signature(scenario, document, template, request):\n    values = _fixture_values(scenario)\n    fixture_request = {\n        **request, "query": values["query"], "headers": values["header"],\n        "cookies": values["cookie"], "body": scenario.get("body"),\n        "body_present": scenario.get("body") is not None, "body_encoding": "json",\n    }\n    try:\n        _, violations, _ = _assess(fixture_request, document, template, values["path"])\n    except EvaluatorUnavailable:\n        return set()\n    return violations\n\n\ndef prepare_frozen_fixture(document, scenario):\n    """Serialize a frozen positive fixture according to its operation."""\n    from urllib.parse import quote, urlencode\n\n    template = scenario["path_template"]\n    path, operation = _operation(document, template, scenario["method"])\n    parameters = {}\n    for raw in [*path.get("parameters", []), *operation.get("parameters", [])]:\n        parameter = _object(document, raw)\n        parameters[parameter["in"], parameter["name"].lower()\n                   if parameter["in"] == "header" else parameter["name"]] = parameter\n    values = _fixture_values(scenario)\n    for location, entries in (scenario.get("parameter_values") or {}).items():\n        for name, value in entries.items():\n            key = name.lower() if location == "header" else name\n            parameter = parameters.get((location, key), {})\n            style = parameter.get("style", "form" if location in {"query", "cookie"} else "simple")\n            explode = parameter.get("explode", style == "form")\n            if isinstance(value, dict) and not value and style == "form" and explode:\n                # An empty exploded object produces no wire parameters.\n                values[location].pop(key, None)\n            elif isinstance(value, list) and (style != "form" or not explode):\n                separator = {"spaceDelimited": " ", "pipeDelimited": "|"}.get(style, ",")\n                values[location][key] = separator.join(map(_literal, value))\n    url_path = template\n    for key, value in values["path"].items():\n        url_path = url_path.replace("{" + key + "}", quote(value, safe=""))\n    if "{" in url_path:\n        raise EvaluatorUnavailable("FROZEN_FIXTURE_PATH_UNSERIALIZABLE")\n    headers = values["header"]\n    if values["cookie"]:\n        headers["cookie"] = "; ".join(f"{k}={v}" for k, v in values["cookie"].items())\n    body = scenario.get("body")\n    if body is not None:\n        content = _object(document, operation.get("requestBody", {})).get("content", {})\n        json_media = [m for m in content if _is_json(m.split(";", 1)[0].strip().lower())]\n        rule = legacy_form_rule(document, template, scenario["method"])\n        if rule and rule["declared_media_type"] in content and isinstance(body, dict):\n            headers["content-type"] = rule["wire_media_type"]\n            form = {}\n            for name, value in body.items():\n                wire = rule["array_fields"].get(name, name)\n                form[wire] = ([_literal(item) for item in value]\n                              if isinstance(value, list) else _literal(value))\n            encoded = urlencode(form, doseq=True)\n            decoded = _decode_form(_query(encoded), content[rule["declared_media_type"]],\n                                   document, rule)\n            if decoded != body:\n                raise EvaluatorDefect("FROZEN_FORM_ROUNDTRIP_MISMATCH")\n            body = encoded.encode()\n        elif json_media:\n            declared = headers.get("content-type", "")\n            if declared not in json_media:\n                headers["content-type"] = min(\n                    json_media, key=lambda media: (media != "application/json", media)\n                )\n            body = json.dumps(body).encode()\n        elif "application/x-www-form-urlencoded" in content and isinstance(body, dict):\n            headers["content-type"] = "application/x-www-form-urlencoded"\n            body = urlencode(body, doseq=True).encode()\n        else:\n            raise EvaluatorUnavailable("FROZEN_FIXTURE_BODY_UNSERIALIZABLE", sorted(content))\n    query = urlencode(values["query"], doseq=True)\n    return normalize_prepared_request(\n        scenario["method"], "http://openapi.test" + url_path + ("?" + query if query else ""),\n        headers, body,\n    )\n\n\ndef prepare_frozen_response(record, baseline):\n    """Serialize JSON strings as JSON and preserve registered response headers."""\n    evidence = record.get("source_evidence") or ()\n    if isinstance(evidence, dict):\n        evidence = [evidence]\n    baseline_evidence = baseline.get("source_evidence") or ()\n    if isinstance(baseline_evidence, dict):\n        baseline_evidence = [baseline_evidence]\n    media_type = next((row["media_type"] for row in [*evidence, *baseline_evidence]\n                       if row.get("media_type")), "")\n    headers = dict(record.get("headers") or {})\n    content_type = next((v for k, v in headers.items() if k.lower() == "content-type"), "")\n    body = record.get("body")\n    if body is None:\n        return headers, b""\n    if not content_type and media_type:\n        headers["Content-Type"] = media_type\n        content_type = media_type\n    if _is_json(content_type.split(";", 1)[0].strip().lower()) or not isinstance(body, str):\n        return headers, json.dumps(body, allow_nan=False).encode()\n    return headers, body.encode()\n\n\ndef audit_frozen_response(document, scenario, record, baseline):\n    """Independently check that a baseline or variation obeys its response contract."""\n    verify_success_status_mapping(document, scenario)\n    _, operation = _operation(document, scenario["path_template"], scenario["method"])\n    code = str(record["status"])\n    responses = operation.get("responses", {})\n    response = responses.get(code, responses.get(code[:1] + "XX", responses.get("default")))\n    if response is None:\n        raise EvaluatorDefect("UNDOCUMENTED_FROZEN_RESPONSE_STATUS", code)\n    response = _object(document, response)\n    headers, _ = prepare_frozen_response(record, baseline)\n    headers = {key.lower(): value for key, value in headers.items()}\n    for name, raw in (response.get("headers") or {}).items():\n        declaration = _object(document, raw)\n        if declaration.get("required") and name.lower() not in headers:\n            raise EvaluatorDefect("MISSING_REQUIRED_FROZEN_RESPONSE_HEADER", name)\n    content = response.get("content") or {}\n    if not content:\n        if record.get("body") is not None:\n            raise EvaluatorDefect("UNDOCUMENTED_FROZEN_RESPONSE_BODY")\n        return\n    media = _select_media(content, headers.get("content-type", ""))\n    if media is None:\n        raise EvaluatorDefect("FROZEN_RESPONSE_MEDIA_TYPE_MISMATCH")\n    errors = _schema_errors(media.get("schema", {}), record.get("body"), document)\n    if errors:\n        raise EvaluatorDefect("FROZEN_RESPONSE_SCHEMA_MISMATCH",\n                              sorted({str(error.validator) for error in errors}))\n\n\ndef classify_request(request, frozen_scenarios, operation_document=None):\n    if not request["origin_allowed"]:\n        raise UnsupportedRoute("UNAUTHORIZED_HOST")\n    matches = []\n    for scenario in frozen_scenarios:\n        if (scenario.get("applicability") == "APPLICABLE"\n                and request["method"] == scenario["method"].upper()):\n            template = scenario["path_template"]\n            path = match_path_template(template, request["path"])\n            if path is not None:\n                matches.append((scenario, template, path))\n    if not matches:\n        raise UnsupportedRoute("REQUEST_OUTSIDE_TARGET_OPERATION")\n    if len({scenario["scenario_id"] for scenario, _, _ in matches}) != len(matches):\n        raise EvaluatorDefect("DUPLICATE_REGISTERED_SCENARIO")\n    if len({template for _, template, _ in matches}) != 1:\n        raise EvaluatorUnavailable("AMBIGUOUS_OPERATION_TEMPLATE")\n    if operation_document is None:\n        raise EvaluatorDefect("MISSING_CANONICAL_OPERATION_DOCUMENT")\n    _, template, path = matches[0]\n    typed, violations, security = _assess(request, operation_document, template, path)\n    candidates = []\n    if violations:\n        for scenario, _, _ in matches:\n            if scenario["expected_validity"] != "INVALID":\n                continue\n            signature = _negative_signature(scenario, operation_document, template, request)\n            # An invalid fixture with multiple discrepancies does not define\n            # a single-violation scenario that the evaluator can credit.\n            if len(signature) == 1 and signature == violations:\n                scope, _, _, validator = next(iter(signature))\n                if NEGATIVE_INTENTS.get(scenario["intent"]) == (scope, validator):\n                    candidates.append(scenario)\n        exact = [s for s in candidates if _exact_fixture(request, path, s)]\n        candidates = exact or candidates\n        status = "SEMANTIC_NEGATIVE_MATCH"\n    else:\n        valid = [s for s, _, _ in matches if s["expected_validity"] == "VALID"]\n        exact = [s for s in valid if _exact_fixture(request, path, s)]\n        candidates = exact or [s for s in valid if s["intent"] == "valid_baseline_request"]\n        status = "EXACT_FROZEN_SCENARIO_REPLAY" if exact else "SEMANTIC_VALID_BASELINE"\n\n    if len(candidates) != 1:\n        raise EvaluatorUnavailable("AMBIGUOUS_OR_UNMAPPED_REQUEST_SEMANTICS", {\n            "candidate_ids": [s["scenario_id"] for s in candidates],\n            "observed_violations": sorted(violations),\n        })\n    matched = candidates[0]\n    verify_success_status_mapping(operation_document, matched)\n    return {\n        "classification_status": status, "classifiable": True,\n        "scenario_id": matched["scenario_id"], "intent": matched["intent"],\n        "expected_validity": matched["expected_validity"], "positive": not violations,\n        "documented_response_mapping": matched.get("documented_response_mapping"),\n        "typed_parameters": typed, "observed_violations": sorted(violations),\n        "security_observation": security,\n    }\n\n\ndef select_frozen_response(classification, baselines, mode="baseline", mutation=None):\n    scenario_id = classification["scenario_id"]\n    mapping = classification.get("documented_response_mapping")\n    if mapping is None:\n        raise EvaluatorUnavailable("NO_DOCUMENTED_RESPONSE_MAPPING")\n    linked = [row for row in baselines if row["scenario_id"] == scenario_id]\n    if len(linked) != 1:\n        raise EvaluatorDefect("FROZEN_BASELINE_LINK_COUNT", len(linked))\n    baseline = linked[0]\n    if int(baseline["status"]) != int(mapping["status"]):\n        raise EvaluatorDefect("FROZEN_BASELINE_STATUS_MISMATCH")\n    if mode == "baseline":\n        return baseline\n    if mode not in {"fault", "variation"} or mutation is None:\n        raise EvaluatorDefect("INVALID_REPLAY_CONFIGURATION")\n    # Only the designated baseline is replaced. Other requests in the same\n    # run still receive their own frozen baseline responses.\n    if mutation.get("baseline_id") != baseline["baseline_id"]:\n        return baseline\n    if mutation.get("scenario_id") not in {None, scenario_id}:\n        raise EvaluatorDefect("MUTATION_SCENARIO_LINK_MISMATCH")\n    return mutation\n'}
ORACLE_CONTRACT_TEXT = '{\n  "$schema": "https://json-schema.org/draft/2020-12/schema",\n  "contract_version": "stage05-oracle-record-contracts-v1.2-fault-payload",\n  "records": {\n    "baselines": {\n      "additionalProperties": false,\n      "properties": {\n        "baseline_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "body": {},\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "headers": {\n          "type": "object"\n        },\n        "scenario_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "source_evidence": {\n          "items": {\n            "type": "object"\n          },\n          "minItems": 1,\n          "type": "array"\n        },\n        "status": {\n          "maximum": 599,\n          "minimum": 100,\n          "type": "integer"\n        },\n        "validation_status": {\n          "enum": [\n            "PASS",\n            "FAIL",\n            "PENDING_REVIEW",\n            "NOT_APPLICABLE",\n            "BLOCKED"\n          ],\n          "type": "string"\n        }\n      },\n      "required": [\n        "baseline_id",\n        "scenario_id",\n        "status",\n        "headers",\n        "body",\n        "source_evidence",\n        "validation_status"\n      ],\n      "type": "object"\n    },\n    "interface_items": {\n      "additionalProperties": false,\n      "properties": {\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "interface_item_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "item_kind": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "location": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "name": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "required": {\n          "type": "boolean"\n        },\n        "review_status": {\n          "enum": [\n            "AUTO_VERIFIED",\n            "REVIEW_REQUIRED",\n            "APPROVED",\n            "REJECTED",\n            "REVISED",\n            "UNRESOLVED"\n          ],\n          "type": "string"\n        },\n        "schema": {\n          "type": [\n            "object",\n            "null"\n          ]\n        },\n        "source_pointer": {\n          "minLength": 1,\n          "type": "string"\n        }\n      },\n      "required": [\n        "interface_item_id",\n        "op_id",\n        "item_kind",\n        "source_pointer",\n        "name",\n        "location",\n        "required",\n        "schema",\n        "review_status"\n      ],\n      "type": "object"\n    },\n    "legitimate_variations": {\n      "additionalProperties": false,\n      "properties": {\n        "baseline_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "body": {},\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "expected_equivalent": {\n          "const": true\n        },\n        "headers": {\n          "type": "object"\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "requirement_ids": {\n          "items": {\n            "minLength": 1,\n            "type": "string"\n          },\n          "type": "array",\n          "uniqueItems": true\n        },\n        "scenario_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "source_evidence": {\n          "type": "object"\n        },\n        "status": {\n          "oneOf": [\n            {\n              "maximum": 599,\n              "minimum": 100,\n              "type": "integer"\n            },\n            {\n              "pattern": "^[1-5][0-9]{2}$",\n              "type": "string"\n            }\n          ]\n        },\n        "transformation": {\n          "type": "object"\n        },\n        "validation_status": {\n          "enum": [\n            "PASS",\n            "FAIL",\n            "PENDING_REVIEW",\n            "NOT_APPLICABLE",\n            "BLOCKED"\n          ],\n          "type": "string"\n        },\n        "variation_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "variation_kind": {\n          "minLength": 1,\n          "type": "string"\n        }\n      },\n      "required": [\n        "baseline_id",\n        "body",\n        "expected_equivalent",\n        "headers",\n        "op_id",\n        "requirement_ids",\n        "source_evidence",\n        "status",\n        "transformation",\n        "validation_status",\n        "variation_id",\n        "variation_kind"\n      ],\n      "type": "object"\n    },\n    "requirements": {\n      "additionalProperties": false,\n      "properties": {\n        "applicability": {\n          "enum": [\n            "APPLICABLE",\n            "CONDITIONAL",\n            "NOT_APPLICABLE",\n            "UNRESOLVED"\n          ],\n          "type": "string"\n        },\n        "condition": {},\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "evidence_sha256": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "evidence_type": {\n          "enum": [\n            "canonical_structured_contract",\n            "l0_operation_projection",\n            "original_source_provenance",\n            "normative_prose",\n            "review_decision"\n          ],\n          "type": "string"\n        },\n        "expected_rule": {\n          "type": "object"\n        },\n        "kind": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "requirement_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "review_status": {\n          "enum": [\n            "AUTO_VERIFIED",\n            "REVIEW_REQUIRED",\n            "APPROVED",\n            "REJECTED",\n            "REVISED",\n            "UNRESOLVED"\n          ],\n          "type": "string"\n        },\n        "source_pointer": {\n          "minLength": 1,\n          "type": "string"\n        }\n      },\n      "required": [\n        "requirement_id",\n        "op_id",\n        "source_pointer",\n        "kind",\n        "expected_rule",\n        "applicability",\n        "evidence_type",\n        "review_status"\n      ],\n      "type": "object"\n    },\n    "response_faults": {\n      "additionalProperties": false,\n      "properties": {\n        "applicability": {\n          "enum": [\n            "APPLICABLE",\n            "CONDITIONAL",\n            "NOT_APPLICABLE",\n            "UNRESOLVED"\n          ],\n          "type": "string"\n        },\n        "applicable": {\n          "const": true\n        },\n        "baseline_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "body": {},\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "expected_detectable": {\n          "const": true\n        },\n        "fault_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "fault_kind": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "headers": {\n          "type": "object"\n        },\n        "mutation": {\n          "type": "object"\n        },\n        "non_equivalent": {\n          "const": true\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "requirement_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "scenario_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "source_evidence": {\n          "type": "object"\n        },\n        "status": {\n          "oneOf": [\n            {\n              "maximum": 599,\n              "minimum": 100,\n              "type": "integer"\n            },\n            {\n              "pattern": "^[1-5][0-9]{2}$",\n              "type": "string"\n            }\n          ]\n        },\n        "validation_evidence": {\n          "type": "object"\n        },\n        "validation_status": {\n          "enum": [\n            "PASS",\n            "FAIL",\n            "PENDING_REVIEW",\n            "NOT_APPLICABLE",\n            "BLOCKED"\n          ],\n          "type": "string"\n        },\n        "violated_requirement_ids": {\n          "items": {\n            "minLength": 1,\n            "type": "string"\n          },\n          "minItems": 1,\n          "type": "array",\n          "uniqueItems": true\n        }\n      },\n      "required": [\n        "applicability",\n        "applicable",\n        "baseline_id",\n        "body",\n        "expected_detectable",\n        "fault_id",\n        "fault_kind",\n        "headers",\n        "mutation",\n        "non_equivalent",\n        "op_id",\n        "requirement_id",\n        "scenario_id",\n        "source_evidence",\n        "status",\n        "validation_evidence",\n        "validation_status",\n        "violated_requirement_ids"\n      ],\n      "type": "object"\n    },\n    "scenarios": {\n      "additionalProperties": false,\n      "properties": {\n        "applicability": {\n          "enum": [\n            "APPLICABLE",\n            "CONDITIONAL",\n            "NOT_APPLICABLE",\n            "UNRESOLVED"\n          ],\n          "type": "string"\n        },\n        "body": {},\n        "diagnostic": {\n          "type": [\n            "string",\n            "null"\n          ]\n        },\n        "documented_response_mapping": {\n          "type": [\n            "object",\n            "null"\n          ]\n        },\n        "expected_validity": {\n          "enum": [\n            "VALID",\n            "INVALID",\n            "CONDITIONAL"\n          ],\n          "type": "string"\n        },\n        "intent": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "method": {\n          "enum": [\n            "get",\n            "put",\n            "post",\n            "delete",\n            "options",\n            "head",\n            "patch",\n            "trace"\n          ],\n          "type": "string"\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "parameter_values": {\n          "type": "object"\n        },\n        "path_template": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "requirement_ids": {\n          "items": {\n            "minLength": 1,\n            "type": "string"\n          },\n          "type": "array",\n          "uniqueItems": true\n        },\n        "scenario_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "validation_status": {\n          "enum": [\n            "PASS",\n            "FAIL",\n            "PENDING_REVIEW",\n            "NOT_APPLICABLE",\n            "BLOCKED"\n          ],\n          "type": "string"\n        }\n      },\n      "required": [\n        "scenario_id",\n        "op_id",\n        "intent",\n        "requirement_ids",\n        "method",\n        "path_template",\n        "parameter_values",\n        "body",\n        "expected_validity",\n        "documented_response_mapping",\n        "applicability",\n        "validation_status"\n      ],\n      "type": "object"\n    },\n    "validation": {\n      "additionalProperties": false,\n      "properties": {\n        "artifact_counts": {\n          "type": "object"\n        },\n        "blocking_issues": {\n          "items": {\n            "type": "object"\n          },\n          "type": "array"\n        },\n        "checks": {\n          "type": "object"\n        },\n        "op_id": {\n          "minLength": 1,\n          "type": "string"\n        },\n        "oracle_status": {\n          "enum": [\n            "DRAFT_PASS",\n            "REVIEW_REQUIRED",\n            "BLOCKED",\n            "FROZEN_PASS"\n          ],\n          "type": "string"\n        }\n      },\n      "required": [\n        "op_id",\n        "oracle_status",\n        "artifact_counts",\n        "checks",\n        "blocking_issues"\n      ],\n      "type": "object"\n    }\n  },\n  "review_decisions_csv": {\n    "allowed_decisions": [\n      "APPROVE",\n      "REJECT",\n      "REVISE",\n      "UNRESOLVED"\n    ],\n    "columns": [\n      "decision_id",\n      "op_id",\n      "artifact_type",\n      "item_id",\n      "source_pointer",\n      "issue_code",\n      "evidence_summary",\n      "proposed_value_json",\n      "decision",\n      "revised_value_json",\n      "reviewer_note",\n      "reviewer_id",\n      "decision_timestamp_utc"\n    ]\n  }\n}\n'
DEVELOPMENT_FIXTURE_TEXT = '{"info":{"title":"Development fixture","version":"1"},"openapi":"3.0.3","paths":{"/users":{"post":{"requestBody":{"content":{"application/json":{"schema":{"properties":{"age":{"maximum":60,"minimum":18,"type":"integer"}},"required":["age"],"type":"object"}}},"required":true},"responses":{"201":{"content":{"application/json":{"schema":{"properties":{"id":{"type":"string"}},"required":["id"],"type":"object"}}},"description":"User created"},"400":{"content":{"application/json":{"schema":{"properties":{"error":{"type":"string"}},"required":["error"],"type":"object"}}},"description":"Invalid age input"}},"summary":"Create a user"}}}}\n'

import importlib.util
import sys

project = PROJECT_ROOT
source = project / "src"
source.mkdir(exist_ok=True)
for name, contents in SHARED_ORACLE_SOURCES.items():
    atomic(source / name, contents.encode())

# Load the same bytes that will be hashed and packaged. This also works
# when this cell is rerun after an earlier builder was imported.
sys.path.insert(0, str(source))
for name in ("routing.py", "oracle_requirements.py", "oracle_scenarios.py",
             "oracle_baselines.py", "oracle_variations.py", "oracle_faults.py", "oracle.py"):
    module_path = source / name
    module_name = module_path.stem
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    exec(compile(module_path.read_bytes(), str(module_path), "exec"), module.__dict__)
import oracle

contracts = json.loads(ORACLE_CONTRACT_TEXT)
atomic(project / "config/oracle_record_contracts.json", ORACLE_CONTRACT_TEXT.encode())
fixture = json.loads(DEVELOPMENT_FIXTURE_TEXT)
def qualify_success_status_mapping(routing):
    """Exercise the same status policy at the builder and evaluator boundaries."""
    scenario = {
        "scenario_id": "status-fixture", "path_template": "/items", "method": "get",
        "applicability": "APPLICABLE", "expected_validity": "VALID",
        "intent": "valid_baseline_request", "parameter_values": {},
        "documented_response_mapping": {"status": 201},
    }
    baseline = {"scenario_id": "status-fixture", "baseline_id": "status-baseline",
                "status": 201, "body": None}
    request = routing.normalize_prepared_request("GET", "http://openapi.test/items", None, None)

    def document(statuses):
        return {"openapi": "3.0.3", "paths": {"/items": {"get": {
            "responses": {code: {"description": "fixture response"} for code in statuses}
        }}}}

    unique = document(["201", "400"])
    assert routing.success_status_mapping(unique, "/items", "get") == {"status": 201}
    routing.audit_frozen_response(unique, scenario, baseline, baseline)
    assert routing.select_frozen_response(
        routing.classify_request(request, [scenario], unique), [baseline]) == baseline

    for statuses in (["201", "204"], ["204", "201"], ["201", "2XX"], ["2XX"]):
        ambiguous = document(statuses)
        for check in (
            lambda: routing.success_status_mapping(ambiguous, "/items", "get"),
            lambda: routing.classify_request(request, [scenario], ambiguous),
            lambda: routing.audit_frozen_response(ambiguous, scenario, baseline, baseline),
        ):
            try:
                check()
            except routing.EvaluatorUnavailable as error:
                if error.code != "AMBIGUOUS_SUCCESS_STATUS_MAPPING":
                    raise
            else:
                raise RuntimeError("Ambiguous success status produced a response mapping")
    assert routing.success_status_mapping(document(["default", "400"]), "/items", "get") is None
    return {"policy": routing.SUCCESS_STATUS_POLICY, "unique_explicit_status": "PASS",
            "ambiguous_statuses_blocked": "PASS", "default_only_unmapped": "PASS"}

status_qualification = qualify_success_status_mapping(oracle.routing)
fixture_package = oracle.build_oracle(fixture, "synthetic-development", "post", "/users", contracts)
from copy import deepcopy
ambiguous_fixture = deepcopy(fixture)
ambiguous_fixture["paths"]["/users"]["post"]["responses"]["204"] = {"description": "Alternate success"}
try:
    oracle.build_oracle(ambiguous_fixture, "synthetic-ambiguous", "post", "/users", contracts)
except oracle.routing.EvaluatorUnavailable as error:
    if error.code != "AMBIGUOUS_SUCCESS_STATUS_MAPPING":
        raise
else:
    raise RuntimeError("The shared oracle builder accepted an ambiguous success status.")

# Exercise the same schema generator used for the corpus request bodies.
allof_value = oracle.oracle_scenarios.valid_value
assert allof_value(fixture, {"allOf": [
    {"description": "Annotation-only branch"},
    {"type": "object", "required": ["count"], "properties": {"count": {"type": "integer"}}},
]}) == {"count": 1}
assert allof_value(fixture, {"allOf": [
    {"type": "string", "enum": ["red", "blue"]},
    {"type": "string", "enum": ["blue", "green"]},
]}) == "blue"
try:
    allof_value(fixture, {"allOf": [{"type": "string"}, {"type": "object"}]})
except oracle.routing.EvaluatorUnavailable as error:
    if error.code != "NO_WITNESS_FOR_ALLOF":
        raise
else:
    raise RuntimeError("Incompatible allOf branches produced a witness.")

if not fixture_package["baselines"] or not fixture_package["response_faults"]:
    raise RuntimeError("The shared builder must qualify the Stage 00 development fixture.")
module_hashes = {name: file_sha(source / name) for name in SHARED_ORACLE_SOURCES}
atomic(project / "reports/audit/feasibility_builder_qualification.json",
       encode({"status": "PASS", "model_outputs_used": False,
               "builder_version": oracle.ORACLE_VERSION, "module_hashes": module_hashes,
               "success_status_qualification": status_qualification,
                "allof_qualification": "PASS",
               "contract_sha256": sha(ORACLE_CONTRACT_TEXT.encode()),
               "fixture_counts": {kind: len(rows) for kind, rows in fixture_package.items()}}))
eligibility_path_cell14 = project / "data/audits/operation_language_prose_eligibility.csv"
construction_path_cell14 = project / "data/audits/l1_construction_records.jsonl"
prose_report_cell14 = load(project / "reports/audit/operation_language_prose_eligibility.json")
source_ledger = file_sha(eligibility_path_cell14)
construction_digest_cell14 = file_sha(construction_path_cell14)
if (prose_report_cell14["status"] != "PASS"
        or source_ledger != prose_report_cell14["eligibility_ledger"]["sha256"]
        or construction_digest_cell14 != prose_report_cell14["input_hashes"]["construction_records"]):
    raise RuntimeError("Cell 14 requires the verified eligibility and construction ledgers from Cell 13.")

eligibility = csv_records(eligibility_path_cell14)
construction_by_id_cell14 = {}
for record in records(construction_path_cell14):
    op_id = record["op_id"]
    if op_id in construction_by_id_cell14 or record["status"] != "PASS":
        raise RuntimeError(f"Invalid or duplicate pair-construction record: {op_id}")
    construction_by_id_cell14[op_id] = record
if set(construction_by_id_cell14) != {row["op_id"] for row in eligibility}:
    raise RuntimeError("Cell 14 construction and eligibility ledgers describe different operations.")


def constructed_slice_path_cell14(root, record, condition):
    name = record[f"{condition.lower()}_relative_path"]
    relative = safe_path(name)
    if (relative.parts[:3] != ("data", "slices", condition)
            or relative.name != f"{record['op_id']}.json"):
        raise RuntimeError(f"Invalid recorded {condition} slice path: {name}")
    path = root / relative
    if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(root.resolve()):
        raise RuntimeError(f"Missing or unsafe recorded {condition} slice: {path}")
    return path

if len(eligibility) != len({row["op_id"] for row in eligibility}):
    raise RuntimeError("Duplicate operation in the feasibility input ledger.")
checkpoint = project / "data/checkpoints/strict_feasibility"
checkpoint.mkdir(parents=True, exist_ok=True)
builder_digest = sha(encode([module_hashes, sha(ORACLE_CONTRACT_TEXT.encode())]))
all_results = []
for start in range(0, len(eligibility), 250):
    chunk = eligibility[start:start + 250]
    destination = checkpoint / f"{start:06d}.json"
    verified_l0_cell14 = {}
    for row in chunk:
        op = row["op_id"]
        construction = construction_by_id_cell14[op]
        if construction["pair_id"] != row["pair_id"]:
            raise RuntimeError(f"Cell 14 pair identity mismatch for {op}")
        if row["prose_eligibility_pass"].lower() == "true":
            path = constructed_slice_path_cell14(project, construction, "L0")
            digest = file_sha(path)
            if digest != construction["l0_sha256"]:
                raise RuntimeError(f"Cell 14 L0 SHA-256 mismatch for {op}: {path}")
            verified_l0_cell14[op] = (path, digest)

    key = sha(encode([source_ledger, construction_digest_cell14, builder_digest, start,
                      [(op, digest) for op, (_, digest) in verified_l0_cell14.items()]]))
    if destination.exists():
        previous = load(destination)
        if previous["input_sha256"] == key:
            all_results.extend(previous["rows"])
            continue
    results = []
    for row in chunk:
        op = row["op_id"]
        passed_prior = row["prose_eligibility_pass"].lower() == "true"
        result = {k: row[k] for k in ("op_id", "pair_id", "spec_id", "family_id", "source_class", "method", "path")}
        result.update(pre_feasibility_eligible=passed_prior, probe_performed=passed_prior,
                      request_constructible=False, response_constructible=False,
                      scenario_count=0, requirement_count=0, response_fault_count=0,
                      evaluator_feasible=False, prior_exclusion_reasons=row["exclusion_reasons"],
                      model_outputs_used=False, diagnostic_count=0, diagnostic_types="",
                      final_status="PRE_FEASIBILITY_INELIGIBLE", oracle_sha256="")
        if passed_prior:
            path, digest = verified_l0_cell14[op]
            document_bytes = path.read_bytes()
            if sha(document_bytes) != digest:
                raise RuntimeError(f"Cell 14 L0 changed after verification: {path}")
            document = json.loads(document_bytes)
            try:
                package = oracle.build_oracle(document, op, row["method"], row["path"], contracts)
            except (oracle.UnsupportedOracle, oracle.routing.UnsupportedRoute,
                    oracle.jsonschema.exceptions.RefResolutionError, RecursionError) as error:
                result.update(final_status="EXCLUDE_EVALUATOR_UNSUPPORTED",
                              diagnostic_count=1, diagnostic_types=str(error)[:1000])
            except Exception as error:
                error.add_note(f"Notebook 03 feasibility failed for {op} ({row['method']} {row['path']})")
                raise
            else:
                result.update(request_constructible=True, response_constructible=True,
                              scenario_count=sum(s["applicability"] == "APPLICABLE" for s in package["scenarios"]),
                              requirement_count=sum(r["applicability"] == "APPLICABLE" for r in package["requirements"]),
                              response_fault_count=len(package["response_faults"]), evaluator_feasible=True,
                              final_status="ELIGIBLE_EVALUATOR_FEASIBLE", oracle_sha256=sha(encode(package)))
        results.append(result)
    atomic(destination, encode({"input_sha256": key, "rows": results}))
    all_results.extend(results)
    print(f"Feasibility checked {len(all_results):,} / {len(eligibility):,}")
csv_write(project / "data/audits/feasibility.csv", list(all_results[0]), all_results)
jsonl(project / "data/audits/feasibility_results.jsonl", all_results)
report_path = project / "reports/audit/evaluator_feasibility_summary.json"
atomic(report_path, encode({"status": "PASS", "model_outputs_used": False,
                            "operation_count": len(all_results),
                            "construction_ledger_sha256": construction_digest_cell14,
                            "prose_eligibility_ledger_sha256": source_ledger,
                            "evaluator_feasible_count": sum(row["evaluator_feasible"] for row in all_results),
                            "module_hashes": module_hashes, "builder_sha256": builder_digest,
                            "ledger_sha256": file_sha(project / "data/audits/feasibility.csv")}))
policy_path = project / "config/stage_03.yaml"
policy = yaml.safe_load(policy_path.read_text())
policy["evaluator_feasibility"] = {"corpus_report_sha256": file_sha(report_path),
                                   "qualification_status": "PASS", "module_hashes": module_hashes}
atomic(policy_path, yaml.safe_dump(policy, sort_keys=False).encode())
print("Stage 03 shared-builder feasibility PASS")



Feasibility checked 250 / 52,922
Feasibility checked 500 / 52,922
Feasibility checked 750 / 52,922
Feasibility checked 1,000 / 52,922
Feasibility checked 1,250 / 52,922
Feasibility checked 1,500 / 52,922
Feasibility checked 1,750 / 52,922
Feasibility checked 2,000 / 52,922
Feasibility checked 2,250 / 52,922
Feasibility checked 2,500 / 52,922
Feasibility checked 2,750 / 52,922
Feasibility checked 3,000 / 52,922
Feasibility checked 3,250 / 52,922
Feasibility checked 3,500 / 52,922
Feasibility checked 3,750 / 52,922
Feasibility checked 4,000 / 52,922
Feasibility checked 4,250 / 52,922
Feasibility checked 4,500 / 52,922
Feasibility checked 4,750 / 52,922
Feasibility checked 5,000 / 52,922
Feasibility checked 5,250 / 52,922
Feasibility checked 5,500 / 52,922
Feasibility checked 5,750 / 52,922
Feasibility checked 6,000 / 52,922
Feasibility checked 6,250 / 52,922
Feasibility checked 6,500 / 52,922
Feasibility checked 6,750 / 52,922
Feasibility checked 7,000 / 52,922
Feasibility checked 7,250 

In [15]:
# Notebook 03 — Cell 15
# Reconcile development exposure and construct the final eligible frame.

from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import json
import os
import re
import time

import yaml


CELL15_STARTED = time.time()

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_03/prose_operation_experiment"
)

PAIR_FRAME_PATH = (
    PROJECT_ROOT
    / "data/inventory/pair_frame_seed.jsonl"
)
CONSTRUCTION_RECORDS_PATH = (
    PROJECT_ROOT
    / "data/audits/l1_construction_records.jsonl"
)
TOKEN_AUDIT_PATH = (
    PROJECT_ROOT
    / "data/audits/token_audit.csv"
)
PROSE_ELIGIBILITY_PATH = (
    PROJECT_ROOT
    / "data/audits/"
    "operation_language_prose_eligibility.csv"
)
FEASIBILITY_LEDGER_PATH = (
    PROJECT_ROOT
    / "data/audits/feasibility.csv"
)
CELL14_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "evaluator_feasibility_summary.json"
)
STAGE_03_POLICY_PATH = (
    PROJECT_ROOT
    / "config/stage_03.yaml"
)

DEVELOPMENT_EXAMPLES_ROOT = (
    PROJECT_ROOT
    / "data/development_examples"
)
DEVELOPMENT_EXPOSURE_PATH = (
    PROJECT_ROOT
    / "data/audits/development_exposure.csv"
)
CUMULATIVE_EXCLUSIONS_PATH = (
    PROJECT_ROOT
    / "data/audits/cumulative_exclusions.csv"
)
FINAL_ELIGIBLE_CSV_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_operations.csv"
)
FINAL_ELIGIBLE_JSONL_PATH = (
    PROJECT_ROOT
    / "data/inventory/eligible_operations.jsonl"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/audit/"
    "final_eligibility_reconciliation.json"
)

EXPECTED_OPERATION_COUNT = expected_eligible_operation_count

FINAL_FRAME_VERSION = (
    "stage03-final-eligible-frame-v1"
)

QUALIFICATION_EVIDENCE_PATHS = [
    PROJECT_ROOT
    / "config/upstream/"
    "stage_00_prompt_reproducibility.json",
    PROJECT_ROOT
    / "reports/audit/"
    "obfuscation_qualification.json",
    PROJECT_ROOT
    / "reports/audit/"
    "pairing_qualification.json",
    PROJECT_ROOT
    / "reports/audit/"
    "feasibility_builder_qualification.json",
]

TEXT_SUFFIXES = {
    ".json",
    ".jsonl",
    ".csv",
    ".yaml",
    ".yml",
    ".txt",
    ".py",
    ".md",
}

CORPUS_IDENTIFIER_PATTERN = re.compile(
    r"\b(?:op|pair|spec|family)_[0-9a-f]{24}\b"
)

for directory in [
    DEVELOPMENT_EXPOSURE_PATH.parent,
    CUMULATIVE_EXCLUSIONS_PATH.parent,
    FINAL_ELIGIBLE_CSV_PATH.parent,
    REPORT_PATH.parent,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def sha256_file(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while block := handle.read(
            chunk_size
        ):
            digest.update(block)

    return digest.hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def canonical_json(value) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )


def read_jsonl(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            value = json.loads(line)

            if not isinstance(value, dict):
                raise RuntimeError(
                    "Non-object JSONL record at "
                    f"{path}:{line_number}."
                )

            yield value


def write_text_atomic(
    path: Path,
    text: str,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        handle.write(text)
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_json_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        json.dump(
            value,
            handle,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_yaml_atomic(
    path: Path,
    value,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        yaml.safe_dump(
            value,
            handle,
            sort_keys=False,
            allow_unicode=True,
        )
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def write_csv_atomic(
    path: Path,
    fieldnames,
    rows,
) -> None:
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            lineterminator="\n",
            extrasaction="ignore",
        )
        writer.writeheader()

        for row in rows:
            writer.writerow(row)

        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary_path,
        path,
    )


def read_csv_by_id(
    path: Path,
    id_field: str = "op_id",
):
    rows = []
    by_id = {}

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        for row in csv.DictReader(handle):
            identifier = str(
                row.get(
                    id_field,
                    "",
                )
            )

            if not identifier:
                raise RuntimeError(
                    f"{path} contains a row "
                    f"without {id_field}."
                )

            if identifier in by_id:
                raise RuntimeError(
                    f"{path} contains duplicate "
                    f"{id_field}: {identifier}."
                )

            rows.append(row)
            by_id[identifier] = row

    return rows, by_id


def parse_boolean(value) -> bool:
    if isinstance(value, bool):
        return value

    normalized = str(
        value
    ).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
        "",
    }:
        return False

    raise ValueError(
        "Cannot parse boolean value: "
        f"{value!r}"
    )


required_paths = [
    PAIR_FRAME_PATH,
    CONSTRUCTION_RECORDS_PATH,
    TOKEN_AUDIT_PATH,
    PROSE_ELIGIBILITY_PATH,
    FEASIBILITY_LEDGER_PATH,
    CELL14_REPORT_PATH,
    STAGE_03_POLICY_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise RuntimeError(
        "Cell 15 required files are missing:\n"
        + "\n".join(missing_paths)
    )

with STAGE_03_POLICY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage_03_policy = (
        yaml.safe_load(handle)
        or {}
    )

with CELL14_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    cell14_report = json.load(handle)

expected_cell14_hash = (
    stage_03_policy
    .get(
        "evaluator_feasibility",
        {},
    )
    .get("corpus_report_sha256")
)
observed_cell14_hash = sha256_file(
    CELL14_REPORT_PATH
)

if (
    not isinstance(
        expected_cell14_hash,
        str,
    )
    or len(expected_cell14_hash) != 64
    or (
        expected_cell14_hash
        != observed_cell14_hash
    )
):
    raise RuntimeError(
        "Cell 14 report does not match the "
        "hash recorded in stage_03.yaml."
    )

if cell14_report.get("status") != "PASS":
    raise RuntimeError(
        "Cell 14 did not report PASS."
    )

expected_feasible_count = int(
    cell14_report.get(
        "evaluator_feasible_count",
        -1,
    )
)

if expected_feasible_count <= 0:
    raise RuntimeError(
        "Cell 14 produced no evaluator-feasible "
        "operations."
    )

pair_frame_by_op_id = {}

for record in read_jsonl(
    PAIR_FRAME_PATH
):
    op_id = str(
        record.get(
            "op_id",
            "",
        )
    )

    if not op_id:
        raise RuntimeError(
            "Pair-frame record has no op_id."
        )

    if op_id in pair_frame_by_op_id:
        raise RuntimeError(
            "Duplicate pair-frame op_id: "
            f"{op_id}."
        )

    pair_frame_by_op_id[op_id] = (
        record
    )

construction_by_op_id = {}

for record in read_jsonl(
    CONSTRUCTION_RECORDS_PATH
):
    op_id = str(
        record.get(
            "op_id",
            "",
        )
    )

    if not op_id:
        raise RuntimeError(
            "Construction record has no op_id."
        )

    if op_id in construction_by_op_id:
        raise RuntimeError(
            "Duplicate construction op_id: "
            f"{op_id}."
        )

    construction_by_op_id[op_id] = (
        record
    )

(
    token_rows,
    token_by_op_id,
) = read_csv_by_id(
    TOKEN_AUDIT_PATH
)

(
    prose_rows,
    prose_by_op_id,
) = read_csv_by_id(
    PROSE_ELIGIBILITY_PATH
)

(
    feasibility_rows,
    feasibility_by_op_id,
) = read_csv_by_id(
    FEASIBILITY_LEDGER_PATH
)

population_sets = {
    "pair_frame": set(
        pair_frame_by_op_id
    ),
    "construction": set(
        construction_by_op_id
    ),
    "token_audit": set(
        token_by_op_id
    ),
    "prose_eligibility": set(
        prose_by_op_id
    ),
    "feasibility": set(
        feasibility_by_op_id
    ),
}

for name, identifiers in (
    population_sets.items()
):
    if (
        len(identifiers)
        != EXPECTED_OPERATION_COUNT
    ):
        raise RuntimeError(
            f"{name} contains "
            f"{len(identifiers)} operations; "
            "expected the Stage 02 candidate count."
        )

if len({
    frozenset(identifiers)
    for identifiers in population_sets.values()
}) != 1:
    raise RuntimeError(
        "Cell 15 input operation populations "
        "do not match."
    )

operation_ids = set(
    pair_frame_by_op_id
)

op_to_spec = {}
op_to_pair = {}
op_to_family = {}
spec_to_operations = defaultdict(set)
pair_to_operations = defaultdict(set)
family_to_operations = defaultdict(set)

for op_id, pair_record in (
    pair_frame_by_op_id.items()
):
    spec_id = str(
        pair_record["spec_id"]
    )
    pair_id = str(
        pair_record["pair_id"]
    )
    family_id = str(
        pair_record["family_id"]
    )

    op_to_spec[op_id] = spec_id
    op_to_pair[op_id] = pair_id
    op_to_family[op_id] = family_id

    spec_to_operations[
        spec_id
    ].add(op_id)
    pair_to_operations[
        pair_id
    ].add(op_id)
    family_to_operations[
        family_id
    ].add(op_id)

# Identify every development/qualification evidence file.
development_evidence_paths = set()

if DEVELOPMENT_EXAMPLES_ROOT.is_dir():
    for path in (
        DEVELOPMENT_EXAMPLES_ROOT
        .rglob("*")
    ):
        if (
            path.is_file()
            and path.suffix.lower()
            in TEXT_SUFFIXES
        ):
            development_evidence_paths.add(
                path
            )

for path in QUALIFICATION_EVIDENCE_PATHS:
    if path.is_file():
        development_evidence_paths.add(
            path
        )


# Exclude fresh API families whose prior pilot members informed evaluator development.
historical_exposure = {"provenance": "Historical Stage 05 pilot manifest used in evaluator development and qualification", "spec_ids": ["spec_021fa49357ff4de18ff298e4", "spec_03dd3d3c13522b2a7bbc545d", "spec_0684f5240fb8c0798c0e2683", "spec_0858ca0e6e3a7a5095fab664", "spec_0a88e1ba003cadff18bf5085", "spec_0c04ce1a181144bff0fdbdef", "spec_0db640e3ef9035d83eac5819", "spec_0e49a16f0548a49a78f2987a", "spec_0ef0fd00b6fcf63c511799ec", "spec_1478e60bcced2da88bbd470b", "spec_1684d74559c6f344d2df5096", "spec_177f1616968cbdaf54da62bb", "spec_17834bbfe703351e176ec7a9", "spec_1fa5a3511d4f732b35084092", "spec_1fb99c79fd635ca6d5bc4565", "spec_20e1ba8e5b95284a929ff1f0", "spec_2117d1e2c4b58c8c7492869d", "spec_2537bd38cc97ec0683861a53", "spec_266b96f6f31097afd2f6019f", "spec_34700632fd99cb0bc9262c4c", "spec_368779ada2d50e5aae3dbe45", "spec_388e2e598a19ed98e69be8aa", "spec_38e24446de1b1fd86acd47b9", "spec_3a77c76f252be983efec76a0", "spec_3c6fba9e2fac4e78b5b184ce", "spec_3de830b073484b0df75bcee9", "spec_3f9da2ec60b22e999190a1b4", "spec_4660da399e8cb967e12eb218", "spec_468faf4e4758081798c9153c", "spec_47738cdf4464f32a45db64c3", "spec_4cbe127a3f4fbcf150b5a621", "spec_4df51875574bec7420a4553e", "spec_4e6db2cc4cb7981eedb8f5ae", "spec_53cb8edaa0b6fea9da3f61cc", "spec_61b708d70ff15dbe6d367ed7", "spec_65ab4d14e7b46b8ee0969d62", "spec_73bb186213db3f816124200b", "spec_73efdf14201a0c55e04d64a8", "spec_7809d3b45d0a78d8ec1b0421", "spec_79e48ba71f963f9056d23417", "spec_79e66dd56c9efb82bededc54", "spec_7a3e0cb3d270aacc68fbc0e9", "spec_7b92b6716a6a6d5538ecc004", "spec_7bd07cc230dc17698c389a41", "spec_7ce1bd2b98e0442e60169da7", "spec_7eeda7b8ecaceed3297f9830", "spec_7f2c428349022cd8e40d6336", "spec_8468bd4eea9a1ad856817a72", "spec_856284011402dfef016f4502", "spec_90e280ff8e466d0b1e1d537b", "spec_91cc90a4052891441fca3821", "spec_91d9a490d54e4cfcfdefca89", "spec_946068b274961ff6bd752f9d", "spec_94d9f52bec10930801fc3474", "spec_9541d6719045e3d62caee87e", "spec_97c2c7e4c86bcfc773be3495", "spec_a3bd02f79aced966d2c0ff06", "spec_a6ca51240456d1bab22ad510", "spec_a74859cf0b37893ef7b39642", "spec_adab1b54a36b76e4585ff259", "spec_afe8db203909f4ab55263948", "spec_b344513139354919a82cee31", "spec_b3615a63fb84176730f8fb56", "spec_baaac331c61242edc99924ee", "spec_bb335faceaa0beb0be08ff27", "spec_c0ffdfe1d2fd5ec6af55bbdf", "spec_c75ba96a91dbe7f1550b7298", "spec_d022985670e8ec98333c6f18", "spec_d42817591c3d7acb1bbce690", "spec_d728c7b3f1a1740c3b43fdab", "spec_d91a47c77e1dc45af626e37f", "spec_d9ff9b400c7e67f558226fdd", "spec_db23918b1b1f8d309594c08a", "spec_db416544285cc6b25789173c", "spec_e11c1142946fbd754da8ff23", "spec_e584c0f4d771a8ce0cef7e4f", "spec_e5f70762a42b1bde29216404", "spec_e6a4daff3be4a584ccb472be", "spec_e79e060e17a2aa7ca0b04473", "spec_f1d1f7f6e23b909f5c0d6a92", "spec_f23989e92b5f46d563064d24", "spec_fed1c0eb00d85ec2a0cda2e6"]}
from pathlib import Path
with (STAGE_02_PROJECT_ROOT / "data/inventory/families.csv").open(newline="", encoding="utf-8") as handle:
    historical_family_ids = sorted({
        row["family_id"] for row in csv.DictReader(handle)
        if row["spec_id"] in historical_exposure["spec_ids"]
    })
exposure_path = DEVELOPMENT_EXAMPLES_ROOT / "historical_evaluator_development_exposure.json"
exposure_path.parent.mkdir(parents=True, exist_ok=True)
exposure_path.write_text(json.dumps({**historical_exposure, "family_ids": historical_family_ids}, indent=2) + "\n")
development_evidence_paths.add(exposure_path)

development_identifier_evidence = (
    defaultdict(set)
)

for evidence_path in sorted(
    development_evidence_paths,
    key=lambda path: str(path),
):
    try:
        text = evidence_path.read_text(
            encoding="utf-8"
        )
    except UnicodeDecodeError:
        continue

    for identifier in (
        CORPUS_IDENTIFIER_PATTERN
        .findall(text)
    ):
        development_identifier_evidence[
            identifier
        ].add(
            str(
                evidence_path.relative_to(
                    PROJECT_ROOT
                )
            ).replace("\\", "/")
        )

development_exposed_operations = (
    defaultdict(
        lambda: {
            "identifiers": set(),
            "evidence_paths": set(),
        }
    )
)

for (
    identifier,
    evidence_paths,
) in development_identifier_evidence.items():
    affected_operations = set()

    if identifier.startswith("op_"):
        if identifier in operation_ids:
            affected_operations.add(
                identifier
            )

    elif identifier.startswith("pair_"):
        affected_operations.update(
            pair_to_operations.get(
                identifier,
                set(),
            )
        )

    elif identifier.startswith("spec_"):
        affected_operations.update(
            spec_to_operations.get(
                identifier,
                set(),
            )
        )

    elif identifier.startswith("family_"):
        affected_operations.update(
            family_to_operations.get(
                identifier,
                set(),
            )
        )

    for op_id in affected_operations:
        development_exposed_operations[
            op_id
        ]["identifiers"].add(
            identifier
        )
        development_exposed_operations[
            op_id
        ]["evidence_paths"].update(
            evidence_paths
        )

development_exposure_rows = []

for op_id in sorted(
    development_exposed_operations
):
    evidence = (
        development_exposed_operations[
            op_id
        ]
    )

    development_exposure_rows.append(
        {
            "op_id": op_id,
            "pair_id": (
                op_to_pair[op_id]
            ),
            "spec_id": (
                op_to_spec[op_id]
            ),
            "family_id": (
                op_to_family[op_id]
            ),
            "matched_identifiers": (
                ";".join(
                    sorted(
                        evidence[
                            "identifiers"
                        ]
                    )
                )
            ),
            "development_uses": (
                "prompt_or_transformation_or_"
                "evaluator_qualification"
            ),
            "evidence_paths": (
                ";".join(
                    sorted(
                        evidence[
                            "evidence_paths"
                        ]
                    )
                )
            ),
            "exclusion_decision": (
                "EXCLUDE_DEVELOPMENT_EXPOSED"
            ),
        }
    )

development_exposure_fieldnames = [
    "op_id",
    "pair_id",
    "spec_id",
    "family_id",
    "matched_identifiers",
    "development_uses",
    "evidence_paths",
    "exclusion_decision",
]

write_csv_atomic(
    DEVELOPMENT_EXPOSURE_PATH,
    development_exposure_fieldnames,
    development_exposure_rows,
)

development_exposed_ids = set(
    development_exposed_operations
)

eligible_records = []
cumulative_exclusion_rows = []

first_exit_status_counts = Counter()
all_reason_counts = Counter()
eligible_source_class_counts = Counter()
eligible_method_counts = Counter()
eligible_family_ids = set()
eligible_spec_ids = set()

for op_id in sorted(
    operation_ids
):
    pair_record = (
        pair_frame_by_op_id[op_id]
    )
    construction_record = (
        construction_by_op_id[op_id]
    )
    token_record = (
        token_by_op_id[op_id]
    )
    prose_record = (
        prose_by_op_id[op_id]
    )
    feasibility_record = (
        feasibility_by_op_id[op_id]
    )

    if not (
        str(pair_record["pair_id"])
        == str(
            construction_record[
                "pair_id"
            ]
        )
        == str(token_record["pair_id"])
        == str(prose_record["pair_id"])
        == str(
            feasibility_record[
                "pair_id"
            ]
        )
    ):
        raise RuntimeError(
            "Pair identity mismatch during "
            f"final reconciliation for {op_id}."
        )

    exclusion_reasons = []

    prose_exclusion_text = str(
        prose_record.get(
            "exclusion_reasons",
            "",
        )
    ).strip()

    if prose_exclusion_text:
        exclusion_reasons.extend(
            reason
            for reason in (
                prose_exclusion_text
                .split(";")
            )
            if reason
        )

    feasibility_status = str(
        feasibility_record.get(
            "final_status",
            "",
        )
    )

    if feasibility_status in {
        "EXCLUDE_EVALUATOR_UNSUPPORTED",
        "EXCLUDE_UNRESOLVED_REFERENCE",
        "EXCLUDE_UNSUPPORTED_SCHEMA",
    }:
        exclusion_reasons.append(
            feasibility_status
        )

    if op_id in development_exposed_ids:
        exclusion_reasons.append(
            "EXCLUDE_DEVELOPMENT_EXPOSED"
        )

    unique_exclusion_reasons = list(
        dict.fromkeys(
            exclusion_reasons
        )
    )

    if (
        feasibility_status
        == "ELIGIBLE_EVALUATOR_FEASIBLE"
        and not unique_exclusion_reasons
    ):
        final_status = "ELIGIBLE"
        first_exit_status = "ELIGIBLE"

    elif (
        "EXCLUDED_PROMPT_LIMIT"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDED_PROMPT_LIMIT"
        )

    elif (
        "EXCLUDED_NONPOSITIVE_PROSE_SCORE"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDED_NONPOSITIVE_PROSE_SCORE"
        )

    elif (
        "EXCLUDED_NO_TRANSFORMED_ASCII_LETTER"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDED_NO_TRANSFORMED_ASCII_LETTER"
        )

    elif (
        "EXCLUDE_UNRESOLVED_REFERENCE"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDE_UNRESOLVED_REFERENCE"
        )

    elif (
        "EXCLUDE_UNSUPPORTED_SCHEMA"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDE_UNSUPPORTED_SCHEMA"
        )

    elif (
        "EXCLUDE_EVALUATOR_UNSUPPORTED"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDE_EVALUATOR_UNSUPPORTED"
        )

    elif (
        "EXCLUDE_DEVELOPMENT_EXPOSED"
        in unique_exclusion_reasons
    ):
        final_status = "INELIGIBLE"
        first_exit_status = (
            "EXCLUDE_DEVELOPMENT_EXPOSED"
        )

    else:
        raise RuntimeError(
            "Operation has no valid final "
            "disposition: "
            f"{op_id}, feasibility="
            f"{feasibility_status}, reasons="
            f"{unique_exclusion_reasons}"
        )

    first_exit_status_counts[
        first_exit_status
    ] += 1

    for reason_index, reason in enumerate(
        unique_exclusion_reasons,
        start=1,
    ):
        all_reason_counts[reason] += 1

        cumulative_exclusion_rows.append(
            {
                "op_id": op_id,
                "pair_id": (
                    pair_record["pair_id"]
                ),
                "spec_id": (
                    pair_record["spec_id"]
                ),
                "family_id": (
                    pair_record["family_id"]
                ),
                "stage": (
                    "03_pair_and_evaluator_"
                    "feasibility"
                ),
                "rule": reason,
                "reason_order": (
                    reason_index
                ),
                "is_first_exit": (
                    reason
                    == first_exit_status
                ),
                "first_exit_status": (
                    first_exit_status
                ),
                "evidence": (
                    "token_audit"
                    if reason
                    == "EXCLUDED_PROMPT_LIMIT"
                    else (
                        "operation_language_"
                        "prose_eligibility"
                        if reason in {
                            "EXCLUDED_NONPOSITIVE_"
                            "PROSE_SCORE",
                            "EXCLUDED_NO_TRANSFORMED_"
                            "ASCII_LETTER",
                        }
                        else (
                            "development_exposure"
                            if reason
                            == "EXCLUDE_DEVELOPMENT_EXPOSED"
                            else "feasibility"
                        )
                    )
                ),
            }
        )

    if final_status != "ELIGIBLE":
        continue

    eligible_record = {
        "frame_version": (
            FINAL_FRAME_VERSION
        ),
        "op_id": op_id,
        "pair_id": (
            pair_record["pair_id"]
        ),
        "spec_id": (
            pair_record["spec_id"]
        ),
        "family_id": (
            pair_record["family_id"]
        ),
        "provider_directory": (
            pair_record[
                "provider_directory"
            ]
        ),
        "source_class": (
            pair_record["source_class"]
        ),
        "method": (
            pair_record["method"]
        ),
        "path": (
            pair_record["path"]
        ),
        "prose_tokens_L0": int(
            pair_record[
                "prose_tokens_L0"
            ]
        ),
        # Published paths use the flat layout consumed by Stages 04 and 05.
        "L0_slice_relative_path": f"data/slices/L0/{op_id}.json",
        # Published paths use the flat layout consumed by Stages 04 and 05.
        "L1_slice_relative_path": f"data/slices/L1/{op_id}.json",
        "L0_sha256": (
            construction_record[
                "l0_sha256"
            ]
        ),
        "L1_sha256": (
            construction_record[
                "l1_sha256"
            ]
        ),
        "annotation_field_count": int(
            construction_record[
                "annotation_field_count"
            ]
        ),
        "changed_field_count": int(
            construction_record[
                "changed_field_count"
            ]
        ),
        "transformed_ascii_letter_count": int(
            construction_record[
                "ascii_letter_count"
            ]
        ),
        "L0_input_tokens": int(
            token_record[
                "l0_input_tokens"
            ]
        ),
        "L1_input_tokens": int(
            token_record[
                "l1_input_tokens"
            ]
        ),
        "token_delta_L1_minus_L0": int(
            token_record[
                "token_delta_l1_minus_l0"
            ]
        ),
        "placeholder_only_flag": (
            parse_boolean(
                prose_record[
                    "placeholder_only_flag"
                ]
            )
        ),
        "language_review_flag": (
            parse_boolean(
                prose_record[
                    "language_review_flag"
                ]
            )
        ),
        "request_constructible": (
            parse_boolean(
                feasibility_record[
                    "request_constructible"
                ]
            )
        ),
        "response_constructible": (
            parse_boolean(
                feasibility_record[
                    "response_constructible"
                ]
            )
        ),
        "scenario_count": int(
            feasibility_record[
                "scenario_count"
            ]
        ),
        "requirement_count": int(
            feasibility_record[
                "requirement_count"
            ]
        ),
        "response_fault_count": int(
            feasibility_record[
                "response_fault_count"
            ]
        ),
        "evaluator_feasible": True,
        "oracle_sha256": feasibility_record["oracle_sha256"],
        "development_exposed": False,
        "final_status": (
            "ELIGIBLE"
        ),
    }

    eligible_records.append(
        eligible_record
    )

    eligible_source_class_counts[
        eligible_record[
            "source_class"
        ]
    ] += 1
    eligible_method_counts[
        eligible_record["method"]
    ] += 1
    eligible_family_ids.add(
        eligible_record["family_id"]
    )
    eligible_spec_ids.add(
        eligible_record["spec_id"]
    )

eligible_records.sort(
    key=lambda record: record["op_id"]
)
cumulative_exclusion_rows.sort(
    key=lambda row: (
        row["op_id"],
        int(row["reason_order"]),
        row["rule"],
    )
)

eligible_fieldnames = [
    "frame_version",
    "op_id",
    "pair_id",
    "spec_id",
    "family_id",
    "provider_directory",
    "source_class",
    "method",
    "path",
    "prose_tokens_L0",
    "L0_slice_relative_path",
    "L1_slice_relative_path",
    "L0_sha256",
    "L1_sha256",
    "annotation_field_count",
    "changed_field_count",
    "transformed_ascii_letter_count",
    "L0_input_tokens",
    "L1_input_tokens",
    "token_delta_L1_minus_L0",
    "placeholder_only_flag",
    "language_review_flag",
    "request_constructible",
    "response_constructible",
    "scenario_count",
    "requirement_count",
    "response_fault_count",
    "evaluator_feasible",
    "oracle_sha256",
    "development_exposed",
    "final_status",
]

write_csv_atomic(
    FINAL_ELIGIBLE_CSV_PATH,
    eligible_fieldnames,
    eligible_records,
)

eligible_jsonl_text = "".join(
    canonical_json(record) + "\n"
    for record in eligible_records
)

write_text_atomic(
    FINAL_ELIGIBLE_JSONL_PATH,
    eligible_jsonl_text,
)

exclusion_fieldnames = [
    "op_id",
    "pair_id",
    "spec_id",
    "family_id",
    "stage",
    "rule",
    "reason_order",
    "is_first_exit",
    "first_exit_status",
    "evidence",
]

write_csv_atomic(
    CUMULATIVE_EXCLUSIONS_PATH,
    exclusion_fieldnames,
    cumulative_exclusion_rows,
)

eligible_count = len(
    eligible_records
)
excluded_operation_count = (
    EXPECTED_OPERATION_COUNT
    - eligible_count
)

eligible_population_sha256 = (
    sha256_text(
        canonical_json(
            [
                record["op_id"]
                for record
                in eligible_records
            ]
        )
    )
)

checks = {
    "full_population_accounted_for": (
        eligible_count
        + excluded_operation_count
        == EXPECTED_OPERATION_COUNT
    ),
    "eligible_count_matches_feasibility_minus_exposure": (
        eligible_count
        == (
            expected_feasible_count
            - len(
                development_exposed_ids
                & {
                    op_id
                    for op_id, row
                    in feasibility_by_op_id.items()
                    if row[
                        "final_status"
                    ]
                    == "ELIGIBLE_EVALUATOR_FEASIBLE"
                }
            )
        )
    ),
    "eligible_operation_ids_unique": (
        len({
            record["op_id"]
            for record in eligible_records
        })
        == eligible_count
    ),
    "eligible_pair_ids_unique": (
        len({
            record["pair_id"]
            for record in eligible_records
        })
        == eligible_count
    ),
    "all_eligible_records_feasible": (
        all(
            record[
                "evaluator_feasible"
            ]
            is True
            for record in eligible_records
        )
    ),
    "no_development_exposed_record_eligible": (
        all(
            record["op_id"]
            not in development_exposed_ids
            for record in eligible_records
        )
    ),
    "all_eligible_prompts_within_limit": (
        all(
            parse_boolean(
                token_by_op_id[
                    record["op_id"]
                ][
                    "pair_within_limit"
                ]
            )
            for record in eligible_records
        )
    ),
    "all_eligible_have_positive_prose": (
        all(
            record[
                "prose_tokens_L0"
            ] > 0
            for record in eligible_records
        )
    ),
    "all_eligible_have_transformed_ascii": (
        all(
            record[
                "transformed_ascii_letter_count"
            ] > 0
            for record in eligible_records
        )
    ),
    "no_model_outputs_used": (
        cell14_report.get(
            "model_outputs_used"
        )
        is False
    ),
    "strata_not_selected": (
        stage_03_policy.get(
            "prohibitions",
            {},
        ).get(
            "strata_selection"
        )
        is True
    ),
    "eligible_frame_nonempty": (
        eligible_count > 0
    ),
}

status = (
    "PASS"
    if all(checks.values())
    else "BLOCKED"
)

report = {
    "status": status,
    "stage": (
        "FINAL_ELIGIBILITY_RECONCILIATION"
    ),
    "frame_version": (
        FINAL_FRAME_VERSION
    ),
    "operation_count": (
        EXPECTED_OPERATION_COUNT
    ),
    "eligible_operation_count": (
        eligible_count
    ),
    "excluded_operation_count": (
        excluded_operation_count
    ),
    "eligible_specification_count": (
        len(eligible_spec_ids)
    ),
    "eligible_family_count": (
        len(eligible_family_ids)
    ),
    "development_exposure": {
        "evidence_file_count": (
            len(
                development_evidence_paths
            )
        ),
        "matched_corpus_identifier_count": (
            len(
                development_identifier_evidence
            )
        ),
        "exposed_operation_count": (
            len(
                development_exposed_ids
            )
        ),
        "policy": (
            "Exclude every operation whose "
            "operation, pair, specification, or "
            "family identifier occurs in a "
            "prompt/transformation/evaluator "
            "development artifact."
        ),
        "ledger_path": str(
            DEVELOPMENT_EXPOSURE_PATH
        ),
        "ledger_sha256": sha256_file(
            DEVELOPMENT_EXPOSURE_PATH
        ),
    },
    "first_exit_status_counts": dict(
        sorted(
            first_exit_status_counts.items()
        )
    ),
    "overlapping_exclusion_reason_counts": dict(
        sorted(
            all_reason_counts.items()
        )
    ),
    "eligible_source_class_counts": dict(
        sorted(
            eligible_source_class_counts.items()
        )
    ),
    "eligible_method_counts": dict(
        sorted(
            eligible_method_counts.items()
        )
    ),
    "eligible_population_sha256": (
        eligible_population_sha256
    ),
    "eligible_operations_csv": {
        "path": str(
            FINAL_ELIGIBLE_CSV_PATH
        ),
        "bytes": (
            FINAL_ELIGIBLE_CSV_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            FINAL_ELIGIBLE_CSV_PATH
        ),
        "record_count": (
            eligible_count
        ),
    },
    "eligible_operations_jsonl": {
        "path": str(
            FINAL_ELIGIBLE_JSONL_PATH
        ),
        "bytes": (
            FINAL_ELIGIBLE_JSONL_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            FINAL_ELIGIBLE_JSONL_PATH
        ),
        "record_count": (
            eligible_count
        ),
    },
    "cumulative_exclusions": {
        "path": str(
            CUMULATIVE_EXCLUSIONS_PATH
        ),
        "bytes": (
            CUMULATIVE_EXCLUSIONS_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            CUMULATIVE_EXCLUSIONS_PATH
        ),
        "row_count": (
            len(
                cumulative_exclusion_rows
            )
        ),
        "unique_excluded_operation_count": (
            len({
                row["op_id"]
                for row
                in cumulative_exclusion_rows
            })
        ),
    },
    "input_hashes": {
        "pair_frame": sha256_file(
            PAIR_FRAME_PATH
        ),
        "construction_records": (
            sha256_file(
                CONSTRUCTION_RECORDS_PATH
            )
        ),
        "token_audit": sha256_file(
            TOKEN_AUDIT_PATH
        ),
        "prose_eligibility": (
            sha256_file(
                PROSE_ELIGIBILITY_PATH
            )
        ),
        "feasibility": sha256_file(
            FEASIBILITY_LEDGER_PATH
        ),
        "cell14_report": (
            observed_cell14_hash
        ),
    },
    "checks": checks,
    "duration_seconds": round(
        time.time()
        - CELL15_STARTED,
        3,
    ),
    "interpretation": (
        "The final Stage 03 frame contains "
        "operation pairs that passed prompt, "
        "treatment, deterministic evaluator "
        "feasibility, and development-exposure "
        "rules. No strata or operation sample "
        "has been selected."
    ),
    "report_path": str(
        REPORT_PATH
    ),
}

write_json_atomic(
    REPORT_PATH,
    report,
)

report_sha256 = sha256_file(
    REPORT_PATH
)

stage_03_policy[
    "development_exposure"
] = {
    "exclude_development_records": (
        True
    ),
    "evidence_file_count": (
        len(
            development_evidence_paths
        )
    ),
    "matched_corpus_identifier_count": (
        len(
            development_identifier_evidence
        )
    ),
    "exposed_operation_count": (
        len(
            development_exposed_ids
        )
    ),
    "ledger_path": (
        str(
            DEVELOPMENT_EXPOSURE_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/")
    ),
    "ledger_sha256": (
        sha256_file(
            DEVELOPMENT_EXPOSURE_PATH
        )
    ),
}

stage_03_policy[
    "final_eligible_frame"
] = {
    "status": status,
    "frame_version": (
        FINAL_FRAME_VERSION
    ),
    "operation_count": (
        EXPECTED_OPERATION_COUNT
    ),
    "eligible_operation_count": (
        eligible_count
    ),
    "excluded_operation_count": (
        excluded_operation_count
    ),
    "eligible_specification_count": (
        len(eligible_spec_ids)
    ),
    "eligible_family_count": (
        len(eligible_family_ids)
    ),
    "eligible_population_sha256": (
        eligible_population_sha256
    ),
    "eligible_csv_path": (
        str(
            FINAL_ELIGIBLE_CSV_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/")
    ),
    "eligible_csv_sha256": (
        sha256_file(
            FINAL_ELIGIBLE_CSV_PATH
        )
    ),
    "eligible_jsonl_path": (
        str(
            FINAL_ELIGIBLE_JSONL_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/")
    ),
    "eligible_jsonl_sha256": (
        sha256_file(
            FINAL_ELIGIBLE_JSONL_PATH
        )
    ),
    "cumulative_exclusions_path": (
        str(
            CUMULATIVE_EXCLUSIONS_PATH
            .relative_to(PROJECT_ROOT)
        ).replace("\\", "/")
    ),
    "cumulative_exclusions_sha256": (
        sha256_file(
            CUMULATIVE_EXCLUSIONS_PATH
        )
    ),
    "report_path": (
        str(
            REPORT_PATH.relative_to(
                PROJECT_ROOT
            )
        ).replace("\\", "/")
    ),
    "report_sha256": (
        report_sha256
    ),
    "strata_selected": False,
    "sampling_performed": False,
    "experimental_generation_authorized": (
        False
    ),
}

write_yaml_atomic(
    STAGE_03_POLICY_PATH,
    stage_03_policy,
)

output = dict(report)
output["report_sha256"] = (
    report_sha256
)
output[
    "updated_stage_03_policy_sha256"
] = sha256_file(
    STAGE_03_POLICY_PATH
)

print(
    json.dumps(
        output,
        indent=2,
        ensure_ascii=False,
    )
)

if status != "PASS":
    raise RuntimeError(
        "Final eligibility reconciliation "
        "failed. Review the checks above."
    )

print(
    "\nNOTEBOOK 03 — CELL 15: PASS"
)

{
  "status": "PASS",
  "stage": "FINAL_ELIGIBILITY_RECONCILIATION",
  "frame_version": "stage03-final-eligible-frame-v1",
  "operation_count": 52922,
  "eligible_operation_count": 18268,
  "excluded_operation_count": 34654,
  "eligible_specification_count": 1261,
  "eligible_family_count": 1029,
  "development_exposure": {
    "evidence_file_count": 11,
    "matched_corpus_identifier_count": 162,
    "exposed_operation_count": 13248,
    "policy": "Exclude every operation whose operation, pair, specification, or family identifier occurs in a prompt/transformation/evaluator development artifact.",
    "ledger_path": "/kaggle/working/fresh/stage_03/prose_operation_experiment/data/audits/development_exposure.csv",
    "ledger_sha256": "9ae60ac65c6c831a6e102a02fdde3445f91ce36ab5b63ecec13da53235b48e93"
  },
  "first_exit_status_counts": {
    "ELIGIBLE": 18268,
    "EXCLUDED_NONPOSITIVE_PROSE_SCORE": 311,
    "EXCLUDED_PROMPT_LIMIT": 4304,
    "EXCLUDE_DEVELOPMENT_EXPOSED": 8495,
    "EXCL

In [16]:
stage02_identity = file_sha(STAGE_02_INTERNAL_MANIFEST_PATH)
stage00_identity = file_sha(STAGE_00_INTERNAL_MANIFEST_PATH)
frame = list(records(PROJECT_ROOT / "data/inventory/eligible_operations.jsonl"))
if len(frame) < 100 or len({row["op_id"] for row in frame}) != len(frame):
    raise RuntimeError("Stage 03 has no valid unique sampling frame of at least 100 operations.")
feas = {row["op_id"]: row for row in records(PROJECT_ROOT / "data/audits/feasibility_results.jsonl")}
release = WORK / "fresh_release/stage_03/prose_operation_experiment"
import shutil
for name in ("data/inventory/eligible_operations.csv", "data/audits/cumulative_exclusions.csv",
             "data/audits/development_exposure.csv", "data/audits/feasibility.csv",
             "data/audits/feasibility_results.jsonl", "data/audits/obfuscation_log.jsonl",
             "data/audits/pair_audit.csv", "data/audits/token_audit.csv", "config/stage_03.yaml",
             "src/obfuscation.py", "src/pairing.py", "data/inventory/family_map.csv",
             "data/development_examples/historical_evaluator_development_exposure.json",
             "config/oracle_record_contracts.json", "reports/audit/feasibility_builder_qualification.json",
             "reports/audit/evaluator_feasibility_summary.json", "reports/audit/final_eligibility_reconciliation.json",
             *(f"src/{name}" for name in SHARED_ORACLE_SOURCES)):
    src = PROJECT_ROOT / name
    dest = release / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(src, dest)
for row in frame:
    op = row["op_id"]
    row["oracle_sha256"] = feas[op]["oracle_sha256"]
    if not row["oracle_sha256"]:
        raise RuntimeError("Eligible operation has no validated oracle digest: " + op)
    for condition in ("L0", "L1"):
        name = f"data/slices/{condition}/{op}.json"
        if row[f"{condition}_slice_relative_path"] != name:
            raise RuntimeError("Run corrected Cell 15 before packaging: " + op)
        construction = construction_by_id_cell14[op]
        src = constructed_slice_path_cell14(PROJECT_ROOT, construction, condition)
        payload = src.read_bytes()
        if (sha(payload) != row[f"{condition}_sha256"]
                or row[f"{condition}_sha256"] != construction[f"{condition.lower()}_sha256"]):
            raise RuntimeError("Pair bytes do not match the final frame: " + op)
        atomic(release / name, payload)
jsonl(release / "data/inventory/eligible_operations.jsonl", frame)
pack("03", release, {"upstream_manifest_sha256": {**load(STAGE_02_INTERNAL_MANIFEST_PATH)["upstream_manifest_sha256"], "02": stage02_identity},
                     "eligible_count": len(frame), "model_generation_performed": False})


Stage 03 PASS. Publish /kaggle/working/stage_03_fresh_dataset


{'stage': '03',
 'status': 'PASS',
 'design_id': 'openapi-fresh-pilot-v1',
 'artifact_count': 36561,
 'upstream_manifest_sha256': {'00': 'a064f3bb9814b3545ed81984c5a5bcd5a5ce236ce621f2fd0d039950efafc431',
  '01': 'e6372a8f281da58f893d9b34c8516d1472f9f7cfb034d52265b1b58582897208',
  '02': '538df541290ea421a1dab61cdd3a78a072d63d4ab109f2d902b805d7e728b5fd'},
 'eligible_count': 18268,
 'model_generation_performed': False,
 'configuration_hash': '05cd60681508b4095248aacb11cfcf17e5ae114fac40fc540e827c166e861d2b',
 'artifact_record_schema': 'openapi-artifact-v1',
 'archive': 'stage_03_fresh.zip',
 'archive_sha256': '6ef4f9419476df1533fe17fdc6d615b90c3da3f410a0821c4b012c589599ddf3',
 'archive_bytes': 250671932,
 'internal_manifest_sha256': '74550c1645772a46e668cbbc4024fb36728cff14a27781694e851483251a9272'}